<a href="https://colab.research.google.com/github/MANIKANTH678/DopamineOS/blob/main/claudesol.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys, subprocess

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'duckdb>=1.1', 'rapidfuzz>=3.6', 'anyascii', 'pyarrow', 'scikit-learn'], check=True)

In [ ]:
import os, time
from contextlib import contextmanager
from pathlib import Path

import duckdb
import joblib
import numpy as np
import pandas as pd
import pyarrow as pa
from anyascii import anyascii
from rapidfuzz import fuzz, process
from rapidfuzz.distance import JaroWinkler
from sklearn.ensemble import HistGradientBoostingClassifier

# ---- paths (env vars let the same notebook run outside Colab) ----
DATA_ZIP  = Path(os.environ.get('ER_DATA_ZIP', '/content/drive/MyDrive/dataset.zip'))
DATA_DIR  = Path(os.environ.get('ER_DATA_DIR', '/content/dataset'))             # where the zip is extracted
CACHE_DIR = Path(os.environ.get('ER_CACHE_DIR',                                 # survives runtime resets
                 '/content/drive/MyDrive/er_cache' if IN_COLAB else 'er_cache'))
WORK_DIR  = Path(os.environ.get('ER_WORK_DIR', '/content/er_work' if IN_COLAB else 'er_work'))  # fast local disk

# ---- knobs ----
MAX_KEY_PAIRS    = 1_000    # drop a blocking key if (#S1 rows) × (#S2+S3 rows) sharing it exceeds this
MAX_CANDS_PER_S1 = 200      # keep at most this many candidates per S1 record (most shared keys first)
N_TRAIN_S1       = 200_000  # S1 entities used for training (all 2.2M are not needed)
N_VAL_S1         = 50_000   # held-out S1 entities for recall / threshold selection
MAX_TRAIN_ROWS   = 4_000_000  # negatives are subsampled above this
BETA             = 1.0      # set to the beta of the competition's F-beta metric
SEED             = 42
BATCH_ROWS       = 1_000_000  # candidate pairs featurized per batch
REBUILD          = False    # True: ignore cached normalized Parquet

CACHE_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)


@contextmanager
def stage(name):
    t = time.time()
    print(f'▶ {name}')
    yield
    print(f'✔ {name}: {time.time() - t:,.1f}s')

In [ ]:
import shutil
import os

# Remove the existing mount folder
if os.path.exists("/content/drive"):
    shutil.rmtree("/content/drive")

# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

if not any(p for p in DATA_DIR.rglob('train_ground_truth.tsv') if '__MACOSX' not in p.parts):
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    with stage('unzip dataset'):
        subprocess.run(['unzip', '-q', '-o', str(DATA_ZIP), '-d', str(DATA_DIR)], check=True)

GT_PATH = next(p for p in DATA_DIR.rglob('train_ground_truth.tsv') if '__MACOSX' not in p.parts)
BASE = GT_PATH.parent.parent
SOURCES = {split: {s: BASE / split / f'{split}_source{s}.tsv' for s in (1, 2, 3)} for split in ('train', 'test')}
for split, files in SOURCES.items():
    for s, p in files.items():
        assert p.exists(), p
        print(f'{p.name:20s} {p.stat().st_size / 2**30:5.2f} GB')

readme = BASE.parent / 'README.md'
if readme.exists():
    print(readme.read_text()[:3000])  # check the submission format described here

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
▶ unzip dataset
✔ unzip dataset: 48.8s
train_source1.tsv     0.20 GB
train_source2.tsv     0.46 GB
train_source3.tsv     0.47 GB
test_source1.tsv      0.16 GB
test_source2.tsv      0.47 GB
test_source3.tsv      0.47 GB
# ML Challenge 2026 Problem Statement

## Business Entity Resolution Challenge

In large-scale commercial platforms, business identity data arrives from multiple independent sources — each contributing partial, noisy fragments of information about the same real-world entities. These fragments share no common identifiers, and the challenge of determining which records refer to the same business is known as Entity Resolution (ER). Your challenge is to build an ML solution that, given business records from 3 independent data sources with noisy and inconsistent fields, determines which records across sources refer to the same real-world business en

In [ ]:
con = duckdb.connect(str(WORK_DIR / 'er.duckdb'))
con.execute(f"SET temp_directory = '{WORK_DIR / 'spill'}'")
con.execute('SET preserve_insertion_order = false')
print('duckdb', duckdb.__version__, '| threads:', con.execute("SELECT current_setting('threads')").fetchone()[0])


def _to_ascii(col):
    # anyascii handles Devanagari/Tamil/Telugu/Bengali and accents: 'मॉडर्न फाइनेंस' -> 'modrn phainems'
    return pa.array([anyascii(s) if s is not None and not s.isascii() else s for s in col.to_pylist()],
                    type=pa.string())


con.create_function('to_ascii', _to_ascii, ['VARCHAR'], 'VARCHAR', type='arrow')

duckdb 1.3.2 | threads: 2


In [ ]:
from pathlib import Path

CACHE_DIR = Path("/content/drive/MyDrive/er_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Cache directory:", CACHE_DIR)
print("Exists:", CACHE_DIR.exists())

Cache directory: /content/drive/MyDrive/er_cache
Exists: True


In [ ]:
LEGAL_AND_FILLER = [
    # legal forms
    'private', 'pvt', 'praivet', 'limited', 'ltd', 'limitad',
    'llp', 'llc', 'l l c', 'inc', 'incorporated',
    'corp', 'corporation', 'co', 'company', 'plc', 'opc',
    'sarl', 'sas', 'sasu', 'sa', 'eurl', 'sci', 'snc',
    'gmbh', 'pllc', 'lp', 'ltda',

    # honorifics / prefixes
    'm s', 'shri', 'shree', 'sri', 'sree', 'shrii', 'smt',

    # articles / connectors
    'the', 'and', 'of', 'et', 'le', 'la', 'les',
    'de', 'du', 'des',
]


ADDR_ABBR = {
    'rd': 'road',
    'st': 'street',
    'ave': 'avenue',
    'av': 'avenue',
    'blvd': 'boulevard',
    'bd': 'boulevard',
    'dr': 'drive',
    'ln': 'lane',
    'ct': 'court',
    'hwy': 'highway',
    'pkwy': 'parkway',
    'pl': 'place',
    'sq': 'square',
    'ter': 'terrace',
    'apt': 'apartment',
    'ste': 'suite',
    'fl': 'floor',
    'bldg': 'building',
    'nagr': 'nagar',
    'mkt': 'market',
    'opp': 'opposite',
}


# ---------------------------------------------------------
# SQL HELPERS
# ---------------------------------------------------------

def sql_clean(col):
    """
    Convert a column to normalized ASCII text.

    Steps:
    1. Replace NULL with ''
    2. Convert non-ASCII text to ASCII
    3. Lowercase
    4. Replace '&' with 'and'
    5. Replace non-alphanumeric characters with spaces
    6. Remove extra spaces
    """

    ascii_expr = (
        f"CASE "
        f"WHEN regexp_matches(coalesce({col}, ''), '[^\\\\x00-\\\\x7F]') "
        f"THEN to_ascii(coalesce({col}, '')) "
        f"ELSE coalesce({col}, '') "
        f"END"
    )

    expr = (
        f"replace("
        f"lower({ascii_expr}), "
        f"'&', "
        f"' and '"
        f")"
    )

    expr = (
        f"regexp_replace("
        f"{expr}, "
        f"'[^a-z0-9]+', "
        f"' ', "
        f"'g'"
        f")"
    )

    return f"trim({expr})"


def sql_squash(expr):
    """
    Remove repeated spaces.
    """
    return f"trim(regexp_replace({expr}, ' +', ' ', 'g'))"


# ---------------------------------------------------------
# LEGAL/FILLER WORDS
# ---------------------------------------------------------

# Python raw string is safe here.
STOP_RE = r'\b(' + '|'.join(LEGAL_AND_FILLER) + r')\b'


def sql_core(expr):
    """
    Remove legal/company filler words.
    """

    # DuckDB must receive \b, not a Python backspace character.
    stop_re_sql = STOP_RE.replace("\\", "\\\\")

    return sql_squash(
        f"regexp_replace("
        f"{expr}, "
        f"'{stop_re_sql}', "
        f"' ', "
        f"'g'"
        f")"
    )


# ---------------------------------------------------------
# PHONETIC / NAME SKELETON
# ---------------------------------------------------------

def sql_skeleton(expr):
    """
    Create a rough phonetic skeleton.

    Rules:
      ph -> f
      w  -> v
      m + consonant -> n + consonant
      remove non-initial vowels
    """

    # ph -> f
    e = f"regexp_replace({expr}, 'ph', 'f', 'g')"

    # w -> v
    e = f"regexp_replace({e}, 'w', 'v', 'g')"

    # m + consonant -> n + consonant
    #
    # IMPORTANT:
    # \\1 ensures Python passes \1 to DuckDB regex replacement.
    e = f"regexp_replace({e}, 'm([kgcjtdsz])', 'n\\\\1', 'g')"

    # Remove vowels except when they are the first character.
    #
    # \\B is required so Python does not interpret \B.
    e = f"regexp_replace({e}, '\\\\B[aeiouy]', '', 'g')"

    return e


# ---------------------------------------------------------
# ADDRESS NORMALIZATION
# ---------------------------------------------------------

def sql_addr(expr):
    """
    Expand common address abbreviations.
    """

    for short, full in ADDR_ABBR.items():

        # \\b creates a word boundary for DuckDB.
        expr = (
            f"regexp_replace("
            f"{expr}, "
            f"'\\\\b{short}\\\\b', "
            f"'{full}', "
            f"'g'"
            f")"
        )

    return expr


# ---------------------------------------------------------
# BUILD RECORD TABLE
# ---------------------------------------------------------

def build_recs(split):
    """
    Create table recs_<split>.

    One row per entity across all sources.
    Results are cached as Parquet.
    """

    pq = CACHE_DIR / f"recs_{split}.parquet"

    # -----------------------------------------------------
    # LOAD CACHE IF AVAILABLE
    # -----------------------------------------------------

    if pq.exists() and not REBUILD:

        print(f"Loading cached {split} records from:")
        print(pq)

        con.execute(
            f"""
            CREATE OR REPLACE TABLE recs_{split}
            AS SELECT *
            FROM read_parquet('{pq}')
            """
        )

        return

    # -----------------------------------------------------
    # BUILD RAW UNION
    # -----------------------------------------------------

    raw_parts = []

    for s, path in SOURCES[split].items():

        raw_parts.append(
            f"""
            SELECT
                {s}::TINYINT AS src,
                *
            FROM read_csv(
                '{path}',
                delim='\\t',
                header=true,
                all_varchar=true,
                quote='"',
                escape='"'
            )
            """
        )

    raw = " UNION ALL ".join(raw_parts)

    # -----------------------------------------------------
    # CREATE NORMALIZED TABLE
    # -----------------------------------------------------

    sql = f"""
    CREATE OR REPLACE TABLE recs_{split} AS

    WITH raw AS (
        {raw}
    ),

    n AS (

        SELECT

            src,

            trim(entity_id) AS entity_id,

            lower(
                trim(
                    coalesce(country, '')
                )
            ) AS country,

            business_name AS name_raw,

            business_address AS addr_raw,

            regexp_matches(
                coalesce(business_name, ''),
                '[^\\\\x00-\\\\x7F]'
            ) AS non_ascii,

            {sql_clean('business_name')}
                AS name_norm,

            {sql_squash(
                sql_addr(
                    sql_clean('business_address')
                )
            )}
                AS addr_norm

        FROM raw
    ),

    c AS (

        SELECT
            *,
            {sql_core('name_norm')} AS name_core

        FROM n
    )

    SELECT

        *,

        {sql_skeleton('name_core')}
            AS name_skel,

        array_to_string(
            list_sort(
                list_distinct(
                    regexp_extract_all(
                        addr_norm,
                        '[0-9]+'
                    )
                )
            ),
            ' '
        ) AS addr_nums

    FROM c

    QUALIFY
        row_number() OVER (
            PARTITION BY entity_id
            ORDER BY src
        ) = 1
    """

    # -----------------------------------------------------
    # EXECUTE
    # -----------------------------------------------------

    print(f"Building recs_{split}...")

    con.execute(sql)

    # -----------------------------------------------------
    # CHECK RECORD COUNTS
    # -----------------------------------------------------

    got = dict(
        con.execute(
            f"""
            SELECT
                src,
                count(*)
            FROM recs_{split}
            GROUP BY src
            ORDER BY src
            """
        ).fetchall()
    )

    # -----------------------------------------------------
    # VERIFY AGAINST SOURCE FILE LINE COUNTS
    # -----------------------------------------------------

    for s, path in SOURCES[split].items():

        with open(path, 'rb') as f:
            lines = sum(1 for _ in f) - 1

        loaded = got.get(s, 0)

        if loaded != lines:

            print(
                f"WARNING: {path.name}: "
                f"{lines:,} data lines but "
                f"{loaded:,} unique records loaded"
            )

        else:

            print(
                f"OK: {path.name}: "
                f"{loaded:,} records"
            )

    # -----------------------------------------------------
    # SAVE PARQUET CACHE
    # -----------------------------------------------------

    print(f"Saving cache: {pq}")

    con.execute(
        f"""
        COPY recs_{split}
        TO '{pq}'
        (FORMAT parquet)
        """
    )

    print(f"Finished {split}")


# =========================================================
# RUN NORMALIZATION
# =========================================================

with stage('normalize train'):
    build_recs('train')


with stage('normalize test'):
    build_recs('test')


# =========================================================
# SHOW COUNTS
# =========================================================

for split in ('train', 'test'):

    result = con.execute(
        f"""
        SELECT
            src,
            count(*)
        FROM recs_{split}
        GROUP BY src
        ORDER BY src
        """
    ).fetchall()

    print(split, result)


# =========================================================
# SAMPLE TRAIN RECORDS
# =========================================================

con.sql(
    """
    SELECT
        src,
        name_raw,
        name_norm,
        name_core,
        name_skel,
        addr_norm,
        addr_nums

    FROM recs_train

    USING SAMPLE 8 ROWS
    """
).df()

▶ normalize train
Building recs_train...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

OK: train_source1.tsv: 2,206,821 records
OK: train_source2.tsv: 5,034,616 records
OK: train_source3.tsv: 5,285,603 records
Saving cache: /content/drive/MyDrive/er_cache/recs_train.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Finished train
✔ normalize train: 436.1s
▶ normalize test
Building recs_test...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

OK: test_source1.tsv: 1,732,544 records
OK: test_source2.tsv: 4,887,273 records
OK: test_source3.tsv: 5,082,316 records
Saving cache: /content/drive/MyDrive/er_cache/recs_test.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Finished test
✔ normalize test: 450.2s
train [(1, 2206821), (2, 5034616), (3, 5285603)]
test [(1, 1732544), (2, 4887273), (3, 5082316)]


,src,name_raw,name_norm,name_core,name_skel,addr_norm,addr_nums
0,1,Western Platinum Prudential Inc,western platinum prudential inc,western platinum prudential inc,vestern platinum prudential inc,1100 hoben road oxford ny,1100
1,1,Sheldon and Whitehead LLC,sheldon and whitehead llc,sheldon and whitehead llc,sheldon and vhitehead llc,ky 57 monroe wilson road bowling green,57
2,2,KQ Cyber Group,kq cyber group,kq cyber group,kq cyber group,mesquite tx 2523 westwood avenue,2523
3,2,ORTHOPEDIC TRUSTED PHYSICIANS INC,orthopedic trusted physicians inc,orthopedic trusted physicians inc,orthopedic trusted fysicians inc,8291 ore trail bauxite ar,8291
4,2,Deque LLC Services,deque llc services,deque llc services,deque llc services,00719 lincoln ave carrollton oh,00719
5,2,શ્રી ફાઇનાન્સ પ્રાઇવેટ લિમિટેડ,sri phainans praivet limited,sri phainans praivet limited,sri fainans praivet limited,a 1 vadodara gujarat,1
6,3,सुप्रीम कंसल्टेंसी स्टील,suprim kmsltemsi stil,suprim kmsltemsi stil,suprim kn\1lten\1i stil,door no 072 first floor bhiwadi alwar rj,072
7,3,Strategic Center Enterprises,strategic center enterprises,strategic center enterprises,strategic center enterprises,,None


In [ ]:
import duckdb
from pathlib import Path

# =========================================================
# DUCKDB CONNECTION
# =========================================================

con = duckdb.connect()

print("DuckDB connection: OK")


# =========================================================
# CACHE DIRECTORY
# =========================================================

CACHE_DIR = Path("/content/drive/MyDrive/er_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Cache directory:", CACHE_DIR)


# =========================================================
# BLOCKING PARAMETERS
# =========================================================

MAX_KEY_PAIRS = 50_000
MAX_CANDS_PER_S1 = 500

print("MAX_KEY_PAIRS:", MAX_KEY_PAIRS)
print("MAX_CANDS_PER_S1:", MAX_CANDS_PER_S1)

DuckDB connection: OK
Cache directory: /content/drive/MyDrive/er_cache
MAX_KEY_PAIRS: 50000
MAX_CANDS_PER_S1: 500


In [ ]:
print("Train cache:")
print(CACHE_DIR / "recs_train.parquet")
print("Exists:", (CACHE_DIR / "recs_train.parquet").exists())

print()

print("Test cache:")
print(CACHE_DIR / "recs_test.parquet")
print("Exists:", (CACHE_DIR / "recs_test.parquet").exists())

Train cache:
/content/drive/MyDrive/er_cache/recs_train.parquet
Exists: True

Test cache:
/content/drive/MyDrive/er_cache/recs_test.parquet
Exists: True


In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE recs_train AS
    SELECT *
    FROM read_parquet('/content/drive/MyDrive/er_cache/recs_train.parquet')
""")

con.execute("""
    CREATE OR REPLACE TABLE recs_test AS
    SELECT *
    FROM read_parquet('/content/drive/MyDrive/er_cache/recs_test.parquet')
""")

print("recs_train:", con.execute("SELECT count(*) FROM recs_train").fetchone()[0])
print("recs_test :", con.execute("SELECT count(*) FROM recs_test").fetchone()[0])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

recs_train: 12527040
recs_test : 11702133


In [ ]:
print(
    con.execute("DESCRIBE recs_train").df()[
        ["column_name", "column_type"]
    ].to_string(index=False)
)

column_name column_type
        src     TINYINT
  entity_id     VARCHAR
    country     VARCHAR
   name_raw     VARCHAR
   addr_raw     VARCHAR
  non_ascii     BOOLEAN
  name_norm     VARCHAR
  addr_norm     VARCHAR
  name_core     VARCHAR
  name_skel     VARCHAR
  addr_nums     VARCHAR


In [ ]:
from contextlib import contextmanager
import time

@contextmanager
def stage(name):
    print(f"\n▶ {name}")
    start = time.time()

    try:
        yield
    finally:
        print(f"✓ {name} completed in {time.time() - start:.2f} seconds")

In [ ]:
with stage('blocking keys (train)'):
    display(build_keys('train'))


▶ blocking keys (train)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,kt,keys,keys_kept,pairs_kept,pairs_dropped,largest_block
0,a2,1251415,1251260,39936020.0,6.489289e+07,29713900
1,n1,116502,112762,157565969.0,3.703224e+10,1122541889
2,n2,1018641,1018396,78941378.0,3.512020e+08,70430346
3,na,6349138,6349086,66460760.0,6.132206e+06,408419
4,px,330063,328241,108683774.0,8.881550e+09,934777030
5,sk,623777,622618,102846117.0,4.780894e+09,934686610


✓ blocking keys (train) completed in 1882.77 seconds


In [ ]:
import duckdb
from pathlib import Path

con = duckdb.connect()

print("con created:", con)

con created: <duckdb.duckdb.DuckDBPyConnection object at 0x78c0f30a15f0>


In [ ]:
MAX_KEY_PAIRS = 50_000
MAX_CANDS_PER_S1 = 500

print("MAX_KEY_PAIRS =", MAX_KEY_PAIRS)
print("MAX_CANDS_PER_S1 =", MAX_CANDS_PER_S1)

MAX_KEY_PAIRS = 50000
MAX_CANDS_PER_S1 = 500


In [ ]:
CACHE_DIR = Path("/content/drive/MyDrive/er_cache")

print("Train cache exists:",
      (CACHE_DIR / "recs_train.parquet").exists())

print("Test cache exists:",
      (CACHE_DIR / "recs_test.parquet").exists())

Train cache exists: True
Test cache exists: True


In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE recs_train AS
    SELECT *
    FROM read_parquet(
        '/content/drive/MyDrive/er_cache/recs_train.parquet'
    )
""")

con.execute("""
    CREATE OR REPLACE TABLE recs_test AS
    SELECT *
    FROM read_parquet(
        '/content/drive/MyDrive/er_cache/recs_test.parquet'
    )
""")

print(
    "recs_train:",
    con.execute("SELECT COUNT(*) FROM recs_train").fetchone()[0]
)

print(
    "recs_test:",
    con.execute("SELECT COUNT(*) FROM recs_test").fetchone()[0]
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

recs_train: 12527040
recs_test: 11702133


In [ ]:
print("con:", "OK" if "con" in globals() else "MISSING")

print(
    con.execute("""
        SELECT table_name
        FROM duckdb_tables()
        WHERE table_name IN ('recs_train', 'recs_test')
        ORDER BY table_name
    """).fetchall()
)

con: OK
[('recs_test',), ('recs_train',)]


In [ ]:
from contextlib import contextmanager
import time

@contextmanager
def stage(name):
    start = time.time()
    print(f"\n▶ {name}")
    try:
        yield
    except Exception as e:
        elapsed = time.time() - start
        print(f"✗ {name} failed after {elapsed:.2f}s")
        raise
    else:
        elapsed = time.time() - start
        print(f"✓ {name} completed in {elapsed:.2f}s")

In [ ]:
KEY_TYPES = ['n1', 'n2', 'px', 'sk', 'a2', 'na']

def rare_tokens(split, text_col, out, min_len):
    con.execute(f"""
        CREATE OR REPLACE TABLE {out} AS
        WITH t AS (
            SELECT DISTINCT
                entity_id,
                country,
                tok
            FROM (
                SELECT
                    entity_id,
                    country,
                    unnest(string_split({text_col}, ' ')) AS tok
                FROM recs_{split}
            )
            WHERE length(tok) >= {min_len}
              AND NOT regexp_matches(tok, '^[0-9]+$')
        ),
        df AS (
            SELECT
                country,
                tok,
                count(*) AS df
            FROM t
            GROUP BY country, tok
        ),
        r AS (
            SELECT
                t.entity_id,
                t.tok,
                row_number() OVER (
                    PARTITION BY t.entity_id
                    ORDER BY df.df, t.tok
                ) AS rn
            FROM t
            JOIN df
              ON t.country = df.country
             AND t.tok = df.tok
        )
        SELECT
            entity_id,
            max(tok) FILTER (WHERE rn = 1) AS r1,
            max(tok) FILTER (WHERE rn = 2) AS r2
        FROM r
        WHERE rn <= 2
        GROUP BY entity_id
    """)


def build_keys(split):
    rare_tokens(
        split,
        "name_core",
        f"nrare_{split}",
        2
    )

    rare_tokens(
        split,
        "addr_norm",
        f"arare_{split}",
        3
    )

    con.execute(f"""
        CREATE OR REPLACE TABLE keys_{split} AS
        WITH b AS (
            SELECT
                r.entity_id,
                r.src,
                r.country,
                replace(r.name_core, ' ', '') AS core_ns,
                replace(r.name_skel, ' ', '') AS skel_ns,
                n.r1 AS n1,
                n.r2 AS n2,
                a.r1 AS a1,
                a.r2 AS a2
            FROM recs_{split} r
            LEFT JOIN nrare_{split} n
                USING (entity_id)
            LEFT JOIN arare_{split} a
                USING (entity_id)
        )

        SELECT DISTINCT
            entity_id,
            src,
            kt,
            k
        FROM (

            SELECT
                entity_id,
                src,
                'n1' AS kt,
                'n1|' || country || '|' || n AS k
            FROM (
                SELECT
                    *,
                    unnest([n1, n2]) AS n
                FROM b
            )
            WHERE n IS NOT NULL

            UNION ALL

            SELECT
                entity_id,
                src,
                'n2' AS kt,
                'n2|' || country || '|' ||
                least(n1, n2) || ' ' ||
                greatest(n1, n2) AS k
            FROM b
            WHERE n1 IS NOT NULL
              AND n2 IS NOT NULL

            UNION ALL

            SELECT
                entity_id,
                src,
                'px' AS kt,
                'px|' || country || '|' ||
                left(core_ns, 6) AS k
            FROM b
            WHERE length(core_ns) >= 4

            UNION ALL

            SELECT
                entity_id,
                src,
                'sk' AS kt,
                'sk|' || country || '|' ||
                left(skel_ns, 8) AS k
            FROM b
            WHERE length(skel_ns) >= 3

            UNION ALL

            SELECT
                entity_id,
                src,
                'a2' AS kt,
                'a2|' || country || '|' ||
                least(a1, a2) || ' ' ||
                greatest(a1, a2) AS k
            FROM b
            WHERE a1 IS NOT NULL
              AND a2 IS NOT NULL

            UNION ALL

            SELECT
                entity_id,
                src,
                'na' AS kt,
                'na|' || country || '|' ||
                n || '|' || a AS k
            FROM (
                SELECT
                    entity_id,
                    src,
                    country,
                    n,
                    unnest([a1, a2]) AS a
                FROM (
                    SELECT
                        *,
                        unnest([n1, n2]) AS n
                    FROM b
                )
            )
            WHERE n IS NOT NULL
              AND a IS NOT NULL
        )
    """)

    con.execute(f"""
        CREATE OR REPLACE TABLE blocks_{split} AS
        SELECT
            k,
            any_value(kt) AS kt,
            count(*) FILTER (WHERE src = 1) AS n1,
            count(*) FILTER (WHERE src > 1) AS n23
        FROM keys_{split}
        GROUP BY k
        HAVING n1 > 0
           AND n23 > 0
    """)

    return con.sql(f"""
        SELECT
            kt,
            count(*) AS keys,
            count(*) FILTER (
                WHERE n1 * n23 <= {MAX_KEY_PAIRS}
            ) AS keys_kept,
            sum(n1 * n23) FILTER (
                WHERE n1 * n23 <= {MAX_KEY_PAIRS}
            ) AS pairs_kept,
            sum(n1 * n23) FILTER (
                WHERE n1 * n23 > {MAX_KEY_PAIRS}
            ) AS pairs_dropped,
            max(n1 * n23) AS largest_block
        FROM blocks_{split}
        GROUP BY kt
        ORDER BY kt
    """).df()

In [ ]:
with stage("blocking keys (train)"):
    train_key_stats = build_keys("train")

display(train_key_stats)


▶ blocking keys (train)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ blocking keys (train) completed in 2288.97s


,kt,keys,keys_kept,pairs_kept,pairs_dropped,largest_block
0,a2,1251415,1251260,39936020.0,6.489289e+07,29713900
1,n1,116502,112762,157565969.0,3.703224e+10,1122541889
2,n2,1018641,1018396,78941378.0,3.512020e+08,70430346
3,na,6349138,6349086,66460760.0,6.132206e+06,408419
4,px,330063,328241,108683774.0,8.881550e+09,934777030
5,sk,623777,622618,102846117.0,4.780894e+09,934686610


In [ ]:
con.close()
print("✅ DuckDB connection closed.")

✅ DuckDB connection closed.


In [ ]:
from pathlib import Path
import shutil

source_db = Path("/content/er_work/er.duckdb")

checkpoint_dir = Path("/content/drive/MyDrive/er_checkpoint")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

checkpoint_db = checkpoint_dir / "er_after_blocking_train.duckdb"

shutil.copy2(source_db, checkpoint_db)

print("✅ CHECKPOINT SAVED")
print("📁", checkpoint_db)
print("💾 Size:", round(checkpoint_db.stat().st_size / (1024**2), 2), "MB")

✅ CHECKPOINT SAVED
📁 /content/drive/MyDrive/er_checkpoint/er_after_blocking_train.duckdb
💾 Size: 3195.51 MB


In [ ]:
print("Checkpoint exists:", checkpoint_db.exists())

Checkpoint exists: True


In [ ]:
checkpoint_info = checkpoint_dir / "CHECKPOINT.txt"

checkpoint_info.write_text(
    "Amazon ML Entity Resolution\n"
    "Checkpoint: blocking keys (train) completed\n"
    "Tables completed: recs_train, nrare_train, arare_train, keys_train, blocks_train\n"
    "Next step: build candidates (train/validation)\n"
)

print("✅ Checkpoint marker saved.")

✅ Checkpoint marker saved.


In [ ]:
from pathlib import Path

# ------------------------------------------------------------
# RESTORE CHECKPOINT DIRECTORY
# ------------------------------------------------------------

CHECKPOINT_DIR = Path("/content/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

CANDIDATE_DIR = CHECKPOINT_DIR / "candidates"
CANDIDATE_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Checkpoint directory:", CHECKPOINT_DIR)
print("✅ Candidate directory:", CANDIDATE_DIR)

# Check what checkpoints/files already exist
print("\nExisting checkpoint files:")
for p in CHECKPOINT_DIR.rglob("*"):
    print(" ", p)

✅ Checkpoint directory: /content/checkpoints
✅ Candidate directory: /content/checkpoints/candidates

Existing checkpoint files:
  /content/checkpoints/candidates


In [ ]:
from pathlib import Path

print("🔍 Searching for SQLite databases...\n")

search_dirs = [
    Path("/content"),
    Path("/content/drive/MyDrive")
]

db_files = []

for base in search_dirs:
    if base.exists():
        for p in base.rglob("*"):
            if p.is_file() and p.suffix.lower() in [".db", ".sqlite", ".sqlite3"]:
                size_mb = p.stat().st_size / (1024 * 1024)
                db_files.append((p, size_mb))

if not db_files:
    print("❌ No SQLite database found.")
else:
    db_files.sort(key=lambda x: x[1], reverse=True)

    for path, size in db_files:
        print(f"{size:,.1f} MB  →  {path}")

🔍 Searching for SQLite databases...

0.0 MB  →  /content/.config/default_configs.db
0.0 MB  →  /content/.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db


In [ ]:
from pathlib import Path

print("🔍 Searching for checkpoint files...\n")

for base in [Path("/content"), Path("/content/drive/MyDrive")]:
    if base.exists():
        for p in base.rglob("*"):
            if p.is_file() and (
                "CHECKPOINT" in p.name.upper()
                or "checkpoint" in str(p.parent).lower()
            ):
                print(p)

🔍 Searching for checkpoint files...



In [ ]:
from pathlib import Path

drive = Path("/content/drive/MyDrive")

print("🔍 Large files in Google Drive:\n")

for p in drive.rglob("*"):
    if p.is_file():
        size_mb = p.stat().st_size / (1024 * 1024)

        if size_mb > 100:
            print(f"{size_mb:,.1f} MB  →  {p}")

🔍 Large files in Google Drive:



In [ ]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive")

print("Searching for your dataset files...\n")

for p in BASE.rglob("*.tsv"):
    size_mb = p.stat().st_size / (1024 * 1024)
    print(f"{size_mb:,.1f} MB  →  {p}")

Searching for your dataset files...



In [ ]:
from pathlib import Path

drive = Path("/content/drive/MyDrive")

print("Google Drive mounted:", drive.exists())
print("\nFiles/folders in MyDrive:")

for p in list(drive.iterdir())[:100]:
    print("📁" if p.is_dir() else "📄", p.name)

Google Drive mounted: False

Files/folders in MyDrive:


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive'

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

drive = Path("/content/drive/MyDrive")

print("✅ Drive exists:", drive.exists())

print("\n📁 MyDrive contents:")
for p in list(drive.iterdir())[:100]:
    print("📁" if p.is_dir() else "📄", p.name)

✅ Drive exists: True

📁 MyDrive contents:
📁 Colab Notebooks
📄 dataset.zip
📁 er_cache
📁 er_checkpoint


In [ ]:
from pathlib import Path

drive = Path("/content/drive/MyDrive")

for folder_name in ["er_checkpoint", "er_cache"]:
    folder = drive / folder_name

    print("\n" + "=" * 70)
    print(f"📁 {folder_name}")
    print("=" * 70)

    if not folder.exists():
        print("❌ Folder not found")
        continue

    for p in folder.rglob("*"):
        if p.is_file():
            size_mb = p.stat().st_size / (1024 * 1024)
            print(f"{size_mb:,.2f} MB  →  {p}")


📁 er_checkpoint
3,195.51 MB  →  /content/drive/MyDrive/er_checkpoint/er_after_blocking_train.duckdb
0.00 MB  →  /content/drive/MyDrive/er_checkpoint/CHECKPOINT.txt

📁 er_cache
1,847.55 MB  →  /content/drive/MyDrive/er_cache/recs_train.parquet
1,769.69 MB  →  /content/drive/MyDrive/er_cache/recs_test.parquet


In [ ]:
checkpoint_dir = drive / "er_checkpoint"

print("\n🔍 Checkpoint files:")

for p in checkpoint_dir.rglob("*"):
    if p.is_file():
        print(p)


🔍 Checkpoint files:
/content/drive/MyDrive/er_checkpoint/er_after_blocking_train.duckdb
/content/drive/MyDrive/er_checkpoint/CHECKPOINT.txt


In [ ]:
import duckdb
from pathlib import Path

# ============================================================
# RECONNECT TO COMPLETED TRAIN BLOCKING CHECKPOINT
# ============================================================

DB_PATH = Path(
    "/content/drive/MyDrive/er_checkpoint/er_after_blocking_train.duckdb"
)

print("Opening:", DB_PATH)
print(f"Size: {DB_PATH.stat().st_size / (1024**3):.2f} GB")

conn = duckdb.connect(str(DB_PATH))

print("✅ DuckDB checkpoint connected!")

Opening: /content/drive/MyDrive/er_checkpoint/er_after_blocking_train.duckdb
Size: 3.12 GB
✅ DuckDB checkpoint connected!


In [ ]:
# ============================================================
# CHECK TABLES
# ============================================================

tables = conn.execute("""
    SHOW TABLES
""").fetchdf()

print(tables.to_string(index=False))

       name
nrare_train
  recs_test
 recs_train


In [ ]:
# ============================================================
# INSPECT THE ACTUAL DUCKDB CHECKPOINT
# ============================================================

print("📋 ALL TABLES IN CHECKPOINT\n")

tables = conn.execute("SHOW TABLES").fetchdf()
print(tables.to_string(index=False))

print("\n" + "=" * 70)
print("📊 ROW COUNTS")
print("=" * 70)

for table in tables["name"]:
    try:
        count = conn.execute(
            f'SELECT COUNT(*) FROM "{table}"'
        ).fetchone()[0]

        print(f"{table:30s} {count:,} rows")
    except Exception as e:
        print(f"{table:30s} ERROR: {e}")

📋 ALL TABLES IN CHECKPOINT

       name
nrare_train
  recs_test
 recs_train

📊 ROW COUNTS
nrare_train                    12,524,513 rows
recs_test                      11,702,133 rows
recs_train                     12,527,040 rows


In [ ]:
# ============================================================
# INSPECT THE REAL CHECKPOINT STRUCTURE
# ============================================================

for table in ["recs_train", "nrare_train", "recs_test"]:

    print("\n" + "=" * 80)
    print(f"TABLE: {table}")
    print("=" * 80)

    # Schema
    schema = conn.execute(
        f'DESCRIBE "{table}"'
    ).fetchdf()

    print("\nCOLUMNS:")
    print(schema.to_string(index=False))

    # Row count
    count = conn.execute(
        f'SELECT COUNT(*) FROM "{table}"'
    ).fetchone()[0]

    print(f"\nROWS: {count:,}")

    # Sample
    print("\nSAMPLE:")
    display(
        conn.execute(
            f'SELECT * FROM "{table}" LIMIT 5'
        ).fetchdf()
    )


TABLE: recs_train

COLUMNS:
column_name column_type null  key default extra
        src     TINYINT  YES None    None  None
  entity_id     VARCHAR  YES None    None  None
    country     VARCHAR  YES None    None  None
   name_raw     VARCHAR  YES None    None  None
   addr_raw     VARCHAR  YES None    None  None
  non_ascii     BOOLEAN  YES None    None  None
  name_norm     VARCHAR  YES None    None  None
  addr_norm     VARCHAR  YES None    None  None
  name_core     VARCHAR  YES None    None  None
  name_skel     VARCHAR  YES None    None  None
  addr_nums     VARCHAR  YES None    None  None

ROWS: 12,527,040

SAMPLE:


,src,entity_id,country,name_raw,addr_raw,non_ascii,name_norm,addr_norm,name_core,name_skel,addr_nums
0,1,S1-100137610,india,Producer Snk Infrastructure Private Limited,"Plot No. 1, New Brahmakshatriya Co. Op. Soc. O...",True,producer snk infrastructure private limited,plot no 1 new brahmakshatriya co op soc opp ko...,producer snk infrastructure private limited,producer snk infrastructure private limited,1
1,1,S1-100138879,us,Stephenie's Seafood,"5712 Diamond Valley Drive, Fort Worth, TX",True,stephenie s seafood,5712 diamond valley drive fort worth tx,stephenie s seafood,stefenie s seafood,5712
2,1,S1-100192380,india,Krishna Tech Limited,"No 491/ W6, Udumalpet, Manupatti Post, 9/6, Ch...",True,krishna tech limited,no 491 w6 udumalpet manupatti post 9 6 check p...,krishna tech limited,krishna tech limited,491 6 9
3,1,S1-100276333,us,Robin Brighthouse Inc,"252 Matteson Avenue, Matteson, IL",True,robin brighthouse inc,252 matteson avenue matteson il,robin brighthouse inc,robin brighthouse inc,252
4,1,S1-100301125,us,Crestline,"706 Timberline Drive, Winston-salem, NC",True,crestline,706 timberline drive winston salem nc,crestline,crestline,706



TABLE: nrare_train

COLUMNS:
column_name column_type null  key default extra
  entity_id     VARCHAR  YES None    None  None
         r1     VARCHAR  YES None    None  None
         r2     VARCHAR  YES None    None  None

ROWS: 12,524,513

SAMPLE:


,entity_id,r1,r2
0,S1-100044592,aparajita,services
1,S1-100048907,peterkin,pippy
2,S1-100062855,gold,industries
3,S1-100067997,dream,solutions
4,S1-100087645,vilavancode,plastics



TABLE: recs_test

COLUMNS:
column_name column_type null  key default extra
        src     TINYINT  YES None    None  None
  entity_id     VARCHAR  YES None    None  None
    country     VARCHAR  YES None    None  None
   name_raw     VARCHAR  YES None    None  None
   addr_raw     VARCHAR  YES None    None  None
  non_ascii     BOOLEAN  YES None    None  None
  name_norm     VARCHAR  YES None    None  None
  addr_norm     VARCHAR  YES None    None  None
  name_core     VARCHAR  YES None    None  None
  name_skel     VARCHAR  YES None    None  None
  addr_nums     VARCHAR  YES None    None  None

ROWS: 11,702,133

SAMPLE:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,src,entity_id,country,name_raw,addr_raw,non_ascii,name_norm,addr_norm,name_core,name_skel,addr_nums
0,1,S1-10013187,india,Black Foundation LLP,"63, Parvati Bhavan, Nehru Road, Vile Parle ( E...",True,black foundation llp,63 parvati bhavan nehru road vile parle east m...,black foundation llp,black foundation llp,63
1,1,S1-100194951,us,"Falzone's National Audio Installation, LLC","515 Tormund Street, Middletown, DE",True,falzone s national audio installation llc,515 tormund street middletown de,falzone s national audio installation llc,falzone s national audio installation llc,515
2,1,S1-100229086,france,Syndicat Maternelle SAS,"13 A Avenue Jean Cordier, Pessac, Nouvelle-Aqu...",True,syndicat maternelle sas,13 a avenue jean cordier pessac nouvelle aquit...,syndicat maternelle sas,syndicat maternelle sas,13
3,1,S1-100318766,france,Te Club SAS,"16 Rue Corneille, Calais, Hauts-de-France",True,te club sas,16 rue corneille calais hauts de france,te club sas,te club sas,16
4,1,S1-100960277,india,South Hospital,"Rahmath Manzil, Mp 25 1975, Near Bus Stand, Ma...",True,south hospital,rahmath manzil mp 25 1975 near bus stand manan...,south hospital,south hospital,1975 25


In [ ]:
# ============================================================
# INSPECT nrare_train
# ============================================================

print("📊 nrare_train sample")
display(
    conn.execute("""
        SELECT *
        FROM nrare_train
        LIMIT 10
    """).fetchdf()
)

print("\n📊 NULL COUNTS")
display(
    conn.execute("""
        SELECT
            COUNT(*) AS total,
            COUNT(*) FILTER (WHERE r1 IS NULL OR r1 = '') AS r1_empty,
            COUNT(*) FILTER (WHERE r2 IS NULL OR r2 = '') AS r2_empty,
            COUNT(DISTINCT r1) AS unique_r1,
            COUNT(DISTINCT r2) AS unique_r2
        FROM nrare_train
    """).fetchdf()
)

print("\n📊 RECORD SOURCE DISTRIBUTION")
display(
    conn.execute("""
        SELECT
            src,
            COUNT(*) AS rows
        FROM recs_train
        GROUP BY src
        ORDER BY src
    """).fetchdf()
)

print("\n📊 nrare ENTITY SAMPLE")
display(
    conn.execute("""
        SELECT
            n.entity_id,
            n.r1,
            n.r2,
            r.src,
            r.country,
            r.name_norm
        FROM nrare_train n
        LEFT JOIN recs_train r
            ON n.entity_id = r.entity_id
        LIMIT 20
    """).fetchdf()
)

📊 nrare_train sample


,entity_id,r1,r2
0,S1-100044592,aparajita,services
1,S1-100048907,peterkin,pippy
2,S1-100062855,gold,industries
3,S1-100067997,dream,solutions
4,S1-100087645,vilavancode,plastics
5,S1-100090442,curnutte,winvest
6,S1-100116988,sixth,place
7,S1-100156870,phyllis,mcdowell
8,S1-100162607,ojaswini,devi
9,S1-10017605,dunn,garcia



📊 NULL COUNTS


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total,r1_empty,r2_empty,unique_r1,unique_r2
0,12524513,0,346449,1333857,92376


...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,src,rows
0,1,2206821
1,2,5034616
2,3,5285603



📊 nrare ENTITY SAMPLE


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,entity_id,r1,r2,src,country,name_norm
0,S1-100137610,snk,infrastructure,1,india,producer snk infrastructure private limited
1,S1-100138879,stephenie,seafood,1,us,stephenie s seafood
2,S1-100192380,krishna,tech,1,india,krishna tech limited
3,S1-100276333,robin,brighthouse,1,us,robin brighthouse inc
4,S1-100301125,crestline,None,1,us,crestline
5,S1-100485852,stag,federal,1,us,stag federal llc
6,S1-100718946,panapuzha,sales,1,india,panapuzha sales private limited
7,S1-100823094,vhs,institutions,1,india,vhs institutions pvt ltd
8,S1-101379147,niece,obic,1,us,niece obic llc
9,S1-101670566,practice,smith,1,us,smith family practice


In [ ]:
# ============================================================
# FINAL CHECK BEFORE CANDIDATE GENERATION
# ============================================================

from pathlib import Path

CHECKPOINT_DIR = Path("/content/drive/MyDrive/er_checkpoint")

# ------------------------------------------------------------
# 1. Read checkpoint marker
# ------------------------------------------------------------

checkpoint_file = CHECKPOINT_DIR / "CHECKPOINT.txt"

print("=" * 80)
print("CHECKPOINT MARKER")
print("=" * 80)

if checkpoint_file.exists():
    print(checkpoint_file.read_text())
else:
    print("❌ CHECKPOINT.txt not found")


# ------------------------------------------------------------
# 2. Inspect r1/r2 frequency
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("R1/R2 FREQUENCY")
print("=" * 80)

freq = conn.execute("""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT r1) AS unique_r1,
        COUNT(DISTINCT r2) AS unique_r2
    FROM nrare_train
""").fetchdf()

display(freq)


# ------------------------------------------------------------
# 3. Most common r1 keys
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TOP R1 KEYS")
print("=" * 80)

display(
    conn.execute("""
        SELECT
            r1,
            COUNT(*) AS n
        FROM nrare_train
        WHERE r1 IS NOT NULL
        GROUP BY r1
        ORDER BY n DESC
        LIMIT 20
    """).fetchdf()
)


# ------------------------------------------------------------
# 4. Most common r2 keys
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TOP R2 KEYS")
print("=" * 80)

display(
    conn.execute("""
        SELECT
            r2,
            COUNT(*) AS n
        FROM nrare_train
        WHERE r2 IS NOT NULL
        GROUP BY r2
        ORDER BY n DESC
        LIMIT 20
    """).fetchdf()
)

CHECKPOINT MARKER
Amazon ML Entity Resolution
Checkpoint: blocking keys (train) completed
Tables completed: recs_train, nrare_train, arare_train, keys_train, blocks_train
Next step: build candidates (train/validation)


R1/R2 FREQUENCY


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,unique_r1,unique_r2
0,12524513,1333857,92376



TOP R1 KEYS


,r1,n
0,vision,24357
1,dermatology,20413
2,nose,20117
3,ankle,20072
4,behavioral,20043
5,orthopedic,19894
6,primary,19742
7,internal,19616
8,womens,19588
9,urgent,19486



TOP R2 KEYS


,r2,n
0,com,366818
1,services,136369
2,center,127875
3,llc,72376
4,partners,69508
5,solutions,66326
6,trading,62247
7,brothers,62193
8,service,56907
9,technologies,55761


In [ ]:
# ============================================================
# CHECK nrare_train SOURCE COVERAGE
# ============================================================

print("=" * 80)
print("nrare_train SOURCE COVERAGE")
print("=" * 80)

result = conn.execute("""
    SELECT
        r.src,
        COUNT(*) AS nrare_rows
    FROM nrare_train n
    JOIN recs_train r
        ON n.entity_id = r.entity_id
    GROUP BY r.src
    ORDER BY r.src
""").fetchdf()

display(result)

print("\n" + "=" * 80)
print("R1/R2 COVERAGE BY SOURCE")
print("=" * 80)

result = conn.execute("""
    SELECT
        r.src,
        COUNT(*) AS total,
        COUNT(*) FILTER (WHERE n.r1 IS NOT NULL) AS has_r1,
        COUNT(*) FILTER (WHERE n.r2 IS NOT NULL) AS has_r2,
        COUNT(*) FILTER (
            WHERE n.r1 IS NOT NULL AND n.r2 IS NOT NULL
        ) AS has_both
    FROM recs_train r
    LEFT JOIN nrare_train n
        ON r.entity_id = n.entity_id
    GROUP BY r.src
    ORDER BY r.src
""").fetchdf()

display(result)

nrare_train SOURCE COVERAGE


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,src,nrare_rows
0,1,2206821
1,2,5033510
2,3,5284182



R1/R2 COVERAGE BY SOURCE


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,src,total,has_r1,has_r2,has_both
0,1,2206821,2206821,2184834,2184834
1,2,5034616,5033510,4907753,4907753
2,3,5285603,5284182,5085477,5085477


In [ ]:
# ============================================================
# CHECK HOW r1/r2 ARE GENERATED
# ============================================================

display(
    conn.execute("""
        SELECT
            r.entity_id,
            r.name_norm,
            n.r1,
            n.r2
        FROM recs_train r
        JOIN nrare_train n
            ON r.entity_id = n.entity_id
        WHERE n.r1 IS NOT NULL
        LIMIT 30
    """).fetchdf()
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,entity_id,name_norm,r1,r2
0,S1-208945372,hudson castaneda and corea pllc,corea,castaneda
1,S1-209182842,haring and velasquez,haring,velasquez
2,S1-209557016,sq builders limited,sq,builders
3,S1-209645316,evolving holdings,evolving,holdings
4,S1-209785936,leatherwood and buford,leatherwood,buford
5,S1-209816132,lin ana k md,ana,lin
6,S1-209904439,high investments private limited,high,investments
7,S1-210259474,pch services private limited,pch,services
8,S1-210474784,grace zion,zion,grace
9,S1-210479156,golden blue btc llc,btc,golden


In [ ]:
# ============================================================
# REBUILD MISSING BLOCKING TABLE
# FROM EXISTING CHECKPOINT
# ============================================================

import time

print("=" * 80)
print("REBUILDING BLOCKING KEYS")
print("=" * 80)

# Remove only a previous incomplete attempt, if one exists.
conn.execute("DROP TABLE IF EXISTS keys_train")

start = time.time()

conn.execute("""
CREATE TABLE keys_train AS
SELECT
    n.entity_id,
    r.src,
    r.country,
    n.r1,
    n.r2,

    -- Combined rare-token key
    CASE
        WHEN n.r1 IS NOT NULL AND n.r2 IS NOT NULL
        THEN r.country || '|' || n.r1 || '|' || n.r2
        WHEN n.r1 IS NOT NULL
        THEN r.country || '|' || n.r1
        ELSE NULL
    END AS rare_key,

    -- Individual keys
    CASE
        WHEN n.r1 IS NOT NULL
        THEN r.country || '|' || n.r1
        ELSE NULL
    END AS r1_key,

    CASE
        WHEN n.r2 IS NOT NULL
        THEN r.country || '|' || n.r2
        ELSE NULL
    END AS r2_key

FROM nrare_train n
JOIN recs_train r
    ON n.entity_id = r.entity_id
""")

elapsed = time.time() - start

print(f"✅ keys_train created in {elapsed:.1f} seconds")

# Check count
count = conn.execute(
    "SELECT COUNT(*) FROM keys_train"
).fetchone()[0]

print(f"✅ rows: {count:,}")

# Inspect
display(
    conn.execute("""
        SELECT *
        FROM keys_train
        LIMIT 10
    """).fetchdf()
)

REBUILDING BLOCKING KEYS


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ keys_train created in 91.0 seconds
✅ rows: 12,524,513


,entity_id,src,country,r1,r2,rare_key,r1_key,r2_key
0,S1-100044592,1,india,aparajita,services,india|aparajita|services,india|aparajita,india|services
1,S1-100048907,1,us,peterkin,pippy,us|peterkin|pippy,us|peterkin,us|pippy
2,S1-100062855,1,india,gold,industries,india|gold|industries,india|gold,india|industries
3,S1-100067997,1,india,dream,solutions,india|dream|solutions,india|dream,india|solutions
4,S1-100087645,1,india,vilavancode,plastics,india|vilavancode|plastics,india|vilavancode,india|plastics
5,S1-100090442,1,us,curnutte,winvest,us|curnutte|winvest,us|curnutte,us|winvest
6,S1-100116988,1,us,sixth,place,us|sixth|place,us|sixth,us|place
7,S1-100156870,1,us,phyllis,mcdowell,us|phyllis|mcdowell,us|phyllis,us|mcdowell
8,S1-100162607,1,india,ojaswini,devi,india|ojaswini|devi,india|ojaswini,india|devi
9,S1-10017605,1,us,dunn,garcia,us|dunn|garcia,us|dunn,us|garcia


In [ ]:
# ============================================================
# CHECK BLOCK SIZE DISTRIBUTION
# ============================================================

print("=" * 80)
print("RARE-KEY BLOCK SIZE DISTRIBUTION")
print("=" * 80)

display(
    conn.execute("""
        SELECT
            COUNT(*) AS number_of_blocks,
            MIN(n) AS smallest_block,
            ROUND(AVG(n), 2) AS average_block,
            MEDIAN(n) AS median_block,
            MAX(n) AS largest_block
        FROM (
            SELECT
                rare_key,
                COUNT(*) AS n
            FROM keys_train
            WHERE rare_key IS NOT NULL
            GROUP BY rare_key
        )
    """).fetchdf()
)

print("\nTOP 20 LARGEST RARE-KEY BLOCKS:")

display(
    conn.execute("""
        SELECT
            rare_key,
            COUNT(*) AS n
        FROM keys_train
        WHERE rare_key IS NOT NULL
        GROUP BY rare_key
        ORDER BY n DESC
        LIMIT 20
    """).fetchdf()
)

RARE-KEY BLOCK SIZE DISTRIBUTION


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,number_of_blocks,smallest_block,average_block,median_block,largest_block
0,3600085,1,3.48,1.0,19897



TOP 20 LARGEST RARE-KEY BLOCKS:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rare_key,n
0,us|ankle|foot,19897
1,us|nose|throat,19619
2,us|physical|therapy,13067
3,us|dental|pediatric,9230
4,us|dentistry|pediatric,9113
5,us|internal|medicine,8403
6,us|alley|back,7584
7,us|edge|straight,7409
8,us|up|rev,6640
9,us|behavioral|health,6571


In [ ]:
print("=" * 80)
print("S1 → S2 / S3 RARE-KEY CANDIDATE DIAGNOSTIC")
print("=" * 80)

q = """
WITH s1 AS (
    SELECT rare_key
    FROM keys_train
    WHERE src = 1
      AND rare_key IS NOT NULL
),
s23 AS (
    SELECT rare_key
    FROM keys_train
    WHERE src IN (2, 3)
      AND rare_key IS NOT NULL
)
SELECT
    COUNT(*) AS s1_rows,
    (SELECT COUNT(*) FROM s23) AS s23_rows,
    COUNT(*) * 1.0 AS s1_key_rows,
    (
        SELECT COUNT(*)
        FROM s1 a
        JOIN s23 b
          ON a.rare_key = b.rare_key
    ) AS rough_candidate_pairs
FROM s1;
"""

result = conn.execute(q).fetchdf()
display(result)

S1 → S2 / S3 RARE-KEY CANDIDATE DIAGNOSTIC


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,s1_rows,s23_rows,s1_key_rows,rough_candidate_pairs
0,2206821,10317692,2206821.0,432643551


In [ ]:
print("=" * 80)
print("ESTIMATING S1 → S2/S3 CANDIDATE PAIRS")
print("=" * 80)

q = """
WITH block_counts AS (
    SELECT
        rare_key,
        SUM(CASE WHEN src = 1 THEN 1 ELSE 0 END) AS s1_count,
        SUM(CASE WHEN src IN (2,3) THEN 1 ELSE 0 END) AS s23_count
    FROM keys_train
    WHERE rare_key IS NOT NULL
    GROUP BY rare_key
)
SELECT
    COUNT(*) AS overlapping_blocks,
    SUM(s1_count * s23_count) AS candidate_pairs,
    MAX(s1_count) AS max_s1_block,
    MAX(s23_count) AS max_s23_block,
    AVG(s1_count * s23_count) AS avg_candidates_per_block
FROM block_counts
WHERE s1_count > 0
  AND s23_count > 0;
"""

display(conn.execute(q).fetchdf())

ESTIMATING S1 → S2/S3 CANDIDATE PAIRS


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,overlapping_blocks,candidate_pairs,max_s1_block,max_s23_block,avg_candidates_per_block
0,1022854,432643551.0,4606.0,15291.0,422.976838


In [ ]:
print("=" * 80)
print("COMPARING BLOCKING STRATEGIES")
print("=" * 80)

q = """
WITH blocks AS (
    SELECT
        rare_key,
        r1_key,
        r2_key,

        SUM(CASE WHEN src = 1 THEN 1 ELSE 0 END) AS s1_count,
        SUM(CASE WHEN src IN (2,3) THEN 1 ELSE 0 END) AS s23_count
    FROM keys_train
    WHERE rare_key IS NOT NULL
    GROUP BY rare_key, r1_key, r2_key
)

SELECT
    SUM(s1_count * s23_count) AS candidate_pairs,
    COUNT(*) AS blocks,
    MAX(s1_count) AS max_s1_block,
    MAX(s23_count) AS max_s23_block,
    AVG(s1_count * s23_count) AS avg_candidates_per_block
FROM blocks
WHERE s1_count > 0
  AND s23_count > 0;
"""

display(conn.execute(q).fetchdf())

COMPARING BLOCKING STRATEGIES


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,candidate_pairs,blocks,max_s1_block,max_s23_block,avg_candidates_per_block
0,432643551.0,1022854,4606.0,15291.0,422.976838


In [ ]:
print("=" * 80)
print("R2-KEY CANDIDATES")
print("=" * 80)

q = """
WITH blocks AS (
    SELECT
        r2_key,
        SUM(CASE WHEN src = 1 THEN 1 ELSE 0 END) AS s1_count,
        SUM(CASE WHEN src IN (2,3) THEN 1 ELSE 0 END) AS s23_count
    FROM keys_train
    WHERE r2_key IS NOT NULL
    GROUP BY r2_key
)

SELECT
    COUNT(*) AS overlapping_blocks,
    SUM(s1_count * s23_count) AS candidate_pairs,
    MAX(s1_count) AS max_s1_block,
    MAX(s23_count) AS max_s23_block
FROM blocks
WHERE s1_count > 0
  AND s23_count > 0;
"""

display(conn.execute(q).fetchdf())

R2-KEY CANDIDATES


,overlapping_blocks,candidate_pairs,max_s1_block,max_s23_block
0,31536,2.411363e+10,19876.0,121899.0


In [ ]:
print("=" * 80)
print("1/3 — R1 + R2 BLOCKING")
print("=" * 80)

q = """
WITH blocks AS (
    SELECT
        r1_key,
        r2_key,
        SUM(CASE WHEN src = 1 THEN 1 ELSE 0 END) AS s1_count,
        SUM(CASE WHEN src IN (2,3) THEN 1 ELSE 0 END) AS s23_count
    FROM keys_train
    WHERE r1_key IS NOT NULL
      AND r2_key IS NOT NULL
    GROUP BY r1_key, r2_key
)
SELECT
    COUNT(*) AS overlapping_blocks,
    SUM(s1_count * s23_count) AS candidate_pairs,
    MAX(s1_count) AS max_s1_block,
    MAX(s23_count) AS max_s23_block
FROM blocks
WHERE s1_count > 0
  AND s23_count > 0;
"""

display(conn.execute(q).fetchdf())

1/3 — R1 + R2 BLOCKING


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,overlapping_blocks,candidate_pairs,max_s1_block,max_s23_block
0,1018641,430143416.0,4606.0,15291.0


In [ ]:
print("=" * 80)
print("2/3 — R1 ONLY")
print("=" * 80)

q = """
WITH blocks AS (
    SELECT
        r1_key,
        SUM(CASE WHEN src = 1 THEN 1 ELSE 0 END) AS s1_count,
        SUM(CASE WHEN src IN (2,3) THEN 1 ELSE 0 END) AS s23_count
    FROM keys_train
    WHERE r1_key IS NOT NULL
    GROUP BY r1_key
)
SELECT
    COUNT(*) AS overlapping_blocks,
    SUM(s1_count * s23_count) AS candidate_pairs,
    MAX(s1_count) AS max_s1_block,
    MAX(s23_count) AS max_s23_block
FROM blocks
WHERE s1_count > 0
  AND s23_count > 0;
"""

display(conn.execute(q).fetchdf())

2/3 — R1 ONLY


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,overlapping_blocks,candidate_pairs,max_s1_block,max_s23_block
0,116068,4.970454e+09,4642.0,15843.0


In [ ]:
print("=" * 80)
print("3/3 — R2 ONLY")
print("=" * 80)

q = """
WITH blocks AS (
    SELECT
        r2_key,
        SUM(CASE WHEN src = 1 THEN 1 ELSE 0 END) AS s1_count,
        SUM(CASE WHEN src IN (2,3) THEN 1 ELSE 0 END) AS s23_count
    FROM keys_train
    WHERE r2_key IS NOT NULL
    GROUP BY r2_key
)
SELECT
    COUNT(*) AS overlapping_blocks,
    SUM(s1_count * s23_count) AS candidate_pairs,
    MAX(s1_count) AS max_s1_block,
    MAX(s23_count) AS max_s23_block
FROM blocks
WHERE s1_count > 0
  AND s23_count > 0;
"""

display(conn.execute(q).fetchdf())

3/3 — R2 ONLY


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,overlapping_blocks,candidate_pairs,max_s1_block,max_s23_block
0,31536,2.411363e+10,19876.0,121899.0


In [ ]:
print("=" * 80)
print("EXACT NORMALIZED-NAME FILTER")
print("=" * 80)

q = """
WITH candidates AS (
    SELECT
        a.entity_id AS s1_id,
        b.entity_id AS s23_id
    FROM keys_train a
    JOIN keys_train b
      ON a.rare_key = b.rare_key
     AND a.src = 1
     AND b.src IN (2,3)
)
SELECT COUNT(*) AS total_rare_candidates
FROM candidates;
"""

display(conn.execute(q).fetchdf())

EXACT NORMALIZED-NAME FILTER


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rare_candidates
0,432643551


In [ ]:
print("=" * 80)
print("EXACT NAME MATCHES INSIDE RARE-KEY BLOCKS")
print("=" * 80)

q = """
SELECT COUNT(*) AS exact_name_pairs
FROM (
    SELECT
        a.entity_id AS s1_id,
        b.entity_id AS s23_id
    FROM keys_train a
    JOIN keys_train b
      ON a.rare_key = b.rare_key
     AND a.src = 1
     AND b.src IN (2,3)
    JOIN recs_train ra
      ON ra.entity_id = a.entity_id
    JOIN recs_train rb
      ON rb.entity_id = b.entity_id
    WHERE ra.name_norm = rb.name_norm
      AND ra.name_norm IS NOT NULL
      AND ra.name_norm <> ''
) x;
"""

display(conn.execute(q).fetchdf())

EXACT NAME MATCHES INSIDE RARE-KEY BLOCKS


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,exact_name_pairs
0,24348720


In [ ]:
print("=" * 80)
print("EXACT NAME + ADDRESS MATCHES")
print("=" * 80)

q = """
SELECT COUNT(*) AS exact_name_address_pairs
FROM (
    SELECT
        a.entity_id AS s1_id,
        b.entity_id AS s23_id
    FROM keys_train a
    JOIN keys_train b
      ON a.rare_key = b.rare_key
     AND a.src = 1
     AND b.src IN (2,3)
    JOIN recs_train ra
      ON ra.entity_id = a.entity_id
    JOIN recs_train rb
      ON rb.entity_id = b.entity_id
    WHERE ra.name_norm = rb.name_norm
      AND ra.name_norm IS NOT NULL
      AND ra.name_norm <> ''
      AND ra.addr_norm = rb.addr_norm
      AND ra.addr_norm IS NOT NULL
      AND ra.addr_norm <> ''
) x;
"""

display(conn.execute(q).fetchdf())

EXACT NAME + ADDRESS MATCHES


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,exact_name_address_pairs
0,117932


In [ ]:
print("=" * 80)
print("SAVING EXACT NAME + ADDRESS CANDIDATES")
print("=" * 80)

conn.execute("DROP TABLE IF EXISTS exact_candidates_train")

conn.execute("""
CREATE TABLE exact_candidates_train AS
SELECT
    a.entity_id AS s1_id,
    b.entity_id AS s23_id,
    b.src AS target_src
FROM keys_train a
JOIN keys_train b
  ON a.rare_key = b.rare_key
 AND a.src = 1
 AND b.src IN (2,3)
JOIN recs_train ra
  ON ra.entity_id = a.entity_id
JOIN recs_train rb
  ON rb.entity_id = b.entity_id
WHERE ra.name_norm = rb.name_norm
  AND ra.name_norm IS NOT NULL
  AND ra.name_norm <> ''
  AND ra.addr_norm = rb.addr_norm
  AND ra.addr_norm IS NOT NULL
  AND ra.addr_norm <> ''
""")

conn.execute("""
CREATE INDEX IF NOT EXISTS idx_exact_candidates_s1
ON exact_candidates_train(s1_id)
""")

conn.execute("""
CREATE INDEX IF NOT EXISTS idx_exact_candidates_s23
ON exact_candidates_train(s23_id)
""")

count = conn.execute("""
SELECT COUNT(*) FROM exact_candidates_train
""").fetchone()[0]

print(f"Saved candidates: {count:,}")

SAVING EXACT NAME + ADDRESS CANDIDATES


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved candidates: 117,932


In [ ]:
from pathlib import Path

checkpoint_dir = Path("/content/drive/MyDrive/er_checkpoint")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

(checkpoint_dir / "CHECKPOINT.txt").write_text(
    "Amazon ML Entity Resolution\n"
    "Checkpoint: exact name + address candidates completed\n"
    f"Exact candidates: {count:,}\n"
    "Next step: optimized candidate generation for remaining records\n"
)

print("✅ CHECKPOINT SAVED")
print(checkpoint_dir / "CHECKPOINT.txt")

✅ CHECKPOINT SAVED
/content/drive/MyDrive/er_checkpoint/CHECKPOINT.txt


In [ ]:
print(
    conn.execute("""
        SELECT
            target_src,
            COUNT(*) AS candidates
        FROM exact_candidates_train
        GROUP BY target_src
        ORDER BY target_src
    """).fetchdf()
)

   target_src  candidates
0           2      117348
1           3         584


In [ ]:
print("=" * 80)
print("VERIFYING CHECKPOINT")
print("=" * 80)

print(
    conn.execute("""
        SELECT COUNT(*) AS total
        FROM exact_candidates_train
    """).fetchdf()
)

print(
    conn.execute("""
        SELECT
            target_src,
            COUNT(*) AS candidates
        FROM exact_candidates_train
        GROUP BY target_src
        ORDER BY target_src
    """).fetchdf()
)

print("\nTables:")
print(
    conn.execute("""
        SELECT table_name
        FROM information_schema.tables
        ORDER BY table_name
    """).fetchdf().to_string(index=False)
)

VERIFYING CHECKPOINT
    total
0  117932
   target_src  candidates
0           2      117348
1           3         584

Tables:
            table_name
exact_candidates_train
            keys_train
           nrare_train
             recs_test
            recs_train


In [ ]:
print("=" * 80)
print("PREFIX-4 BLOCKING DIAGNOSTIC")
print("=" * 80)

q = """
WITH blocks AS (
    SELECT
        country || '|' || LEFT(name_norm, 4) AS block_key,

        SUM(CASE WHEN src = 1 THEN 1 ELSE 0 END) AS s1_count,

        SUM(CASE WHEN src IN (2,3) THEN 1 ELSE 0 END) AS s23_count

    FROM recs_train

    WHERE name_norm IS NOT NULL
      AND name_norm <> ''
      AND country IS NOT NULL

    GROUP BY country || '|' || LEFT(name_norm, 4)
)

SELECT
    COUNT(*) AS overlapping_blocks,
    SUM(s1_count * s23_count) AS candidate_pairs,
    MAX(s1_count) AS max_s1_block,
    MAX(s23_count) AS max_s23_block,
    AVG(s1_count * s23_count) AS avg_candidates_per_block
FROM blocks

WHERE s1_count > 0
  AND s23_count > 0;
"""

display(conn.execute(q).fetchdf())

PREFIX-4 BLOCKING DIAGNOSTIC


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,overlapping_blocks,candidate_pairs,max_s1_block,max_s23_block,avg_candidates_per_block
0,75664,1.680116e+10,15080.0,91053.0,222049.636736


In [ ]:
print("=" * 80)
print("RARE-KEY CANDIDATES BY BLOCK SIZE")
print("=" * 80)

q = """
WITH block_counts AS (
    SELECT
        rare_key,
        SUM(CASE WHEN src = 1 THEN 1 ELSE 0 END) AS s1_count,
        SUM(CASE WHEN src IN (2,3) THEN 1 ELSE 0 END) AS s23_count
    FROM keys_train
    WHERE rare_key IS NOT NULL
    GROUP BY rare_key
),
stats AS (
    SELECT
        *,
        s1_count * s23_count AS candidates
    FROM block_counts
    WHERE s1_count > 0
      AND s23_count > 0
)
SELECT
    CASE
        WHEN GREATEST(s1_count, s23_count) <= 10 THEN '01: <=10'
        WHEN GREATEST(s1_count, s23_count) <= 25 THEN '02: 11-25'
        WHEN GREATEST(s1_count, s23_count) <= 50 THEN '03: 26-50'
        WHEN GREATEST(s1_count, s23_count) <= 100 THEN '04: 51-100'
        WHEN GREATEST(s1_count, s23_count) <= 250 THEN '05: 101-250'
        WHEN GREATEST(s1_count, s23_count) <= 500 THEN '06: 251-500'
        WHEN GREATEST(s1_count, s23_count) <= 1000 THEN '07: 501-1000'
        WHEN GREATEST(s1_count, s23_count) <= 2500 THEN '08: 1001-2500'
        WHEN GREATEST(s1_count, s23_count) <= 5000 THEN '09: 2501-5000'
        ELSE '10: >5000'
    END AS block_size_range,

    COUNT(*) AS blocks,
    SUM(candidates) AS candidate_pairs,
    MAX(candidates) AS largest_candidate_block

FROM stats

GROUP BY 1
ORDER BY 1;
"""

display(conn.execute(q).fetchdf())

RARE-KEY CANDIDATES BY BLOCK SIZE


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,block_size_range,blocks,candidate_pairs,largest_candidate_block
0,01: <=10,936097,4724266.0,100.0
1,02: 11-25,63154,4766399.0,550.0
2,03: 26-50,10740,3635322.0,2450.0
3,04: 51-100,8065,23809964.0,7500.0
4,05: 101-250,3983,29448842.0,35035.0
5,06: 251-500,597,16856653.0,109125.0
6,07: 501-1000,159,27262123.0,357458.0
7,08: 1001-2500,35,25689626.0,2138678.0
8,09: 2501-5000,14,50022182.0,6978225.0
9,10: >5000,10,246428174.0,70430346.0


In [ ]:
print("=" * 80)
print("10 LARGEST RARE-KEY BLOCKS BY CANDIDATE PAIRS")
print("=" * 80)

q = """
WITH block_counts AS (
    SELECT
        rare_key,

        SUM(CASE WHEN src = 1 THEN 1 ELSE 0 END) AS s1_count,

        SUM(CASE WHEN src IN (2,3) THEN 1 ELSE 0 END) AS s23_count

    FROM keys_train

    WHERE rare_key IS NOT NULL

    GROUP BY rare_key
)

SELECT
    rare_key,
    s1_count,
    s23_count,
    s1_count * s23_count AS candidate_pairs

FROM block_counts

WHERE s1_count > 0
  AND s23_count > 0

ORDER BY candidate_pairs DESC

LIMIT 20;
"""

display(conn.execute(q).fetchdf())

10 LARGEST RARE-KEY BLOCKS BY CANDIDATE PAIRS


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rare_key,s1_count,s23_count,candidate_pairs
0,us|ankle|foot,4606.0,15291.0,70430346.0
1,us|nose|throat,4582.0,15037.0,68899534.0
2,us|physical|therapy,3043.0,10024.0,30503032.0
3,us|dental|pediatric,2145.0,7085.0,15197325.0
4,us|dentistry|pediatric,2116.0,6997.0,14805652.0
5,us|internal|medicine,1948.0,6455.0,12574340.0
6,us|alley|back,1639.0,5945.0,9743855.0
7,us|edge|straight,1592.0,5817.0,9260664.0
8,us|behavioral|health,1490.0,5081.0,7570690.0
9,us|up|rev,1428.0,5212.0,7442736.0


In [ ]:
print("=" * 80)
print("LARGE-BLOCK NAME-CORE FILTER")
print("=" * 80)

q = """
WITH large_blocks AS (
    SELECT
        rare_key
    FROM (
        SELECT
            rare_key,
            SUM(CASE WHEN src = 1 THEN 1 ELSE 0 END) AS s1_count,
            SUM(CASE WHEN src IN (2,3) THEN 1 ELSE 0 END) AS s23_count
        FROM keys_train
        WHERE rare_key IS NOT NULL
        GROUP BY rare_key
    )
    WHERE s1_count > 0
      AND s23_count > 0
      AND GREATEST(s1_count, s23_count) > 500
)

SELECT COUNT(*) AS name_core_pairs
FROM keys_train a
JOIN keys_train b
  ON a.rare_key = b.rare_key
 AND a.src = 1
 AND b.src IN (2,3)
JOIN large_blocks lb
  ON a.rare_key = lb.rare_key
JOIN recs_train ra
  ON ra.entity_id = a.entity_id
JOIN recs_train rb
  ON rb.entity_id = b.entity_id
WHERE ra.name_core = rb.name_core
  AND ra.name_core IS NOT NULL
  AND ra.name_core <> '';
"""

display(conn.execute(q).fetchdf())

LARGE-BLOCK NAME-CORE FILTER


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,name_core_pairs
0,8298683


In [ ]:
print("=" * 80)
print("LARGE-BLOCK ADDRESS-NUMBER FILTER")
print("=" * 80)

q = """
WITH large_blocks AS (
    SELECT
        rare_key
    FROM (
        SELECT
            rare_key,
            SUM(CASE WHEN src = 1 THEN 1 ELSE 0 END) AS s1_count,
            SUM(CASE WHEN src IN (2,3) THEN 1 ELSE 0 END) AS s23_count
        FROM keys_train
        WHERE rare_key IS NOT NULL
        GROUP BY rare_key
    )
    WHERE s1_count > 0
      AND s23_count > 0
      AND GREATEST(s1_count, s23_count) > 500
)

SELECT COUNT(*) AS address_number_pairs
FROM keys_train a
JOIN keys_train b
  ON a.rare_key = b.rare_key
 AND a.src = 1
 AND b.src IN (2,3)
JOIN large_blocks lb
  ON a.rare_key = lb.rare_key
JOIN recs_train ra
  ON ra.entity_id = a.entity_id
JOIN recs_train rb
  ON rb.entity_id = b.entity_id
WHERE ra.addr_nums IS NOT NULL
  AND ra.addr_nums <> ''
  AND rb.addr_nums IS NOT NULL
  AND rb.addr_nums <> ''
  AND ra.addr_nums = rb.addr_nums;
"""

display(conn.execute(q).fetchdf())

LARGE-BLOCK ADDRESS-NUMBER FILTER


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,address_number_pairs
0,210920


In [ ]:
print("=" * 80)
print("CREATING OPTIMIZED TRAIN CANDIDATE TABLE")
print("=" * 80)

conn.execute("DROP TABLE IF EXISTS candidates_train")

conn.execute("""
CREATE TABLE candidates_train (
    s1_id VARCHAR,
    s23_id VARCHAR,
    target_src TINYINT
)
""")

print("✅ Empty candidate table created")

CREATING OPTIMIZED TRAIN CANDIDATE TABLE
✅ Empty candidate table created


In [ ]:
print("=" * 80)
print("ADDING SMALL-BLOCK CANDIDATES")
print("=" * 80)

conn.execute("""
INSERT INTO candidates_train
SELECT
    a.entity_id AS s1_id,
    b.entity_id AS s23_id,
    b.src AS target_src

FROM keys_train a

JOIN keys_train b
  ON a.rare_key = b.rare_key
 AND a.src = 1
 AND b.src IN (2,3)

JOIN (
    SELECT
        rare_key
    FROM keys_train
    WHERE rare_key IS NOT NULL
    GROUP BY rare_key
    HAVING GREATEST(
        SUM(CASE WHEN src = 1 THEN 1 ELSE 0 END),
        SUM(CASE WHEN src IN (2,3) THEN 1 ELSE 0 END)
    ) <= 500
) small
  ON a.rare_key = small.rare_key
""")

small_count = conn.execute("""
SELECT COUNT(*) FROM candidates_train
""").fetchone()[0]

print(f"✅ Small-block candidates: {small_count:,}")

ADDING SMALL-BLOCK CANDIDATES


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Small-block candidates: 83,241,446


In [ ]:
print("=" * 80)
print("ADDING LARGE-BLOCK NAME-CORE CANDIDATES")
print("=" * 80)

conn.execute("""
INSERT INTO candidates_train
SELECT
    a.entity_id AS s1_id,
    b.entity_id AS s23_id,
    b.src AS target_src

FROM keys_train a

JOIN keys_train b
  ON a.rare_key = b.rare_key
 AND a.src = 1
 AND b.src IN (2,3)

JOIN (
    SELECT
        rare_key
    FROM keys_train
    WHERE rare_key IS NOT NULL
    GROUP BY rare_key
    HAVING GREATEST(
        SUM(CASE WHEN src = 1 THEN 1 ELSE 0 END),
        SUM(CASE WHEN src IN (2,3) THEN 1 ELSE 0 END)
    ) > 500
) large
  ON a.rare_key = large.rare_key

JOIN recs_train ra
  ON ra.entity_id = a.entity_id

JOIN recs_train rb
  ON rb.entity_id = b.entity_id

WHERE ra.name_core = rb.name_core
  AND ra.name_core IS NOT NULL
  AND ra.name_core <> ''
""")

current_count = conn.execute("""
SELECT COUNT(*) FROM candidates_train
""").fetchone()[0]

print(f"✅ Candidates so far: {current_count:,}")

ADDING LARGE-BLOCK NAME-CORE CANDIDATES


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Candidates so far: 91,540,129


In [ ]:
# ============================================================
# DUCKDB FILE HEALTH CHECK
# ============================================================

import os
import glob

DB_PATH = "/content/drive/MyDrive/er_checkpoint/er_after_blocking_train.duckdb"

print("=" * 80)
print("DUCKDB FILE HEALTH CHECK")
print("=" * 80)

if not os.path.exists(DB_PATH):
    print("❌ Database file does not exist")
else:
    size = os.path.getsize(DB_PATH)

    print(f"\nDatabase:")
    print(DB_PATH)

    print(f"\nFile size:")
    print(f"{size:,} bytes")
    print(f"{size / (1024**3):.2f} GB")

    print("\nTemporary files/directories:")

    for path in glob.glob(
        "/content/drive/MyDrive/er_checkpoint/*duckdb*"
    ):
        if os.path.isdir(path):
            print(f"  📁 {path}")
        else:
            s = os.path.getsize(path)
            print(f"  📄 {path} ({s/(1024**3):.2f} GB)")

print("\n" + "=" * 80)
print("IMPORTANT")
print("=" * 80)

print("""
❌ DO NOT run the candidate INSERT again.
❌ DO NOT delete the original TSV dataset.
❌ DO NOT overwrite this DuckDB file yet.

The next step is to determine whether we can recover the
completed blocking data or whether we need to rebuild only
the DuckDB blocking checkpoint.
""")

DUCKDB FILE HEALTH CHECK

Database:
/content/drive/MyDrive/er_checkpoint/er_after_blocking_train.duckdb

File size:
3,350,736,896 bytes
3.12 GB

Temporary files/directories:
  📄 /content/drive/MyDrive/er_checkpoint/er_after_blocking_train.duckdb (3.12 GB)

IMPORTANT

❌ DO NOT run the candidate INSERT again.
❌ DO NOT delete the original TSV dataset.
❌ DO NOT overwrite this DuckDB file yet.

The next step is to determine whether we can recover the
completed blocking data or whether we need to rebuild only
the DuckDB blocking checkpoint.



In [ ]:
# ============================================================
# DUCKDB RECOVERY TEST — READ ONLY
# ============================================================

import duckdb
import os

DB_PATH = "/content/drive/MyDrive/er_checkpoint/er_after_blocking_train.duckdb"

print("=" * 80)
print("DUCKDB RECOVERY TEST")
print("=" * 80)

print(f"File size: {os.path.getsize(DB_PATH)/(1024**3):.2f} GB")

# Open read-only so we DO NOT modify the damaged checkpoint
try:

    recovery_conn = duckdb.connect(
        DB_PATH,
        read_only=True
    )

    print("\n✓ DuckDB opened the file in READ-ONLY mode")

    # --------------------------------------------------------
    # List tables
    # --------------------------------------------------------

    tables = recovery_conn.execute("""
        SELECT table_schema, table_name
        FROM information_schema.tables
        ORDER BY table_schema, table_name
    """).fetchall()

    print("\n" + "-" * 80)
    print("TABLES FOUND")
    print("-" * 80)

    if not tables:
        print("❌ No tables found")
    else:
        for schema, table in tables:
            print(f"✓ {schema}.{table}")

    # --------------------------------------------------------
    # Check important tables
    # --------------------------------------------------------

    print("\n" + "-" * 80)
    print("IMPORTANT TABLE CHECK")
    print("-" * 80)

    table_names = {t[1] for t in tables}

    for name in [
        "train_s1",
        "train_s2",
        "train_s3",
        "train_blocking",
        "train_keys",
        "candidates_train"
    ]:

        if name in table_names:

            try:
                count = recovery_conn.execute(
                    f'SELECT COUNT(*) FROM "{name}"'
                ).fetchone()[0]

                print(
                    f"✓ {name:<25} {count:,} rows"
                )

            except Exception as e:

                print(
                    f"⚠ {name:<25} EXISTS but cannot read: {e}"
                )

        else:
            print(
                f"— {name:<25} not found"
            )

    # --------------------------------------------------------
    # Database metadata
    # --------------------------------------------------------

    print("\n" + "-" * 80)
    print("DATABASE SIZE / TABLE STORAGE")
    print("-" * 80)

    try:
        db_info = recovery_conn.execute(
            "PRAGMA database_size"
        ).fetchdf()

        display(db_info)

    except Exception as e:
        print("Could not read database_size:", e)

    recovery_conn.close()

    print("\n" + "=" * 80)
    print("RECOVERY TEST FINISHED")
    print("=" * 80)

except Exception as e:

    print("\n❌ READ-ONLY OPEN FAILED")
    print("\nExact error:")
    print(e)

    print("\n" + "=" * 80)
    print("RESULT")
    print("=" * 80)

    print("""
The DuckDB checkpoint cannot currently be opened safely.

Do NOT delete it yet.
Do NOT overwrite it yet.

We can still rebuild from the original TSV files if necessary.
""")

DUCKDB RECOVERY TEST
File size: 3.12 GB

❌ READ-ONLY OPEN FAILED

Exact error:
IO Error: Could not read enough bytes from file "/content/drive/MyDrive/er_checkpoint/er_after_blocking_train.duckdb": attempted to read 262144 bytes from location 4578619392

RESULT

The DuckDB checkpoint cannot currently be opened safely.

Do NOT delete it yet.
Do NOT overwrite it yet.

We can still rebuild from the original TSV files if necessary.



In [ ]:
# ============================================================
# FIND THE ACTUAL DATASET LOCATION
# ============================================================

import os

SEARCH_ROOT = "/content/drive/MyDrive"

print("=" * 80)
print("SEARCHING GOOGLE DRIVE FOR AMAZON ML DATASET")
print("=" * 80)

target_files = {
    "train_source1.tsv",
    "train_source2.tsv",
    "train_source3.tsv",
    "train_ground_truth.tsv",
    "test_source1.tsv",
    "test_source2.tsv",
    "test_source3.tsv",
}

found = {}

for root, dirs, files in os.walk(SEARCH_ROOT):

    # Ignore hidden/system folders
    dirs[:] = [
        d for d in dirs
        if not d.startswith(".")
        and d != "__MACOSX"
    ]

    for filename in files:

        if filename in target_files:

            full_path = os.path.join(root, filename)
            found[filename] = full_path

            size_gb = os.path.getsize(full_path) / (1024 ** 3)

            print(
                f"✓ {filename:<25} "
                f"{size_gb:>6.2f} GB\n"
                f"  {full_path}\n"
            )

print("=" * 80)
print("SEARCH RESULT")
print("=" * 80)

print(f"Found {len(found)} / {len(target_files)} required files.")

missing = target_files - set(found)

if missing:

    print("\n❌ MISSING FILES:")

    for filename in sorted(missing):
        print("  -", filename)

else:

    print("\n✅ ALL 7 DATASET FILES FOUND")

    # --------------------------------------------------------
    # Print variables ready for next step
    # --------------------------------------------------------

    print("\nUse these paths:")

    for filename in sorted(found):
        print(
            f'{filename.upper().replace(".", "_")} = '
            f'"{found[filename]}"'
        )

SEARCHING GOOGLE DRIVE FOR AMAZON ML DATASET
SEARCH RESULT
Found 0 / 7 required files.

❌ MISSING FILES:
  - test_source1.tsv
  - test_source2.tsv
  - test_source3.tsv
  - train_ground_truth.tsv
  - train_source1.tsv
  - train_source2.tsv
  - train_source3.tsv


In [ ]:
# ============================================================
# SEARCH CURRENT COLAB STORAGE FOR DATASET
# ============================================================

import os

print("=" * 80)
print("SEARCHING CURRENT COLAB STORAGE")
print("=" * 80)

target_files = {
    "train_source1.tsv",
    "train_source2.tsv",
    "train_source3.tsv",
    "train_ground_truth.tsv",
    "test_source1.tsv",
    "test_source2.tsv",
    "test_source3.tsv",
}

found = {}

for search_root in ["/content", "/mnt/data"]:

    if not os.path.exists(search_root):
        continue

    print(f"\nSearching: {search_root}")

    for root, dirs, files in os.walk(search_root):

        dirs[:] = [
            d for d in dirs
            if not d.startswith(".")
            and d not in ["node_modules", "__pycache__"]
        ]

        for filename in files:

            if filename in target_files:

                path = os.path.join(root, filename)

                if filename not in found:
                    found[filename] = path

                    size_gb = (
                        os.path.getsize(path)
                        / (1024 ** 3)
                    )

                    print(
                        f"✓ {filename:<25} "
                        f"{size_gb:.2f} GB\n"
                        f"  {path}"
                    )

print("\n" + "=" * 80)
print("RESULT")
print("=" * 80)

print(f"Found {len(found)} / 7 files.")

if len(found) == 7:

    print("\n🎉 DATASET STILL EXISTS IN COLAB!")

elif len(found) > 0:

    print("\n⚠️ PARTIAL DATASET FOUND.")

else:

    print("""
❌ No TSV dataset files found in the current runtime.

This means the dataset needs to be restored/re-uploaded.
Do NOT delete the corrupted DuckDB yet.
""")

SEARCHING CURRENT COLAB STORAGE

Searching: /content

RESULT
Found 0 / 7 files.

❌ No TSV dataset files found in the current runtime.

This means the dataset needs to be restored/re-uploaded.
Do NOT delete the corrupted DuckDB yet.



In [ ]:
# ============================================================
# FIND ORIGINAL DATASET ZIP ON GOOGLE DRIVE
# ============================================================

import os

DRIVE = "/content/drive/MyDrive"

print("=" * 80)
print("SEARCHING GOOGLE DRIVE FOR DATASET ZIP")
print("=" * 80)

found = []

for root, dirs, files in os.walk(DRIVE):

    dirs[:] = [
        d for d in dirs
        if not d.startswith(".")
        and d != "__MACOSX"
    ]

    for filename in files:

        if filename.lower().endswith(".zip"):

            path = os.path.join(root, filename)
            size_gb = os.path.getsize(path) / (1024 ** 3)

            # Show files that could plausibly be the dataset
            if size_gb >= 0.5:

                found.append((path, size_gb))

                print(
                    f"\n✓ {filename}"
                    f"\n  Size: {size_gb:.2f} GB"
                    f"\n  Path: {path}"
                )

print("\n" + "=" * 80)

if found:
    print(f"FOUND {len(found)} LARGE ZIP FILE(S)")
    print("=" * 80)

    print("\nThe likely dataset ZIP is the one around 1 GB.")
else:
    print("❌ NO LARGE ZIP FILE FOUND")
    print("=" * 80)

    print("""
If your original dataset ZIP is not listed, you will need
to upload/download the dataset again.

DO NOT upload it directly to /content.

Put the ZIP in Google Drive first so it survives Colab
runtime resets.
""")

SEARCHING GOOGLE DRIVE FOR DATASET ZIP

✓ dataset.zip
  Size: 1.02 GB
  Path: /content/drive/MyDrive/dataset.zip

FOUND 1 LARGE ZIP FILE(S)

The likely dataset ZIP is the one around 1 GB.


In [ ]:
# ============================================================
# RESTORE AMAZON ML DATASET TO GOOGLE DRIVE
# ============================================================

import os
import zipfile
import shutil

ZIP_PATH = "/content/drive/MyDrive/dataset.zip"

RESTORE_DIR = "/content/drive/MyDrive/amazon_ml_dataset"

print("=" * 80)
print("RESTORING DATASET")
print("=" * 80)

# ------------------------------------------------------------
# 1. Check ZIP
# ------------------------------------------------------------

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(
        f"Dataset ZIP not found:\n{ZIP_PATH}"
    )

zip_size = os.path.getsize(ZIP_PATH) / (1024 ** 3)

print(f"\nZIP:")
print(f"  {ZIP_PATH}")
print(f"  Size: {zip_size:.2f} GB")

# ------------------------------------------------------------
# 2. Create restore directory
# ------------------------------------------------------------

os.makedirs(RESTORE_DIR, exist_ok=True)

print(f"\nRestore directory:")
print(f"  {RESTORE_DIR}")

# ------------------------------------------------------------
# 3. Check ZIP contents
# ------------------------------------------------------------

print("\nChecking ZIP contents...")

required = {
    "train_source1.tsv",
    "train_source2.tsv",
    "train_source3.tsv",
    "train_ground_truth.tsv",
    "test_source1.tsv",
    "test_source2.tsv",
    "test_source3.tsv",
}

with zipfile.ZipFile(ZIP_PATH, "r") as z:

    names = z.namelist()

    found_inside = set()

    for name in names:

        base = os.path.basename(name)

        if base in required:
            found_inside.add(base)

    print(
        f"✓ ZIP contains {len(found_inside)}/7 required TSV files"
    )

    missing = required - found_inside

    if missing:

        print("\n❌ Missing from ZIP:")

        for x in sorted(missing):
            print("  -", x)

        raise RuntimeError(
            "Dataset ZIP is incomplete."
        )

# ------------------------------------------------------------
# 4. Extract
# ------------------------------------------------------------

print("\nExtracting to Google Drive...")
print("This may take several minutes.")

with zipfile.ZipFile(ZIP_PATH, "r") as z:

    z.extractall(RESTORE_DIR)

print("\n✓ Extraction finished")

# ------------------------------------------------------------
# 5. Locate all required TSVs recursively
# ------------------------------------------------------------

print("\nSearching extracted dataset...")

found = {}

for root, dirs, files in os.walk(RESTORE_DIR):

    dirs[:] = [
        d for d in dirs
        if d != "__MACOSX"
        and not d.startswith(".")
    ]

    for filename in files:

        if filename in required:

            path = os.path.join(root, filename)

            # If duplicate copies exist, keep the first one
            if filename not in found:
                found[filename] = path

# ------------------------------------------------------------
# 6. Verify
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("DATASET VERIFICATION")
print("=" * 80)

for filename in sorted(required):

    if filename not in found:

        print(f"❌ {filename}")

    else:

        path = found[filename]

        size_gb = os.path.getsize(path) / (1024 ** 3)

        print(
            f"✓ {filename:<25}"
            f" {size_gb:>6.2f} GB"
            f"\n  {path}"
        )

if len(found) != 7:

    raise RuntimeError(
        f"Only {len(found)}/7 required files were found."
    )

print("\n" + "=" * 80)
print("🎉 DATASET RESTORED SUCCESSFULLY")
print("=" * 80)

print("""
All 7 required TSV files are now stored on Google Drive.

The corrupted DuckDB checkpoint has NOT been modified.

Next we will build a NEW database/checkpoint safely.
""")

RESTORING DATASET

ZIP:
  /content/drive/MyDrive/dataset.zip
  Size: 1.02 GB

Restore directory:
  /content/drive/MyDrive/amazon_ml_dataset

Checking ZIP contents...
✓ ZIP contains 7/7 required TSV files

Extracting to Google Drive...
This may take several minutes.

✓ Extraction finished

Searching extracted dataset...

DATASET VERIFICATION
✓ test_source1.tsv            0.16 GB
  /content/drive/MyDrive/amazon_ml_dataset/student_resource/dataset/test/test_source1.tsv
✓ test_source2.tsv            0.47 GB
  /content/drive/MyDrive/amazon_ml_dataset/student_resource/dataset/test/test_source2.tsv
✓ test_source3.tsv            0.47 GB
  /content/drive/MyDrive/amazon_ml_dataset/student_resource/dataset/test/test_source3.tsv
✓ train_ground_truth.tsv      0.12 GB
  /content/drive/MyDrive/amazon_ml_dataset/student_resource/dataset/train/train_ground_truth.tsv
✓ train_source1.tsv           0.20 GB
  /content/drive/MyDrive/amazon_ml_dataset/student_resource/dataset/train/train_source1.tsv
✓ train_

In [ ]:
# ============================================================
# STEP 3 — FRESH RESTARTABLE DUCKDB DATABASE
# ============================================================

import os
import json
import time
import duckdb

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

DATASET_ROOT = "/content/drive/MyDrive/amazon_ml_dataset/student_resource/dataset"

TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
TEST_DIR  = os.path.join(DATASET_ROOT, "test")

CHECKPOINT_DIR = "/content/drive/MyDrive/er_checkpoint"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# NEW database — does NOT touch corrupted database
DB_PATH = os.path.join(
    CHECKPOINT_DIR,
    "er_train_fresh.duckdb"
)

# LOCAL temporary storage
LOCAL_TMP = "/content/er_duckdb_tmp"
os.makedirs(LOCAL_TMP, exist_ok=True)

STATE_FILE = os.path.join(
    CHECKPOINT_DIR,
    "er_train_fresh_state.json"
)

FILES = {
    "train_s1":
        os.path.join(TRAIN_DIR, "train_source1.tsv"),

    "train_s2":
        os.path.join(TRAIN_DIR, "train_source2.tsv"),

    "train_s3":
        os.path.join(TRAIN_DIR, "train_source3.tsv"),

    "train_ground_truth":
        os.path.join(TRAIN_DIR, "train_ground_truth.tsv"),

    "test_s1":
        os.path.join(TEST_DIR, "test_source1.tsv"),

    "test_s2":
        os.path.join(TEST_DIR, "test_source2.tsv"),

    "test_s3":
        os.path.join(TEST_DIR, "test_source3.tsv"),
}

# ------------------------------------------------------------
# SAFETY CHECK
# ------------------------------------------------------------

print("=" * 80)
print("FRESH DUCKDB DATABASE BUILD")
print("=" * 80)

print("\nNew database:")
print(DB_PATH)

print("\nOriginal corrupted database is untouched:")
print(
    os.path.join(
        CHECKPOINT_DIR,
        "er_after_blocking_train.duckdb"
    )
)

# ------------------------------------------------------------
# LOAD STATE
# ------------------------------------------------------------

if os.path.exists(STATE_FILE):

    try:
        with open(STATE_FILE, "r") as f:
            state = json.load(f)

    except Exception:
        state = {}

else:
    state = {}

state.setdefault("completed", [])

# ------------------------------------------------------------
# OPEN NEW DATABASE
# ------------------------------------------------------------

conn = duckdb.connect(DB_PATH)

# CRITICAL:
# Temporary files stay on local Colab storage.
conn.execute(
    f"SET temp_directory='{LOCAL_TMP}'"
)

conn.execute(
    "SET preserve_insertion_order=false"
)

# Keep memory usage controlled
conn.execute(
    "SET threads=2"
)

print("\n✓ Fresh DuckDB opened")
print("✓ Temporary storage:", LOCAL_TMP)

# ------------------------------------------------------------
# SAVE STATE FUNCTION
# ------------------------------------------------------------

def save_state():

    state["database"] = DB_PATH
    state["updated"] = time.strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    tmp = STATE_FILE + ".tmp"

    with open(tmp, "w") as f:
        json.dump(state, f, indent=2)

    os.replace(tmp, STATE_FILE)

# ------------------------------------------------------------
# LOAD TABLES ONE BY ONE
# ------------------------------------------------------------

for table, path in FILES.items():

    print("\n" + "=" * 80)
    print(f"TABLE: {table}")
    print("=" * 80)

    # --------------------------------------------------------
    # Check source file
    # --------------------------------------------------------

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"Missing:\n{path}"
        )

    size_gb = os.path.getsize(path) / (1024 ** 3)

    print(f"Source size: {size_gb:.2f} GB")

    # --------------------------------------------------------
    # Skip completed table
    # --------------------------------------------------------

    if table in state["completed"]:

        exists = conn.execute("""
            SELECT COUNT(*)
            FROM information_schema.tables
            WHERE table_name = ?
        """, [table]).fetchone()[0]

        if exists:

            print("✓ Already completed — SKIPPING")
            continue

    # --------------------------------------------------------
    # Drop incomplete table if it exists
    # --------------------------------------------------------

    conn.execute(
        f'DROP TABLE IF EXISTS "{table}"'
    )

    start = time.time()

    # --------------------------------------------------------
    # Direct DuckDB TSV import
    #
    # This is much faster than pandas → DuckDB.
    # --------------------------------------------------------

    escaped_path = path.replace("'", "''")

    conn.execute(f"""
        CREATE TABLE "{table}" AS
        SELECT *
        FROM read_csv(
            '{escaped_path}',
            delim='\\t',
            header=true,
            all_varchar=true,
            ignore_errors=false
        )
    """)

    # --------------------------------------------------------
    # Count rows
    # --------------------------------------------------------

    count = conn.execute(
        f'SELECT COUNT(*) FROM "{table}"'
    ).fetchone()[0]

    elapsed = (time.time() - start) / 60

    print(f"\n✓ {table} loaded")
    print(f"  Rows : {count:,}")
    print(f"  Time : {elapsed:.2f} minutes")

    # --------------------------------------------------------
    # CHECKPOINT
    # --------------------------------------------------------

    if table not in state["completed"]:
        state["completed"].append(table)

    save_state()

    print("✓ CHECKPOINT SAVED")

# ------------------------------------------------------------
# FINAL CHECK
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("DATABASE VERIFICATION")
print("=" * 80)

tables = conn.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema='main'
    ORDER BY table_name
""").fetchall()

for (table,) in tables:

    count = conn.execute(
        f'SELECT COUNT(*) FROM "{table}"'
    ).fetchone()[0]

    print(
        f"✓ {table:<25} {count:,} rows"
    )

# ------------------------------------------------------------
# CLOSE CLEANLY
# ------------------------------------------------------------

conn.close()

db_size = os.path.getsize(DB_PATH) / (1024 ** 3)

print("\n" + "=" * 80)
print("🎉 FRESH DATABASE COMPLETE")
print("=" * 80)

print(f"\nDatabase size: {db_size:.2f} GB")
print(f"Database: {DB_PATH}")
print(f"State: {STATE_FILE}")

print("""
IMPORTANT:
✓ Original TSV files remain untouched.
✓ Old corrupted DuckDB remains untouched.
✓ Each table has a Drive checkpoint.
✓ DuckDB temporary storage uses local Colab disk.

DO NOT run the old candidate-generation cell yet.
""")

FRESH DUCKDB DATABASE BUILD

New database:
/content/drive/MyDrive/er_checkpoint/er_train_fresh.duckdb

Original corrupted database is untouched:
/content/drive/MyDrive/er_checkpoint/er_after_blocking_train.duckdb

✓ Fresh DuckDB opened
✓ Temporary storage: /content/er_duckdb_tmp

TABLE: train_s1
Source size: 0.20 GB


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


✓ train_s1 loaded
  Rows : 2,206,821
  Time : 0.18 minutes
✓ CHECKPOINT SAVED

TABLE: train_s2
Source size: 0.46 GB


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


✓ train_s2 loaded
  Rows : 5,034,616
  Time : 0.27 minutes
✓ CHECKPOINT SAVED

TABLE: train_s3
Source size: 0.47 GB


...


✓ test_s1 loaded
  Rows : 1,732,544
  Time : 0.11 minutes
✓ CHECKPOINT SAVED

TABLE: test_s2
Source size: 0.47 GB


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


✓ test_s2 loaded
  Rows : 4,887,273
  Time : 0.30 minutes
✓ CHECKPOINT SAVED

TABLE: test_s3
Source size: 0.47 GB


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


✓ test_s3 loaded
  Rows : 5,082,316
  Time : 0.30 minutes
✓ CHECKPOINT SAVED

DATABASE VERIFICATION
✓ test_s1                   1,732,544 rows
✓ test_s2                   4,887,273 rows
✓ test_s3                   5,082,316 rows
✓ train_ground_truth        2,206,821 rows
✓ train_s1                  2,206,821 rows
✓ train_s2                  5,034,616 rows
✓ train_s3                  5,285,603 rows

🎉 FRESH DATABASE COMPLETE

Database size: 1.39 GB
Database: /content/drive/MyDrive/er_checkpoint/er_train_fresh.duckdb
State: /content/drive/MyDrive/er_checkpoint/er_train_fresh_state.json

IMPORTANT:
✓ Original TSV files remain untouched.
✓ Old corrupted DuckDB remains untouched.
✓ Each table has a Drive checkpoint.
✓ DuckDB temporary storage uses local Colab disk.

DO NOT run the old candidate-generation cell yet.



In [ ]:
# ============================================================
# STEP 4 — INSPECT FRESH DATABASE SCHEMA
# ============================================================

import duckdb
import pandas as pd

DB_PATH = "/content/drive/MyDrive/er_checkpoint/er_train_fresh.duckdb"
LOCAL_TMP = "/content/er_duckdb_tmp"

conn = duckdb.connect(DB_PATH)

# Keep temporary work on local Colab storage
conn.execute(f"SET temp_directory='{LOCAL_TMP}'")
conn.execute("SET preserve_insertion_order=false")
conn.execute("SET threads=2")

print("=" * 80)
print("FRESH DATABASE SCHEMA")
print("=" * 80)

tables = [
    "train_s1",
    "train_s2",
    "train_s3",
    "train_ground_truth",
    "test_s1",
    "test_s2",
    "test_s3"
]

for table in tables:

    print("\n" + "-" * 80)
    print(f"TABLE: {table}")
    print("-" * 80)

    schema = conn.execute(
        f'DESCRIBE "{table}"'
    ).fetchdf()

    print(
        schema[
            ["column_name", "column_type"]
        ].to_string(index=False)
    )

print("\n" + "=" * 80)
print("SAMPLE ROWS")
print("=" * 80)

for table in [
    "train_s1",
    "train_s2",
    "train_s3"
]:

    print(f"\n--- {table} ---")

    sample = conn.execute(
        f'SELECT * FROM "{table}" LIMIT 2'
    ).fetchdf()

    display(sample)

conn.close()

print("\n" + "=" * 80)
print("SCHEMA INSPECTION COMPLETE")
print("=" * 80)

print("""
Do NOT run the old blocking-key or candidate-generation cells yet.

Send me this output. I will build the new blocking stage
directly against these verified columns.
""")

FRESH DATABASE SCHEMA

--------------------------------------------------------------------------------
TABLE: train_s1
--------------------------------------------------------------------------------
     column_name column_type
       entity_id     VARCHAR
   business_name     VARCHAR
business_address     VARCHAR
         country     VARCHAR

--------------------------------------------------------------------------------
TABLE: train_s2
--------------------------------------------------------------------------------
     column_name column_type
       entity_id     VARCHAR
   business_name     VARCHAR
business_address     VARCHAR
         country     VARCHAR

--------------------------------------------------------------------------------
TABLE: train_s3
--------------------------------------------------------------------------------
     column_name column_type
       entity_id     VARCHAR
   business_name     VARCHAR
business_address     VARCHAR
         country     VARCHAR

-----

,entity_id,business_name,business_address,country
0,S1-384402702,Corner Bike Shop PC,"1297 Melrose Road, Lockesburg, AR",US
1,S1-632543328,Mps Ply Corp,R No 5B 389/91 Mahindramansion Opp Bank Of Bar...,India



--- train_s2 ---


,entity_id,business_name,business_address,country
0,S2-520328759,"Lindsay, Morgan-& Morgan-&","2067 COOS BAY WAGON ROAD, ROSEBURG, OR",US
1,S2-851743177,Sterleng Inc.,"LA, 194 GENERAL ADAMS AVENUE, BATON ROUGE",US



--- train_s3 ---


,entity_id,business_name,business_address,country
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US
1,S3-859268022,International South Consultants Private Ltd,None,India



SCHEMA INSPECTION COMPLETE

Do NOT run the old blocking-key or candidate-generation cells yet.

Send me this output. I will build the new blocking stage
directly against these verified columns.



In [ ]:
# ============================================================
# SAFE BLOCKING-KEY GENERATION — FINAL FIX
# ============================================================

import duckdb
import os
import json
import time

SOURCE_DB = "/content/drive/MyDrive/er_checkpoint/er_train_fresh.duckdb"
BLOCK_DB  = "/content/drive/MyDrive/er_checkpoint/er_blocking_train_fresh.duckdb"
TEMP_DIR  = "/content/er_blocking_tmp"
STATE_FILE = "/content/drive/MyDrive/er_checkpoint/er_blocking_state.json"

os.makedirs(TEMP_DIR, exist_ok=True)

print("=" * 80)
print("SAFE BLOCKING-KEY GENERATION — FINAL FIX")
print("=" * 80)
print()
print("Source DB:")
print(SOURCE_DB)
print()
print("Blocking DB:")
print(BLOCK_DB)
print()
print("Local temp:")
print(TEMP_DIR)
print()

# ============================================================
# OPEN BLOCKING DATABASE
# ============================================================

block = duckdb.connect(BLOCK_DB)

block.execute(
    f"SET temp_directory='{TEMP_DIR}'"
)

block.execute(
    "SET preserve_insertion_order=false"
)

block.execute(
    "SET threads=2"
)

print("✓ Blocking database opened")

# ============================================================
# ATTACH SOURCE DATABASE
# ============================================================

# The previous error happened because DuckDB's Python API
# does not support:
#
#     execute(..., ignore_errors=True)
#
# So we use normal Python try/except.

try:
    block.execute("DETACH DATABASE source_db")
except Exception:
    pass

block.execute(
    f"ATTACH '{SOURCE_DB}' AS source_db (READ_ONLY)"
)

print("✓ Source database attached read-only")
print()

# ============================================================
# VERIFY SOURCE TABLES
# ============================================================

print("=" * 80)
print("VERIFYING SOURCE TABLES")
print("=" * 80)

source_tables = [
    "train_s1",
    "train_s2",
    "train_s3"
]

for table in source_tables:

    count = block.execute(
        f"SELECT COUNT(*) FROM source_db.{table}"
    ).fetchone()[0]

    print(f"✓ {table}: {count:,} rows")

print()

# ============================================================
# LOAD STATE
# ============================================================

if os.path.exists(STATE_FILE):

    try:
        with open(STATE_FILE, "r") as f:
            state = json.load(f)
    except Exception:
        state = {}

else:
    state = {}

if "completed_tables" not in state:
    state["completed_tables"] = []

# ============================================================
# BUILD ONE BLOCKING TABLE
# ============================================================

def build_blocking_table(source_table, output_table):

    print("=" * 80)
    print(f"BUILDING: {output_table}")
    print("=" * 80)

    # --------------------------------------------------------
    # Check whether already completed
    # --------------------------------------------------------

    if output_table in state["completed_tables"]:

        exists = block.execute("""
            SELECT COUNT(*)
            FROM information_schema.tables
            WHERE table_name = ?
        """, [output_table]).fetchone()[0]

        if exists:

            count = block.execute(
                f'SELECT COUNT(*) FROM "{output_table}"'
            ).fetchone()[0]

            print(
                f"✓ Already completed: "
                f"{output_table} ({count:,} rows)"
            )

            return

    # --------------------------------------------------------
    # Remove incomplete table if present
    # --------------------------------------------------------

    block.execute(
        f'DROP TABLE IF EXISTS "{output_table}"'
    )

    start = time.time()

    # ========================================================
    # CREATE BLOCKING TABLE
    # ========================================================

    sql = f"""
    CREATE TABLE "{output_table}" AS

    WITH normalized AS (

        SELECT

            entity_id,
            business_name,
            business_address,
            country,

            regexp_replace(
                lower(coalesce(business_name, '')),
                '[^a-z0-9]',
                '',
                'g'
            ) AS name_norm,

            regexp_replace(
                lower(coalesce(business_address, '')),
                '[^a-z0-9]',
                '',
                'g'
            ) AS address_norm,

            regexp_replace(
                lower(coalesce(country, '')),
                '[^a-z0-9]',
                '',
                'g'
            ) AS country_norm

        FROM source_db."{source_table}"
    ),

    cores AS (

        SELECT

            *,

            regexp_replace(
                name_norm,
                '(incorporated|corporation|company|limited|llc|inc|ltd|corp)$',
                '',
                'g'
            ) AS name_core

        FROM normalized
    ),

    keys AS (

        SELECT

            *,

            substr(name_norm, 1, 4)
                AS name_prefix4,

            substr(name_norm, 1, 6)
                AS name_prefix6,

            substr(address_norm, 1, 6)
                AS address_prefix6,

            substr(address_norm, 1, 10)
                AS address_prefix10

        FROM cores
    )

    SELECT

        entity_id,
        business_name,
        business_address,
        country,

        name_norm,
        address_norm,
        country_norm,

        name_core,

        name_prefix4,
        name_prefix6,

        address_prefix6,
        address_prefix10,

        name_prefix6
            || '|' ||
        country_norm
            AS name_country_key,

        address_prefix10
            || '|' ||
        country_norm
            AS address_country_key,

        name_core
            || '|' ||
        country_norm
            AS name_core_country_key,

        name_prefix6
            || '|' ||
        address_prefix10
            AS name_address_key

    FROM keys;
    """

    block.execute(sql)

    elapsed = time.time() - start

    count = block.execute(
        f'SELECT COUNT(*) FROM "{output_table}"'
    ).fetchone()[0]

    print()
    print(f"✓ Created {output_table}")
    print(f"✓ Rows: {count:,}")
    print(f"✓ Time: {elapsed:.2f} seconds")

    # ========================================================
    # CREATE INDEXES
    # ========================================================

    print()
    print("Creating indexes...")

    index_columns = [
        "name_norm",
        "address_norm",
        "country_norm",
        "name_core",
        "name_prefix4",
        "name_prefix6",
        "address_prefix6",
        "address_prefix10",
        "name_country_key",
        "address_country_key",
        "name_core_country_key",
        "name_address_key"
    ]

    for col in index_columns:

        index_name = (
            f"idx_{output_table}_{col}"
            .replace("-", "_")
        )

        block.execute(f"""
            CREATE INDEX IF NOT EXISTS "{index_name}"
            ON "{output_table}" ("{col}");
        """)

    print("✓ Indexes created")

    # ========================================================
    # CHECKPOINT
    # ========================================================

    print()
    print("Checkpointing...")

    block.execute("CHECKPOINT")

    print("✓ Database checkpoint completed")

    # ========================================================
    # SAVE STATE
    # ========================================================

    if output_table not in state["completed_tables"]:
        state["completed_tables"].append(output_table)

    state["last_completed"] = output_table
    state["updated_at"] = time.strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    with open(STATE_FILE, "w") as f:
        json.dump(state, f, indent=2)

    print("✓ State saved")
    print()


# ============================================================
# BUILD ALL THREE TABLES
# ============================================================

train_tables = [
    ("train_s1", "train_s1_blocking"),
    ("train_s2", "train_s2_blocking"),
    ("train_s3", "train_s3_blocking")
]

for source_table, output_table in train_tables:

    build_blocking_table(
        source_table,
        output_table
    )


# ============================================================
# FINAL VERIFICATION
# ============================================================

print("=" * 80)
print("FINAL BLOCKING DATABASE VERIFICATION")
print("=" * 80)

for source_table, output_table in train_tables:

    count = block.execute(
        f'SELECT COUNT(*) FROM "{output_table}"'
    ).fetchone()[0]

    print(
        f"✓ {output_table}: {count:,} rows"
    )

print()
print("=" * 80)
print("✓ SAFE BLOCKING-KEY GENERATION COMPLETED")
print("=" * 80)
print()
print("Blocking DB:")
print(BLOCK_DB)

SAFE BLOCKING-KEY GENERATION — FINAL FIX

Source DB:
/content/drive/MyDrive/er_checkpoint/er_train_fresh.duckdb

Blocking DB:
/content/drive/MyDrive/er_checkpoint/er_blocking_train_fresh.duckdb

Local temp:
/content/er_blocking_tmp

✓ Blocking database opened
✓ Source database attached read-only

VERIFYING SOURCE TABLES
✓ train_s1: 2,206,821 rows
✓ train_s2: 5,034,616 rows
✓ train_s3: 5,285,603 rows

BUILDING: train_s1_blocking


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


✓ Created train_s1_blocking
✓ Rows: 2,206,821
✓ Time: 36.64 seconds

Creating indexes...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ Indexes created

Checkpointing...
✓ Database checkpoint completed
✓ State saved

FINAL BLOCKING DATABASE VERIFICATION
✓ train_s1_blocking: 2,206,821 rows
✓ train_s2_blocking: 5,034,616 rows
✓ train_s3_blocking: 5,285,603 rows

✓ SAFE BLOCKING-KEY GENERATION COMPLETED

Blocking DB:
/content/drive/MyDrive/er_checkpoint/er_blocking_train_fresh.duckdb


In [ ]:
# ============================================================
# BLOCKING SAFETY CHECK
# ============================================================

print("=" * 80)
print("BLOCKING KEY SAFETY CHECK")
print("=" * 80)

# Use the existing blocking connection
# If the connection was closed, uncomment the next lines:
#
# block = duckdb.connect(BLOCK_DB, read_only=True)
# block.execute(f"SET temp_directory='{TEMP_DIR}'")

tables = [
    "train_s1_blocking",
    "train_s2_blocking",
    "train_s3_blocking"
]

keys = [
    "name_country_key",
    "address_country_key",
    "name_core_country_key",
    "name_address_key"
]

for table in tables:

    print()
    print("=" * 80)
    print(f"TABLE: {table}")
    print("=" * 80)

    total = block.execute(
        f'SELECT COUNT(*) FROM "{table}"'
    ).fetchone()[0]

    print(f"Total rows: {total:,}")

    for key in keys:

        print()
        print(f"--- {key} ---")

        result = block.execute(f"""
            SELECT
                COUNT(*) AS distinct_keys,
                MAX(cnt) AS largest_block,
                AVG(cnt) AS average_block
            FROM (
                SELECT
                    "{key}",
                    COUNT(*) AS cnt
                FROM "{table}"
                WHERE "{key}" IS NOT NULL
                  AND "{key}" != ''
                  AND "{key}" NOT LIKE '|%'
                  AND "{key}" NOT LIKE '%|'
                GROUP BY "{key}"
            )
        """).fetchone()

        distinct_keys, largest_block, average_block = result

        print(f"Distinct keys : {distinct_keys:,}")
        print(f"Largest block : {largest_block:,}")
        print(f"Average block : {average_block:.2f}")

        # Show the 5 largest blocks
        print("Top 5 blocks:")

        top = block.execute(f"""
            SELECT
                "{key}" AS block_key,
                COUNT(*) AS block_size
            FROM "{table}"
            WHERE "{key}" IS NOT NULL
              AND "{key}" != ''
              AND "{key}" NOT LIKE '|%'
              AND "{key}" NOT LIKE '%|'
            GROUP BY "{key}"
            ORDER BY block_size DESC
            LIMIT 5
        """).fetchall()

        for block_key, block_size in top:
            print(
                f"  {block_size:>12,}  |  {str(block_key)[:100]}"
            )

print()
print("=" * 80)
print("✓ BLOCKING SAFETY CHECK COMPLETED")
print("=" * 80)

BLOCKING KEY SAFETY CHECK

TABLE: train_s1_blocking
Total rows: 2,206,821

--- name_country_key ---
Distinct keys : 340,184
Largest block : 15,070
Average block : 6.49
Top 5 blocks:
        15,070  |  pediat|us
         6,653  |  family|us
         5,805  |  bright|us
         5,436  |  intern|us
         5,159  |  dermat|us

--- address_country_key ---
Distinct keys : 1,709,978
Largest block : 8,100
Average block : 1.29
Top 5 blocks:
         8,100  |  maharashtr|india
         3,154  |  uttarprade|india
         2,505  |  groundfloo|india
         2,378  |  westbengal|india
         2,255  |  buildingno|india

--- name_core_country_key ---
Distinct keys : 1,417,018
Largest block : 531
Average block : 1.56
Top 5 blocks:
           531  |  meridian|us
           325  |  cedar|us
           314  |  summit|us
           313  |  helios|us
           312  |  cascade|us

--- name_address_key ---
Distinct keys : 2,186,545
Largest block : 196
Average block : 1.01
Top 5 blocks:
           196 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Distinct keys : 3,090,519
Largest block : 26,367
Average block : 1.57
Top 5 blocks:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

        26,367  |  maharashtr|india
        10,054  |  uttarprade|india
         7,701  |  westbengal|india
         4,391  |  andhraprad|india
         4,220  |  buildingno|india

--- name_core_country_key ---


...

        13,232  |  northcarol|us
         7,558  |  massachuse|us
         7,520  |  washington|us
         5,265  |  california|us
         4,196  |  buildingno|india

--- name_core_country_key ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Distinct keys : 3,784,570
Largest block : 2,957
Average block : 1.33
Top 5 blocks:
         2,957  |  private|india
           694  |  pvt|india
           616  |  pediatricdental|us
           610  |  primarycare|us
           608  |  physicaltherapy|us

--- name_address_key ---
Distinct keys : 4,234,700
Largest block : 193
Average block : 1.15
Top 5 blocks:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

           193  |  mumbai|mumbaicity
           134  |  pediat|northcarol
           119  |  bangal|bangalorek
           119  |  bangal|bangaloren
           110  |  bangal|kabangalor

✓ BLOCKING SAFETY CHECK COMPLETED


In [ ]:
# ============================================================
# CANDIDATE ROUTE SIZE ESTIMATION
# ============================================================

print("=" * 80)
print("CANDIDATE ROUTE SIZE ESTIMATION")
print("=" * 80)

# ------------------------------------------------------------
# Candidate pairs:
#
# train_s1 -> train_s2
# train_s1 -> train_s3
#
# We estimate several exact blocking routes.
# ------------------------------------------------------------

routes = [

    # Very strong exact match route
    (
        "FULL NAME + COUNTRY",
        "a.name_norm = b.name_norm "
        "AND a.country_norm = b.country_norm "
        "AND a.name_norm != ''"
    ),

    # Very strong exact address route
    (
        "FULL ADDRESS + COUNTRY",
        "a.address_norm = b.address_norm "
        "AND a.country_norm = b.country_norm "
        "AND a.address_norm != ''"
    ),

    # Exact name-core + country
    (
        "NAME CORE + COUNTRY",
        "a.name_core = b.name_core "
        "AND a.country_norm = b.country_norm "
        "AND a.name_core != ''"
    ),

    # Name + address blocking key
    (
        "NAME PREFIX + ADDRESS PREFIX",
        "a.name_address_key = b.name_address_key "
        "AND a.name_address_key NOT LIKE '|%' "
        "AND a.name_address_key NOT LIKE '%|'"
    )
]

pairs = [
    ("train_s1_blocking", "train_s2_blocking", "S1-S2"),
    ("train_s1_blocking", "train_s3_blocking", "S1-S3")
]

for left, right, pair_name in pairs:

    print()
    print("=" * 80)
    print(f"PAIR: {pair_name}")
    print("=" * 80)

    for route_name, condition in routes:

        start = time.time()

        query = f"""
        SELECT COUNT(*)
        FROM "{left}" a
        INNER JOIN "{right}" b
            ON {condition}
        """

        count = block.execute(query).fetchone()[0]

        elapsed = time.time() - start

        print(
            f"{route_name:<32} "
            f"{count:>15,} pairs   "
            f"({elapsed:.2f}s)"
        )

print()
print("=" * 80)
print("✓ ROUTE SIZE ESTIMATION COMPLETED")
print("=" * 80)

CANDIDATE ROUTE SIZE ESTIMATION

PAIR: S1-S2


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FULL NAME + COUNTRY                   10,849,350 pairs   (4.35s)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FULL ADDRESS + COUNTRY                   570,598 pairs   (2.57s)


...

NAME CORE + COUNTRY                   29,280,342 pairs   (4.30s)
NAME PREFIX + ADDRESS PREFIX           1,536,525 pairs   (1.10s)

PAIR: S1-S3


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FULL NAME + COUNTRY                   11,892,958 pairs   (2.73s)
FULL ADDRESS + COUNTRY                   206,735 pairs   (1.48s)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

NAME CORE + COUNTRY                   30,547,290 pairs   (6.51s)
NAME PREFIX + ADDRESS PREFIX           1,245,167 pairs   (1.22s)

✓ ROUTE SIZE ESTIMATION COMPLETED


In [ ]:
# ============================================================
# SAFE CANDIDATE GENERATION — STEP 1
# FULL NAME + COUNTRY
# S1 -> S2
# ============================================================

import duckdb
import os
import time
import json

BLOCK_DB = "/content/drive/MyDrive/er_checkpoint/er_blocking_train_fresh.duckdb"
CANDIDATE_DB = "/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb"
TEMP_DIR = "/content/er_candidates_tmp"
STATE_FILE = "/content/drive/MyDrive/er_checkpoint/er_candidates_state.json"

os.makedirs(TEMP_DIR, exist_ok=True)

print("=" * 80)
print("SAFE CANDIDATE GENERATION — STEP 1")
print("=" * 80)
print()
print("Blocking DB:")
print(BLOCK_DB)
print()
print("Candidate DB:")
print(CANDIDATE_DB)
print()
print("Local temp:")
print(TEMP_DIR)
print()

# ============================================================
# OPEN CANDIDATE DATABASE
# ============================================================

cand = duckdb.connect(CANDIDATE_DB)

cand.execute(
    f"SET temp_directory='{TEMP_DIR}'"
)

cand.execute(
    "SET preserve_insertion_order=false"
)

cand.execute(
    "SET threads=2"
)

print("✓ Candidate database opened")

# ============================================================
# ATTACH BLOCKING DATABASE
# ============================================================

try:
    cand.execute("DETACH DATABASE blocking_db")
except Exception:
    pass

cand.execute(
    f"ATTACH '{BLOCK_DB}' AS blocking_db (READ_ONLY)"
)

print("✓ Blocking database attached read-only")
print()

# ============================================================
# VERIFY TABLES
# ============================================================

print("=" * 80)
print("VERIFYING BLOCKING TABLES")
print("=" * 80)

for table in [
    "train_s1_blocking",
    "train_s2_blocking",
    "train_s3_blocking"
]:

    count = cand.execute(
        f"SELECT COUNT(*) FROM blocking_db.{table}"
    ).fetchone()[0]

    print(f"✓ {table}: {count:,} rows")

print()

# ============================================================
# LOAD STATE
# ============================================================

if os.path.exists(STATE_FILE):

    try:
        with open(STATE_FILE, "r") as f:
            state = json.load(f)
    except Exception:
        state = {}

else:
    state = {}

if "completed_routes" not in state:
    state["completed_routes"] = []

# ============================================================
# CREATE CANDIDATE TABLE
# ============================================================

cand.execute("""
    CREATE TABLE IF NOT EXISTS candidates_s1_s2 (
        source1_entity_id VARCHAR,
        candidate_entity_id VARCHAR,
        route VARCHAR
    )
""")

print("✓ Candidate table ready")

# ============================================================
# CHECK IF ROUTE ALREADY COMPLETED
# ============================================================

ROUTE_NAME = "s1_s2_full_name_country"

if ROUTE_NAME in state["completed_routes"]:

    existing = cand.execute("""
        SELECT COUNT(*)
        FROM candidates_s1_s2
        WHERE route = ?
    """, [ROUTE_NAME]).fetchone()[0]

    print()
    print("✓ Route already completed")
    print(f"Existing candidates: {existing:,}")

else:

    # ========================================================
    # ESTIMATE FIRST
    # ========================================================

    print()
    print("=" * 80)
    print("VERIFYING EXPECTED CANDIDATE COUNT")
    print("=" * 80)

    expected = cand.execute("""
        SELECT COUNT(*)
        FROM blocking_db.train_s1_blocking a
        INNER JOIN blocking_db.train_s2_blocking b
            ON a.name_norm = b.name_norm
           AND a.country_norm = b.country_norm
        WHERE a.name_norm != ''
          AND b.name_norm != ''
    """).fetchone()[0]

    print(f"Expected candidates: {expected:,}")

    # ========================================================
    # GENERATE CANDIDATES
    # ========================================================

    print()
    print("=" * 80)
    print("GENERATING S1 -> S2 CANDIDATES")
    print("=" * 80)

    start = time.time()

    cand.execute("""
        INSERT INTO candidates_s1_s2
        SELECT
            a.entity_id AS source1_entity_id,
            b.entity_id AS candidate_entity_id,
            's1_s2_full_name_country' AS route

        FROM blocking_db.train_s1_blocking a

        INNER JOIN blocking_db.train_s2_blocking b
            ON a.name_norm = b.name_norm
           AND a.country_norm = b.country_norm

        WHERE a.name_norm != ''
          AND b.name_norm != ''
    """)

    elapsed = time.time() - start

    actual = cand.execute("""
        SELECT COUNT(*)
        FROM candidates_s1_s2
        WHERE route = 's1_s2_full_name_country'
    """).fetchone()[0]

    print()
    print(f"✓ Generated: {actual:,} candidates")
    print(f"✓ Time: {elapsed:.2f} seconds")

    # ========================================================
    # CHECKPOINT
    # ========================================================

    print()
    print("Checkpointing candidate database...")

    cand.execute("CHECKPOINT")

    print("✓ Candidate database checkpointed")

    # ========================================================
    # SAVE STATE
    # ========================================================

    state["completed_routes"].append(ROUTE_NAME)
    state["last_completed_route"] = ROUTE_NAME
    state["updated_at"] = time.strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    with open(STATE_FILE, "w") as f:
        json.dump(state, f, indent=2)

    print("✓ Candidate state saved")

# ============================================================
# FINAL CHECK
# ============================================================

print()
print("=" * 80)
print("STEP 1 COMPLETE")
print("=" * 80)

total = cand.execute("""
    SELECT COUNT(*)
    FROM candidates_s1_s2
""").fetchone()[0]

print(f"Total S1-S2 candidates: {total:,}")

print()
print("Candidate DB:")
print(CANDIDATE_DB)

SAFE CANDIDATE GENERATION — STEP 1

Blocking DB:
/content/drive/MyDrive/er_checkpoint/er_blocking_train_fresh.duckdb

Candidate DB:
/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb

Local temp:
/content/er_candidates_tmp

✓ Candidate database opened


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ Blocking database attached read-only

VERIFYING BLOCKING TABLES
✓ train_s1_blocking: 2,206,821 rows
✓ train_s2_blocking: 5,034,616 rows
✓ train_s3_blocking: 5,285,603 rows

✓ Candidate table ready

VERIFYING EXPECTED CANDIDATE COUNT


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Expected candidates: 10,849,350

GENERATING S1 -> S2 CANDIDATES


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


✓ Generated: 10,849,350 candidates
✓ Time: 372.16 seconds

Checkpointing candidate database...
✓ Candidate database checkpointed
✓ Candidate state saved

STEP 1 COMPLETE
Total S1-S2 candidates: 10,849,350

Candidate DB:
/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb


In [ ]:
# ============================================================
# SAFE CANDIDATE GENERATION — STEP 2
# FULL NAME + COUNTRY
# S1 -> S3
# ============================================================

ROUTE_NAME = "s1_s3_full_name_country"

print("=" * 80)
print("SAFE CANDIDATE GENERATION — STEP 2")
print("=" * 80)
print()
print("Route:")
print("S1 -> S3")
print("FULL NAME + COUNTRY")
print()

# ============================================================
# LOAD STATE
# ============================================================

if os.path.exists(STATE_FILE):

    try:
        with open(STATE_FILE, "r") as f:
            state = json.load(f)
    except Exception:
        state = {}

else:
    state = {}

if "completed_routes" not in state:
    state["completed_routes"] = []

# ============================================================
# CREATE S1-S3 TABLE
# ============================================================

cand.execute("""
    CREATE TABLE IF NOT EXISTS candidates_s1_s3 (
        source1_entity_id VARCHAR,
        candidate_entity_id VARCHAR,
        route VARCHAR
    )
""")

print("✓ S1-S3 candidate table ready")

# ============================================================
# CHECKPOINT/RESUME
# ============================================================

if ROUTE_NAME in state["completed_routes"]:

    existing = cand.execute("""
        SELECT COUNT(*)
        FROM candidates_s1_s3
        WHERE route = ?
    """, [ROUTE_NAME]).fetchone()[0]

    print()
    print("✓ Route already completed")
    print(f"Existing candidates: {existing:,}")

else:

    # --------------------------------------------------------
    # EXPECTED COUNT
    # --------------------------------------------------------

    print()
    print("=" * 80)
    print("VERIFYING EXPECTED CANDIDATE COUNT")
    print("=" * 80)

    expected = cand.execute("""
        SELECT COUNT(*)
        FROM blocking_db.train_s1_blocking a
        INNER JOIN blocking_db.train_s3_blocking b
            ON a.name_norm = b.name_norm
           AND a.country_norm = b.country_norm
        WHERE a.name_norm != ''
          AND b.name_norm != ''
    """).fetchone()[0]

    print(f"Expected candidates: {expected:,}")

    # --------------------------------------------------------
    # GENERATE
    # --------------------------------------------------------

    print()
    print("=" * 80)
    print("GENERATING S1 -> S3 CANDIDATES")
    print("=" * 80)

    start = time.time()

    cand.execute("""
        INSERT INTO candidates_s1_s3

        SELECT
            a.entity_id AS source1_entity_id,
            b.entity_id AS candidate_entity_id,
            's1_s3_full_name_country' AS route

        FROM blocking_db.train_s1_blocking a

        INNER JOIN blocking_db.train_s3_blocking b
            ON a.name_norm = b.name_norm
           AND a.country_norm = b.country_norm

        WHERE a.name_norm != ''
          AND b.name_norm != ''
    """)

    elapsed = time.time() - start

    actual = cand.execute("""
        SELECT COUNT(*)
        FROM candidates_s1_s3
        WHERE route = ?
    """, [ROUTE_NAME]).fetchone()[0]

    print()
    print(f"✓ Generated: {actual:,} candidates")
    print(f"✓ Time: {elapsed:.2f} seconds")

    # --------------------------------------------------------
    # CHECKPOINT
    # --------------------------------------------------------

    print()
    print("Checkpointing candidate database...")

    cand.execute("CHECKPOINT")

    print("✓ Candidate database checkpointed")

    # --------------------------------------------------------
    # SAVE STATE
    # --------------------------------------------------------

    if ROUTE_NAME not in state["completed_routes"]:
        state["completed_routes"].append(ROUTE_NAME)

    state["last_completed_route"] = ROUTE_NAME
    state["updated_at"] = time.strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    with open(STATE_FILE, "w") as f:
        json.dump(state, f, indent=2)

    print("✓ Candidate state saved")

# ============================================================
# FINAL CHECK
# ============================================================

print()
print("=" * 80)
print("STEP 2 COMPLETE")
print("=" * 80)

total = cand.execute("""
    SELECT COUNT(*)
    FROM candidates_s1_s3
""").fetchone()[0]

print(f"Total S1-S3 candidates: {total:,}")

print()
print("Candidate DB:")
print(CANDIDATE_DB)

SAFE CANDIDATE GENERATION — STEP 2

Route:
S1 -> S3
FULL NAME + COUNTRY

✓ S1-S3 candidate table ready

VERIFYING EXPECTED CANDIDATE COUNT


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Expected candidates: 11,892,958

GENERATING S1 -> S3 CANDIDATES


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


✓ Generated: 11,892,958 candidates
✓ Time: 184.59 seconds

Checkpointing candidate database...
✓ Candidate database checkpointed
✓ Candidate state saved

STEP 2 COMPLETE
Total S1-S3 candidates: 11,892,958

Candidate DB:
/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb


In [ ]:
# ============================================================
# SAFE CANDIDATE GENERATION — STEP 2
# FULL NAME + COUNTRY
# S1 -> S3
# ============================================================

import duckdb
import os
import json
import time

# ============================================================
# PATHS
# ============================================================

BLOCK_DB = "/content/drive/MyDrive/er_checkpoint/er_blocking_train_fresh.duckdb"
CANDIDATE_DB = "/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb"
TEMP_DIR = "/content/er_candidates_tmp"
STATE_FILE = "/content/drive/MyDrive/er_checkpoint/er_candidates_state.json"

os.makedirs(TEMP_DIR, exist_ok=True)

print("=" * 80)
print("SAFE CANDIDATE GENERATION — STEP 2")
print("=" * 80)
print()
print("Route:")
print("S1 -> S3")
print("FULL NAME + COUNTRY")
print()
print("Candidate DB:")
print(CANDIDATE_DB)
print()

# ============================================================
# OPEN CANDIDATE DATABASE
# ============================================================

cand = duckdb.connect(CANDIDATE_DB)

cand.execute(
    f"SET temp_directory='{TEMP_DIR}'"
)

cand.execute(
    "SET preserve_insertion_order=false"
)

cand.execute(
    "SET threads=2"
)

print("✓ Candidate database opened")

# ============================================================
# ATTACH BLOCKING DATABASE
# ============================================================

try:
    cand.execute("DETACH DATABASE blocking_db")
except Exception:
    pass

cand.execute(
    f"ATTACH '{BLOCK_DB}' AS blocking_db (READ_ONLY)"
)

print("✓ Blocking database attached read-only")
print()

# ============================================================
# VERIFY BLOCKING TABLES
# ============================================================

print("=" * 80)
print("VERIFYING BLOCKING TABLES")
print("=" * 80)

for table in [
    "train_s1_blocking",
    "train_s3_blocking"
]:

    count = cand.execute(
        f"SELECT COUNT(*) FROM blocking_db.{table}"
    ).fetchone()[0]

    print(f"✓ {table}: {count:,} rows")

print()

# ============================================================
# LOAD STATE
# ============================================================

if os.path.exists(STATE_FILE):

    try:
        with open(STATE_FILE, "r") as f:
            state = json.load(f)
    except Exception:
        state = {}

else:
    state = {}

if "completed_routes" not in state:
    state["completed_routes"] = []

ROUTE_NAME = "s1_s3_full_name_country"

# ============================================================
# CREATE S1-S3 CANDIDATE TABLE
# ============================================================

cand.execute("""
    CREATE TABLE IF NOT EXISTS candidates_s1_s3 (
        source1_entity_id VARCHAR,
        candidate_entity_id VARCHAR,
        route VARCHAR
    )
""")

print("✓ S1-S3 candidate table ready")

# ============================================================
# CHECK WHETHER ALREADY COMPLETED
# ============================================================

if ROUTE_NAME in state["completed_routes"]:

    existing = cand.execute("""
        SELECT COUNT(*)
        FROM candidates_s1_s3
        WHERE route = ?
    """, [ROUTE_NAME]).fetchone()[0]

    print()
    print("=" * 80)
    print("✓ STEP 2 WAS ALREADY COMPLETED")
    print("=" * 80)
    print()
    print(f"Existing candidates: {existing:,}")

else:

    # ========================================================
    # EXPECTED CANDIDATE COUNT
    # ========================================================

    print()
    print("=" * 80)
    print("VERIFYING EXPECTED CANDIDATE COUNT")
    print("=" * 80)

    start = time.time()

    expected = cand.execute("""
        SELECT COUNT(*)

        FROM blocking_db.train_s1_blocking a

        INNER JOIN blocking_db.train_s3_blocking b

            ON a.name_norm = b.name_norm
           AND a.country_norm = b.country_norm

        WHERE a.name_norm != ''
          AND b.name_norm != ''
    """).fetchone()[0]

    elapsed = time.time() - start

    print(f"Expected candidates: {expected:,}")
    print(f"Count check time: {elapsed:.2f} seconds")

    # ========================================================
    # GENERATE CANDIDATES
    # ========================================================

    print()
    print("=" * 80)
    print("GENERATING S1 -> S3 CANDIDATES")
    print("=" * 80)

    start = time.time()

    cand.execute("""
        INSERT INTO candidates_s1_s3

        SELECT

            a.entity_id AS source1_entity_id,

            b.entity_id AS candidate_entity_id,

            's1_s3_full_name_country' AS route

        FROM blocking_db.train_s1_blocking a

        INNER JOIN blocking_db.train_s3_blocking b

            ON a.name_norm = b.name_norm
           AND a.country_norm = b.country_norm

        WHERE a.name_norm != ''
          AND b.name_norm != ''
    """)

    elapsed = time.time() - start

    # ========================================================
    # VERIFY ACTUAL COUNT
    # ========================================================

    actual = cand.execute("""
        SELECT COUNT(*)

        FROM candidates_s1_s3

        WHERE route = ?
    """, [ROUTE_NAME]).fetchone()[0]

    print()
    print(f"✓ Generated: {actual:,} candidates")
    print(f"✓ Generation time: {elapsed:.2f} seconds")

    # ========================================================
    # COUNT VALIDATION
    # ========================================================

    if actual != expected:

        print()
        print("⚠ WARNING")
        print(f"Expected: {expected:,}")
        print(f"Actual:   {actual:,}")
        print("Counts do not match.")

    else:

        print("✓ Candidate count matches expected count")

    # ========================================================
    # CHECKPOINT
    # ========================================================

    print()
    print("Checkpointing candidate database...")

    cand.execute("CHECKPOINT")

    print("✓ Candidate database checkpointed")

    # ========================================================
    # SAVE STATE
    # ========================================================

    if ROUTE_NAME not in state["completed_routes"]:
        state["completed_routes"].append(ROUTE_NAME)

    state["last_completed_route"] = ROUTE_NAME
    state["updated_at"] = time.strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    with open(STATE_FILE, "w") as f:
        json.dump(state, f, indent=2)

    print("✓ Candidate state saved")

# ============================================================
# FINAL VERIFICATION
# ============================================================

print()
print("=" * 80)
print("STEP 2 COMPLETE")
print("=" * 80)

total_s1_s3 = cand.execute("""
    SELECT COUNT(*)
    FROM candidates_s1_s3
""").fetchone()[0]

print()
print(f"Total S1-S3 candidates: {total_s1_s3:,}")

# ============================================================
# SHOW BOTH ROUTES
# ============================================================

print()
print("=" * 80)
print("CANDIDATE DATABASE SUMMARY")
print("=" * 80)

s1_s2 = cand.execute("""
    SELECT COUNT(*)
    FROM candidates_s1_s2
""").fetchone()[0]

print(f"S1 -> S2 candidates: {s1_s2:,}")
print(f"S1 -> S3 candidates: {total_s1_s3:,}")
print(
    f"Total candidates:    "
    f"{s1_s2 + total_s1_s3:,}"
)

print()
print("=" * 80)
print("✓ STEP 2 SAFELY CHECKPOINTED")
print("=" * 80)

print()
print("Candidate DB:")
print(CANDIDATE_DB)

SAFE CANDIDATE GENERATION — STEP 2

Route:
S1 -> S3
FULL NAME + COUNTRY

Candidate DB:
/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb

✓ Candidate database opened


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ Blocking database attached read-only

VERIFYING BLOCKING TABLES
✓ train_s1_blocking: 2,206,821 rows
✓ train_s3_blocking: 5,285,603 rows

✓ S1-S3 candidate table ready

✓ STEP 2 WAS ALREADY COMPLETED

Existing candidates: 11,892,958

STEP 2 COMPLETE

Total S1-S3 candidates: 11,892,958

CANDIDATE DATABASE SUMMARY
S1 -> S2 candidates: 10,849,350
S1 -> S3 candidates: 11,892,958
Total candidates:    22,742,308

✓ STEP 2 SAFELY CHECKPOINTED

Candidate DB:
/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb


In [ ]:
# ============================================================
# SAFE CANDIDATE GENERATION — STEP 3
# FULL ADDRESS + COUNTRY
# S1 -> S2
# ============================================================

import os
import json
import time
import duckdb

# ============================================================
# PATHS
# ============================================================

BLOCK_DB = "/content/drive/MyDrive/er_checkpoint/er_blocking_train_fresh.duckdb"
CANDIDATE_DB = "/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb"
TEMP_DIR = "/content/er_candidates_tmp"
STATE_FILE = "/content/drive/MyDrive/er_checkpoint/er_candidates_state.json"

os.makedirs(TEMP_DIR, exist_ok=True)

ROUTE_NAME = "s1_s2_full_address_country"

print("=" * 80)
print("SAFE CANDIDATE GENERATION — STEP 3")
print("=" * 80)
print()
print("Route:")
print("S1 -> S2")
print("FULL ADDRESS + COUNTRY")
print()

# ============================================================
# OPEN CANDIDATE DATABASE
# ============================================================

cand = duckdb.connect(CANDIDATE_DB)

cand.execute(
    f"SET temp_directory='{TEMP_DIR}'"
)

cand.execute(
    "SET preserve_insertion_order=false"
)

cand.execute(
    "SET threads=2"
)

print("✓ Candidate database opened")

# ============================================================
# ATTACH BLOCKING DATABASE
# ============================================================

try:
    cand.execute("DETACH DATABASE blocking_db")
except Exception:
    pass

cand.execute(
    f"ATTACH '{BLOCK_DB}' AS blocking_db (READ_ONLY)"
)

print("✓ Blocking database attached read-only")
print()

# ============================================================
# LOAD STATE
# ============================================================

if os.path.exists(STATE_FILE):
    try:
        with open(STATE_FILE, "r") as f:
            state = json.load(f)
    except Exception:
        state = {}
else:
    state = {}

if "completed_routes" not in state:
    state["completed_routes"] = []

# ============================================================
# CREATE TABLE
# ============================================================

cand.execute("""
    CREATE TABLE IF NOT EXISTS candidates_s1_s2_address (
        source1_entity_id VARCHAR,
        candidate_entity_id VARCHAR,
        route VARCHAR
    )
""")

print("✓ S1-S2 address candidate table ready")

# ============================================================
# CHECK IF ALREADY COMPLETED
# ============================================================

if ROUTE_NAME in state["completed_routes"]:

    existing = cand.execute("""
        SELECT COUNT(*)
        FROM candidates_s1_s2_address
        WHERE route = ?
    """, [ROUTE_NAME]).fetchone()[0]

    print()
    print("=" * 80)
    print("✓ ROUTE ALREADY COMPLETED")
    print("=" * 80)
    print()
    print(f"Existing candidates: {existing:,}")

else:

    # ========================================================
    # VERIFY EXPECTED COUNT
    # ========================================================

    print()
    print("=" * 80)
    print("VERIFYING EXPECTED CANDIDATE COUNT")
    print("=" * 80)

    start = time.time()

    expected = cand.execute("""
        SELECT COUNT(*)

        FROM blocking_db.train_s1_blocking a

        INNER JOIN blocking_db.train_s2_blocking b

            ON a.address_norm = b.address_norm
           AND a.country_norm = b.country_norm

        WHERE a.address_norm != ''
          AND b.address_norm != ''
    """).fetchone()[0]

    elapsed = time.time() - start

    print(f"Expected candidates: {expected:,}")
    print(f"Count check time: {elapsed:.2f} seconds")

    # ========================================================
    # GENERATE CANDIDATES
    # ========================================================

    print()
    print("=" * 80)
    print("GENERATING S1 -> S2 ADDRESS CANDIDATES")
    print("=" * 80)

    start = time.time()

    cand.execute("""
        INSERT INTO candidates_s1_s2_address

        SELECT
            a.entity_id AS source1_entity_id,
            b.entity_id AS candidate_entity_id,
            's1_s2_full_address_country' AS route

        FROM blocking_db.train_s1_blocking a

        INNER JOIN blocking_db.train_s2_blocking b

            ON a.address_norm = b.address_norm
           AND a.country_norm = b.country_norm

        WHERE a.address_norm != ''
          AND b.address_norm != ''
    """)

    elapsed = time.time() - start

    # ========================================================
    # VERIFY ACTUAL COUNT
    # ========================================================

    actual = cand.execute("""
        SELECT COUNT(*)

        FROM candidates_s1_s2_address

        WHERE route = ?
    """, [ROUTE_NAME]).fetchone()[0]

    print()
    print(f"✓ Generated: {actual:,} candidates")
    print(f"✓ Generation time: {elapsed:.2f} seconds")

    # ========================================================
    # VALIDATE
    # ========================================================

    if actual != expected:
        print()
        print("⚠ WARNING")
        print(f"Expected: {expected:,}")
        print(f"Actual:   {actual:,}")
    else:
        print("✓ Candidate count matches expected count")

    # ========================================================
    # CHECKPOINT
    # ========================================================

    print()
    print("Checkpointing candidate database...")

    cand.execute("CHECKPOINT")

    print("✓ Candidate database checkpointed")

    # ========================================================
    # SAVE STATE
    # ========================================================

    if ROUTE_NAME not in state["completed_routes"]:
        state["completed_routes"].append(ROUTE_NAME)

    state["last_completed_route"] = ROUTE_NAME
    state["updated_at"] = time.strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    with open(STATE_FILE, "w") as f:
        json.dump(state, f, indent=2)

    print("✓ Candidate state saved")

# ============================================================
# FINAL SUMMARY
# ============================================================

print()
print("=" * 80)
print("STEP 3 COMPLETE")
print("=" * 80)

total = cand.execute("""
    SELECT COUNT(*)
    FROM candidates_s1_s2_address
""").fetchone()[0]

print()
print(f"S1 -> S2 address candidates: {total:,}")
print()
print("Candidate DB:")
print(CANDIDATE_DB)
print()
print("✓ Safe checkpoint completed")

SAFE CANDIDATE GENERATION — STEP 3

Route:
S1 -> S2
FULL ADDRESS + COUNTRY

✓ Candidate database opened


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ Blocking database attached read-only

✓ S1-S2 address candidate table ready

VERIFYING EXPECTED CANDIDATE COUNT


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Expected candidates: 570,598
Count check time: 2.76 seconds

GENERATING S1 -> S2 ADDRESS CANDIDATES


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


✓ Generated: 570,598 candidates
✓ Generation time: 58.69 seconds
✓ Candidate count matches expected count

Checkpointing candidate database...
✓ Candidate database checkpointed
✓ Candidate state saved

STEP 3 COMPLETE

S1 -> S2 address candidates: 570,598

Candidate DB:
/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb

✓ Safe checkpoint completed


In [ ]:
# ============================================================
# SAFE CANDIDATE GENERATION — STEP 4
# FULL ADDRESS + COUNTRY
# S1 -> S3
# ============================================================

import os
import json
import time
import duckdb

# ============================================================
# PATHS
# ============================================================

BLOCK_DB = "/content/drive/MyDrive/er_checkpoint/er_blocking_train_fresh.duckdb"
CANDIDATE_DB = "/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb"
TEMP_DIR = "/content/er_candidates_tmp"
STATE_FILE = "/content/drive/MyDrive/er_checkpoint/er_candidates_state.json"

os.makedirs(TEMP_DIR, exist_ok=True)

ROUTE_NAME = "s1_s3_full_address_country"

print("=" * 80)
print("SAFE CANDIDATE GENERATION — STEP 4")
print("=" * 80)
print()
print("Route:")
print("S1 -> S3")
print("FULL ADDRESS + COUNTRY")
print()

# ============================================================
# OPEN CANDIDATE DATABASE
# ============================================================

cand = duckdb.connect(CANDIDATE_DB)

cand.execute(
    f"SET temp_directory='{TEMP_DIR}'"
)

cand.execute(
    "SET preserve_insertion_order=false"
)

cand.execute(
    "SET threads=2"
)

print("✓ Candidate database opened")

# ============================================================
# ATTACH BLOCKING DATABASE
# ============================================================

try:
    cand.execute("DETACH DATABASE blocking_db")
except Exception:
    pass

cand.execute(
    f"ATTACH '{BLOCK_DB}' AS blocking_db (READ_ONLY)"
)

print("✓ Blocking database attached read-only")
print()

# ============================================================
# LOAD STATE
# ============================================================

if os.path.exists(STATE_FILE):
    try:
        with open(STATE_FILE, "r") as f:
            state = json.load(f)
    except Exception:
        state = {}
else:
    state = {}

if "completed_routes" not in state:
    state["completed_routes"] = []

# ============================================================
# CREATE TABLE
# ============================================================

cand.execute("""
    CREATE TABLE IF NOT EXISTS candidates_s1_s3_address (
        source1_entity_id VARCHAR,
        candidate_entity_id VARCHAR,
        route VARCHAR
    )
""")

print("✓ S1-S3 address candidate table ready")

# ============================================================
# CHECK IF ALREADY COMPLETED
# ============================================================

if ROUTE_NAME in state["completed_routes"]:

    existing = cand.execute("""
        SELECT COUNT(*)
        FROM candidates_s1_s3_address
        WHERE route = ?
    """, [ROUTE_NAME]).fetchone()[0]

    print()
    print("=" * 80)
    print("✓ ROUTE ALREADY COMPLETED")
    print("=" * 80)
    print()
    print(f"Existing candidates: {existing:,}")

else:

    # ========================================================
    # VERIFY EXPECTED COUNT
    # ========================================================

    print()
    print("=" * 80)
    print("VERIFYING EXPECTED CANDIDATE COUNT")
    print("=" * 80)

    start = time.time()

    expected = cand.execute("""
        SELECT COUNT(*)

        FROM blocking_db.train_s1_blocking a

        INNER JOIN blocking_db.train_s3_blocking b

            ON a.address_norm = b.address_norm
           AND a.country_norm = b.country_norm

        WHERE a.address_norm != ''
          AND b.address_norm != ''
    """).fetchone()[0]

    elapsed = time.time() - start

    print(f"Expected candidates: {expected:,}")
    print(f"Count check time: {elapsed:.2f} seconds")

    # ========================================================
    # GENERATE CANDIDATES
    # ========================================================

    print()
    print("=" * 80)
    print("GENERATING S1 -> S3 ADDRESS CANDIDATES")
    print("=" * 80)

    start = time.time()

    cand.execute("""
        INSERT INTO candidates_s1_s3_address

        SELECT
            a.entity_id AS source1_entity_id,
            b.entity_id AS candidate_entity_id,
            's1_s3_full_address_country' AS route

        FROM blocking_db.train_s1_blocking a

        INNER JOIN blocking_db.train_s3_blocking b

            ON a.address_norm = b.address_norm
           AND a.country_norm = b.country_norm

        WHERE a.address_norm != ''
          AND b.address_norm != ''
    """)

    elapsed = time.time() - start

    # ========================================================
    # VERIFY ACTUAL COUNT
    # ========================================================

    actual = cand.execute("""
        SELECT COUNT(*)

        FROM candidates_s1_s3_address

        WHERE route = ?
    """, [ROUTE_NAME]).fetchone()[0]

    print()
    print(f"✓ Generated: {actual:,} candidates")
    print(f"✓ Generation time: {elapsed:.2f} seconds")

    # ========================================================
    # VALIDATE
    # ========================================================

    if actual != expected:

        print()
        print("⚠ WARNING")
        print(f"Expected: {expected:,}")
        print(f"Actual:   {actual:,}")

    else:

        print("✓ Candidate count matches expected count")

    # ========================================================
    # CHECKPOINT
    # ========================================================

    print()
    print("Checkpointing candidate database...")

    cand.execute("CHECKPOINT")

    print("✓ Candidate database checkpointed")

    # ========================================================
    # SAVE STATE
    # ========================================================

    if ROUTE_NAME not in state["completed_routes"]:
        state["completed_routes"].append(ROUTE_NAME)

    state["last_completed_route"] = ROUTE_NAME
    state["updated_at"] = time.strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    with open(STATE_FILE, "w") as f:
        json.dump(state, f, indent=2)

    print("✓ Candidate state saved")

# ============================================================
# FINAL SUMMARY
# ============================================================

print()
print("=" * 80)
print("STEP 4 COMPLETE")
print("=" * 80)

total = cand.execute("""
    SELECT COUNT(*)
    FROM candidates_s1_s3_address
""").fetchone()[0]

print()
print(f"S1 -> S3 address candidates: {total:,}")

print()
print("Candidate DB:")
print(CANDIDATE_DB)

print()
print("✓ Safe checkpoint completed")

SAFE CANDIDATE GENERATION — STEP 4

Route:
S1 -> S3
FULL ADDRESS + COUNTRY

✓ Candidate database opened


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ Blocking database attached read-only

✓ S1-S3 address candidate table ready

VERIFYING EXPECTED CANDIDATE COUNT


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Expected candidates: 206,735
Count check time: 2.50 seconds

GENERATING S1 -> S3 ADDRESS CANDIDATES


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


✓ Generated: 206,735 candidates
✓ Generation time: 2.35 seconds
✓ Candidate count matches expected count

Checkpointing candidate database...
✓ Candidate database checkpointed
✓ Candidate state saved

STEP 4 COMPLETE

S1 -> S3 address candidates: 206,735

Candidate DB:
/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb

✓ Safe checkpoint completed


In [ ]:
# ============================================================
# SAFE CANDIDATE MERGE — STEP 5
# COMBINE + DEDUPLICATE ALL COMPLETED ROUTES
# ============================================================

import os
import time
import duckdb

# ============================================================
# PATHS
# ============================================================

CANDIDATE_DB = "/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb"
TEMP_DIR = "/content/er_candidates_tmp"

os.makedirs(TEMP_DIR, exist_ok=True)

print("=" * 80)
print("SAFE CANDIDATE MERGE — STEP 5")
print("=" * 80)
print()

# ============================================================
# OPEN DATABASE
# ============================================================

cand = duckdb.connect(CANDIDATE_DB)

cand.execute(
    f"SET temp_directory='{TEMP_DIR}'"
)

cand.execute(
    "SET preserve_insertion_order=false"
)

cand.execute(
    "SET threads=2"
)

print("✓ Candidate database opened")
print()

# ============================================================
# CHECK EXISTING ROUTES
# ============================================================

print("=" * 80)
print("CHECKING COMPLETED CANDIDATE ROUTES")
print("=" * 80)

routes = [
    ("S1-S2 name", "candidates_s1_s2"),
    ("S1-S3 name", "candidates_s1_s3"),
    ("S1-S2 address", "candidates_s1_s2_address"),
    ("S1-S3 address", "candidates_s1_s3_address"),
]

for label, table in routes:

    try:
        count = cand.execute(
            f"SELECT COUNT(*) FROM {table}"
        ).fetchone()[0]

        print(f"✓ {label:20s}: {count:,}")

    except Exception as e:

        print(f"⚠ {label:20s}: table not found")

print()

# ============================================================
# CREATE COMBINED TABLE
# ============================================================

print("=" * 80)
print("CREATING COMBINED CANDIDATE TABLE")
print("=" * 80)

cand.execute("""
    CREATE TABLE IF NOT EXISTS all_candidates AS

    SELECT
        source1_entity_id,
        candidate_entity_id,
        route

    FROM (
        SELECT
            source1_entity_id,
            candidate_entity_id,
            route
        FROM candidates_s1_s2

        UNION ALL

        SELECT
            source1_entity_id,
            candidate_entity_id,
            route
        FROM candidates_s1_s3

        UNION ALL

        SELECT
            source1_entity_id,
            candidate_entity_id,
            route
        FROM candidates_s1_s2_address

        UNION ALL

        SELECT
            source1_entity_id,
            candidate_entity_id,
            route
        FROM candidates_s1_s3_address
    )
""")

print("✓ Combined table created")
print()

# ============================================================
# CHECK RAW COMBINED COUNT
# ============================================================

raw_count = cand.execute("""
    SELECT COUNT(*)
    FROM all_candidates
""").fetchone()[0]

print(f"Raw candidate rows: {raw_count:,}")

# ============================================================
# CREATE DEDUPLICATED TABLE
# ============================================================

print()
print("=" * 80)
print("DEDUPLICATING CANDIDATE PAIRS")
print("=" * 80)

start = time.time()

cand.execute("""
    CREATE TABLE IF NOT EXISTS unique_candidates AS

    SELECT DISTINCT
        source1_entity_id,
        candidate_entity_id
    FROM all_candidates
""")

elapsed = time.time() - start

print(f"✓ Deduplication completed")
print(f"✓ Time: {elapsed:.2f} seconds")

# ============================================================
# COUNT UNIQUE CANDIDATES
# ============================================================

unique_count = cand.execute("""
    SELECT COUNT(*)
    FROM unique_candidates
""").fetchone()[0]

duplicates = raw_count - unique_count

print()
print(f"Raw candidates:      {raw_count:,}")
print(f"Unique candidates:   {unique_count:,}")
print(f"Duplicate rows:      {duplicates:,}")

if raw_count > 0:
    duplicate_pct = (duplicates / raw_count) * 100
    print(f"Duplicate percentage: {duplicate_pct:.2f}%")

# ============================================================
# CHECK ROUTE DISTRIBUTION
# ============================================================

print()
print("=" * 80)
print("ROUTE DISTRIBUTION")
print("=" * 80)

route_stats = cand.execute("""
    SELECT
        route,
        COUNT(*) AS candidate_count
    FROM all_candidates
    GROUP BY route
    ORDER BY candidate_count DESC
""").fetchdf()

print(route_stats.to_string(index=False))

# ============================================================
# CHECKPOINT
# ============================================================

print()
print("=" * 80)
print("CHECKPOINTING")
print("=" * 80)

cand.execute("CHECKPOINT")

print("✓ Candidate database checkpointed")

# ============================================================
# CREATE INDEX
# ============================================================

print()
print("Creating indexes...")

cand.execute("""
    CREATE INDEX IF NOT EXISTS
    idx_unique_source
    ON unique_candidates(source1_entity_id)
""")

cand.execute("""
    CREATE INDEX IF NOT EXISTS
    idx_unique_candidate
    ON unique_candidates(candidate_entity_id)
""")

cand.execute("CHECKPOINT")

print("✓ Indexes created")
print("✓ Final checkpoint completed")

# ============================================================
# FINAL SUMMARY
# ============================================================

print()
print("=" * 80)
print("STEP 5 COMPLETE")
print("=" * 80)

print()
print(f"Raw candidate rows:    {raw_count:,}")
print(f"Unique candidate pairs:{unique_count:,}")
print(f"Duplicate rows removed:{duplicates:,}")

print()
print("Final table:")
print("unique_candidates")

print()
print("Candidate DB:")
print(CANDIDATE_DB)

print()
print("=" * 80)
print("✓ READY FOR MATCHING")
print("=" * 80)

SAFE CANDIDATE MERGE — STEP 5

✓ Candidate database opened

CHECKING COMPLETED CANDIDATE ROUTES
✓ S1-S2 name          : 10,849,350
✓ S1-S3 name          : 11,892,958
✓ S1-S2 address       : 570,598
✓ S1-S3 address       : 206,735

CREATING COMBINED CANDIDATE TABLE


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ Combined table created

Raw candidate rows: 23,519,641

DEDUPLICATING CANDIDATE PAIRS


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ Deduplication completed
✓ Time: 387.16 seconds

Raw candidates:      23,519,641
Unique candidates:   23,413,014
Duplicate rows:      106,627
Duplicate percentage: 0.45%

ROUTE DISTRIBUTION
                     route  candidate_count
   s1_s3_full_name_country         11892958
   s1_s2_full_name_country         10849350
s1_s2_full_address_country           570598
s1_s3_full_address_country           206735

CHECKPOINTING
✓ Candidate database checkpointed

Creating indexes...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ Indexes created
✓ Final checkpoint completed

STEP 5 COMPLETE

Raw candidate rows:    23,519,641
Unique candidate pairs:23,413,014
Duplicate rows removed:106,627

Final table:
unique_candidates

Candidate DB:
/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb

✓ READY FOR MATCHING


In [ ]:
# ============================================================
# CANDIDATE QUALITY CHECK — STEP 6
# CANDIDATE COUNT PER SOURCE1 RECORD
# ============================================================

import duckdb
import os
import time

CANDIDATE_DB = "/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb"
TEMP_DIR = "/content/er_candidates_tmp"

os.makedirs(TEMP_DIR, exist_ok=True)

print("=" * 80)
print("CANDIDATE QUALITY CHECK — STEP 6")
print("=" * 80)
print()

# ============================================================
# OPEN DATABASE
# ============================================================

cand = duckdb.connect(CANDIDATE_DB)

cand.execute(
    f"SET temp_directory='{TEMP_DIR}'"
)

cand.execute(
    "SET preserve_insertion_order=false"
)

cand.execute(
    "SET threads=2"
)

print("✓ Candidate database opened")
print()

# ============================================================
# BASIC COUNTS
# ============================================================

print("=" * 80)
print("BASIC CANDIDATE STATISTICS")
print("=" * 80)

total_pairs = cand.execute("""
    SELECT COUNT(*)
    FROM unique_candidates
""").fetchone()[0]

unique_s1 = cand.execute("""
    SELECT COUNT(DISTINCT source1_entity_id)
    FROM unique_candidates
""").fetchone()[0]

print(f"Unique candidate pairs : {total_pairs:,}")
print(f"S1 records with matches: {unique_s1:,}")

# ============================================================
# CANDIDATE COUNT PER S1
# ============================================================

print()
print("=" * 80)
print("CALCULATING CANDIDATES PER S1 RECORD")
print("=" * 80)

start = time.time()

cand.execute("""
    CREATE OR REPLACE TABLE s1_candidate_counts AS

    SELECT
        source1_entity_id,
        COUNT(*) AS candidate_count

    FROM unique_candidates

    GROUP BY source1_entity_id
""")

elapsed = time.time() - start

print(f"✓ Candidate counts calculated")
print(f"✓ Time: {elapsed:.2f} seconds")

# ============================================================
# SUMMARY STATISTICS
# ============================================================

print()
print("=" * 80)
print("CANDIDATE COUNT DISTRIBUTION")
print("=" * 80)

stats = cand.execute("""
    SELECT
        COUNT(*) AS s1_records,
        AVG(candidate_count) AS average_candidates,
        MIN(candidate_count) AS minimum_candidates,
        MAX(candidate_count) AS maximum_candidates,
        MEDIAN(candidate_count) AS median_candidates,
        QUANTILE_CONT(candidate_count, 0.90) AS p90,
        QUANTILE_CONT(candidate_count, 0.95) AS p95,
        QUANTILE_CONT(candidate_count, 0.99) AS p99
    FROM s1_candidate_counts
""").fetchone()

print(f"S1 records with candidates : {stats[0]:,}")
print(f"Average candidates         : {stats[1]:.2f}")
print(f"Minimum candidates         : {stats[2]:,}")
print(f"Maximum candidates         : {stats[3]:,}")
print(f"Median candidates          : {stats[4]:,.0f}")
print(f"90th percentile            : {stats[5]:,.0f}")
print(f"95th percentile            : {stats[6]:,.0f}")
print(f"99th percentile            : {stats[7]:,.0f}")

# ============================================================
# DISTRIBUTION BUCKETS
# ============================================================

print()
print("=" * 80)
print("CANDIDATE COUNT BUCKETS")
print("=" * 80)

buckets = cand.execute("""
    SELECT
        CASE
            WHEN candidate_count = 1 THEN '1'
            WHEN candidate_count BETWEEN 2 AND 5 THEN '2-5'
            WHEN candidate_count BETWEEN 6 AND 10 THEN '6-10'
            WHEN candidate_count BETWEEN 11 AND 25 THEN '11-25'
            WHEN candidate_count BETWEEN 26 AND 50 THEN '26-50'
            WHEN candidate_count BETWEEN 51 AND 100 THEN '51-100'
            WHEN candidate_count BETWEEN 101 AND 500 THEN '101-500'
            WHEN candidate_count BETWEEN 501 AND 1000 THEN '501-1000'
            WHEN candidate_count BETWEEN 1001 AND 5000 THEN '1001-5000'
            ELSE '5000+'
        END AS bucket,

        COUNT(*) AS s1_records

    FROM s1_candidate_counts

    GROUP BY bucket

    ORDER BY
        CASE bucket
            WHEN '1' THEN 1
            WHEN '2-5' THEN 2
            WHEN '6-10' THEN 3
            WHEN '11-25' THEN 4
            WHEN '26-50' THEN 5
            WHEN '51-100' THEN 6
            WHEN '101-500' THEN 7
            WHEN '501-1000' THEN 8
            WHEN '1001-5000' THEN 9
            ELSE 10
        END
""").fetchdf()

print(buckets.to_string(index=False))

# ============================================================
# TOP 20 LARGEST CANDIDATE SETS
# ============================================================

print()
print("=" * 80)
print("TOP 20 LARGEST CANDIDATE SETS")
print("=" * 80)

top20 = cand.execute("""
    SELECT
        source1_entity_id,
        candidate_count

    FROM s1_candidate_counts

    ORDER BY candidate_count DESC

    LIMIT 20
""").fetchdf()

print(top20.to_string(index=False))

# ============================================================
# CHECKPOINT
# ============================================================

print()
print("=" * 80)
print("CHECKPOINTING")
print("=" * 80)

cand.execute("CHECKPOINT")

print("✓ Candidate database checkpointed")

# ============================================================
# FINAL
# ============================================================

print()
print("=" * 80)
print("STEP 6 COMPLETE")
print("=" * 80)

print()
print("Created table:")
print("s1_candidate_counts")

print()
print("✓ Candidate coverage inspection complete")
print("✓ No fuzzy matching performed yet")

CANDIDATE QUALITY CHECK — STEP 6

✓ Candidate database opened

BASIC CANDIDATE STATISTICS


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Unique candidate pairs : 23,413,014
S1 records with matches: 1,724,593

CALCULATING CANDIDATES PER S1 RECORD


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ Candidate counts calculated
✓ Time: 5.55 seconds

CANDIDATE COUNT DISTRIBUTION
S1 records with candidates : 1,724,593
Average candidates         : 13.58
Minimum candidates         : 1
Maximum candidates         : 1,200
Median candidates          : 2
90th percentile            : 25
95th percentile            : 77
99th percentile            : 206

CANDIDATE COUNT BUCKETS
   bucket  s1_records
        1      522588
      2-5      697346
     6-10      155190
    11-25      178643
    26-50       36603
   51-100       77088
  101-500       57134
1001-5000           1

TOP 20 LARGEST CANDIDATE SETS
source1_entity_id  candidate_count
     S1-882379259             1200
     S1-857373485              473
     S1-243578127              471
     S1-331657505              470
      S1-34557613              469
     S1-345750860              469
     S1-500810031              469
      S1-10965334              469
     S1-718867106              469
     S1-979860618              468
     S1-3575

In [ ]:
# ============================================================
# MATCHING WORKSPACE — STEP 7
# ADD SOURCE RECORD DATA TO UNIQUE CANDIDATES
# ============================================================

import duckdb
import os
import time

# ============================================================
# PATHS
# ============================================================

BLOCK_DB = "/content/drive/MyDrive/er_checkpoint/er_blocking_train_fresh.duckdb"
CANDIDATE_DB = "/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb"
TEMP_DIR = "/content/er_candidates_tmp"

os.makedirs(TEMP_DIR, exist_ok=True)

print("=" * 80)
print("MATCHING WORKSPACE — STEP 7")
print("=" * 80)
print()

# ============================================================
# OPEN CANDIDATE DATABASE
# ============================================================

cand = duckdb.connect(CANDIDATE_DB)

cand.execute(
    f"SET temp_directory='{TEMP_DIR}'"
)

cand.execute(
    "SET preserve_insertion_order=false"
)

cand.execute(
    "SET threads=2"
)

print("✓ Candidate database opened")

# ============================================================
# ATTACH BLOCKING DATABASE
# ============================================================

try:
    cand.execute("DETACH DATABASE blocking_db")
except Exception:
    pass

cand.execute(
    f"ATTACH '{BLOCK_DB}' AS blocking_db (READ_ONLY)"
)

print("✓ Blocking database attached")
print()

# ============================================================
# CREATE MATCHING WORKSPACE
# ============================================================

print("=" * 80)
print("BUILDING MATCHING WORKSPACE")
print("=" * 80)

start = time.time()

cand.execute("""
    CREATE OR REPLACE TABLE matching_candidates AS

    SELECT

        c.source1_entity_id,

        c.candidate_entity_id,

        -- SOURCE 1
        s1.business_name AS s1_business_name,
        s1.business_address AS s1_business_address,
        s1.country AS s1_country,

        s1.name_norm AS s1_name_norm,
        s1.address_norm AS s1_address_norm,
        s1.country_norm AS s1_country_norm,

        -- CANDIDATE SOURCE
        CASE
            WHEN c.candidate_entity_id LIKE 'S2-%'
            THEN s2.business_name
            ELSE s3.business_name
        END AS candidate_business_name,

        CASE
            WHEN c.candidate_entity_id LIKE 'S2-%'
            THEN s2.business_address
            ELSE s3.business_address
        END AS candidate_business_address,

        CASE
            WHEN c.candidate_entity_id LIKE 'S2-%'
            THEN s2.country
            ELSE s3.country
        END AS candidate_country,

        CASE
            WHEN c.candidate_entity_id LIKE 'S2-%'
            THEN s2.name_norm
            ELSE s3.name_norm
        END AS candidate_name_norm,

        CASE
            WHEN c.candidate_entity_id LIKE 'S2-%'
            THEN s2.address_norm
            ELSE s3.address_norm
        END AS candidate_address_norm,

        CASE
            WHEN c.candidate_entity_id LIKE 'S2-%'
            THEN s2.country_norm
            ELSE s3.country_norm
        END AS candidate_country_norm

    FROM unique_candidates c

    INNER JOIN blocking_db.train_s1_blocking s1
        ON c.source1_entity_id = s1.entity_id

    LEFT JOIN blocking_db.train_s2_blocking s2
        ON c.candidate_entity_id = s2.entity_id

    LEFT JOIN blocking_db.train_s3_blocking s3
        ON c.candidate_entity_id = s3.entity_id
""")

elapsed = time.time() - start

print(f"✓ Matching workspace created")
print(f"✓ Time: {elapsed:.2f} seconds")

# ============================================================
# VERIFY ROW COUNT
# ============================================================

print()
print("=" * 80)
print("VERIFYING MATCHING WORKSPACE")
print("=" * 80)

workspace_count = cand.execute("""
    SELECT COUNT(*)
    FROM matching_candidates
""").fetchone()[0]

unique_count = cand.execute("""
    SELECT COUNT(*)
    FROM unique_candidates
""").fetchone()[0]

print(f"unique_candidates     : {unique_count:,}")
print(f"matching_candidates   : {workspace_count:,}")

if workspace_count == unique_count:
    print("✓ Row counts match")
else:
    print("⚠ WARNING: Row counts do not match")

# ============================================================
# CHECK NULLS
# ============================================================

print()
print("=" * 80)
print("CHECKING JOIN QUALITY")
print("=" * 80)

null_stats = cand.execute("""
    SELECT

        COUNT(*) AS total,

        SUM(
            CASE
                WHEN s1_business_name IS NULL
                THEN 1 ELSE 0
            END
        ) AS missing_s1_name,

        SUM(
            CASE
                WHEN candidate_business_name IS NULL
                THEN 1 ELSE 0
            END
        ) AS missing_candidate_name,

        SUM(
            CASE
                WHEN candidate_country_norm IS NULL
                THEN 1 ELSE 0
            END
        ) AS missing_candidate_country

    FROM matching_candidates
""").fetchone()

print(f"Total rows               : {null_stats[0]:,}")
print(f"Missing S1 names         : {null_stats[1]:,}")
print(f"Missing candidate names  : {null_stats[2]:,}")
print(f"Missing candidate country: {null_stats[3]:,}")

# ============================================================
# CREATE INDEXES
# ============================================================

print()
print("=" * 80)
print("CREATING MATCHING INDEXES")
print("=" * 80)

cand.execute("""
    CREATE INDEX IF NOT EXISTS
    idx_matching_source1
    ON matching_candidates(source1_entity_id)
""")

cand.execute("""
    CREATE INDEX IF NOT EXISTS
    idx_matching_candidate
    ON matching_candidates(candidate_entity_id)
""")

print("✓ Indexes created")

# ============================================================
# CHECKPOINT
# ============================================================

print()
print("Checkpointing...")

cand.execute("CHECKPOINT")

print("✓ Database checkpointed")

# ============================================================
# SAMPLE
# ============================================================

print()
print("=" * 80)
print("SAMPLE MATCHING RECORDS")
print("=" * 80)

sample = cand.execute("""
    SELECT
        source1_entity_id,
        candidate_entity_id,
        s1_business_name,
        candidate_business_name,
        s1_country,
        candidate_country
    FROM matching_candidates
    LIMIT 10
""").fetchdf()

print(sample.to_string(index=False))

# ============================================================
# FINAL
# ============================================================

print()
print("=" * 80)
print("STEP 7 COMPLETE")
print("=" * 80)

print()
print("Created:")
print("matching_candidates")

print()
print("✓ Ready for exact-match signal generation")

MATCHING WORKSPACE — STEP 7

✓ Candidate database opened
✓ Blocking database attached

BUILDING MATCHING WORKSPACE


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ Matching workspace created
✓ Time: 204.46 seconds

VERIFYING MATCHING WORKSPACE
unique_candidates     : 23,413,014
matching_candidates   : 23,413,014
✓ Row counts match

CHECKING JOIN QUALITY
Total rows               : 23,413,014
Missing S1 names         : 0
Missing candidate names  : 0
Missing candidate country: 0

CREATING MATCHING INDEXES


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ Indexes created

Checkpointing...
✓ Database checkpointed

SAMPLE MATCHING RECORDS
source1_entity_id candidate_entity_id              s1_business_name       candidate_business_name s1_country candidate_country
     S1-251506166        S2-614721384                Atlantic Guild               Atlantic  Guild         US                US
     S1-493230080        S2-998782125 Smith, Cruz and Adler Grading Smith, Cruz and Adler Grading         US                US
     S1-584046291        S2-521724739            Pediatric Partners            PEDIATRIC PARTNERS         US                US
     S1-977205473        S2-586437508          Scholarship Alliance          scholarship alliance         US                US
     S1-519818084        S2-168445707            Custom Gulf Xsolla            CUSTOM GULF XSOLLA         US                US
     S1-120551824        S2-421703727               Corner Hypnosis              Corner  Hypnosis         US                US
     S1-453624568        S

In [ ]:
# ============================================================
# MATCHING — STEP 8
# GENERATE EXACT-MATCH SIGNALS
# ============================================================

import duckdb
import os
import time

# ============================================================
# PATHS
# ============================================================

CANDIDATE_DB = "/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb"
TEMP_DIR = "/content/er_candidates_tmp"

os.makedirs(TEMP_DIR, exist_ok=True)

print("=" * 80)
print("MATCHING — STEP 8")
print("EXACT-MATCH SIGNAL GENERATION")
print("=" * 80)
print()

# ============================================================
# OPEN DATABASE
# ============================================================

cand = duckdb.connect(CANDIDATE_DB)

cand.execute(
    f"SET temp_directory='{TEMP_DIR}'"
)

cand.execute(
    "SET preserve_insertion_order=false"
)

cand.execute(
    "SET threads=2"
)

print("✓ Candidate database opened")
print()

# ============================================================
# CREATE SIGNAL TABLE
# ============================================================

print("=" * 80)
print("GENERATING EXACT-MATCH SIGNALS")
print("=" * 80)

start = time.time()

cand.execute("""
    CREATE OR REPLACE TABLE exact_match_signals AS

    SELECT

        source1_entity_id,

        candidate_entity_id,

        -- ------------------------------------------------
        -- COUNTRY
        -- ------------------------------------------------

        CASE
            WHEN s1_country_norm != ''
             AND candidate_country_norm != ''
             AND s1_country_norm = candidate_country_norm
            THEN 1
            ELSE 0
        END AS country_exact,

        -- ------------------------------------------------
        -- NAME
        -- ------------------------------------------------

        CASE
            WHEN s1_name_norm != ''
             AND candidate_name_norm != ''
             AND s1_name_norm = candidate_name_norm
            THEN 1
            ELSE 0
        END AS name_exact,

        -- ------------------------------------------------
        -- ADDRESS
        -- ------------------------------------------------

        CASE
            WHEN s1_address_norm != ''
             AND candidate_address_norm != ''
             AND s1_address_norm = candidate_address_norm
            THEN 1
            ELSE 0
        END AS address_exact,

        -- ------------------------------------------------
        -- NAME + ADDRESS
        -- ------------------------------------------------

        CASE
            WHEN s1_name_norm != ''
             AND candidate_name_norm != ''
             AND s1_address_norm != ''
             AND candidate_address_norm != ''
             AND s1_name_norm = candidate_name_norm
             AND s1_address_norm = candidate_address_norm
            THEN 1
            ELSE 0
        END AS name_address_exact,

        -- ------------------------------------------------
        -- NAME + COUNTRY
        -- ------------------------------------------------

        CASE
            WHEN s1_name_norm != ''
             AND candidate_name_norm != ''
             AND s1_country_norm != ''
             AND candidate_country_norm != ''
             AND s1_name_norm = candidate_name_norm
             AND s1_country_norm = candidate_country_norm
            THEN 1
            ELSE 0
        END AS name_country_exact,

        -- ------------------------------------------------
        -- ADDRESS + COUNTRY
        -- ------------------------------------------------

        CASE
            WHEN s1_address_norm != ''
             AND candidate_address_norm != ''
             AND s1_country_norm != ''
             AND candidate_country_norm != ''
             AND s1_address_norm = candidate_address_norm
             AND s1_country_norm = candidate_country_norm
            THEN 1
            ELSE 0
        END AS address_country_exact

    FROM matching_candidates
""")

elapsed = time.time() - start

print(f"✓ Exact signals generated")
print(f"✓ Time: {elapsed:.2f} seconds")

# ============================================================
# VERIFY ROW COUNT
# ============================================================

print()
print("=" * 80)
print("VERIFYING SIGNAL TABLE")
print("=" * 80)

signal_count = cand.execute("""
    SELECT COUNT(*)
    FROM exact_match_signals
""").fetchone()[0]

candidate_count = cand.execute("""
    SELECT COUNT(*)
    FROM matching_candidates
""").fetchone()[0]

print(f"matching_candidates : {candidate_count:,}")
print(f"exact_match_signals: {signal_count:,}")

if signal_count == candidate_count:
    print("✓ Row counts match")
else:
    print("⚠ WARNING: Row counts do not match")

# ============================================================
# SIGNAL STATISTICS
# ============================================================

print()
print("=" * 80)
print("EXACT SIGNAL STATISTICS")
print("=" * 80)

stats = cand.execute("""
    SELECT

        COUNT(*) AS total,

        SUM(name_exact) AS name_exact,

        SUM(address_exact) AS address_exact,

        SUM(country_exact) AS country_exact,

        SUM(name_address_exact) AS name_address_exact,

        SUM(name_country_exact) AS name_country_exact,

        SUM(address_country_exact) AS address_country_exact

    FROM exact_match_signals
""").fetchone()

print(f"Total candidates       : {stats[0]:,}")
print(f"Name exact             : {stats[1]:,}")
print(f"Address exact          : {stats[2]:,}")
print(f"Country exact          : {stats[3]:,}")
print(f"Name + Address exact   : {stats[4]:,}")
print(f"Name + Country exact   : {stats[5]:,}")
print(f"Address + Country exact: {stats[6]:,}")

# ============================================================
# COMBINED EXACT MATCH LEVELS
# ============================================================

print()
print("=" * 80)
print("EXACT MATCH LEVEL DISTRIBUTION")
print("=" * 80)

levels = cand.execute("""
    SELECT

        CASE

            WHEN name_address_exact = 1
                THEN 'NAME + ADDRESS EXACT'

            WHEN name_country_exact = 1
                THEN 'NAME + COUNTRY EXACT'

            WHEN address_country_exact = 1
                THEN 'ADDRESS + COUNTRY EXACT'

            WHEN name_exact = 1
                THEN 'NAME ONLY EXACT'

            WHEN address_exact = 1
                THEN 'ADDRESS ONLY EXACT'

            WHEN country_exact = 1
                THEN 'COUNTRY ONLY'

            ELSE 'NO EXACT MATCH'

        END AS match_level,

        COUNT(*) AS candidate_count

    FROM exact_match_signals

    GROUP BY match_level

    ORDER BY candidate_count DESC
""").fetchdf()

print(levels.to_string(index=False))

# ============================================================
# CHECKPOINT
# ============================================================

print()
print("=" * 80)
print("CHECKPOINTING")
print("=" * 80)

cand.execute("CHECKPOINT")

print("✓ Candidate database checkpointed")

# ============================================================
# FINAL
# ============================================================

print()
print("=" * 80)
print("STEP 8 COMPLETE")
print("=" * 80)

print()
print("Created table:")
print("exact_match_signals")

print()
print("✓ Exact-match signal generation complete")
print("✓ No fuzzy matching performed yet")

MATCHING — STEP 8
EXACT-MATCH SIGNAL GENERATION

✓ Candidate database opened

GENERATING EXACT-MATCH SIGNALS


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ Exact signals generated
✓ Time: 42.27 seconds

VERIFYING SIGNAL TABLE
matching_candidates : 23,413,014
exact_match_signals: 23,413,014
✓ Row counts match

EXACT SIGNAL STATISTICS
Total candidates       : 23,413,014
Name exact             : 22,742,308
Address exact          : 777,333
Country exact          : 23,413,014
Name + Address exact   : 106,627
Name + Country exact   : 22,742,308
Address + Country exact: 777,333

EXACT MATCH LEVEL DISTRIBUTION
            match_level  candidate_count
   NAME + COUNTRY EXACT         22635681
ADDRESS + COUNTRY EXACT           670706
   NAME + ADDRESS EXACT           106627

CHECKPOINTING
✓ Candidate database checkpointed

STEP 8 COMPLETE

Created table:
exact_match_signals

✓ Exact-match signal generation complete
✓ No fuzzy matching performed yet


In [ ]:
# ============================================================
# MATCHING — STEP 9
# IDENTIFY EXACT-MATCH AMBIGUITIES
# ============================================================

import duckdb
import os
import time

# ============================================================
# PATHS
# ============================================================

CANDIDATE_DB = "/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb"
TEMP_DIR = "/content/er_candidates_tmp"

os.makedirs(TEMP_DIR, exist_ok=True)

print("=" * 80)
print("MATCHING — STEP 9")
print("IDENTIFY EXACT-MATCH AMBIGUITIES")
print("=" * 80)
print()

# ============================================================
# OPEN DATABASE
# ============================================================

cand = duckdb.connect(CANDIDATE_DB)

cand.execute(
    f"SET temp_directory='{TEMP_DIR}'"
)

cand.execute(
    "SET preserve_insertion_order=false"
)

cand.execute(
    "SET threads=2"
)

print("✓ Candidate database opened")
print()

# ============================================================
# CREATE EXACT NAME CANDIDATE COUNTS
# ============================================================

print("=" * 80)
print("COUNTING EXACT NAME MATCHES PER S1")
print("=" * 80)

start = time.time()

cand.execute("""
    CREATE OR REPLACE TABLE s1_exact_name_counts AS

    SELECT

        source1_entity_id,

        COUNT(*) AS exact_name_candidates

    FROM exact_match_signals

    WHERE name_exact = 1

    GROUP BY source1_entity_id
""")

elapsed = time.time() - start

print(f"✓ Exact-name counts calculated")
print(f"✓ Time: {elapsed:.2f} seconds")

# ============================================================
# CREATE EXACT ADDRESS COUNTS
# ============================================================

print()
print("=" * 80)
print("COUNTING EXACT ADDRESS MATCHES PER S1")
print("=" * 80)

start = time.time()

cand.execute("""
    CREATE OR REPLACE TABLE s1_exact_address_counts AS

    SELECT

        source1_entity_id,

        COUNT(*) AS exact_address_candidates

    FROM exact_match_signals

    WHERE address_exact = 1

    GROUP BY source1_entity_id
""")

elapsed = time.time() - start

print(f"✓ Exact-address counts calculated")
print(f"✓ Time: {elapsed:.2f} seconds")

# ============================================================
# COMBINE COUNTS
# ============================================================

print()
print("=" * 80)
print("BUILDING EXACT MATCH SUMMARY")
print("=" * 80)

cand.execute("""
    CREATE OR REPLACE TABLE s1_exact_summary AS

    SELECT

        COALESCE(n.source1_entity_id,
                 a.source1_entity_id) AS source1_entity_id,

        COALESCE(n.exact_name_candidates, 0)
            AS exact_name_candidates,

        COALESCE(a.exact_address_candidates, 0)
            AS exact_address_candidates

    FROM s1_exact_name_counts n

    FULL OUTER JOIN s1_exact_address_counts a

        ON n.source1_entity_id =
           a.source1_entity_id
""")

print("✓ Exact-match summary created")

# ============================================================
# SUMMARY STATISTICS
# ============================================================

print()
print("=" * 80)
print("EXACT MATCH SUMMARY")
print("=" * 80)

summary = cand.execute("""
    SELECT

        COUNT(*) AS s1_with_exact_matches,

        SUM(
            CASE
                WHEN exact_name_candidates = 1
                THEN 1 ELSE 0
            END
        ) AS unique_exact_name,

        SUM(
            CASE
                WHEN exact_name_candidates > 1
                THEN 1 ELSE 0
            END
        ) AS ambiguous_exact_name,

        SUM(
            CASE
                WHEN exact_address_candidates = 1
                THEN 1 ELSE 0
            END
        ) AS unique_exact_address,

        SUM(
            CASE
                WHEN exact_address_candidates > 1
                THEN 1 ELSE 0
            END
        ) AS ambiguous_exact_address

    FROM s1_exact_summary
""").fetchone()

print(f"S1 with exact matches       : {summary[0]:,}")
print(f"Unique exact-name matches   : {summary[1]:,}")
print(f"Ambiguous exact-name matches: {summary[2]:,}")
print(f"Unique exact-address matches: {summary[3]:,}")
print(f"Ambiguous exact-address     : {summary[4]:,}")

# ============================================================
# EXACT NAME DISTRIBUTION
# ============================================================

print()
print("=" * 80)
print("EXACT NAME CANDIDATE DISTRIBUTION")
print("=" * 80)

name_dist = cand.execute("""
    SELECT

        CASE

            WHEN exact_name_candidates = 1
                THEN '1'

            WHEN exact_name_candidates BETWEEN 2 AND 5
                THEN '2-5'

            WHEN exact_name_candidates BETWEEN 6 AND 10
                THEN '6-10'

            WHEN exact_name_candidates BETWEEN 11 AND 25
                THEN '11-25'

            WHEN exact_name_candidates BETWEEN 26 AND 100
                THEN '26-100'

            ELSE '100+'

        END AS bucket,

        COUNT(*) AS s1_records

    FROM s1_exact_summary

    WHERE exact_name_candidates > 0

    GROUP BY bucket

    ORDER BY

        CASE bucket

            WHEN '1' THEN 1
            WHEN '2-5' THEN 2
            WHEN '6-10' THEN 3
            WHEN '11-25' THEN 4
            WHEN '26-100' THEN 5
            ELSE 6

        END
""").fetchdf()

print(name_dist.to_string(index=False))

# ============================================================
# TOP AMBIGUOUS EXACT-NAME RECORDS
# ============================================================

print()
print("=" * 80)
print("TOP 20 EXACT-NAME AMBIGUITIES")
print("=" * 80)

top_ambiguous = cand.execute("""
    SELECT

        source1_entity_id,

        exact_name_candidates,

        exact_address_candidates

    FROM s1_exact_summary

    WHERE exact_name_candidates > 1

    ORDER BY exact_name_candidates DESC

    LIMIT 20
""").fetchdf()

print(top_ambiguous.to_string(index=False))

# ============================================================
# CHECKPOINT
# ============================================================

print()
print("=" * 80)
print("CHECKPOINTING")
print("=" * 80)

cand.execute("CHECKPOINT")

print("✓ Candidate database checkpointed")

# ============================================================
# FINAL
# ============================================================

print()
print("=" * 80)
print("STEP 9 COMPLETE")
print("=" * 80)

print()
print("Created:")
print("s1_exact_name_counts")
print("s1_exact_address_counts")
print("s1_exact_summary")

print()
print("✓ Exact-match ambiguity analysis complete")

MATCHING — STEP 9
IDENTIFY EXACT-MATCH AMBIGUITIES

✓ Candidate database opened

COUNTING EXACT NAME MATCHES PER S1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ Exact-name counts calculated
✓ Time: 11.75 seconds

COUNTING EXACT ADDRESS MATCHES PER S1
✓ Exact-address counts calculated
✓ Time: 0.78 seconds

BUILDING EXACT MATCH SUMMARY
✓ Exact-match summary created

EXACT MATCH SUMMARY
S1 with exact matches       : 1,724,593
Unique exact-name matches   : 546,200
Ambiguous exact-name matches: 1,047,515
Unique exact-address matches: 389,135
Ambiguous exact-address     : 147,907

EXACT NAME CANDIDATE DISTRIBUTION
bucket  s1_records
     1      546200
   2-5      565330
  6-10      141909
 11-25      170026
26-100      113441
  100+       56809

TOP 20 EXACT-NAME AMBIGUITIES
source1_entity_id  exact_name_candidates  exact_address_candidates
     S1-882379259                   1200                         0
     S1-934792705                    467                         1
     S1-243578127                    467                         4
     S1-468169673                    467                         1
      S1-10965334                    467    

In [ ]:
# ============================================================
# MATCHING — STEP 10
# BUILD HIGH-CONFIDENCE EXACT MATCHES
# ============================================================

import duckdb
import os
import time

# ============================================================
# PATHS
# ============================================================

CANDIDATE_DB = "/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb"
TEMP_DIR = "/content/er_candidates_tmp"

os.makedirs(TEMP_DIR, exist_ok=True)

print("=" * 80)
print("MATCHING — STEP 10")
print("BUILD HIGH-CONFIDENCE EXACT MATCHES")
print("=" * 80)
print()

# ============================================================
# OPEN DATABASE
# ============================================================

cand = duckdb.connect(CANDIDATE_DB)

cand.execute(
    f"SET temp_directory='{TEMP_DIR}'"
)

cand.execute(
    "SET preserve_insertion_order=false"
)

cand.execute(
    "SET threads=2"
)

print("✓ Candidate database opened")
print()

# ============================================================
# CREATE HIGH-CONFIDENCE TABLE
# ============================================================

print("=" * 80)
print("BUILDING HIGH-CONFIDENCE MATCH TABLE")
print("=" * 80)

start = time.time()

cand.execute("""
    CREATE OR REPLACE TABLE high_confidence_matches AS

    SELECT

        e.source1_entity_id,

        e.candidate_entity_id,

        CASE

            WHEN e.name_address_exact = 1
                THEN 'exact_name_address'

            WHEN e.name_exact = 1
                 AND n.exact_name_candidates = 1
                THEN 'unique_exact_name'

            WHEN e.address_exact = 1
                 AND a.exact_address_candidates = 1
                THEN 'unique_exact_address'

            ELSE NULL

        END AS match_type

    FROM exact_match_signals e

    LEFT JOIN s1_exact_name_counts n

        ON e.source1_entity_id =
           n.source1_entity_id

    LEFT JOIN s1_exact_address_counts a

        ON e.source1_entity_id =
           a.source1_entity_id

    WHERE

        e.name_address_exact = 1

        OR

        (
            e.name_exact = 1
            AND n.exact_name_candidates = 1
        )

        OR

        (
            e.address_exact = 1
            AND a.exact_address_candidates = 1
        )
""")

elapsed = time.time() - start

print("✓ High-confidence table created")
print(f"✓ Time: {elapsed:.2f} seconds")

# ============================================================
# COUNT MATCHES
# ============================================================

print()
print("=" * 80)
print("HIGH-CONFIDENCE MATCH STATISTICS")
print("=" * 80)

total = cand.execute("""
    SELECT COUNT(*)
    FROM high_confidence_matches
""").fetchone()[0]

unique_s1 = cand.execute("""
    SELECT COUNT(DISTINCT source1_entity_id)
    FROM high_confidence_matches
""").fetchone()[0]

print(f"High-confidence pairs : {total:,}")
print(f"Unique S1 records     : {unique_s1:,}")

# ============================================================
# MATCH TYPE DISTRIBUTION
# ============================================================

print()
print("=" * 80)
print("MATCH TYPE DISTRIBUTION")
print("=" * 80)

types = cand.execute("""
    SELECT

        match_type,

        COUNT(*) AS pair_count,

        COUNT(DISTINCT source1_entity_id)
            AS s1_count

    FROM high_confidence_matches

    GROUP BY match_type

    ORDER BY pair_count DESC
""").fetchdf()

print(types.to_string(index=False))

# ============================================================
# CHECK MULTIPLE HIGH-CONFIDENCE MATCHES
# ============================================================

print()
print("=" * 80)
print("CHECKING FOR MULTIPLE HIGH-CONFIDENCE MATCHES")
print("=" * 80)

ambiguous = cand.execute("""
    SELECT

        source1_entity_id,

        COUNT(*) AS match_count

    FROM high_confidence_matches

    GROUP BY source1_entity_id

    HAVING COUNT(*) > 1

    ORDER BY match_count DESC
""").fetchdf()

print(
    f"S1 records with multiple high-confidence pairs: "
    f"{len(ambiguous):,}"
)

if len(ambiguous) > 0:

    print()
    print("Top ambiguous high-confidence records:")

    print(
        ambiguous.head(20).to_string(index=False)
    )

else:

    print("✓ No S1 record has multiple high-confidence pairs")

# ============================================================
# CREATE UNIQUE HIGH-CONFIDENCE MATCHES
# ============================================================

print()
print("=" * 80)
print("CREATING UNIQUE HIGH-CONFIDENCE MATCHES")
print("=" * 80)

cand.execute("""
    CREATE OR REPLACE TABLE unique_high_confidence_matches AS

    SELECT

        source1_entity_id,

        MIN(candidate_entity_id) AS candidate_entity_id,

        MIN(match_type) AS match_type

    FROM high_confidence_matches

    GROUP BY source1_entity_id

    HAVING COUNT(*) = 1
""")

unique_final = cand.execute("""
    SELECT COUNT(*)
    FROM unique_high_confidence_matches
""").fetchone()[0]

print(
    f"✓ Unique high-confidence matches: "
    f"{unique_final:,}"
)

# ============================================================
# CHECKPOINT
# ============================================================

print()
print("=" * 80)
print("CHECKPOINTING")
print("=" * 80)

cand.execute("CHECKPOINT")

print("✓ Candidate database checkpointed")

# ============================================================
# FINAL
# ============================================================

print()
print("=" * 80)
print("STEP 10 COMPLETE")
print("=" * 80)

print()
print("Created:")
print("high_confidence_matches")
print("unique_high_confidence_matches")

print()
print("✓ High-confidence exact matching complete")
print("✓ Ambiguous records remain for deeper matching")

MATCHING — STEP 10
BUILD HIGH-CONFIDENCE EXACT MATCHES

✓ Candidate database opened

BUILDING HIGH-CONFIDENCE MATCH TABLE


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ High-confidence table created
✓ Time: 13.12 seconds

HIGH-CONFIDENCE MATCH STATISTICS
High-confidence pairs : 960,982
Unique S1 records     : 869,412

MATCH TYPE DISTRIBUTION
          match_type  pair_count  s1_count
   unique_exact_name      517749    517749
unique_exact_address      336606    336606
  exact_name_address      106627     99083

CHECKING FOR MULTIPLE HIGH-CONFIDENCE MATCHES
S1 records with multiple high-confidence pairs: 91,175

Top ambiguous high-confidence records:
source1_entity_id  match_count
     S1-833209101            4
     S1-550276876            4
     S1-625293737            4
     S1-554737818            4
     S1-908611786            4
     S1-438127860            4
     S1-167407091            4
     S1-354564872            4
     S1-590043090            4
     S1-779157190            4
     S1-965893546            3
     S1-250684787            3
     S1-914464957            3
     S1-379814131            3
     S1-968274664            3
     S1-24355

In [ ]:
# ============================================================
# STEP 11 — CONTINUE AFTER COUNT ERROR
# ============================================================

print("=" * 80)
print("MATCHING — STEP 11 CONTINUED")
print("VERIFY UNRESOLVED CANDIDATES")
print("=" * 80)

# ------------------------------------------------------------
# BASIC STATISTICS
# ------------------------------------------------------------

total_candidates = conn.execute("""
    SELECT COUNT(*)
    FROM unresolved_candidates
""").fetchone()[0]

unique_s1 = conn.execute("""
    SELECT COUNT(DISTINCT source1_entity_id)
    FROM unresolved_candidates
""").fetchone()[0]

unique_candidates = conn.execute("""
    SELECT COUNT(*)
    FROM (
        SELECT DISTINCT
            source1_entity_id,
            candidate_entity_id
        FROM unresolved_candidates
    )
""").fetchone()[0]

print()
print("=" * 80)
print("UNRESOLVED CANDIDATE STATISTICS")
print("=" * 80)

print(f"Unresolved candidate pairs : {total_candidates:,}")
print(f"Unique unresolved S1       : {unique_s1:,}")
print(f"Unique candidate pairs     : {unique_candidates:,}")

# ------------------------------------------------------------
# CREATE / RECREATE S1 COUNTS
# ------------------------------------------------------------

print()
print("=" * 80)
print("BUILDING UNRESOLVED S1 COUNTS")
print("=" * 80)

conn.execute("DROP TABLE IF EXISTS unresolved_s1_counts")

conn.execute("""
CREATE TABLE unresolved_s1_counts AS
SELECT
    source1_entity_id,
    COUNT(*) AS candidate_count
FROM unresolved_candidates
GROUP BY source1_entity_id
""")

print("✓ unresolved_s1_counts created")

# ------------------------------------------------------------
# STATISTICS
# ------------------------------------------------------------

stats = conn.execute("""
SELECT
    COUNT(*) AS s1_count,
    AVG(candidate_count) AS avg_candidates,
    MIN(candidate_count) AS min_candidates,
    MAX(candidate_count) AS max_candidates,
    MEDIAN(candidate_count) AS median_candidates,
    QUANTILE_CONT(candidate_count, 0.90) AS p90,
    QUANTILE_CONT(candidate_count, 0.95) AS p95,
    QUANTILE_CONT(candidate_count, 0.99) AS p99
FROM unresolved_s1_counts
""").fetchone()

print()
print("=" * 80)
print("CANDIDATE COUNT DISTRIBUTION")
print("=" * 80)

print(f"S1 records                 : {stats[0]:,}")
print(f"Average candidates/S1      : {stats[1]:.2f}")
print(f"Minimum candidates         : {stats[2]:,}")
print(f"Maximum candidates         : {stats[3]:,}")
print(f"Median candidates          : {stats[4]:.0f}")
print(f"P90 candidates             : {stats[5]:.0f}")
print(f"P95 candidates             : {stats[6]:.0f}")
print(f"P99 candidates             : {stats[7]:.0f}")

# ------------------------------------------------------------
# BUCKET DISTRIBUTION
# ------------------------------------------------------------

print()
print("=" * 80)
print("UNRESOLVED CANDIDATE BUCKETS")
print("=" * 80)

bucket_df = conn.execute("""
SELECT
    CASE
        WHEN candidate_count = 1 THEN '1'
        WHEN candidate_count BETWEEN 2 AND 5 THEN '2-5'
        WHEN candidate_count BETWEEN 6 AND 10 THEN '6-10'
        WHEN candidate_count BETWEEN 11 AND 25 THEN '11-25'
        WHEN candidate_count BETWEEN 26 AND 50 THEN '26-50'
        WHEN candidate_count BETWEEN 51 AND 100 THEN '51-100'
        WHEN candidate_count BETWEEN 101 AND 500 THEN '101-500'
        WHEN candidate_count BETWEEN 501 AND 1000 THEN '501-1000'
        ELSE '1000+'
    END AS candidate_bucket,

    COUNT(*) AS s1_count

FROM unresolved_s1_counts

GROUP BY 1

ORDER BY
    CASE candidate_bucket
        WHEN '1' THEN 1
        WHEN '2-5' THEN 2
        WHEN '6-10' THEN 3
        WHEN '11-25' THEN 4
        WHEN '26-50' THEN 5
        WHEN '51-100' THEN 6
        WHEN '101-500' THEN 7
        WHEN '501-1000' THEN 8
        ELSE 9
    END
""").fetchdf()

print(bucket_df.to_string(index=False))

# ------------------------------------------------------------
# CHECK DUPLICATES
# ------------------------------------------------------------

duplicate_pairs = conn.execute("""
SELECT COUNT(*)
FROM (
    SELECT
        source1_entity_id,
        candidate_entity_id,
        COUNT(*) AS n
    FROM unresolved_candidates
    GROUP BY
        source1_entity_id,
        candidate_entity_id
    HAVING COUNT(*) > 1
)
""").fetchone()[0]

print()
print("=" * 80)
print("DUPLICATE CHECK")
print("=" * 80)

print(f"Duplicate candidate pairs: {duplicate_pairs:,}")

# ------------------------------------------------------------
# INDEXES
# ------------------------------------------------------------

print()
print("=" * 80)
print("CREATING INDEXES")
print("=" * 80)

conn.execute("""
CREATE INDEX IF NOT EXISTS idx_unresolved_source1
ON unresolved_candidates(source1_entity_id)
""")

conn.execute("""
CREATE INDEX IF NOT EXISTS idx_unresolved_candidate
ON unresolved_candidates(candidate_entity_id)
""")

conn.execute("""
CREATE INDEX IF NOT EXISTS idx_unresolved_name
ON unresolved_candidates(s1_name_norm)
""")

conn.commit()

print("✓ Indexes created")

# ------------------------------------------------------------
# CHECKPOINT
# ------------------------------------------------------------

print()
print("=" * 80)
print("CHECKPOINTING")
print("=" * 80)

state = {
    "step": 11,
    "status": "complete",
    "database": CANDIDATE_DB,
    "table": "unresolved_candidates",
    "total_candidates": int(total_candidates),
    "unique_s1": int(unique_s1),
    "unique_candidate_pairs": int(unique_candidates),
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S")
}

with open(CANDIDATE_STATE, "w") as f:
    json.dump(state, f, indent=2)

print("✓ State file updated")
print("✓ Candidate database checkpointed")

print()
print("=" * 80)
print("STEP 11 COMPLETE")
print("=" * 80)

print()
print("Created:")
print("  unresolved_candidates")
print("  unresolved_s1_counts")

print()
print("✓ High-confidence matches excluded")
print("✓ Duplicate check completed")
print("✓ Checkpoint saved")
print("✓ Ready for Step 12")

MATCHING — STEP 11 CONTINUED
VERIFY UNRESOLVED CANDIDATES


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


UNRESOLVED CANDIDATE STATISTICS
Unresolved candidate pairs : 17,952,660
Unique unresolved S1       : 946,356
Unique candidate pairs     : 17,952,660

BUILDING UNRESOLVED S1 COUNTS


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...


DUPLICATE CHECK
Duplicate candidate pairs: 0

CREATING INDEXES


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ Indexes created

CHECKPOINTING
✓ State file updated
✓ Candidate database checkpointed

STEP 11 COMPLETE

Created:
  unresolved_candidates
  unresolved_s1_counts

✓ High-confidence matches excluded
✓ Duplicate check completed
✓ Checkpoint saved
✓ Ready for Step 12


In [ ]:
# ============================================================
# MATCHING — STEP 12
# BUILD FUZZY MATCHING WORKSPACE
# ============================================================

import duckdb
import os
import json
import time

print("=" * 80)
print("MATCHING — STEP 12")
print("BUILD FUZZY MATCHING WORKSPACE")
print("=" * 80)

CANDIDATE_DB = "/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb"
CANDIDATE_STATE = "/content/drive/MyDrive/er_checkpoint/er_candidates_state.json"

conn = duckdb.connect(CANDIDATE_DB)

print("✓ Candidate database opened")

# ------------------------------------------------------------
# CHECK REQUIRED TABLE
# ------------------------------------------------------------

print()
print("=" * 80)
print("CHECKING REQUIRED TABLE")
print("=" * 80)

tables = conn.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
""").fetchdf()

existing = set(tables["table_name"].tolist())

if "unresolved_candidates" not in existing:
    raise RuntimeError(
        "unresolved_candidates table not found. "
        "Step 11 must be completed first."
    )

print("✓ unresolved_candidates found")

# ------------------------------------------------------------
# CREATE COMPACT FUZZY WORKSPACE
# ------------------------------------------------------------

print()
print("=" * 80)
print("CREATING FUZZY WORKSPACE")
print("=" * 80)

start = time.time()

conn.execute("DROP TABLE IF EXISTS fuzzy_candidates")

conn.execute("""
CREATE TABLE fuzzy_candidates AS
SELECT
    source1_entity_id,
    candidate_entity_id,

    s1_name_norm,
    candidate_name_norm,

    s1_address_norm,
    candidate_address_norm,

    s1_country_norm,
    candidate_country_norm

FROM unresolved_candidates
""")

elapsed = time.time() - start

print(f"✓ fuzzy_candidates created")
print(f"✓ Time: {elapsed:.2f} seconds")

# ------------------------------------------------------------
# VERIFY ROW COUNT
# ------------------------------------------------------------

fuzzy_count = conn.execute("""
    SELECT COUNT(*)
    FROM fuzzy_candidates
""").fetchone()[0]

unresolved_count = conn.execute("""
    SELECT COUNT(*)
    FROM unresolved_candidates
""").fetchone()[0]

print()
print("=" * 80)
print("ROW COUNT VERIFICATION")
print("=" * 80)

print(f"unresolved_candidates : {unresolved_count:,}")
print(f"fuzzy_candidates      : {fuzzy_count:,}")

if fuzzy_count != unresolved_count:
    raise RuntimeError(
        "Row count mismatch! fuzzy_candidates does not match "
        "unresolved_candidates."
    )

print("✓ Row counts match exactly")

# ------------------------------------------------------------
# CHECK NULLS
# ------------------------------------------------------------

print()
print("=" * 80)
print("NULL CHECK")
print("=" * 80)

null_stats = conn.execute("""
SELECT
    SUM(CASE WHEN s1_name_norm IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN candidate_name_norm IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN s1_address_norm IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN candidate_address_norm IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN s1_country_norm IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN candidate_country_norm IS NULL THEN 1 ELSE 0 END)
FROM fuzzy_candidates
""").fetchone()

print(f"S1 name NULL              : {null_stats[0]:,}")
print(f"Candidate name NULL       : {null_stats[1]:,}")
print(f"S1 address NULL            : {null_stats[2]:,}")
print(f"Candidate address NULL     : {null_stats[3]:,}")
print(f"S1 country NULL            : {null_stats[4]:,}")
print(f"Candidate country NULL     : {null_stats[5]:,}")

# ------------------------------------------------------------
# SAMPLE
# ------------------------------------------------------------

print()
print("=" * 80)
print("SAMPLE FUZZY CANDIDATES")
print("=" * 80)

sample = conn.execute("""
SELECT
    source1_entity_id,
    candidate_entity_id,
    s1_name_norm,
    candidate_name_norm,
    s1_address_norm,
    candidate_address_norm,
    s1_country_norm,
    candidate_country_norm
FROM fuzzy_candidates
LIMIT 10
""").fetchdf()

print(sample.to_string(index=False))

# ------------------------------------------------------------
# INDEXES
# ------------------------------------------------------------

print()
print("=" * 80)
print("CREATING INDEXES")
print("=" * 80)

conn.execute("""
CREATE INDEX IF NOT EXISTS idx_fuzzy_source1
ON fuzzy_candidates(source1_entity_id)
""")

conn.execute("""
CREATE INDEX IF NOT EXISTS idx_fuzzy_candidate
ON fuzzy_candidates(candidate_entity_id)
""")

conn.commit()

print("✓ Indexes created")

# ------------------------------------------------------------
# CHECKPOINT
# ------------------------------------------------------------

print()
print("=" * 80)
print("CHECKPOINTING")
print("=" * 80)

state = {
    "step": 12,
    "status": "workspace_complete",
    "database": CANDIDATE_DB,
    "table": "fuzzy_candidates",
    "rows": int(fuzzy_count),
    "source_table": "unresolved_candidates",
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S")
}

with open(CANDIDATE_STATE, "w") as f:
    json.dump(state, f, indent=2)

print("✓ State file updated")
print("✓ Candidate database checkpointed")

print()
print("=" * 80)
print("STEP 12 WORKSPACE COMPLETE")
print("=" * 80)

print()
print("Created:")
print("  fuzzy_candidates")

print()
print("✓ 17.95M unresolved pairs prepared")
print("✓ No rows lost")
print("✓ No duplicate pairs")
print("✓ Ready for batched RapidFuzz scoring")

MATCHING — STEP 12
BUILD FUZZY MATCHING WORKSPACE
✓ Candidate database opened

CHECKING REQUIRED TABLE
✓ unresolved_candidates found

CREATING FUZZY WORKSPACE


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ fuzzy_candidates created
✓ Time: 77.48 seconds

ROW COUNT VERIFICATION
unresolved_candidates : 17,952,660
fuzzy_candidates      : 17,952,660
✓ Row counts match exactly

NULL CHECK
S1 name NULL              : 0
Candidate name NULL       : 0
S1 address NULL            : 0
Candidate address NULL     : 0
S1 country NULL            : 0
Candidate country NULL     : 0

SAMPLE FUZZY CANDIDATES
source1_entity_id candidate_entity_id                 s1_name_norm          candidate_name_norm                                              s1_address_norm                candidate_address_norm s1_country_norm candidate_country_norm
      S1-25837265        S2-606954109              blueicecreamllc              blueicecreamllc                                           92qeastwenatcheewa                  powell465woodardploh              us                     us
     S1-306110843        S2-309055779              veteranscouncil              veteranscouncil                              newbergor22175hi

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ Indexes created

CHECKPOINTING
✓ State file updated
✓ Candidate database checkpointed

STEP 12 WORKSPACE COMPLETE

Created:
  fuzzy_candidates

✓ 17.95M unresolved pairs prepared
✓ No rows lost
✓ No duplicate pairs
✓ Ready for batched RapidFuzz scoring


In [ ]:
# ============================================================
# MATCHING — STEP 13
# RAPIDFUZZ TEST + FIRST BATCH
# ============================================================

import duckdb
import os
import sys
import time

print("=" * 80)
print("MATCHING — STEP 13")
print("RAPIDFUZZ TEST + FIRST BATCH")
print("=" * 80)

# ------------------------------------------------------------
# INSTALL / IMPORT RAPIDFUZZ
# ------------------------------------------------------------

print()
print("=" * 80)
print("CHECKING RAPIDFUZZ")
print("=" * 80)

try:
    from rapidfuzz import fuzz
    print("✓ RapidFuzz already installed")

except ImportError:
    print("RapidFuzz not found.")
    print("Installing...")

    import subprocess

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "rapidfuzz"
    ])

    from rapidfuzz import fuzz

    print("✓ RapidFuzz installed successfully")

# ------------------------------------------------------------
# CONNECT
# ------------------------------------------------------------

CANDIDATE_DB = "/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb"

conn = duckdb.connect(CANDIDATE_DB)

print()
print("✓ Candidate database opened")

# ------------------------------------------------------------
# VERIFY FUZZY TABLE
# ------------------------------------------------------------

count = conn.execute("""
    SELECT COUNT(*)
    FROM fuzzy_candidates
""").fetchone()[0]

print(f"✓ fuzzy_candidates rows: {count:,}")

# ------------------------------------------------------------
# LOAD SMALL TEST BATCH
# ------------------------------------------------------------

BATCH_SIZE = 10_000

print()
print("=" * 80)
print(f"LOADING TEST BATCH ({BATCH_SIZE:,} ROWS)")
print("=" * 80)

start = time.time()

df = conn.execute(f"""
SELECT
    source1_entity_id,
    candidate_entity_id,

    s1_name_norm,
    candidate_name_norm,

    s1_address_norm,
    candidate_address_norm,

    s1_country_norm,
    candidate_country_norm

FROM fuzzy_candidates

LIMIT {BATCH_SIZE}
""").fetchdf()

load_time = time.time() - start

print(f"✓ Loaded {len(df):,} rows")
print(f"✓ Load time: {load_time:.2f} seconds")

# ------------------------------------------------------------
# FUZZY SCORING
# ------------------------------------------------------------

print()
print("=" * 80)
print("CALCULATING FUZZY SIMILARITY")
print("=" * 80)

start = time.time()

name_scores = []
address_scores = []

for name1, name2, addr1, addr2 in zip(
    df["s1_name_norm"],
    df["candidate_name_norm"],
    df["s1_address_norm"],
    df["candidate_address_norm"]
):

    # Business-name similarity
    name_score = fuzz.ratio(
        str(name1),
        str(name2)
    )

    # Address similarity
    address_score = fuzz.ratio(
        str(addr1),
        str(addr2)
    )

    name_scores.append(name_score)
    address_scores.append(address_score)

df["name_score"] = name_scores
df["address_score"] = address_scores

score_time = time.time() - start

print(f"✓ Scored {len(df):,} pairs")
print(f"✓ Scoring time: {score_time:.2f} seconds")

# ------------------------------------------------------------
# COMBINED SCORE
# ------------------------------------------------------------

df["combined_score"] = (
    df["name_score"] * 0.65
    +
    df["address_score"] * 0.35
)

# ------------------------------------------------------------
# SCORE STATISTICS
# ------------------------------------------------------------

print()
print("=" * 80)
print("FUZZY SCORE STATISTICS")
print("=" * 80)

print(
    f"Name score     : "
    f"min={df['name_score'].min():.1f}, "
    f"median={df['name_score'].median():.1f}, "
    f"max={df['name_score'].max():.1f}"
)

print(
    f"Address score  : "
    f"min={df['address_score'].min():.1f}, "
    f"median={df['address_score'].median():.1f}, "
    f"max={df['address_score'].max():.1f}"
)

print(
    f"Combined score : "
    f"min={df['combined_score'].min():.1f}, "
    f"median={df['combined_score'].median():.1f}, "
    f"max={df['combined_score'].max():.1f}"
)

# ------------------------------------------------------------
# TOP MATCHES
# ------------------------------------------------------------

print()
print("=" * 80)
print("TOP 20 FUZZY MATCHES")
print("=" * 80)

top = df.sort_values(
    "combined_score",
    ascending=False
).head(20)

display(
    top[
        [
            "source1_entity_id",
            "candidate_entity_id",
            "s1_name_norm",
            "candidate_name_norm",
            "name_score",
            "address_score",
            "combined_score"
        ]
    ]
)

# ------------------------------------------------------------
# SCORE DISTRIBUTION
# ------------------------------------------------------------

print()
print("=" * 80)
print("COMBINED SCORE DISTRIBUTION")
print("=" * 80)

bins = [
    (0, 49),
    (50, 59),
    (60, 69),
    (70, 79),
    (80, 89),
    (90, 94),
    (95, 99),
    (100, 100)
]

for low, high in bins:

    if high == 100:
        n = ((df["combined_score"] >= low) &
             (df["combined_score"] <= high)).sum()
    else:
        n = ((df["combined_score"] >= low) &
             (df["combined_score"] < high + 1)).sum()

    print(f"{low:>3}-{high:<3}: {n:,}")

print()
print("=" * 80)
print("STEP 13 TEST COMPLETE")
print("=" * 80)

print()
print("✓ RapidFuzz works")
print("✓ 10,000 candidates successfully scored")
print("✓ No database modification performed")
print("✓ Ready to evaluate fuzzy-scoring performance")

MATCHING — STEP 13
RAPIDFUZZ TEST + FIRST BATCH

CHECKING RAPIDFUZZ
RapidFuzz not found.
Installing...
✓ RapidFuzz installed successfully

✓ Candidate database opened
✓ fuzzy_candidates rows: 17,952,660

LOADING TEST BATCH (10,000 ROWS)
✓ Loaded 10,000 rows
✓ Load time: 0.03 seconds

CALCULATING FUZZY SIMILARITY
✓ Scored 10,000 pairs
✓ Scoring time: 0.13 seconds

FUZZY SCORE STATISTICS
Name score     : min=100.0, median=100.0, max=100.0
Address score  : min=0.0, median=30.8, max=100.0
Combined score : min=65.0, median=75.8, max=100.0

TOP 20 FUZZY MATCHES


,source1_entity_id,candidate_entity_id,s1_name_norm,candidate_name_norm,name_score,address_score,combined_score
2736,S1-912229154,S2-236562425,veteranscentervllc,veteranscentervllc,100.0,100.000000,100.000000
2738,S1-555289306,S2-531369018,himadrienterprisesprivatelimited,himadrienterprisesprivatelimited,100.0,100.000000,100.000000
8808,S1-994451577,S2-716024107,harewoodospreyllc,harewoodospreyllc,100.0,100.000000,100.000000
4403,S1-948241969,S2-573033308,brightdeli,brightdeli,100.0,100.000000,100.000000
5364,S1-105320683,S2-257703023,strozierandmartinezllc,strozierandmartinezllc,100.0,100.000000,100.000000
1048,S1-923494608,S2-779837321,sapphirellc,sapphirellc,100.0,100.000000,100.000000
1453,S1-658410583,S2-55121360,rarliabilityllp,rarliabilityllp,100.0,100.000000,100.000000
1412,S1-124028550,S2-961133043,interstateguild,interstateguild,100.0,100.000000,100.000000
6485,S1-495041553,S2-867314829,alifvidyalayaprivatelimited,alifvidyalayaprivatelimited,100.0,100.000000,100.000000
8823,S1-345182581,S2-58725692,fzpsecurewisekeyllc,fzpsecurewisekeyllc,100.0,100.000000,100.000000



COMBINED SCORE DISTRIBUTION
  0-49 : 0
 50-59 : 0
 60-69 : 232
 70-79 : 9,005
 80-89 : 417
 90-94 : 109
 95-99 : 226
100-100: 11

STEP 13 TEST COMPLETE

✓ RapidFuzz works
✓ 10,000 candidates successfully scored
✓ No database modification performed
✓ Ready to evaluate fuzzy-scoring performance


In [ ]:
# ============================================================
# MATCHING — STEP 14
# BATCH FUZZY SCORING
# ============================================================

import duckdb
import os
import time
import json
import sys
import subprocess
from rapidfuzz import fuzz

print("=" * 80)
print("MATCHING — STEP 14")
print("BATCH FUZZY SCORING")
print("=" * 80)

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

CANDIDATE_DB = "/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb"
STATE_FILE = "/content/drive/MyDrive/er_checkpoint/er_candidates_state.json"

BATCH_SIZE = 100_000

# ------------------------------------------------------------
# CONNECT
# ------------------------------------------------------------

conn = duckdb.connect(CANDIDATE_DB)

print("✓ Candidate database opened")

# ------------------------------------------------------------
# VERIFY SOURCE
# ------------------------------------------------------------

total_rows = conn.execute("""
    SELECT COUNT(*)
    FROM fuzzy_candidates
""").fetchone()[0]

print(f"✓ Fuzzy candidates: {total_rows:,}")

# ------------------------------------------------------------
# CREATE SCORE TABLE
# ------------------------------------------------------------

print()
print("=" * 80)
print("PREPARING FUZZY SCORE TABLE")
print("=" * 80)

conn.execute("""
CREATE TABLE IF NOT EXISTS fuzzy_scores (
    source1_entity_id VARCHAR,
    candidate_entity_id VARCHAR,

    name_score DOUBLE,
    address_score DOUBLE,
    combined_score DOUBLE,

    score_type VARCHAR
)
""")

# ------------------------------------------------------------
# CHECK ALREADY PROCESSED ROWS
# ------------------------------------------------------------

processed = conn.execute("""
    SELECT COUNT(*)
    FROM fuzzy_scores
""").fetchone()[0]

print(f"Already scored: {processed:,}")

if processed > total_rows:
    raise RuntimeError(
        "fuzzy_scores contains more rows than fuzzy_candidates."
    )

# ------------------------------------------------------------
# CREATE TEMP ROW NUMBER WORKSPACE
# ------------------------------------------------------------

print()
print("=" * 80)
print("PREPARING BATCH WORKSPACE")
print("=" * 80)

conn.execute("DROP TABLE IF EXISTS fuzzy_batch_source")

conn.execute("""
CREATE TEMP TABLE fuzzy_batch_source AS
SELECT
    ROW_NUMBER() OVER (
        ORDER BY source1_entity_id, candidate_entity_id
    ) AS row_id,

    source1_entity_id,
    candidate_entity_id,

    s1_name_norm,
    candidate_name_norm,

    s1_address_norm,
    candidate_address_norm,

    s1_country_norm,
    candidate_country_norm

FROM fuzzy_candidates
""")

print("✓ Batch workspace ready")

# ------------------------------------------------------------
# DETERMINE START POSITION
# ------------------------------------------------------------

start_row = processed + 1

print()
print(f"Starting from row: {start_row:,}")

# ------------------------------------------------------------
# PROCESS BATCHES
# ------------------------------------------------------------

overall_start = time.time()

batch_number = ((start_row - 1) // BATCH_SIZE) + 1

while start_row <= total_rows:

    batch_start_time = time.time()

    end_row = min(
        start_row + BATCH_SIZE - 1,
        total_rows
    )

    print()
    print("=" * 80)
    print(
        f"BATCH {batch_number} "
        f"| rows {start_row:,} - {end_row:,} "
        f"of {total_rows:,}"
    )
    print("=" * 80)

    # --------------------------------------------------------
    # LOAD BATCH
    # --------------------------------------------------------

    df = conn.execute(f"""
        SELECT
            source1_entity_id,
            candidate_entity_id,

            s1_name_norm,
            candidate_name_norm,

            s1_address_norm,
            candidate_address_norm,

            s1_country_norm,
            candidate_country_norm

        FROM fuzzy_batch_source

        WHERE row_id BETWEEN {start_row} AND {end_row}

        ORDER BY row_id
    """).fetchdf()

    if len(df) == 0:
        print("No rows returned. Stopping.")
        break

    print(f"✓ Loaded {len(df):,} rows")

    # --------------------------------------------------------
    # SCORE
    # --------------------------------------------------------

    name_scores = []
    address_scores = []
    combined_scores = []
    score_types = []

    score_start = time.time()

    for name1, name2, addr1, addr2 in zip(
        df["s1_name_norm"],
        df["candidate_name_norm"],
        df["s1_address_norm"],
        df["candidate_address_norm"]
    ):

        name1 = str(name1)
        name2 = str(name2)
        addr1 = str(addr1)
        addr2 = str(addr2)

        # Name similarity
        ns = fuzz.ratio(name1, name2)

        # Address similarity
        ads = fuzz.ratio(addr1, addr2)

        # ----------------------------------------------------
        # SCORING LOGIC
        # ----------------------------------------------------
        #
        # Exact name:
        #   Address becomes the important discriminator.
        #
        # Exact address:
        #   Name becomes the important discriminator.
        #
        # Neither exact:
        #   Use balanced score.
        #

        if ns == 100 and ads < 100:
            combined = (
                ns * 0.30 +
                ads * 0.70
            )
            score_type = "exact_name_fuzzy_address"

        elif ads == 100 and ns < 100:
            combined = (
                ns * 0.70 +
                ads * 0.30
            )
            score_type = "fuzzy_name_exact_address"

        elif ns == 100 and ads == 100:
            combined = 100.0
            score_type = "exact_name_exact_address"

        else:
            combined = (
                ns * 0.60 +
                ads * 0.40
            )
            score_type = "fuzzy_name_fuzzy_address"

        name_scores.append(ns)
        address_scores.append(ads)
        combined_scores.append(combined)
        score_types.append(score_type)

    score_time = time.time() - score_start

    df["name_score"] = name_scores
    df["address_score"] = address_scores
    df["combined_score"] = combined_scores
    df["score_type"] = score_types

    print(f"✓ Scored {len(df):,} rows")
    print(f"✓ Scoring time: {score_time:.2f} sec")

    # --------------------------------------------------------
    # INSERT SCORES
    # --------------------------------------------------------

    conn.register("batch_df", df)

    conn.execute("""
        INSERT INTO fuzzy_scores
        SELECT
            source1_entity_id,
            candidate_entity_id,
            name_score,
            address_score,
            combined_score,
            score_type
        FROM batch_df
    """)

    conn.unregister("batch_df")

    conn.commit()

    # --------------------------------------------------------
    # BATCH STATISTICS
    # --------------------------------------------------------

    print()
    print("Score distribution:")

    print(
        f"  Name median    : "
        f"{df['name_score'].median():.2f}"
    )

    print(
        f"  Address median : "
        f"{df['address_score'].median():.2f}"
    )

    print(
        f"  Combined median: "
        f"{df['combined_score'].median():.2f}"
    )

    print(
        f"  Combined max   : "
        f"{df['combined_score'].max():.2f}"
    )

    # --------------------------------------------------------
    # PROGRESS
    # --------------------------------------------------------

    processed_now = conn.execute("""
        SELECT COUNT(*)
        FROM fuzzy_scores
    """).fetchone()[0]

    elapsed = time.time() - overall_start

    rate = (
        processed_now / elapsed
        if elapsed > 0
        else 0
    )

    remaining = total_rows - processed_now

    eta_seconds = (
        remaining / rate
        if rate > 0
        else 0
    )

    print()
    print(
        f"✓ Total scored      : "
        f"{processed_now:,} / {total_rows:,}"
    )

    print(
        f"✓ Progress          : "
        f"{processed_now / total_rows * 100:.2f}%"
    )

    print(
        f"✓ Processing speed  : "
        f"{rate:,.0f} rows/sec"
    )

    print(
        f"✓ Estimated remaining: "
        f"{eta_seconds / 60:.1f} minutes"
    )

    # --------------------------------------------------------
    # CHECKPOINT
    # --------------------------------------------------------

    state = {
        "step": 14,
        "status": "running",
        "database": CANDIDATE_DB,
        "table": "fuzzy_scores",
        "total_rows": int(total_rows),
        "processed_rows": int(processed_now),
        "batch_size": int(BATCH_SIZE),
        "updated_at": time.strftime(
            "%Y-%m-%d %H:%M:%S"
        )
    }

    with open(STATE_FILE, "w") as f:
        json.dump(state, f, indent=2)

    print("✓ Batch checkpoint saved")

    # --------------------------------------------------------
    # NEXT
    # --------------------------------------------------------

    start_row = end_row + 1
    batch_number += 1

    # Free batch memory
    del df

# ------------------------------------------------------------
# FINAL VERIFICATION
# ------------------------------------------------------------

final_count = conn.execute("""
    SELECT COUNT(*)
    FROM fuzzy_scores
""").fetchone()[0]

print()
print("=" * 80)
print("FINAL FUZZY SCORING VERIFICATION")
print("=" * 80)

print(f"Expected rows : {total_rows:,}")
print(f"Scored rows   : {final_count:,}")

if final_count != total_rows:
    print("⚠ Fuzzy scoring is incomplete.")
else:
    print("✓ ALL FUZZY CANDIDATES SCORED")

# ------------------------------------------------------------
# SCORE TYPE DISTRIBUTION
# ------------------------------------------------------------

print()
print("=" * 80)
print("SCORE TYPE DISTRIBUTION")
print("=" * 80)

score_type_df = conn.execute("""
SELECT
    score_type,
    COUNT(*) AS pair_count
FROM fuzzy_scores
GROUP BY score_type
ORDER BY pair_count DESC
""").fetchdf()

print(score_type_df.to_string(index=False))

# ------------------------------------------------------------
# FINAL CHECKPOINT
# ------------------------------------------------------------

if final_count == total_rows:

    state = {
        "step": 14,
        "status": "complete",
        "database": CANDIDATE_DB,
        "table": "fuzzy_scores",
        "rows": int(final_count),
        "completed_at": time.strftime(
            "%Y-%m-%d %H:%M:%S"
        )
    }

    with open(STATE_FILE, "w") as f:
        json.dump(state, f, indent=2)

    print()
    print("✓ Final checkpoint saved")

print()
print("=" * 80)
print("STEP 14 COMPLETE")
print("=" * 80)

MATCHING — STEP 14
BATCH FUZZY SCORING
✓ Candidate database opened
✓ Fuzzy candidates: 17,952,660

PREPARING FUZZY SCORE TABLE
Already scored: 0

PREPARING BATCH WORKSPACE


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ Batch workspace ready

Starting from row: 1

BATCH 1 | rows 1 - 100,000 of 17,952,660
✓ Loaded 100,000 rows
✓ Scored 100,000 rows
✓ Scoring time: 0.42 sec

Score distribution:
  Name median    : 100.00
  Address median : 30.77
  Combined median: 51.54
  Combined max   : 100.00

✓ Total scored      : 100,000 / 17,952,660
✓ Progress          : 0.56%
✓ Processing speed  : 2,242 rows/sec
✓ Estimated remaining: 132.7 minutes
✓ Batch checkpoint saved

BATCH 2 | rows 100,001 - 200,000 of 17,952,660
✓ Loaded 100,000 rows
✓ Scored 100,000 rows
✓ Scoring time: 0.26 sec

Score distribution:
  Name median    : 100.00
  Address median : 31.03
  Combined median: 51.69
  Combined max   : 100.00

✓ Total scored      : 200,000 / 17,952,660
✓ Progress          : 1.11%
✓ Processing speed  : 4,406 rows/sec
✓ Estimated remaining: 67.1 minutes
✓ Batch checkpoint saved

BATCH 3 | rows 200,001 - 300,000 of 17,952,660
✓ Loaded 100,000 rows
✓ Scored 100,000 rows
✓ Scoring time: 0.24 sec

Score distribution:
 

In [ ]:
# ============================================================
# MATCHING — STEP 15
# RANK FUZZY CANDIDATES
# ============================================================

import duckdb
import json
import time

print("=" * 80)
print("MATCHING — STEP 15")
print("RANK FUZZY CANDIDATES")
print("=" * 80)

CANDIDATE_DB = "/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb"
STATE_FILE = "/content/drive/MyDrive/er_checkpoint/er_candidates_state.json"

conn = duckdb.connect(CANDIDATE_DB)

print("✓ Candidate database opened")

# ------------------------------------------------------------
# VERIFY FUZZY SCORES
# ------------------------------------------------------------

fuzzy_count = conn.execute("""
    SELECT COUNT(*)
    FROM fuzzy_scores
""").fetchone()[0]

print(f"✓ Fuzzy scores: {fuzzy_count:,}")

if fuzzy_count != 17_952_660:
    raise RuntimeError(
        f"Expected 17,952,660 fuzzy scores, found {fuzzy_count:,}"
    )

# ------------------------------------------------------------
# CREATE RANKED TABLE
# ------------------------------------------------------------

print()
print("=" * 80)
print("RANKING CANDIDATES PER S1")
print("=" * 80)

start = time.time()

conn.execute("""
DROP TABLE IF EXISTS ranked_fuzzy_matches
""")

conn.execute("""
CREATE TABLE ranked_fuzzy_matches AS

SELECT
    source1_entity_id,
    candidate_entity_id,

    name_score,
    address_score,
    combined_score,
    score_type,

    ROW_NUMBER() OVER (
        PARTITION BY source1_entity_id
        ORDER BY
            combined_score DESC,
            name_score DESC,
            address_score DESC,
            candidate_entity_id
    ) AS match_rank

FROM fuzzy_scores
""")

elapsed = time.time() - start

print(f"✓ ranked_fuzzy_matches created")
print(f"✓ Time: {elapsed:.2f} seconds")

# ------------------------------------------------------------
# VERIFY
# ------------------------------------------------------------

ranked_count = conn.execute("""
    SELECT COUNT(*)
    FROM ranked_fuzzy_matches
""").fetchone()[0]

print()
print("=" * 80)
print("ROW COUNT VERIFICATION")
print("=" * 80)

print(f"Fuzzy scores       : {fuzzy_count:,}")
print(f"Ranked fuzzy rows  : {ranked_count:,}")

if ranked_count != fuzzy_count:
    raise RuntimeError("Row count mismatch!")

print("✓ Row counts match")

# ------------------------------------------------------------
# CREATE TOP-2 SUMMARY
# ------------------------------------------------------------

print()
print("=" * 80)
print("BUILDING TOP-2 MATCH SUMMARY")
print("=" * 80)

conn.execute("""
DROP TABLE IF EXISTS fuzzy_match_summary
""")

conn.execute("""
CREATE TABLE fuzzy_match_summary AS

SELECT
    source1_entity_id,

    MAX(
        CASE
            WHEN match_rank = 1
            THEN candidate_entity_id
        END
    ) AS best_candidate,

    MAX(
        CASE
            WHEN match_rank = 1
            THEN combined_score
        END
    ) AS best_score,

    MAX(
        CASE
            WHEN match_rank = 1
            THEN name_score
        END
    ) AS best_name_score,

    MAX(
        CASE
            WHEN match_rank = 1
            THEN address_score
        END
    ) AS best_address_score,

    MAX(
        CASE
            WHEN match_rank = 2
            THEN candidate_entity_id
        END
    ) AS second_candidate,

    MAX(
        CASE
            WHEN match_rank = 2
            THEN combined_score
        END
    ) AS second_score

FROM ranked_fuzzy_matches

GROUP BY source1_entity_id
""")

# ------------------------------------------------------------
# ADD SCORE MARGIN
# ------------------------------------------------------------

conn.execute("""
ALTER TABLE fuzzy_match_summary
ADD COLUMN score_margin DOUBLE
""")

conn.execute("""
UPDATE fuzzy_match_summary
SET score_margin =
    CASE
        WHEN second_score IS NULL
        THEN best_score
        ELSE best_score - second_score
    END
""")

conn.commit()

print("✓ fuzzy_match_summary created")

# ------------------------------------------------------------
# SUMMARY STATISTICS
# ------------------------------------------------------------

summary_stats = conn.execute("""
SELECT
    COUNT(*) AS s1_count,

    AVG(best_score) AS avg_best_score,

    MEDIAN(best_score) AS median_best_score,

    MIN(best_score) AS min_best_score,

    MAX(best_score) AS max_best_score,

    AVG(score_margin) AS avg_margin,

    MEDIAN(score_margin) AS median_margin,

    MIN(score_margin) AS min_margin,

    MAX(score_margin) AS max_margin

FROM fuzzy_match_summary
""").fetchone()

print()
print("=" * 80)
print("FUZZY MATCH SUMMARY")
print("=" * 80)

print(f"S1 records                    : {summary_stats[0]:,}")
print(f"Average best score            : {summary_stats[1]:.2f}")
print(f"Median best score             : {summary_stats[2]:.2f}")
print(f"Minimum best score            : {summary_stats[3]:.2f}")
print(f"Maximum best score            : {summary_stats[4]:.2f}")
print()
print(f"Average score margin          : {summary_stats[5]:.2f}")
print(f"Median score margin           : {summary_stats[6]:.2f}")
print(f"Minimum score margin          : {summary_stats[7]:.2f}")
print(f"Maximum score margin          : {summary_stats[8]:.2f}")

# ------------------------------------------------------------
# BEST SCORE BUCKETS
# ------------------------------------------------------------

print()
print("=" * 80)
print("BEST SCORE DISTRIBUTION")
print("=" * 80)

best_buckets = conn.execute("""
SELECT
    CASE
        WHEN best_score >= 99 THEN '99-100'
        WHEN best_score >= 95 THEN '95-99'
        WHEN best_score >= 90 THEN '90-95'
        WHEN best_score >= 85 THEN '85-90'
        WHEN best_score >= 80 THEN '80-85'
        WHEN best_score >= 70 THEN '70-80'
        WHEN best_score >= 60 THEN '60-70'
        ELSE '<60'
    END AS score_bucket,

    COUNT(*) AS s1_count

FROM fuzzy_match_summary

GROUP BY 1

ORDER BY
    CASE score_bucket
        WHEN '99-100' THEN 1
        WHEN '95-99' THEN 2
        WHEN '90-95' THEN 3
        WHEN '85-90' THEN 4
        WHEN '80-85' THEN 5
        WHEN '70-80' THEN 6
        WHEN '60-70' THEN 7
        ELSE 8
    END
""").fetchdf()

print(best_buckets.to_string(index=False))

# ------------------------------------------------------------
# MARGIN DISTRIBUTION
# ------------------------------------------------------------

print()
print("=" * 80)
print("SCORE MARGIN DISTRIBUTION")
print("=" * 80)

margin_buckets = conn.execute("""
SELECT
    CASE
        WHEN score_margin >= 20 THEN '20+'
        WHEN score_margin >= 10 THEN '10-20'
        WHEN score_margin >= 5 THEN '5-10'
        WHEN score_margin >= 2 THEN '2-5'
        WHEN score_margin >= 1 THEN '1-2'
        ELSE '<1'
    END AS margin_bucket,

    COUNT(*) AS s1_count

FROM fuzzy_match_summary

GROUP BY 1

ORDER BY
    CASE margin_bucket
        WHEN '20+' THEN 1
        WHEN '10-20' THEN 2
        WHEN '5-10' THEN 3
        WHEN '2-5' THEN 4
        WHEN '1-2' THEN 5
        ELSE 6
    END
""").fetchdf()

print(margin_buckets.to_string(index=False))

# ------------------------------------------------------------
# MOST AMBIGUOUS CASES
# ------------------------------------------------------------

print()
print("=" * 80)
print("MOST AMBIGUOUS MATCHES")
print("=" * 80)

ambiguous = conn.execute("""
SELECT
    source1_entity_id,
    best_candidate,
    best_score,
    second_candidate,
    second_score,
    score_margin,
    best_name_score,
    best_address_score
FROM fuzzy_match_summary
ORDER BY
    score_margin ASC,
    best_score DESC
LIMIT 20
""").fetchdf()

print(ambiguous.to_string(index=False))

# ------------------------------------------------------------
# TOP STRONG MATCHES
# ------------------------------------------------------------

print()
print("=" * 80)
print("STRONGEST FUZZY MATCHES")
print("=" * 80)

strong = conn.execute("""
SELECT
    source1_entity_id,
    best_candidate,
    best_score,
    second_score,
    score_margin,
    best_name_score,
    best_address_score
FROM fuzzy_match_summary
ORDER BY
    best_score DESC,
    score_margin DESC
LIMIT 20
""").fetchdf()

print(strong.to_string(index=False))

# ------------------------------------------------------------
# INDEXES
# ------------------------------------------------------------

print()
print("=" * 80)
print("CREATING INDEXES")
print("=" * 80)

conn.execute("""
CREATE INDEX IF NOT EXISTS idx_ranked_fuzzy_source1
ON ranked_fuzzy_matches(source1_entity_id)
""")

conn.execute("""
CREATE INDEX IF NOT EXISTS idx_summary_source1
ON fuzzy_match_summary(source1_entity_id)
""")

conn.execute("""
CREATE INDEX IF NOT EXISTS idx_summary_best_candidate
ON fuzzy_match_summary(best_candidate)
""")

conn.commit()

print("✓ Indexes created")

# ------------------------------------------------------------
# CHECKPOINT
# ------------------------------------------------------------

state = {
    "step": 15,
    "status": "complete",
    "database": CANDIDATE_DB,
    "tables": [
        "ranked_fuzzy_matches",
        "fuzzy_match_summary"
    ],
    "s1_records": int(summary_stats[0]),
    "created_at": time.strftime(
        "%Y-%m-%d %H:%M:%S"
    )
}

with open(STATE_FILE, "w") as f:
    json.dump(state, f, indent=2)

print("✓ State file updated")
print("✓ Candidate database checkpointed")

print()
print("=" * 80)
print("STEP 15 COMPLETE")
print("=" * 80)

print()
print("Created:")
print("  ranked_fuzzy_matches")
print("  fuzzy_match_summary")

print()
print("✓ Best candidate identified for every unresolved S1")
print("✓ Second-best candidate identified")
print("✓ Confidence margin calculated")
print("✓ Ready for confidence classification")

MATCHING — STEP 15
RANK FUZZY CANDIDATES
✓ Candidate database opened
✓ Fuzzy scores: 17,952,660

RANKING CANDIDATES PER S1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ ranked_fuzzy_matches created
✓ Time: 52.20 seconds

ROW COUNT VERIFICATION
Fuzzy scores       : 17,952,660
Ranked fuzzy rows  : 17,952,660
✓ Row counts match

BUILDING TOP-2 MATCH SUMMARY


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ fuzzy_match_summary created

FUZZY MATCH SUMMARY
S1 records                    : 946,356
Average best score            : 78.15
Median best score             : 84.04
Minimum best score            : 30.00
Maximum best score            : 100.00

Average score margin          : 12.36
Median score margin           : 5.50
Minimum score margin          : 0.00
Maximum score margin          : 69.71

BEST SCORE DISTRIBUTION
score_bucket  s1_count
      99-100     10978
       95-99    140581
       90-95    192503
       85-90    114067
       80-85     63609
       70-80     77306
       60-70    131492
         <60    215820

SCORE MARGIN DISTRIBUTION
margin_bucket  s1_count
          20+    239936
        10-20    117295
         5-10    136930
          2-5    181452
          1-2     95235
           <1    175508

MOST AMBIGUOUS MATCHES
source1_entity_id best_candidate  best_score second_candidate  second_score  score_margin  best_name_score  best_address_score
     S1-724125828   S2-5771

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ Indexes created
✓ State file updated
✓ Candidate database checkpointed

STEP 15 COMPLETE

Created:
  ranked_fuzzy_matches
  fuzzy_match_summary

✓ Best candidate identified for every unresolved S1
✓ Second-best candidate identified
✓ Confidence margin calculated
✓ Ready for confidence classification


In [ ]:
# ============================================================
# MATCHING — STEP 16
# INSPECT GROUND TRUTH FORMAT
# ============================================================

import duckdb

CANDIDATE_DB = "/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb"
SOURCE_DB = "/content/drive/MyDrive/er_checkpoint/er_train_fresh.duckdb"

conn = duckdb.connect(CANDIDATE_DB)

print("=" * 80)
print("MATCHING — STEP 16")
print("GROUND TRUTH INSPECTION")
print("=" * 80)

# ------------------------------------------------------------
# ATTACH SOURCE DATABASE
# ------------------------------------------------------------

try:
    conn.execute("DETACH DATABASE source_db")
except Exception:
    pass

conn.execute(
    f"ATTACH '{SOURCE_DB}' AS source_db (READ_ONLY)"
)

print("✓ Source database attached")

# ------------------------------------------------------------
# CHECK GROUND TRUTH TABLE
# ------------------------------------------------------------

print()
print("=" * 80)
print("GROUND TRUTH SCHEMA")
print("=" * 80)

schema = conn.execute("""
    PRAGMA table_info(source_db.train_ground_truth)
""").fetchdf()

print(schema.to_string(index=False))

# ------------------------------------------------------------
# SAMPLE
# ------------------------------------------------------------

print()
print("=" * 80)
print("GROUND TRUTH SAMPLE")
print("=" * 80)

sample = conn.execute("""
SELECT
    source1_entity_id,
    matched_entity_ids
FROM source_db.train_ground_truth
LIMIT 20
""").fetchdf()

print(sample.to_string(index=False))

# ------------------------------------------------------------
# COUNT
# ------------------------------------------------------------

count = conn.execute("""
SELECT COUNT(*)
FROM source_db.train_ground_truth
""").fetchone()[0]

print()
print("=" * 80)
print("GROUND TRUTH COUNT")
print("=" * 80)

print(f"Ground-truth rows: {count:,}")

print()
print("=" * 80)
print("STEP 16 INSPECTION COMPLETE")
print("=" * 80)

print()
print("✓ No matching scores modified")
print("✓ No candidate tables modified")
print("✓ Ground-truth format inspected")

MATCHING — STEP 16
GROUND TRUTH INSPECTION
✓ Source database attached

GROUND TRUTH SCHEMA
 cid               name    type  notnull dflt_value    pk
   0  source1_entity_id VARCHAR    False       None False
   1 matched_entity_ids VARCHAR    False       None False

GROUND TRUTH SAMPLE
source1_entity_id                                                                        matched_entity_ids
      S1-85939751             S2-414041380,S2-324599416,S2-190847807,S2-400062645,S3-124958878,S3-642931674
     S1-638981683             S2-922247107,S3-205977453,S3-158862194,S3-573890630,S3-392084374,S3-707741256
     S1-832807052                                       S2-622713906,S2-397298716,S2-592961179,S3-475641609
     S1-534445545                           S2-289004688,S2-132351593,S2-18576064,S2-370146438,S3-794331824
     S1-640729105                                                                              S2-702729959
     S1-789555951                                                 

In [ ]:
# ============================================================
# MATCHING — STEP 17
# GROUND-TRUTH EVALUATION OF FUZZY MATCHES
# ============================================================

import duckdb
import pandas as pd

CANDIDATE_DB = "/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb"
SOURCE_DB = "/content/drive/MyDrive/er_checkpoint/er_train_fresh.duckdb"

conn = duckdb.connect(CANDIDATE_DB)

print("=" * 80)
print("MATCHING — STEP 17")
print("GROUND-TRUTH EVALUATION")
print("=" * 80)

# ------------------------------------------------------------
# ATTACH SOURCE DATABASE
# ------------------------------------------------------------

try:
    conn.execute("DETACH DATABASE source_db")
except Exception:
    pass

conn.execute(
    f"ATTACH '{SOURCE_DB}' AS source_db (READ_ONLY)"
)

print("✓ Source database attached")


# ------------------------------------------------------------
# 1. CREATE PARSED GROUND TRUTH
# ------------------------------------------------------------

print()
print("=" * 80)
print("1. PARSING GROUND TRUTH")
print("=" * 80)

conn.execute("""
CREATE OR REPLACE TEMP TABLE ground_truth_pairs AS
SELECT
    source1_entity_id,
    TRIM(matched_entity_id) AS matched_entity_id
FROM (
    SELECT
        source1_entity_id,
        UNNEST(
            STRING_SPLIT(
                COALESCE(matched_entity_ids, ''),
                ','
            )
        ) AS matched_entity_id
    FROM source_db.train_ground_truth
)
WHERE TRIM(matched_entity_id) <> ''
""")

gt_pairs = conn.execute("""
SELECT COUNT(*)
FROM ground_truth_pairs
""").fetchone()[0]

gt_s1 = conn.execute("""
SELECT COUNT(DISTINCT source1_entity_id)
FROM ground_truth_pairs
""").fetchone()[0]

print(f"Ground-truth pairs:       {gt_pairs:,}")
print(f"S1 records with matches:  {gt_s1:,}")


# ------------------------------------------------------------
# 2. CHECK GROUND TRUTH MATCH COUNT DISTRIBUTION
# ------------------------------------------------------------

print()
print("=" * 80)
print("2. GROUND-TRUTH MATCH COUNT DISTRIBUTION")
print("=" * 80)

gt_counts = conn.execute("""
SELECT
    source1_entity_id,
    COUNT(*) AS true_match_count
FROM ground_truth_pairs
GROUP BY source1_entity_id
""")

gt_counts_df = gt_counts.fetchdf()

print(gt_counts_df["true_match_count"].describe())

print()
print("Distribution:")

distribution = conn.execute("""
SELECT
    CASE
        WHEN cnt = 1 THEN '1'
        WHEN cnt BETWEEN 2 AND 5 THEN '2-5'
        WHEN cnt BETWEEN 6 AND 10 THEN '6-10'
        WHEN cnt BETWEEN 11 AND 25 THEN '11-25'
        WHEN cnt BETWEEN 26 AND 100 THEN '26-100'
        ELSE '100+'
    END AS bucket,
    COUNT(*) AS s1_count
FROM (
    SELECT
        source1_entity_id,
        COUNT(*) AS cnt
    FROM ground_truth_pairs
    GROUP BY source1_entity_id
)
GROUP BY bucket
ORDER BY
    CASE bucket
        WHEN '1' THEN 1
        WHEN '2-5' THEN 2
        WHEN '6-10' THEN 3
        WHEN '11-25' THEN 4
        WHEN '26-100' THEN 5
        ELSE 6
    END
""").fetchdf()

print(distribution.to_string(index=False))


# ------------------------------------------------------------
# 3. EVALUATE BEST FUZZY CANDIDATE
# ------------------------------------------------------------

print()
print("=" * 80)
print("3. BEST FUZZY CANDIDATE ACCURACY")
print("=" * 80)

conn.execute("""
CREATE OR REPLACE TEMP TABLE fuzzy_best_eval AS
SELECT
    f.source1_entity_id,
    f.best_candidate_entity_id,
    f.best_score,
    f.best_name_score,
    f.best_address_score,
    f.second_candidate_entity_id,
    f.second_score,
    f.score_margin,

    CASE
        WHEN g.matched_entity_id IS NOT NULL THEN 1
        ELSE 0
    END AS best_is_correct

FROM fuzzy_match_summary f

LEFT JOIN ground_truth_pairs g
    ON f.source1_entity_id = g.source1_entity_id
   AND f.best_candidate_entity_id = g.matched_entity_id
""")

# Important:
# The LEFT JOIN above can produce duplicate rows when an S1 has
# multiple ground-truth matches.
#
# Collapse it to exactly one row per S1.

conn.execute("""
CREATE OR REPLACE TEMP TABLE fuzzy_best_eval_one AS
SELECT
    source1_entity_id,
    MAX(best_is_correct) AS best_is_correct,
    MAX(best_score) AS best_score,
    MAX(best_name_score) AS best_name_score,
    MAX(best_address_score) AS best_address_score,
    MAX(score_margin) AS score_margin,
    MAX(second_score) AS second_score
FROM fuzzy_best_eval
GROUP BY source1_entity_id
""")

eval_stats = conn.execute("""
SELECT
    COUNT(*) AS unresolved_s1,
    SUM(best_is_correct) AS correct_best,
    ROUND(
        100.0 * SUM(best_is_correct) / COUNT(*),
        2
    ) AS best_accuracy_pct
FROM fuzzy_best_eval_one
""").fetchdf()

print(eval_stats.to_string(index=False))


# ------------------------------------------------------------
# 4. ACCURACY BY BEST SCORE
# ------------------------------------------------------------

print()
print("=" * 80)
print("4. ACCURACY BY BEST SCORE")
print("=" * 80)

score_eval = conn.execute("""
SELECT
    CASE
        WHEN best_score >= 99 THEN '99-100'
        WHEN best_score >= 95 THEN '95-99'
        WHEN best_score >= 90 THEN '90-95'
        WHEN best_score >= 85 THEN '85-90'
        WHEN best_score >= 80 THEN '80-85'
        WHEN best_score >= 70 THEN '70-80'
        WHEN best_score >= 60 THEN '60-70'
        ELSE '<60'
    END AS score_bucket,

    COUNT(*) AS s1_count,

    SUM(best_is_correct) AS correct_count,

    ROUND(
        100.0 * SUM(best_is_correct) / COUNT(*),
        2
    ) AS accuracy_pct

FROM fuzzy_best_eval_one

GROUP BY score_bucket

ORDER BY
    CASE score_bucket
        WHEN '99-100' THEN 1
        WHEN '95-99' THEN 2
        WHEN '90-95' THEN 3
        WHEN '85-90' THEN 4
        WHEN '80-85' THEN 5
        WHEN '70-80' THEN 6
        WHEN '60-70' THEN 7
        ELSE 8
    END
""").fetchdf()

print(score_eval.to_string(index=False))


# ------------------------------------------------------------
# 5. ACCURACY BY SCORE MARGIN
# ------------------------------------------------------------

print()
print("=" * 80)
print("5. ACCURACY BY SCORE MARGIN")
print("=" * 80)

margin_eval = conn.execute("""
SELECT
    CASE
        WHEN score_margin >= 20 THEN '20+'
        WHEN score_margin >= 10 THEN '10-20'
        WHEN score_margin >= 5 THEN '5-10'
        WHEN score_margin >= 2 THEN '2-5'
        WHEN score_margin >= 1 THEN '1-2'
        ELSE '<1'
    END AS margin_bucket,

    COUNT(*) AS s1_count,

    SUM(best_is_correct) AS correct_count,

    ROUND(
        100.0 * SUM(best_is_correct) / COUNT(*),
        2
    ) AS accuracy_pct

FROM fuzzy_best_eval_one

GROUP BY margin_bucket

ORDER BY
    CASE margin_bucket
        WHEN '20+' THEN 1
        WHEN '10-20' THEN 2
        WHEN '5-10' THEN 3
        WHEN '2-5' THEN 4
        WHEN '1-2' THEN 5
        ELSE 6
    END
""").fetchdf()

print(margin_eval.to_string(index=False))


# ------------------------------------------------------------
# 6. COMBINED SCORE + MARGIN
# ------------------------------------------------------------

print()
print("=" * 80)
print("6. HIGH-CONFIDENCE REGION ANALYSIS")
print("=" * 80)

combined_eval = conn.execute("""
SELECT
    COUNT(*) AS s1_count,
    SUM(best_is_correct) AS correct_count,
    ROUND(
        100.0 * SUM(best_is_correct) / COUNT(*),
        2
    ) AS accuracy_pct
FROM fuzzy_best_eval_one
WHERE best_score >= 90
  AND score_margin >= 5
""").fetchdf()

print("Condition: best_score >= 90 AND margin >= 5")
print(combined_eval.to_string(index=False))


combined_eval2 = conn.execute("""
SELECT
    COUNT(*) AS s1_count,
    SUM(best_is_correct) AS correct_count,
    ROUND(
        100.0 * SUM(best_is_correct) / COUNT(*),
        2
    ) AS accuracy_pct
FROM fuzzy_best_eval_one
WHERE best_score >= 95
  AND score_margin >= 10
""").fetchdf()

print()
print("Condition: best_score >= 95 AND margin >= 10")
print(combined_eval2.to_string(index=False))


# ------------------------------------------------------------
# 7. TIED BEST SCORES
# ------------------------------------------------------------

print()
print("=" * 80)
print("7. TIED / AMBIGUOUS BEST SCORES")
print("=" * 80)

tie_eval = conn.execute("""
SELECT
    COUNT(*) AS tied_s1,
    SUM(best_is_correct) AS correct_best,
    ROUND(
        100.0 * SUM(best_is_correct) / COUNT(*),
        2
    ) AS accuracy_pct
FROM fuzzy_best_eval_one
WHERE score_margin = 0
""").fetchdf()

print(tie_eval.to_string(index=False))


# ------------------------------------------------------------
# 8. OVERALL COVERAGE
# ------------------------------------------------------------

print()
print("=" * 80)
print("8. OVERALL TRAINING COVERAGE")
print("=" * 80)

total_s1 = conn.execute("""
SELECT COUNT(*)
FROM source_db.train_source1
""").fetchone()[0]

gt_available = conn.execute("""
SELECT COUNT(*)
FROM source_db.train_ground_truth
WHERE matched_entity_ids IS NOT NULL
  AND TRIM(matched_entity_ids) <> ''
""").fetchone()[0]

print(f"Total S1 records:                 {total_s1:,}")
print(f"S1 records with ground truth:     {gt_available:,}")
print(f"S1 records evaluated by fuzzy:    {len(fuzzy_best_eval_one_df := conn.execute('SELECT * FROM fuzzy_best_eval_one').fetchdf()):,}")


print()
print("=" * 80)
print("STEP 17 COMPLETE")
print("=" * 80)

print()
print("✓ Ground truth parsed")
print("✓ Fuzzy best candidates evaluated")
print("✓ Score accuracy measured")
print("✓ Margin accuracy measured")
print("✓ Tie behavior measured")
print()
print("IMPORTANT:")
print("Do NOT choose final thresholds yet.")
print("Use these results to calibrate the matching rules.")

MATCHING — STEP 17
GROUND-TRUTH EVALUATION
✓ Source database attached

1. PARSING GROUND TRUTH


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Ground-truth pairs:       7,638,365
S1 records with matches:  2,083,574

2. GROUND-TRUTH MATCH COUNT DISTRIBUTION
count    2.083574e+06
mean     3.665992e+00
std      1.526294e+00
min      1.000000e+00
25%      3.000000e+00
50%      4.000000e+00
75%      5.000000e+00
max      1.100000e+01
Name: true_match_count, dtype: float64

Distribution:
bucket  s1_count
     1    119157
   2-5   1712125
  6-10    252255
 11-25        37

3. BEST FUZZY CANDIDATE ACCURACY


BinderException: Binder Error: Table "f" does not have a column named "best_candidate_entity_id"

Candidate bindings: : "best_candidate"

LINE 22:    AND f.best_candidate_entity_id = g.matched_entity_id
                ^

In [ ]:
# ============================================================
# MATCHING — STEP 17A
# FIX FUZZY GROUND-TRUTH EVALUATION
# ============================================================

print("=" * 80)
print("MATCHING — STEP 17A")
print("GROUND-TRUTH EVALUATION — FIXED")
print("=" * 80)


# ------------------------------------------------------------
# 1. VERIFY FUZZY SUMMARY COLUMNS
# ------------------------------------------------------------

print()
print("=" * 80)
print("1. FUZZY MATCH SUMMARY COLUMNS")
print("=" * 80)

summary_schema = conn.execute("""
    DESCRIBE fuzzy_match_summary
""").fetchdf()

print(summary_schema.to_string(index=False))


# ------------------------------------------------------------
# 2. CREATE EVALUATION TABLE
# ------------------------------------------------------------

print()
print("=" * 80)
print("2. EVALUATING BEST FUZZY CANDIDATE")
print("=" * 80)

conn.execute("""
CREATE OR REPLACE TEMP TABLE fuzzy_best_eval AS

SELECT
    f.source1_entity_id,

    f.best_candidate,
    f.best_score,
    f.best_name_score,
    f.best_address_score,

    f.second_candidate,
    f.second_score,
    f.score_margin,

    CASE
        WHEN g.matched_entity_id IS NOT NULL
        THEN 1
        ELSE 0
    END AS best_is_correct

FROM fuzzy_match_summary f

LEFT JOIN ground_truth_pairs g
    ON f.source1_entity_id = g.source1_entity_id
   AND f.best_candidate = g.matched_entity_id
""")


# ------------------------------------------------------------
# 3. COLLAPSE TO ONE ROW PER S1
# ------------------------------------------------------------

conn.execute("""
CREATE OR REPLACE TEMP TABLE fuzzy_best_eval_one AS

SELECT
    source1_entity_id,

    MAX(best_is_correct) AS best_is_correct,

    MAX(best_score) AS best_score,
    MAX(best_name_score) AS best_name_score,
    MAX(best_address_score) AS best_address_score,

    MAX(score_margin) AS score_margin,
    MAX(second_score) AS second_score

FROM fuzzy_best_eval

GROUP BY source1_entity_id
""")


# ------------------------------------------------------------
# 4. OVERALL ACCURACY
# ------------------------------------------------------------

print()
print("=" * 80)
print("3. OVERALL BEST-CANDIDATE ACCURACY")
print("=" * 80)

overall = conn.execute("""
SELECT
    COUNT(*) AS evaluated_s1,
    SUM(best_is_correct) AS correct_best,
    COUNT(*) - SUM(best_is_correct) AS incorrect_best,

    ROUND(
        100.0 * SUM(best_is_correct) / COUNT(*),
        2
    ) AS accuracy_pct

FROM fuzzy_best_eval_one
""").fetchdf()

print(overall.to_string(index=False))


# ------------------------------------------------------------
# 5. ACCURACY BY BEST SCORE
# ------------------------------------------------------------

print()
print("=" * 80)
print("4. ACCURACY BY BEST SCORE")
print("=" * 80)

score_eval = conn.execute("""
SELECT

    CASE
        WHEN best_score >= 99 THEN '99-100'
        WHEN best_score >= 95 THEN '95-99'
        WHEN best_score >= 90 THEN '90-95'
        WHEN best_score >= 85 THEN '85-90'
        WHEN best_score >= 80 THEN '80-85'
        WHEN best_score >= 70 THEN '70-80'
        WHEN best_score >= 60 THEN '60-70'
        ELSE '<60'
    END AS score_bucket,

    COUNT(*) AS s1_count,

    SUM(best_is_correct) AS correct_count,

    COUNT(*) - SUM(best_is_correct) AS incorrect_count,

    ROUND(
        100.0 * SUM(best_is_correct) / COUNT(*),
        2
    ) AS accuracy_pct

FROM fuzzy_best_eval_one

GROUP BY score_bucket

ORDER BY
    CASE score_bucket
        WHEN '99-100' THEN 1
        WHEN '95-99' THEN 2
        WHEN '90-95' THEN 3
        WHEN '85-90' THEN 4
        WHEN '80-85' THEN 5
        WHEN '70-80' THEN 6
        WHEN '60-70' THEN 7
        ELSE 8
    END
""").fetchdf()

print(score_eval.to_string(index=False))


# ------------------------------------------------------------
# 6. ACCURACY BY SCORE MARGIN
# ------------------------------------------------------------

print()
print("=" * 80)
print("5. ACCURACY BY SCORE MARGIN")
print("=" * 80)

margin_eval = conn.execute("""
SELECT

    CASE
        WHEN score_margin >= 20 THEN '20+'
        WHEN score_margin >= 10 THEN '10-20'
        WHEN score_margin >= 5 THEN '5-10'
        WHEN score_margin >= 2 THEN '2-5'
        WHEN score_margin >= 1 THEN '1-2'
        ELSE '<1'
    END AS margin_bucket,

    COUNT(*) AS s1_count,

    SUM(best_is_correct) AS correct_count,

    COUNT(*) - SUM(best_is_correct) AS incorrect_count,

    ROUND(
        100.0 * SUM(best_is_correct) / COUNT(*),
        2
    ) AS accuracy_pct

FROM fuzzy_best_eval_one

GROUP BY margin_bucket

ORDER BY
    CASE margin_bucket
        WHEN '20+' THEN 1
        WHEN '10-20' THEN 2
        WHEN '5-10' THEN 3
        WHEN '2-5' THEN 4
        WHEN '1-2' THEN 5
        ELSE 6
    END
""").fetchdf()

print(margin_eval.to_string(index=False))


# ------------------------------------------------------------
# 7. HIGH-CONFIDENCE COMBINATIONS
# ------------------------------------------------------------

print()
print("=" * 80)
print("6. COMBINED SCORE + MARGIN")
print("=" * 80)

conditions = [
    ("score >= 95 AND margin >= 10",
     "best_score >= 95 AND score_margin >= 10"),

    ("score >= 95 AND margin >= 5",
     "best_score >= 95 AND score_margin >= 5"),

    ("score >= 90 AND margin >= 10",
     "best_score >= 90 AND score_margin >= 10"),

    ("score >= 90 AND margin >= 5",
     "best_score >= 90 AND score_margin >= 5"),

    ("score >= 85 AND margin >= 10",
     "best_score >= 85 AND score_margin >= 10"),

    ("score >= 85 AND margin >= 5",
     "best_score >= 85 AND score_margin >= 5"),

    ("score >= 80 AND margin >= 10",
     "best_score >= 80 AND score_margin >= 10"),
]

for label, condition in conditions:

    result = conn.execute(f"""
        SELECT
            COUNT(*) AS s1_count,
            SUM(best_is_correct) AS correct_count,
            COUNT(*) - SUM(best_is_correct) AS incorrect_count,

            ROUND(
                100.0 * SUM(best_is_correct) / COUNT(*),
                2
            ) AS accuracy_pct

        FROM fuzzy_best_eval_one

        WHERE {condition}
    """).fetchone()

    print(
        f"{label:35s} | "
        f"S1={result[0]:>9,} | "
        f"Correct={result[1]:>9,} | "
        f"Accuracy={result[3]:>6.2f}%"
    )


# ------------------------------------------------------------
# 8. TIED BEST SCORES
# ------------------------------------------------------------

print()
print("=" * 80)
print("7. TIED BEST SCORES")
print("=" * 80)

tie_eval = conn.execute("""
SELECT

    COUNT(*) AS tied_s1,

    SUM(best_is_correct) AS correct_count,

    COUNT(*) - SUM(best_is_correct) AS incorrect_count,

    ROUND(
        100.0 * SUM(best_is_correct) / COUNT(*),
        2
    ) AS accuracy_pct

FROM fuzzy_best_eval_one

WHERE score_margin = 0
""").fetchdf()

print(tie_eval.to_string(index=False))


# ------------------------------------------------------------
# 9. VERY HIGH SCORE BUT LOW MARGIN
# ------------------------------------------------------------

print()
print("=" * 80)
print("8. HIGH SCORE + LOW MARGIN")
print("=" * 80)

high_score_low_margin = conn.execute("""
SELECT

    CASE
        WHEN score_margin = 0 THEN 'margin = 0'
        WHEN score_margin < 1 THEN 'margin < 1'
        WHEN score_margin < 2 THEN 'margin 1-2'
        WHEN score_margin < 5 THEN 'margin 2-5'
        ELSE 'margin >= 5'
    END AS margin_bucket,

    COUNT(*) AS s1_count,

    SUM(best_is_correct) AS correct_count,

    ROUND(
        100.0 * SUM(best_is_correct) / COUNT(*),
        2
    ) AS accuracy_pct

FROM fuzzy_best_eval_one

WHERE best_score >= 95

GROUP BY margin_bucket

ORDER BY
    CASE margin_bucket
        WHEN 'margin = 0' THEN 1
        WHEN 'margin < 1' THEN 2
        WHEN 'margin 1-2' THEN 3
        WHEN 'margin 2-5' THEN 4
        ELSE 5
    END
""").fetchdf()

print(high_score_low_margin.to_string(index=False))


# ------------------------------------------------------------
# 10. FINAL
# ------------------------------------------------------------

print()
print("=" * 80)
print("STEP 17A COMPLETE")
print("=" * 80)

print()
print("✓ Ground truth already parsed")
print("✓ Best fuzzy candidates evaluated")
print("✓ Accuracy by score calculated")
print("✓ Accuracy by margin calculated")
print("✓ Score + margin combinations tested")
print("✓ Ties evaluated")

print()
print("IMPORTANT:")
print("Do NOT select a final threshold yet.")
print("Paste the output here so we can calibrate the final rules.")

MATCHING — STEP 17A
GROUND-TRUTH EVALUATION — FIXED

1. FUZZY MATCH SUMMARY COLUMNS
       column_name column_type null  key default extra
 source1_entity_id     VARCHAR  YES None    None  None
    best_candidate     VARCHAR  YES None    None  None
        best_score      DOUBLE  YES None    None  None
   best_name_score      DOUBLE  YES None    None  None
best_address_score      DOUBLE  YES None    None  None
  second_candidate     VARCHAR  YES None    None  None
      second_score      DOUBLE  YES None    None  None
      score_margin      DOUBLE  YES None    None  None

2. EVALUATING BEST FUZZY CANDIDATE


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


3. OVERALL BEST-CANDIDATE ACCURACY
 evaluated_s1  correct_best  incorrect_best  accuracy_pct
       946356      639674.0        306682.0         67.59

4. ACCURACY BY BEST SCORE
score_bucket  s1_count  correct_count  incorrect_count  accuracy_pct
      99-100     10978        10978.0              0.0        100.00
       95-99    140581       140225.0            356.0         99.75
       90-95    192503       191938.0            565.0         99.71
       85-90    114067       113621.0            446.0         99.61
       80-85     63609        63118.0            491.0         99.23
       70-80     77306        71068.0           6238.0         91.93
       60-70    131492        41300.0          90192.0         31.41
         <60    215820         7426.0         208394.0          3.44

5. ACCURACY BY SCORE MARGIN
margin_bucket  s1_count  correct_count  incorrect_count  accuracy_pct
          20+    239936       233667.0           6269.0         97.39
        10-20    117295       1

In [ ]:
# ============================================================
# MATCHING — STEP 18
# CONFIDENCE TIER CALIBRATION
# ============================================================

print("=" * 80)
print("MATCHING — STEP 18")
print("CONFIDENCE TIER CALIBRATION")
print("=" * 80)


# ------------------------------------------------------------
# HELPER FUNCTION
# ------------------------------------------------------------

def evaluate_rule(label, condition):

    result = conn.execute(f"""
        SELECT
            COUNT(*) AS selected_s1,

            SUM(best_is_correct) AS correct_s1,

            COUNT(*) - SUM(best_is_correct) AS incorrect_s1,

            ROUND(
                100.0 * SUM(best_is_correct) / COUNT(*),
                2
            ) AS accuracy_pct

        FROM fuzzy_best_eval_one

        WHERE {condition}
    """).fetchone()

    total = conn.execute("""
        SELECT COUNT(*)
        FROM fuzzy_best_eval_one
    """).fetchone()[0]

    selected = result[0]

    coverage = (
        100.0 * selected / total
        if total > 0 else 0
    )

    print(
        f"{label:45s} | "
        f"Selected={selected:>9,} | "
        f"Coverage={coverage:>6.2f}% | "
        f"Correct={result[1]:>9,} | "
        f"Accuracy={result[3]:>6.2f}%"
    )


# ------------------------------------------------------------
# 1. SCORE-ONLY RULES
# ------------------------------------------------------------

print()
print("=" * 80)
print("1. SCORE-ONLY RULES")
print("=" * 80)

evaluate_rule(
    "Score >= 95",
    "best_score >= 95"
)

evaluate_rule(
    "Score >= 90",
    "best_score >= 90"
)

evaluate_rule(
    "Score >= 85",
    "best_score >= 85"
)

evaluate_rule(
    "Score >= 80",
    "best_score >= 80"
)

evaluate_rule(
    "Score >= 75",
    "best_score >= 75"
)

evaluate_rule(
    "Score >= 70",
    "best_score >= 70"
)


# ------------------------------------------------------------
# 2. SCORE + MARGIN
# ------------------------------------------------------------

print()
print("=" * 80)
print("2. SCORE + MARGIN RULES")
print("=" * 80)

evaluate_rule(
    "Score >= 95 AND margin >= 10",
    "best_score >= 95 AND score_margin >= 10"
)

evaluate_rule(
    "Score >= 95 AND margin >= 5",
    "best_score >= 95 AND score_margin >= 5"
)

evaluate_rule(
    "Score >= 90 AND margin >= 10",
    "best_score >= 90 AND score_margin >= 10"
)

evaluate_rule(
    "Score >= 90 AND margin >= 5",
    "best_score >= 90 AND score_margin >= 5"
)

evaluate_rule(
    "Score >= 85 AND margin >= 10",
    "best_score >= 85 AND score_margin >= 10"
)

evaluate_rule(
    "Score >= 85 AND margin >= 5",
    "best_score >= 85 AND score_margin >= 5"
)

evaluate_rule(
    "Score >= 80 AND margin >= 10",
    "best_score >= 80 AND score_margin >= 10"
)

evaluate_rule(
    "Score >= 80 AND margin >= 5",
    "best_score >= 80 AND score_margin >= 5"
)


# ------------------------------------------------------------
# 3. VERY HIGH CONFIDENCE
# ------------------------------------------------------------

print()
print("=" * 80)
print("3. VERY HIGH CONFIDENCE")
print("=" * 80)

evaluate_rule(
    "Score >= 95",
    "best_score >= 95"
)

evaluate_rule(
    "Score >= 90 AND margin >= 5",
    "best_score >= 90 AND score_margin >= 5"
)


# ------------------------------------------------------------
# 4. MEDIUM CONFIDENCE
# ------------------------------------------------------------

print()
print("=" * 80)
print("4. MEDIUM CONFIDENCE")
print("=" * 80)

evaluate_rule(
    "80 <= score < 90 AND margin >= 5",
    "best_score >= 80 AND best_score < 90 AND score_margin >= 5"
)

evaluate_rule(
    "80 <= score < 90 AND margin >= 10",
    "best_score >= 80 AND best_score < 90 AND score_margin >= 10"
)

evaluate_rule(
    "70 <= score < 80 AND margin >= 10",
    "best_score >= 70 AND best_score < 80 AND score_margin >= 10"
)

evaluate_rule(
    "70 <= score < 80 AND margin >= 20",
    "best_score >= 70 AND best_score < 80 AND score_margin >= 20"
)


# ------------------------------------------------------------
# 5. LOW CONFIDENCE
# ------------------------------------------------------------

print()
print("=" * 80)
print("5. LOW CONFIDENCE")
print("=" * 80)

evaluate_rule(
    "Score < 70",
    "best_score < 70"
)

evaluate_rule(
    "Score < 80",
    "best_score < 80"
)


# ------------------------------------------------------------
# 6. SCORE/MARGIN CROSS-TABLE
# ------------------------------------------------------------

print()
print("=" * 80)
print("6. SCORE × MARGIN ACCURACY")
print("=" * 80)

cross = conn.execute("""
SELECT

    CASE
        WHEN best_score >= 95 THEN '95+'
        WHEN best_score >= 90 THEN '90-95'
        WHEN best_score >= 85 THEN '85-90'
        WHEN best_score >= 80 THEN '80-85'
        WHEN best_score >= 70 THEN '70-80'
        ELSE '<70'
    END AS score_band,

    CASE
        WHEN score_margin >= 10 THEN '10+'
        WHEN score_margin >= 5 THEN '5-10'
        WHEN score_margin >= 2 THEN '2-5'
        WHEN score_margin >= 1 THEN '1-2'
        ELSE '<1'
    END AS margin_band,

    COUNT(*) AS s1_count,

    SUM(best_is_correct) AS correct_count,

    ROUND(
        100.0 * SUM(best_is_correct) / COUNT(*),
        2
    ) AS accuracy_pct

FROM fuzzy_best_eval_one

GROUP BY
    score_band,
    margin_band

ORDER BY
    CASE score_band
        WHEN '95+' THEN 1
        WHEN '90-95' THEN 2
        WHEN '85-90' THEN 3
        WHEN '80-85' THEN 4
        WHEN '70-80' THEN 5
        ELSE 6
    END,

    CASE margin_band
        WHEN '10+' THEN 1
        WHEN '5-10' THEN 2
        WHEN '2-5' THEN 3
        WHEN '1-2' THEN 4
        ELSE 5
    END
""").fetchdf()

print(cross.to_string(index=False))


# ------------------------------------------------------------
# 7. FINAL SUMMARY
# ------------------------------------------------------------

print()
print("=" * 80)
print("STEP 18 COMPLETE")
print("=" * 80)

print()
print("✓ Score thresholds evaluated")
print("✓ Margin thresholds evaluated")
print("✓ Coverage calculated")
print("✓ Score × margin cross-table calculated")

print()
print("Paste the complete output here.")
print("We will use it to design the final confidence tiers.")

MATCHING — STEP 18
CONFIDENCE TIER CALIBRATION

1. SCORE-ONLY RULES
Score >= 95                                   | Selected=  151,559 | Coverage= 16.02% | Correct=  151,203 | Accuracy= 99.77%
Score >= 90                                   | Selected=  344,062 | Coverage= 36.36% | Correct=  343,141 | Accuracy= 99.73%
Score >= 85                                   | Selected=  458,129 | Coverage= 48.41% | Correct=  456,762 | Accuracy= 99.70%
Score >= 80                                   | Selected=  521,738 | Coverage= 55.13% | Correct=  519,880 | Accuracy= 99.64%
Score >= 75                                   | Selected=  561,660 | Coverage= 59.35% | Correct=  558,519 | Accuracy= 99.44%
Score >= 70                                   | Selected=  599,044 | Coverage= 63.30% | Correct=  590,948 | Accuracy= 98.65%

2. SCORE + MARGIN RULES
Score >= 95 AND margin >= 10                  | Selected=   71,225 | Coverage=  7.53% | Correct=   71,044 | Accuracy= 99.75%
Score >= 95 AND margin >= 5     

In [ ]:
# ============================================================
# MATCHING — STEP 19
# CREATE CALIBRATED FUZZY CONFIDENCE TIERS
# ============================================================

print("=" * 80)
print("MATCHING — STEP 19")
print("CALIBRATED FUZZY CONFIDENCE TIERS")
print("=" * 80)


# ------------------------------------------------------------
# 1. CREATE CALIBRATED MATCH TABLE
# ------------------------------------------------------------

print()
print("=" * 80)
print("1. CREATING CALIBRATED FUZZY MATCHES")
print("=" * 80)

conn.execute("""
CREATE OR REPLACE TABLE calibrated_fuzzy_matches AS

SELECT

    source1_entity_id,

    best_candidate AS candidate_entity_id,

    best_score,

    best_name_score,

    best_address_score,

    second_candidate,

    second_score,

    score_margin,

    CASE

        WHEN best_score >= 85
            THEN 'A_VERY_HIGH'

        WHEN best_score >= 80
            THEN 'B_HIGH'

        WHEN best_score >= 70
             AND score_margin >= 20
            THEN 'C_MODERATE'

        ELSE 'UNRESOLVED'

    END AS confidence_tier

FROM fuzzy_match_summary
""")


# ------------------------------------------------------------
# 2. TIER DISTRIBUTION
# ------------------------------------------------------------

print()
print("=" * 80)
print("2. CONFIDENCE TIER DISTRIBUTION")
print("=" * 80)

tier_stats = conn.execute("""
SELECT

    confidence_tier,

    COUNT(*) AS s1_count,

    ROUND(
        100.0 * COUNT(*) /
        SUM(COUNT(*)) OVER (),
        2
    ) AS pct_of_unresolved

FROM calibrated_fuzzy_matches

GROUP BY confidence_tier

ORDER BY
    CASE confidence_tier
        WHEN 'A_VERY_HIGH' THEN 1
        WHEN 'B_HIGH' THEN 2
        WHEN 'C_MODERATE' THEN 3
        ELSE 4
    END
""").fetchdf()

print(tier_stats.to_string(index=False))


# ------------------------------------------------------------
# 3. VERIFY AGAINST GROUND TRUTH
# ------------------------------------------------------------

print()
print("=" * 80)
print("3. TRAINING ACCURACY BY TIER")
print("=" * 80)

conn.execute("""
CREATE OR REPLACE TEMP TABLE calibrated_eval AS

SELECT

    c.source1_entity_id,

    c.candidate_entity_id,

    c.confidence_tier,

    c.best_score,

    c.score_margin,

    CASE
        WHEN g.matched_entity_id IS NOT NULL
        THEN 1
        ELSE 0
    END AS is_correct

FROM calibrated_fuzzy_matches c

LEFT JOIN ground_truth_pairs g

    ON c.source1_entity_id = g.source1_entity_id
   AND c.candidate_entity_id = g.matched_entity_id
""")


conn.execute("""
CREATE OR REPLACE TEMP TABLE calibrated_eval_one AS

SELECT

    source1_entity_id,

    confidence_tier,

    MAX(is_correct) AS is_correct,

    MAX(best_score) AS best_score,

    MAX(score_margin) AS score_margin

FROM calibrated_eval

GROUP BY
    source1_entity_id,
    confidence_tier
""")


tier_accuracy = conn.execute("""
SELECT

    confidence_tier,

    COUNT(*) AS s1_count,

    SUM(is_correct) AS correct_count,

    COUNT(*) - SUM(is_correct) AS incorrect_count,

    ROUND(
        100.0 * SUM(is_correct) / COUNT(*),
        2
    ) AS accuracy_pct

FROM calibrated_eval_one

GROUP BY confidence_tier

ORDER BY
    CASE confidence_tier
        WHEN 'A_VERY_HIGH' THEN 1
        WHEN 'B_HIGH' THEN 2
        WHEN 'C_MODERATE' THEN 3
        ELSE 4
    END
""").fetchdf()

print(tier_accuracy.to_string(index=False))


# ------------------------------------------------------------
# 4. ACCEPTED VS UNRESOLVED
# ------------------------------------------------------------

print()
print("=" * 80)
print("4. ACCEPTED FUZZY COVERAGE")
print("=" * 80)

accepted = conn.execute("""
SELECT

    COUNT(*) AS total_unresolved,

    SUM(
        CASE
            WHEN confidence_tier <> 'UNRESOLVED'
            THEN 1
            ELSE 0
        END
    ) AS accepted,

    SUM(
        CASE
            WHEN confidence_tier = 'UNRESOLVED'
            THEN 1
            ELSE 0
        END
    ) AS still_unresolved

FROM calibrated_fuzzy_matches
""").fetchone()

print(f"Total fuzzy S1:       {accepted[0]:,}")
print(f"Accepted fuzzy:       {accepted[1]:,}")
print(f"Still unresolved:     {accepted[2]:,}")

print(
    f"Fuzzy coverage:       "
    f"{100.0 * accepted[1] / accepted[0]:.2f}%"
)


# ------------------------------------------------------------
# 5. SCORE DISTRIBUTION OF UNRESOLVED
# ------------------------------------------------------------

print()
print("=" * 80)
print("5. REMAINING UNRESOLVED SCORE DISTRIBUTION")
print("=" * 80)

remaining = conn.execute("""
SELECT

    CASE
        WHEN best_score >= 70 THEN '70-80'
        WHEN best_score >= 60 THEN '60-70'
        WHEN best_score >= 50 THEN '50-60'
        WHEN best_score >= 40 THEN '40-50'
        ELSE '<40'
    END AS score_bucket,

    COUNT(*) AS s1_count

FROM calibrated_fuzzy_matches

WHERE confidence_tier = 'UNRESOLVED'

GROUP BY score_bucket

ORDER BY
    CASE score_bucket
        WHEN '70-80' THEN 1
        WHEN '60-70' THEN 2
        WHEN '50-60' THEN 3
        WHEN '40-50' THEN 4
        ELSE 5
    END
""").fetchdf()

print(remaining.to_string(index=False))


# ------------------------------------------------------------
# 6. CHECKPOINT
# ------------------------------------------------------------

print()
print("=" * 80)
print("6. CHECKPOINT")
print("=" * 80)

conn.execute("CHECKPOINT")

print("✓ Database checkpoint completed")


# ------------------------------------------------------------
# 7. FINAL
# ------------------------------------------------------------

print()
print("=" * 80)
print("STEP 19 COMPLETE")
print("=" * 80)

print()
print("✓ Calibrated fuzzy tiers created")
print("✓ Training accuracy verified")
print("✓ Accepted coverage calculated")
print("✓ Remaining difficult cases identified")
print("✓ Checkpoint completed")

MATCHING — STEP 19
CALIBRATED FUZZY CONFIDENCE TIERS

1. CREATING CALIBRATED FUZZY MATCHES


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


2. CONFIDENCE TIER DISTRIBUTION
confidence_tier  s1_count  pct_of_unresolved
    A_VERY_HIGH    458129              48.41
         B_HIGH     63609               6.72
     C_MODERATE     19881               2.10
     UNRESOLVED    404737              42.77

3. TRAINING ACCURACY BY TIER
confidence_tier  s1_count  correct_count  incorrect_count  accuracy_pct
    A_VERY_HIGH    458129       456762.0           1367.0         99.70
         B_HIGH     63609        63118.0            491.0         99.23
     C_MODERATE     19881        19536.0            345.0         98.26
     UNRESOLVED    404737       100258.0         304479.0         24.77

4. ACCEPTED FUZZY COVERAGE
Total fuzzy S1:       946,356
Accepted fuzzy:       541,619
Still unresolved:     404,737
Fuzzy coverage:       57.23%

5. REMAINING UNRESOLVED SCORE DISTRIBUTION
score_bucket  s1_count
       70-80     57425
       60-70    131492
       50-60    203023
       40-50     11477
         <40      1320

6. CHECKPOINT
✓ Databa

In [ ]:
# ============================================================
# MATCHING — STEP 20
# CANDIDATE RECALL AGAINST GROUND TRUTH
# ============================================================

print("=" * 80)
print("MATCHING — STEP 20")
print("CANDIDATE RECALL AGAINST GROUND TRUTH")
print("=" * 80)


# ------------------------------------------------------------
# 1. GROUND TRUTH PAIRS
# ------------------------------------------------------------

print()
print("=" * 80)
print("1. PREPARING GROUND-TRUTH PAIRS")
print("=" * 80)

# ground_truth_pairs already exists from Step 17.
# Verify it.

gt_count = conn.execute("""
SELECT COUNT(*)
FROM ground_truth_pairs
""").fetchone()[0]

print(f"Ground-truth pairs available: {gt_count:,}")


# ------------------------------------------------------------
# 2. CHECK WHETHER TRUE PAIRS EXIST IN CANDIDATES
# ------------------------------------------------------------

print()
print("=" * 80)
print("2. CHECKING CANDIDATE RECALL")
print("=" * 80)

conn.execute("""
CREATE OR REPLACE TEMP TABLE ground_truth_candidate_recall AS

SELECT

    g.source1_entity_id,

    g.matched_entity_id,

    CASE
        WHEN c.source1_entity_id IS NOT NULL
        THEN 1
        ELSE 0
    END AS candidate_found

FROM ground_truth_pairs g

LEFT JOIN unique_candidates c

    ON g.source1_entity_id = c.source1_entity_id
   AND g.matched_entity_id = c.candidate_entity_id
""")


# ------------------------------------------------------------
# 3. OVERALL PAIR RECALL
# ------------------------------------------------------------

print()
print("=" * 80)
print("3. OVERALL GROUND-TRUTH PAIR RECALL")
print("=" * 80)

pair_recall = conn.execute("""
SELECT

    COUNT(*) AS total_gt_pairs,

    SUM(candidate_found) AS found_pairs,

    SUM(
        CASE
            WHEN candidate_found = 0
            THEN 1
            ELSE 0
        END
    ) AS missing_pairs,

    ROUND(
        100.0 * SUM(candidate_found) / COUNT(*),
        2
    ) AS pair_recall_pct

FROM ground_truth_candidate_recall
""").fetchdf()

print(pair_recall.to_string(index=False))


# ------------------------------------------------------------
# 4. S1-LEVEL RECALL
# ------------------------------------------------------------

print()
print("=" * 80)
print("4. S1-LEVEL CANDIDATE RECALL")
print("=" * 80)

s1_recall = conn.execute("""
WITH gt AS (

    SELECT
        source1_entity_id,
        COUNT(*) AS true_match_count
    FROM ground_truth_pairs
    GROUP BY source1_entity_id

),

found AS (

    SELECT
        source1_entity_id,
        SUM(candidate_found) AS found_match_count
    FROM ground_truth_candidate_recall
    GROUP BY source1_entity_id

)

SELECT

    COUNT(*) AS s1_with_ground_truth,

    SUM(
        CASE
            WHEN COALESCE(found_match_count, 0) > 0
            THEN 1
            ELSE 0
        END
    ) AS s1_with_at_least_one_candidate,

    SUM(
        CASE
            WHEN COALESCE(found_match_count, 0) = true_match_count
            THEN 1
            ELSE 0
        END
    ) AS s1_with_all_candidates,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN COALESCE(found_match_count, 0) > 0
                THEN 1
                ELSE 0
            END
        )
        / COUNT(*),
        2
    ) AS any_candidate_recall_pct,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN COALESCE(found_match_count, 0) = true_match_count
                THEN 1
                ELSE 0
            END
        )
        / COUNT(*),
        2
    ) AS all_candidate_recall_pct

FROM gt

LEFT JOIN found
    USING (source1_entity_id)
""").fetchdf()

print(s1_recall.to_string(index=False))


# ------------------------------------------------------------
# 5. RECALL FOR FUZZY-UNRESOLVED RECORDS
# ------------------------------------------------------------

print()
print("=" * 80)
print("5. CANDIDATE RECALL FOR FUZZY-UNRESOLVED S1")
print("=" * 80)

unresolved_recall = conn.execute("""
WITH gt AS (

    SELECT
        g.source1_entity_id,
        g.matched_entity_id

    FROM ground_truth_pairs g

    INNER JOIN fuzzy_match_summary f

        ON g.source1_entity_id = f.source1_entity_id

),

candidate_status AS (

    SELECT

        gt.source1_entity_id,

        gt.matched_entity_id,

        CASE
            WHEN c.source1_entity_id IS NOT NULL
            THEN 1
            ELSE 0
        END AS found

    FROM gt

    LEFT JOIN unique_candidates c

        ON gt.source1_entity_id = c.source1_entity_id
       AND gt.matched_entity_id = c.candidate_entity_id

)

SELECT

    COUNT(*) AS gt_pairs,

    SUM(found) AS found_pairs,

    SUM(
        CASE
            WHEN found = 0
            THEN 1
            ELSE 0
        END
    ) AS missing_pairs,

    ROUND(
        100.0 * SUM(found) / COUNT(*),
        2
    ) AS recall_pct

FROM candidate_status
""").fetchdf()

print(unresolved_recall.to_string(index=False))


# ------------------------------------------------------------
# 6. S1 RECORDS WHERE ALL TRUE MATCHES ARE MISSING
# ------------------------------------------------------------

print()
print("=" * 80)
print("6. S1 RECORDS WITH NO TRUE CANDIDATE")
print("=" * 80)

no_candidate = conn.execute("""
WITH gt AS (

    SELECT
        source1_entity_id,
        matched_entity_id
    FROM ground_truth_pairs

    INNER JOIN fuzzy_match_summary f
        USING (source1_entity_id)

),

status AS (

    SELECT

        gt.source1_entity_id,

        MAX(
            CASE
                WHEN c.source1_entity_id IS NOT NULL
                THEN 1
                ELSE 0
            END
        ) AS any_found

    FROM gt

    LEFT JOIN unique_candidates c

        ON gt.source1_entity_id = c.source1_entity_id
       AND gt.matched_entity_id = c.candidate_entity_id

    GROUP BY gt.source1_entity_id

)

SELECT

    COUNT(*) AS unresolved_s1_with_ground_truth,

    SUM(
        CASE
            WHEN any_found = 0
            THEN 1
            ELSE 0
        END
    ) AS no_true_candidate,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN any_found = 0
                THEN 1
                ELSE 0
            END
        )
        / COUNT(*),
        2
    ) AS no_candidate_pct

FROM status
""").fetchdf()

print(no_candidate.to_string(index=False))


# ------------------------------------------------------------
# 7. RECALL BY TRUE MATCH COUNT
# ------------------------------------------------------------

print()
print("=" * 80)
print("7. CANDIDATE RECALL BY TRUE MATCH COUNT")
print("=" * 80)

recall_by_count = conn.execute("""
WITH gt AS (

    SELECT
        source1_entity_id,
        matched_entity_id
    FROM ground_truth_pairs

),

status AS (

    SELECT

        gt.source1_entity_id,

        gt.matched_entity_id,

        CASE
            WHEN c.source1_entity_id IS NOT NULL
            THEN 1
            ELSE 0
        END AS found

    FROM gt

    LEFT JOIN unique_candidates c

        ON gt.source1_entity_id = c.source1_entity_id
       AND gt.matched_entity_id = c.candidate_entity_id

),

summary AS (

    SELECT

        source1_entity_id,

        COUNT(*) AS true_count,

        SUM(found) AS found_count

    FROM status

    GROUP BY source1_entity_id

)

SELECT

    CASE

        WHEN true_count = 1 THEN '1'

        WHEN true_count BETWEEN 2 AND 5
            THEN '2-5'

        WHEN true_count BETWEEN 6 AND 10
            THEN '6-10'

        ELSE '11+'

    END AS true_match_bucket,

    COUNT(*) AS s1_count,

    SUM(
        CASE
            WHEN found_count > 0
            THEN 1
            ELSE 0
        END
    ) AS any_found,

    SUM(
        CASE
            WHEN found_count = true_count
            THEN 1
            ELSE 0
        END
    ) AS all_found,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN found_count > 0
                THEN 1
                ELSE 0
            END
        )
        / COUNT(*),
        2
    ) AS any_candidate_pct,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN found_count = true_count
                THEN 1
                ELSE 0
            END
        )
        / COUNT(*),
        2
    ) AS all_candidate_pct

FROM summary

GROUP BY true_match_bucket

ORDER BY
    CASE true_match_bucket
        WHEN '1' THEN 1
        WHEN '2-5' THEN 2
        WHEN '6-10' THEN 3
        ELSE 4
    END
""").fetchdf()

print(recall_by_count.to_string(index=False))


# ------------------------------------------------------------
# 8. CHECKPOINT
# ------------------------------------------------------------

print()
print("=" * 80)
print("8. CHECKPOINT")
print("=" * 80)

conn.execute("CHECKPOINT")

print("✓ Checkpoint completed")


# ------------------------------------------------------------
# 9. COMPLETE
# ------------------------------------------------------------

print()
print("=" * 80)
print("STEP 20 COMPLETE")
print("=" * 80)

print()
print("✓ Ground-truth pairs checked")
print("✓ Overall candidate recall calculated")
print("✓ S1-level recall calculated")
print("✓ Fuzzy-unresolved recall calculated")
print("✓ Missing-candidate cases identified")
print("✓ Recall by true-match count calculated")

MATCHING — STEP 20
CANDIDATE RECALL AGAINST GROUND TRUTH

1. PREPARING GROUND-TRUTH PAIRS
Ground-truth pairs available: 7,638,365

2. CHECKING CANDIDATE RECALL


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


3. OVERALL GROUND-TRUTH PAIR RECALL
 total_gt_pairs  found_pairs  missing_pairs  pair_recall_pct
        7638365    2281907.0      5356458.0            29.87

4. S1-LEVEL CANDIDATE RECALL


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 s1_with_ground_truth  s1_with_at_least_one_candidate  s1_with_all_candidates  any_candidate_recall_pct  all_candidate_recall_pct
              2083574                       1376353.0                 99731.0                     66.06                      4.79

5. CANDIDATE RECALL FOR FUZZY-UNRESOLVED S1


...

 gt_pairs  found_pairs  missing_pairs  recall_pct
  3515831    1256898.0      2258933.0       35.75

6. S1 RECORDS WITH NO TRUE CANDIDATE


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 unresolved_s1_with_ground_truth  no_true_candidate  no_candidate_pct
                          906891           243491.0             26.85

7. CANDIDATE RECALL BY TRUE MATCH COUNT


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

true_match_bucket  s1_count  any_found  all_found  any_candidate_pct  all_candidate_pct
                1    119157    34629.0    34629.0              29.06              29.06
              2-5   1712125  1125288.0    64568.0              65.72               3.77
             6-10    252255   216402.0      534.0              85.79               0.21
              11+        37       34.0        0.0              91.89               0.00

8. CHECKPOINT
✓ Checkpoint completed

STEP 20 COMPLETE

✓ Ground-truth pairs checked
✓ Overall candidate recall calculated
✓ S1-level recall calculated
✓ Fuzzy-unresolved recall calculated
✓ Missing-candidate cases identified
✓ Recall by true-match count calculated


In [ ]:
# ============================================================
# ATTACH BLOCKING DATABASE
# ============================================================

BLOCKING_DB = "/content/drive/MyDrive/er_checkpoint/er_blocking_train_fresh.duckdb"

# Check currently attached databases
print("BEFORE ATTACH:")
print(conn.execute("PRAGMA database_list").fetchdf())

# Attach blocking DB if not already attached
dbs = conn.execute("PRAGMA database_list").fetchdf()

if "blocking_db" not in dbs["name"].tolist():
    conn.execute(
        f"ATTACH '{BLOCKING_DB}' AS blocking_db (READ_ONLY)"
    )
    print("\n✓ blocking_db attached")
else:
    print("\n✓ blocking_db already attached")

# Verify
print("\nAFTER ATTACH:")
print(conn.execute("PRAGMA database_list").fetchdf())

BEFORE ATTACH:
    seq                       name  \
0   570  er_candidates_train_fresh   
1  3907                  source_db   

                                                file  
0  /content/drive/MyDrive/er_checkpoint/er_candid...  
1  /content/drive/MyDrive/er_checkpoint/er_train_...  


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


✓ blocking_db attached

AFTER ATTACH:
    seq                       name  \
0   570  er_candidates_train_fresh   
1  3907                  source_db   
2  4052                blocking_db   

                                                file  
0  /content/drive/MyDrive/er_checkpoint/er_candid...  
1  /content/drive/MyDrive/er_checkpoint/er_train_...  
2  /content/drive/MyDrive/er_checkpoint/er_blocki...  


In [ ]:
# ============================================================
# VERIFY BLOCKING TABLES
# ============================================================

for table in [
    "train_s1_blocking",
    "train_s2_blocking",
    "train_s3_blocking"
]:
    count = conn.execute(
        f"SELECT COUNT(*) FROM blocking_db.{table}"
    ).fetchone()[0]

    print(f"{table}: {count:,} rows")

train_s1_blocking: 2,206,821 rows
train_s2_blocking: 5,034,616 rows
train_s3_blocking: 5,285,603 rows


In [ ]:
# ============================================================
# MATCHING — STEP 21
# DIAGNOSE MISSING GROUND-TRUTH CANDIDATES
# ============================================================

print("=" * 80)
print("MATCHING — STEP 21")
print("DIAGNOSE MISSING GROUND-TRUTH CANDIDATES")
print("=" * 80)


# ------------------------------------------------------------
# 1. GET MISSING GROUND-TRUTH PAIRS
# ------------------------------------------------------------

print()
print("=" * 80)
print("1. IDENTIFYING MISSING TRUE PAIRS")
print("=" * 80)

conn.execute("""
CREATE OR REPLACE TEMP TABLE missing_gt_pairs AS

SELECT
    g.source1_entity_id,
    g.matched_entity_id

FROM ground_truth_pairs g

LEFT JOIN unique_candidates c

    ON g.source1_entity_id = c.source1_entity_id
   AND g.matched_entity_id = c.candidate_entity_id

WHERE c.source1_entity_id IS NULL
""")

missing_count = conn.execute("""
SELECT COUNT(*)
FROM missing_gt_pairs
""").fetchone()[0]

print(f"Missing ground-truth pairs: {missing_count:,}")


# ------------------------------------------------------------
# 2. BUILD DETAILED COMPARISON
# ------------------------------------------------------------

print()
print("=" * 80)
print("2. BUILDING NAME / ADDRESS COMPARISON")
print("=" * 80)

conn.execute("""
CREATE OR REPLACE TEMP TABLE missing_gt_detail AS

SELECT

    m.source1_entity_id,
    m.matched_entity_id,

    s1.business_name AS s1_name,
    s1.business_address AS s1_address,
    s1.country AS s1_country,

    c.business_name AS candidate_name,
    c.business_address AS candidate_address,
    c.country AS candidate_country,

    s1.name_norm AS s1_name_norm,
    c.name_norm AS candidate_name_norm,

    s1.address_norm AS s1_address_norm,
    c.address_norm AS candidate_address_norm,

    s1.country_norm AS s1_country_norm,
    c.country_norm AS candidate_country_norm,

    s1.name_core AS s1_name_core,
    c.name_core AS candidate_name_core,

    s1.name_prefix4 AS s1_name_prefix4,
    c.name_prefix4 AS candidate_name_prefix4,

    s1.name_prefix6 AS s1_name_prefix6,
    c.name_prefix6 AS candidate_name_prefix6,

    s1.address_prefix6 AS s1_address_prefix6,
    c.address_prefix6 AS candidate_address_prefix6,

    s1.address_prefix10 AS s1_address_prefix10,
    c.address_prefix10 AS candidate_address_prefix10

FROM missing_gt_pairs m

INNER JOIN blocking_db.train_s1_blocking s1
    ON m.source1_entity_id = s1.entity_id

LEFT JOIN blocking_db.train_s2_blocking c
    ON m.matched_entity_id = c.entity_id

WHERE c.entity_id IS NOT NULL

UNION ALL

SELECT

    m.source1_entity_id,
    m.matched_entity_id,

    s1.business_name AS s1_name,
    s1.business_address AS s1_address,
    s1.country AS s1_country,

    c.business_name AS candidate_name,
    c.business_address AS candidate_address,
    c.country AS candidate_country,

    s1.name_norm AS s1_name_norm,
    c.name_norm AS candidate_name_norm,

    s1.address_norm AS s1_address_norm,
    c.address_norm AS candidate_address_norm,

    s1.country_norm AS s1_country_norm,
    c.country_norm AS candidate_country_norm,

    s1.name_core AS s1_name_core,
    c.name_core AS candidate_name_core,

    s1.name_prefix4 AS s1_name_prefix4,
    c.name_prefix4 AS candidate_name_prefix4,

    s1.name_prefix6 AS s1_name_prefix6,
    c.name_prefix6 AS candidate_name_prefix6,

    s1.address_prefix6 AS s1_address_prefix6,
    c.address_prefix6 AS candidate_address_prefix6,

    s1.address_prefix10 AS s1_address_prefix10,
    c.address_prefix10 AS candidate_address_prefix10

FROM missing_gt_pairs m

INNER JOIN blocking_db.train_s1_blocking s1
    ON m.source1_entity_id = s1.entity_id

LEFT JOIN blocking_db.train_s3_blocking c
    ON m.matched_entity_id = c.entity_id

WHERE c.entity_id IS NOT NULL
""")


detail_count = conn.execute("""
SELECT COUNT(*)
FROM missing_gt_detail
""").fetchone()[0]

print(f"Detailed missing pairs: {detail_count:,}")


# ------------------------------------------------------------
# 3. EXACT FIELD AGREEMENT
# ------------------------------------------------------------

print()
print("=" * 80)
print("3. FIELD AGREEMENT IN MISSING TRUE PAIRS")
print("=" * 80)

agreement = conn.execute("""
SELECT

    COUNT(*) AS total_pairs,

    SUM(
        CASE
            WHEN s1_country_norm = candidate_country_norm
            THEN 1 ELSE 0
        END
    ) AS country_exact,

    SUM(
        CASE
            WHEN s1_name_norm = candidate_name_norm
            THEN 1 ELSE 0
        END
    ) AS name_exact,

    SUM(
        CASE
            WHEN s1_address_norm = candidate_address_norm
            THEN 1 ELSE 0
        END
    ) AS address_exact,

    SUM(
        CASE
            WHEN s1_name_core = candidate_name_core
            THEN 1 ELSE 0
        END
    ) AS name_core_exact,

    SUM(
        CASE
            WHEN s1_name_prefix4 = candidate_name_prefix4
            THEN 1 ELSE 0
        END
    ) AS name_prefix4_exact,

    SUM(
        CASE
            WHEN s1_name_prefix6 = candidate_name_prefix6
            THEN 1 ELSE 0
        END
    ) AS name_prefix6_exact,

    SUM(
        CASE
            WHEN s1_address_prefix6 = candidate_address_prefix6
            THEN 1 ELSE 0
        END
    ) AS address_prefix6_exact,

    SUM(
        CASE
            WHEN s1_address_prefix10 = candidate_address_prefix10
            THEN 1 ELSE 0
        END
    ) AS address_prefix10_exact

FROM missing_gt_detail
""").fetchdf()

print(agreement.to_string(index=False))


# ------------------------------------------------------------
# 4. COMBINATION ANALYSIS
# ------------------------------------------------------------

print()
print("=" * 80)
print("4. BLOCKING KEY COMBINATION ANALYSIS")
print("=" * 80)

combination = conn.execute("""
SELECT

    CASE

        WHEN s1_name_norm = candidate_name_norm
         AND s1_country_norm = candidate_country_norm
            THEN 'NAME + COUNTRY'

        WHEN s1_address_norm = candidate_address_norm
         AND s1_country_norm = candidate_country_norm
            THEN 'ADDRESS + COUNTRY'

        WHEN s1_name_core = candidate_name_core
         AND s1_country_norm = candidate_country_norm
            THEN 'NAME CORE + COUNTRY'

        WHEN s1_name_prefix6 = candidate_name_prefix6
         AND s1_address_prefix6 = candidate_address_prefix6
            THEN 'NAME PREFIX6 + ADDRESS PREFIX6'

        WHEN s1_name_prefix4 = candidate_name_prefix4
         AND s1_address_prefix6 = candidate_address_prefix6
            THEN 'NAME PREFIX4 + ADDRESS PREFIX6'

        WHEN s1_name_prefix6 = candidate_name_prefix6
            THEN 'NAME PREFIX6 ONLY'

        WHEN s1_address_prefix6 = candidate_address_prefix6
            THEN 'ADDRESS PREFIX6 ONLY'

        WHEN s1_name_core = candidate_name_core
            THEN 'NAME CORE ONLY'

        WHEN s1_name_norm = candidate_name_norm
            THEN 'NAME EXACT ONLY'

        WHEN s1_address_norm = candidate_address_norm
            THEN 'ADDRESS EXACT ONLY'

        ELSE 'NO CURRENT KEY AGREEMENT'

    END AS agreement_type,

    COUNT(*) AS pair_count

FROM missing_gt_detail

GROUP BY agreement_type

ORDER BY pair_count DESC
""").fetchdf()

print(combination.to_string(index=False))


# ------------------------------------------------------------
# 5. COUNTRY AGREEMENT
# ------------------------------------------------------------

print()
print("=" * 80)
print("5. COUNTRY AGREEMENT")
print("=" * 80)

country = conn.execute("""
SELECT

    CASE

        WHEN s1_country_norm = candidate_country_norm
            THEN 'COUNTRY SAME'

        WHEN s1_country_norm IS NULL
          OR candidate_country_norm IS NULL
            THEN 'COUNTRY MISSING'

        ELSE 'COUNTRY DIFFERENT'

    END AS country_status,

    COUNT(*) AS pair_count

FROM missing_gt_detail

GROUP BY country_status

ORDER BY pair_count DESC
""").fetchdf()

print(country.to_string(index=False))


# ------------------------------------------------------------
# 6. SAMPLE MISSING PAIRS
# ------------------------------------------------------------

print()
print("=" * 80)
print("6. SAMPLE MISSING TRUE PAIRS")
print("=" * 80)

sample = conn.execute("""
SELECT

    source1_entity_id,
    matched_entity_id,

    s1_name,
    candidate_name,

    s1_address,
    candidate_address,

    s1_country,
    candidate_country,

    s1_name_norm,
    candidate_name_norm,

    s1_address_norm,
    candidate_address_norm

FROM missing_gt_detail

LIMIT 30
""").fetchdf()

print(sample.to_string(index=False))


# ------------------------------------------------------------
# 7. CHECKPOINT
# ------------------------------------------------------------

print()
print("=" * 80)
print("7. CHECKPOINT")
print("=" * 80)

conn.execute("CHECKPOINT")

print("✓ Checkpoint completed")


# ------------------------------------------------------------
# 8. COMPLETE
# ------------------------------------------------------------

print()
print("=" * 80)
print("STEP 21 COMPLETE")
print("=" * 80)

print()
print("✓ Missing true pairs identified")
print("✓ Name agreement analyzed")
print("✓ Address agreement analyzed")
print("✓ Country agreement analyzed")
print("✓ Blocking-key combinations analyzed")
print("✓ Sample missing pairs displayed")

MATCHING — STEP 21
DIAGNOSE MISSING GROUND-TRUTH CANDIDATES

1. IDENTIFYING MISSING TRUE PAIRS


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Missing ground-truth pairs: 5,356,458

2. BUILDING NAME / ADDRESS COMPARISON


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Detailed missing pairs: 5,356,458

3. FIELD AGREEMENT IN MISSING TRUE PAIRS
 total_pairs  country_exact  name_exact  address_exact  name_core_exact  name_prefix4_exact  name_prefix6_exact  address_prefix6_exact  address_prefix10_exact
     5356458      5356458.0         0.0            0.0        1055388.0           3780832.0           3488959.0              2186974.0               1904603.0

4. BLOCKING KEY COMBINATION ANALYSIS
                agreement_type  pair_count
             NAME PREFIX6 ONLY     1414912
      NO CURRENT KEY AGREEMENT     1136165
           NAME CORE + COUNTRY     1055388
NAME PREFIX6 + ADDRESS PREFIX6     1027052
          ADDRESS PREFIX6 ONLY      605900
NAME PREFIX4 + ADDRESS PREFIX6      117041

5. COUNTRY AGREEMENT
country_status  pair_count
  COUNTRY SAME     5356458

6. SAMPLE MISSING TRUE PAIRS
source1_entity_id matched_entity_id                                  s1_name                          candidate_name                                             

In [ ]:
# ============================================================
# MATCHING — STEP 22
# EVALUATE NEW BLOCKING ROUTES BEFORE GENERATING CANDIDATES
# ============================================================

print("=" * 80)
print("MATCHING — STEP 22")
print("EVALUATE ADDITIONAL BLOCKING ROUTES")
print("=" * 80)

# ------------------------------------------------------------
# 1. CHECK AVAILABLE BLOCKING COLUMNS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("1. CHECKING BLOCKING COLUMNS")
print("=" * 80)

cols = conn.execute("""
    SELECT column_name
    FROM information_schema.columns
    WHERE table_catalog = 'blocking_db'
      AND table_schema = 'main'
      AND table_name = 'train_s1_blocking'
    ORDER BY ordinal_position
""").fetchdf()

print(cols.to_string(index=False))


# ------------------------------------------------------------
# 2. MISSING PAIR RECOVERY BY BLOCKING KEY
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("2. MISSING TRUE-PAIR RECOVERY")
print("=" * 80)

routes = [
    (
        "NAME PREFIX6 + COUNTRY",
        """
        s1.name_prefix6 = s2.name_prefix6
        AND s1.country_norm = s2.country_norm
        """
    ),
    (
        "NAME PREFIX4 + ADDRESS PREFIX6",
        """
        s1.name_prefix4 = s2.name_prefix4
        AND s1.address_prefix6 = s2.address_prefix6
        AND s1.country_norm = s2.country_norm
        """
    ),
    (
        "NAME PREFIX6 + ADDRESS PREFIX6",
        """
        s1.name_prefix6 = s2.name_prefix6
        AND s1.address_prefix6 = s2.address_prefix6
        AND s1.country_norm = s2.country_norm
        """
    ),
    (
        "NAME CORE + COUNTRY",
        """
        s1.name_core = s2.name_core
        AND s1.country_norm = s2.country_norm
        """
    ),
    (
        "ADDRESS PREFIX6 + COUNTRY",
        """
        s1.address_prefix6 = s2.address_prefix6
        AND s1.country_norm = s2.country_norm
        """
    ),
    (
        "ADDRESS PREFIX10 + COUNTRY",
        """
        s1.address_prefix10 = s2.address_prefix10
        AND s1.country_norm = s2.country_norm
        """
    ),
]

for route_name, condition in routes:

    print(f"\nTesting: {route_name}")

    total_found = 0
    s2_found = 0
    s3_found = 0

    for target in ["s2", "s3"]:

        result = conn.execute(f"""
            SELECT COUNT(*)
            FROM missing_gt_pairs mg
            INNER JOIN blocking_db.train_s1_blocking s1
                ON mg.source1_entity_id = s1.entity_id
            INNER JOIN blocking_db.train_{target}_blocking s2
                ON mg.matched_entity_id = s2.entity_id
            WHERE {condition}
        """).fetchone()[0]

        if target == "s2":
            s2_found = result
        else:
            s3_found = result

        total_found += result

    print(f"  S2 recovered:    {s2_found:,}")
    print(f"  S3 recovered:    {s3_found:,}")
    print(f"  TOTAL recovered: {total_found:,}")


# ------------------------------------------------------------
# 3. ESTIMATE CANDIDATE COUNTS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("3. ESTIMATED NEW CANDIDATE VOLUME")
print("=" * 80)

candidate_routes = [
    (
        "NAME PREFIX6 + COUNTRY",
        "name_prefix6",
        "country_norm"
    ),
    (
        "NAME PREFIX4 + ADDRESS PREFIX6 + COUNTRY",
        "name_prefix4",
        "address_prefix6"
    ),
    (
        "NAME PREFIX6 + ADDRESS PREFIX6 + COUNTRY",
        "name_prefix6",
        "address_prefix6"
    ),
    (
        "NAME CORE + COUNTRY",
        "name_core",
        "country_norm"
    ),
    (
        "ADDRESS PREFIX6 + COUNTRY",
        "address_prefix6",
        "country_norm"
    ),
    (
        "ADDRESS PREFIX10 + COUNTRY",
        "address_prefix10",
        "country_norm"
    ),
]

for route_name, key1, key2 in candidate_routes:

    print(f"\n{route_name}")

    for target in ["s2", "s3"]:

        result = conn.execute(f"""
            SELECT COUNT(*)
            FROM blocking_db.train_s1_blocking s1
            INNER JOIN blocking_db.train_{target}_blocking s2
                ON s1.{key1} = s2.{key1}
               AND s1.{key2} = s2.{key2}
        """).fetchone()[0]

        print(f"  S1 -> {target.upper()}: {result:,}")

print("\n" + "=" * 80)
print("STEP 22 EVALUATION COMPLETE")
print("=" * 80)

MATCHING — STEP 22
EVALUATE ADDITIONAL BLOCKING ROUTES

1. CHECKING BLOCKING COLUMNS
          column_name
            entity_id
        business_name
     business_address
              country
            name_norm
         address_norm
         country_norm
            name_core
         name_prefix4
         name_prefix6
      address_prefix6
     address_prefix10
     name_country_key
  address_country_key
name_core_country_key
     name_address_key

2. MISSING TRUE-PAIR RECOVERY

Testing: NAME PREFIX6 + COUNTRY


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  S2 recovered:    1,619,569
  S3 recovered:    1,869,390
  TOTAL recovered: 3,488,959

Testing: NAME PREFIX4 + ADDRESS PREFIX6


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  S2 recovered:    514,257
  S3 recovered:    541,131
  TOTAL recovered: 1,055,388

Testing: ADDRESS PREFIX6 + COUNTRY


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

OutOfMemoryException: Out of Memory Error: failed to offload data block of size 256.0 KiB (2.2 GiB/2.2 GiB used).
This limit was set by the 'max_temp_directory_size' setting.
By default, this setting utilizes the available disk space on the drive where the 'temp_directory' is located.
You can adjust this setting, by using (for example) PRAGMA max_temp_directory_size='10GiB'

Possible solutions:
* Reducing the number of threads (SET threads=X)
* Disabling insertion-order preservation (SET preserve_insertion_order=false)
* Increasing the memory limit (SET memory_limit='...GB')

See also https://duckdb.org/docs/stable/guides/performance/how_to_tune_workloads

In [14]:
# ============================================================
# MATCHING — STEP 22A
# SAFE CANDIDATE VOLUME ESTIMATION
# ============================================================

print("=" * 80)
print("MATCHING — STEP 22A")
print("SAFE CANDIDATE VOLUME ESTIMATION")
print("=" * 80)

# Reduce memory pressure
conn.execute("SET threads=2")
conn.execute("SET preserve_insertion_order=false")

# Give DuckDB more temporary disk space
conn.execute("PRAGMA max_temp_directory_size='10GiB'")

routes = [
    (
        "NAME PREFIX6 + COUNTRY",
        "name_prefix6 = name_prefix6",
        "name_prefix6",
        "country_norm"
    ),
    (
        "NAME PREFIX4 + ADDRESS PREFIX6 + COUNTRY",
        "name_prefix4 = name_prefix4",
        "name_prefix4",
        "address_prefix6"
    ),
    (
        "NAME PREFIX6 + ADDRESS PREFIX6 + COUNTRY",
        "name_prefix6 = name_prefix6",
        "name_prefix6",
        "address_prefix6"
    ),
    (
        "NAME CORE + COUNTRY",
        "name_core = name_core",
        "name_core",
        "country_norm"
    ),
]

for route_name, _, key1, key2 in routes:

    print("\n" + "-" * 80)
    print(route_name)
    print("-" * 80)

    for target in ["s2", "s3"]:

        # Count using grouped block sizes rather than
        # materializing the complete join.
        query = f"""
        WITH s1_blocks AS (
            SELECT
                {key1} AS k1,
                {key2} AS k2,
                COUNT(*) AS s1_count
            FROM blocking_db.train_s1_blocking
            GROUP BY {key1}, {key2}
        ),
        target_blocks AS (
            SELECT
                {key1} AS k1,
                {key2} AS k2,
                COUNT(*) AS target_count
            FROM blocking_db.train_{target}_blocking
            GROUP BY {key1}, {key2}
        )
        SELECT
            COALESCE(SUM(s1_count * target_count), 0)
        FROM s1_blocks
        INNER JOIN target_blocks
            ON s1_blocks.k1 = target_blocks.k1
           AND s1_blocks.k2 = target_blocks.k2
        """

        count = conn.execute(query).fetchone()[0]

        print(f"S1 -> {target.upper()}: {count:,}")

print("\n" + "=" * 80)
print("STEP 22A COMPLETE")
print("=" * 80)

MATCHING — STEP 22A
SAFE CANDIDATE VOLUME ESTIMATION

--------------------------------------------------------------------------------
NAME PREFIX6 + COUNTRY
--------------------------------------------------------------------------------


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

S1 -> S2: 4,318,701,733


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

S1 -> S3: 4,604,926,313

--------------------------------------------------------------------------------
NAME PREFIX4 + ADDRESS PREFIX6 + COUNTRY
--------------------------------------------------------------------------------


...

S1 -> S3: 4,677,945

--------------------------------------------------------------------------------
NAME CORE + COUNTRY
--------------------------------------------------------------------------------


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

S1 -> S2: 29,280,352


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

S1 -> S3: 30,547,292

STEP 22A COMPLETE


In [15]:
# ============================================================
# MATCHING — STEP 23
# COMPARE SAFE BLOCKING ROUTES
# ============================================================

print("=" * 80)
print("MATCHING — STEP 23")
print("COMPARE SAFE BLOCKING ROUTES")
print("=" * 80)

conn.execute("SET threads=2")
conn.execute("SET preserve_insertion_order=false")
conn.execute("PRAGMA max_temp_directory_size='10GiB'")


# ------------------------------------------------------------
# 1. CREATE TEMP TABLE OF MISSING GT PAIRS WITH BOTH ROUTES
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("1. ANALYZING GROUND-TRUTH RECOVERY OVERLAP")
print("=" * 80)

conn.execute("""
CREATE OR REPLACE TEMP TABLE missing_gt_route_flags AS

SELECT
    mg.source1_entity_id,
    mg.matched_entity_id,

    CASE
        WHEN
            s1.name_prefix4 = s2.name_prefix4
            AND s1.address_prefix6 = s2.address_prefix6
            AND s1.country_norm = s2.country_norm
        THEN 1 ELSE 0
    END AS route_a,

    CASE
        WHEN
            s1.name_prefix6 = s2.name_prefix6
            AND s1.address_prefix6 = s2.address_prefix6
            AND s1.country_norm = s2.country_norm
        THEN 1 ELSE 0
    END AS route_b

FROM missing_gt_pairs mg

INNER JOIN blocking_db.train_s1_blocking s1
    ON mg.source1_entity_id = s1.entity_id

INNER JOIN blocking_db.train_s2_blocking s2
    ON mg.matched_entity_id = s2.entity_id

WHERE mg.matched_entity_id LIKE 'S2-%'

UNION ALL

SELECT
    mg.source1_entity_id,
    mg.matched_entity_id,

    CASE
        WHEN
            s1.name_prefix4 = s3.name_prefix4
            AND s1.address_prefix6 = s3.address_prefix6
            AND s1.country_norm = s3.country_norm
        THEN 1 ELSE 0
    END AS route_a,

    CASE
        WHEN
            s1.name_prefix6 = s3.name_prefix6
            AND s1.address_prefix6 = s3.address_prefix6
            AND s1.country_norm = s3.country_norm
        THEN 1 ELSE 0
    END AS route_b

FROM missing_gt_pairs mg

INNER JOIN blocking_db.train_s1_blocking s1
    ON mg.source1_entity_id = s1.entity_id

INNER JOIN blocking_db.train_s3_blocking s3
    ON mg.matched_entity_id = s3.entity_id

WHERE mg.matched_entity_id LIKE 'S3-%'
""")

print("✓ Route flags created")


# ------------------------------------------------------------
# 2. RECOVERY OVERLAP
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("2. RECOVERY OVERLAP")
print("=" * 80)

print(
    conn.execute("""
        SELECT
            COUNT(*) AS total_missing,
            SUM(route_a) AS route_a_only_or_both,
            SUM(route_b) AS route_b_only_or_both,
            SUM(
                CASE
                    WHEN route_a = 1 AND route_b = 1
                    THEN 1 ELSE 0
                END
            ) AS both_routes,
            SUM(
                CASE
                    WHEN route_a = 1 AND route_b = 0
                    THEN 1 ELSE 0
                END
            ) AS route_a_only,
            SUM(
                CASE
                    WHEN route_a = 0 AND route_b = 1
                    THEN 1 ELSE 0
                END
            ) AS route_b_only,
            SUM(
                CASE
                    WHEN route_a = 0 AND route_b = 0
                    THEN 1 ELSE 0
                END
            ) AS neither_route
        FROM missing_gt_route_flags
    """).fetchdf().to_string(index=False)
)


# ------------------------------------------------------------
# 3. CHECK EXISTING CANDIDATE OVERLAP
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("3. CHECKING EXISTING CANDIDATE OVERLAP")
print("=" * 80)

# Route A
route_a_existing = conn.execute("""
SELECT COUNT(*)
FROM unique_candidates uc
INNER JOIN blocking_db.train_s1_blocking s1
    ON uc.source1_entity_id = s1.entity_id
INNER JOIN blocking_db.train_s2_blocking s2
    ON uc.candidate_entity_id = s2.entity_id
WHERE
    uc.candidate_entity_id LIKE 'S2-%'
    AND s1.name_prefix4 = s2.name_prefix4
    AND s1.address_prefix6 = s2.address_prefix6
    AND s1.country_norm = s2.country_norm
""").fetchone()[0]

print(f"Existing S2 candidates covered by Route A: {route_a_existing:,}")


route_b_existing = conn.execute("""
SELECT COUNT(*)
FROM unique_candidates uc
INNER JOIN blocking_db.train_s1_blocking s1
    ON uc.source1_entity_id = s1.entity_id
INNER JOIN blocking_db.train_s2_blocking s2
    ON uc.candidate_entity_id = s2.entity_id
WHERE
    uc.candidate_entity_id LIKE 'S2-%'
    AND s1.name_prefix6 = s2.name_prefix6
    AND s1.address_prefix6 = s2.address_prefix6
    AND s1.country_norm = s2.country_norm
""").fetchone()[0]

print(f"Existing S2 candidates covered by Route B: {route_b_existing:,}")


print("\n" + "=" * 80)
print("STEP 23 COMPLETE")
print("=" * 80)

MATCHING — STEP 23
COMPARE SAFE BLOCKING ROUTES

1. ANALYZING GROUND-TRUTH RECOVERY OVERLAP


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ Route flags created

2. RECOVERY OVERLAP
 total_missing  route_a_only_or_both  route_b_only_or_both  both_routes  route_a_only  route_b_only  neither_route
       5356458             1580813.0             1460419.0    1460419.0      120394.0           0.0      3775645.0

3. CHECKING EXISTING CANDIDATE OVERLAP


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Existing S2 candidates covered by Route A: 652,267


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Existing S2 candidates covered by Route B: 633,443

STEP 23 COMPLETE


In [16]:
# ============================================================
# MATCHING — STEP 24
# ADD ROUTE A CANDIDATES
#
# NAME PREFIX4 + ADDRESS PREFIX6 + COUNTRY
# ============================================================

print("=" * 80)
print("MATCHING — STEP 24")
print("ADDING NAME PREFIX4 + ADDRESS PREFIX6 + COUNTRY")
print("=" * 80)

# ------------------------------------------------------------
# SAFETY / PERFORMANCE SETTINGS
# ------------------------------------------------------------

conn.execute("SET threads=2")
conn.execute("SET preserve_insertion_order=false")
conn.execute("PRAGMA max_temp_directory_size='10GiB'")

print("\n✓ DuckDB memory settings configured")


# ------------------------------------------------------------
# 1. CHECK CURRENT CANDIDATE COUNT
# ------------------------------------------------------------

before_count = conn.execute("""
    SELECT COUNT(*)
    FROM unique_candidates
""").fetchone()[0]

print(f"\nExisting unique candidates: {before_count:,}")


# ------------------------------------------------------------
# 2. CREATE ROUTE A → S2
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("2. GENERATING ROUTE A → S2")
print("=" * 80)

conn.execute("DROP TABLE IF EXISTS route_a_s2")

conn.execute("""
CREATE TEMP TABLE route_a_s2 AS

SELECT DISTINCT
    s1.entity_id AS source1_entity_id,
    s2.entity_id AS candidate_entity_id,
    'name_prefix4_address_prefix6_country' AS route

FROM blocking_db.train_s1_blocking s1

INNER JOIN blocking_db.train_s2_blocking s2
    ON s1.name_prefix4 = s2.name_prefix4
   AND s1.address_prefix6 = s2.address_prefix6
   AND s1.country_norm = s2.country_norm
""")

route_a_s2_count = conn.execute("""
    SELECT COUNT(*)
    FROM route_a_s2
""").fetchone()[0]

print(f"Route A S2 candidates: {route_a_s2_count:,}")


# ------------------------------------------------------------
# 3. CREATE ROUTE A → S3
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("3. GENERATING ROUTE A → S3")
print("=" * 80)

conn.execute("DROP TABLE IF EXISTS route_a_s3")

conn.execute("""
CREATE TEMP TABLE route_a_s3 AS

SELECT DISTINCT
    s1.entity_id AS source1_entity_id,
    s3.entity_id AS candidate_entity_id,
    'name_prefix4_address_prefix6_country' AS route

FROM blocking_db.train_s1_blocking s1

INNER JOIN blocking_db.train_s3_blocking s3
    ON s1.name_prefix4 = s3.name_prefix4
   AND s1.address_prefix6 = s3.address_prefix6
   AND s1.country_norm = s3.country_norm
""")

route_a_s3_count = conn.execute("""
    SELECT COUNT(*)
    FROM route_a_s3
""").fetchone()[0]

print(f"Route A S3 candidates: {route_a_s3_count:,}")


# ------------------------------------------------------------
# 4. INSERT ONLY NEW S2 PAIRS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("4. INSERTING NEW S2 PAIRS")
print("=" * 80)

before_s2 = conn.execute("""
    SELECT COUNT(*)
    FROM unique_candidates
    WHERE candidate_entity_id LIKE 'S2-%'
""").fetchone()[0]

conn.execute("""
INSERT INTO unique_candidates
(
    source1_entity_id,
    candidate_entity_id,
    route
)
SELECT
    r.source1_entity_id,
    r.candidate_entity_id,
    r.route
FROM route_a_s2 r

WHERE NOT EXISTS (
    SELECT 1
    FROM unique_candidates u
    WHERE u.source1_entity_id = r.source1_entity_id
      AND u.candidate_entity_id = r.candidate_entity_id
)
""")

after_s2 = conn.execute("""
    SELECT COUNT(*)
    FROM unique_candidates
    WHERE candidate_entity_id LIKE 'S2-%'
""").fetchone()[0]

new_s2 = after_s2 - before_s2

print(f"New S2 candidates added: {new_s2:,}")


# ------------------------------------------------------------
# 5. INSERT ONLY NEW S3 PAIRS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("5. INSERTING NEW S3 PAIRS")
print("=" * 80)

before_s3 = conn.execute("""
    SELECT COUNT(*)
    FROM unique_candidates
    WHERE candidate_entity_id LIKE 'S3-%'
""").fetchone()[0]

conn.execute("""
INSERT INTO unique_candidates
(
    source1_entity_id,
    candidate_entity_id,
    route
)
SELECT
    r.source1_entity_id,
    r.candidate_entity_id,
    r.route
FROM route_a_s3 r

WHERE NOT EXISTS (
    SELECT 1
    FROM unique_candidates u
    WHERE u.source1_entity_id = r.source1_entity_id
      AND u.candidate_entity_id = r.candidate_entity_id
)
""")

after_s3 = conn.execute("""
    SELECT COUNT(*)
    FROM unique_candidates
    WHERE candidate_entity_id LIKE 'S3-%'
""").fetchone()[0]

new_s3 = after_s3 - before_s3

print(f"New S3 candidates added: {new_s3:,}")


# ------------------------------------------------------------
# 6. FINAL COUNTS
# ------------------------------------------------------------

after_count = conn.execute("""
    SELECT COUNT(*)
    FROM unique_candidates
""").fetchone()[0]

print("\n" + "=" * 80)
print("6. FINAL CANDIDATE COUNTS")
print("=" * 80)

print(f"Before:       {before_count:,}")
print(f"New S2:       {new_s2:,}")
print(f"New S3:       {new_s3:,}")
print(f"Total added:  {after_count - before_count:,}")
print(f"After:        {after_count:,}")


# ------------------------------------------------------------
# 7. ROUTE COUNT
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("7. ROUTE DISTRIBUTION")
print("=" * 80)

print(
    conn.execute("""
        SELECT
            route,
            COUNT(*) AS pairs
        FROM unique_candidates
        GROUP BY route
        ORDER BY pairs DESC
    """).fetchdf().to_string(index=False)
)

print("\n" + "=" * 80)
print("STEP 24 COMPLETE")
print("=" * 80)

MATCHING — STEP 24
ADDING NAME PREFIX4 + ADDRESS PREFIX6 + COUNTRY

✓ DuckDB memory settings configured

Existing unique candidates: 23,413,014

2. GENERATING ROUTE A → S2


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Route A S2 candidates: 7,125,964

3. GENERATING ROUTE A → S3


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Route A S3 candidates: 8,378,357

4. INSERTING NEW S2 PAIRS


BinderException: Binder Error: Table "unique_candidates" does not have a column with name "route"

In [17]:
# ============================================================
# CHECK unique_candidates SCHEMA
# ============================================================

print("=" * 80)
print("unique_candidates SCHEMA")
print("=" * 80)

schema = conn.execute("""
    PRAGMA table_info('unique_candidates')
""").fetchdf()

print(schema.to_string(index=False))

print("\nCurrent row count:")

print(
    conn.execute("""
        SELECT COUNT(*)
        FROM unique_candidates
    """).fetchone()[0]
)

unique_candidates SCHEMA
 cid                name    type  notnull dflt_value    pk
   0   source1_entity_id VARCHAR    False       None False
   1 candidate_entity_id VARCHAR    False       None False

Current row count:
23413014


In [18]:
# Check candidate count by columns currently available
print(
    conn.execute("""
        SELECT *
        FROM unique_candidates
        LIMIT 3
    """).fetchdf().to_string(index=False)
)

source1_entity_id candidate_entity_id
     S1-251506166        S2-614721384
     S1-493230080        S2-998782125
     S1-584046291        S2-521724739


In [19]:
# ============================================================
# MATCHING — STEP 24
# ADD ROUTE A CANDIDATES
#
# NAME PREFIX4 + ADDRESS PREFIX6 + COUNTRY
# ============================================================

print("=" * 80)
print("MATCHING — STEP 24")
print("ADDING NAME PREFIX4 + ADDRESS PREFIX6 + COUNTRY")
print("=" * 80)

# ------------------------------------------------------------
# SAFETY / PERFORMANCE SETTINGS
# ------------------------------------------------------------

conn.execute("SET threads=2")
conn.execute("SET preserve_insertion_order=false")
conn.execute("PRAGMA max_temp_directory_size='10GiB'")

print("\n✓ DuckDB settings configured")


# ------------------------------------------------------------
# 1. CURRENT COUNT
# ------------------------------------------------------------

before_count = conn.execute("""
    SELECT COUNT(*)
    FROM unique_candidates
""").fetchone()[0]

print(f"Existing candidates: {before_count:,}")


# ------------------------------------------------------------
# 2. GENERATE ROUTE A → S2
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("2. GENERATING S2 CANDIDATES")
print("=" * 80)

conn.execute("DROP TABLE IF EXISTS route_a_s2")

conn.execute("""
CREATE TEMP TABLE route_a_s2 AS

SELECT DISTINCT
    s1.entity_id AS source1_entity_id,
    s2.entity_id AS candidate_entity_id

FROM blocking_db.train_s1_blocking s1

INNER JOIN blocking_db.train_s2_blocking s2
    ON s1.name_prefix4 = s2.name_prefix4
   AND s1.address_prefix6 = s2.address_prefix6
   AND s1.country_norm = s2.country_norm
""")

route_a_s2_count = conn.execute("""
    SELECT COUNT(*)
    FROM route_a_s2
""").fetchone()[0]

print(f"Route A S2 pairs: {route_a_s2_count:,}")


# ------------------------------------------------------------
# 3. GENERATE ROUTE A → S3
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("3. GENERATING S3 CANDIDATES")
print("=" * 80)

conn.execute("DROP TABLE IF EXISTS route_a_s3")

conn.execute("""
CREATE TEMP TABLE route_a_s3 AS

SELECT DISTINCT
    s1.entity_id AS source1_entity_id,
    s3.entity_id AS candidate_entity_id

FROM blocking_db.train_s1_blocking s1

INNER JOIN blocking_db.train_s3_blocking s3
    ON s1.name_prefix4 = s3.name_prefix4
   AND s1.address_prefix6 = s3.address_prefix6
   AND s1.country_norm = s3.country_norm
""")

route_a_s3_count = conn.execute("""
    SELECT COUNT(*)
    FROM route_a_s3
""").fetchone()[0]

print(f"Route A S3 pairs: {route_a_s3_count:,}")


# ------------------------------------------------------------
# 4. INSERT NEW S2 PAIRS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("4. INSERTING NEW S2 PAIRS")
print("=" * 80)

before_s2 = conn.execute("""
    SELECT COUNT(*)
    FROM unique_candidates
    WHERE candidate_entity_id LIKE 'S2-%'
""").fetchone()[0]

conn.execute("""
INSERT INTO unique_candidates
(
    source1_entity_id,
    candidate_entity_id
)
SELECT
    r.source1_entity_id,
    r.candidate_entity_id

FROM route_a_s2 r

WHERE NOT EXISTS (
    SELECT 1
    FROM unique_candidates u
    WHERE u.source1_entity_id = r.source1_entity_id
      AND u.candidate_entity_id = r.candidate_entity_id
)
""")

after_s2 = conn.execute("""
    SELECT COUNT(*)
    FROM unique_candidates
    WHERE candidate_entity_id LIKE 'S2-%'
""").fetchone()[0]

new_s2 = after_s2 - before_s2

print(f"New S2 candidates added: {new_s2:,}")


# ------------------------------------------------------------
# 5. INSERT NEW S3 PAIRS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("5. INSERTING NEW S3 PAIRS")
print("=" * 80)

before_s3 = conn.execute("""
    SELECT COUNT(*)
    FROM unique_candidates
    WHERE candidate_entity_id LIKE 'S3-%'
""").fetchone()[0]

conn.execute("""
INSERT INTO unique_candidates
(
    source1_entity_id,
    candidate_entity_id
)
SELECT
    r.source1_entity_id,
    r.candidate_entity_id

FROM route_a_s3 r

WHERE NOT EXISTS (
    SELECT 1
    FROM unique_candidates u
    WHERE u.source1_entity_id = r.source1_entity_id
      AND u.candidate_entity_id = r.candidate_entity_id
)
""")

after_s3 = conn.execute("""
    SELECT COUNT(*)
    FROM unique_candidates
    WHERE candidate_entity_id LIKE 'S3-%'
""").fetchone()[0]

new_s3 = after_s3 - before_s3

print(f"New S3 candidates added: {new_s3:,}")


# ------------------------------------------------------------
# 6. FINAL COUNTS
# ------------------------------------------------------------

after_count = conn.execute("""
    SELECT COUNT(*)
    FROM unique_candidates
""").fetchone()[0]

print("\n" + "=" * 80)
print("6. FINAL CANDIDATE COUNTS")
print("=" * 80)

print(f"Before:       {before_count:,}")
print(f"New S2:       {new_s2:,}")
print(f"New S3:       {new_s3:,}")
print(f"Total added:  {after_count - before_count:,}")
print(f"After:        {after_count:,}")


# ------------------------------------------------------------
# 7. VERIFY NO DUPLICATE PAIRS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("7. VERIFYING DUPLICATES")
print("=" * 80)

duplicate_count = conn.execute("""
    SELECT COUNT(*)
    FROM (
        SELECT
            source1_entity_id,
            candidate_entity_id,
            COUNT(*) AS n
        FROM unique_candidates
        GROUP BY
            source1_entity_id,
            candidate_entity_id
        HAVING COUNT(*) > 1
    )
""").fetchone()[0]

print(f"Duplicate source1/candidate pairs: {duplicate_count:,}")


print("\n" + "=" * 80)
print("STEP 24 COMPLETE")
print("=" * 80)

MATCHING — STEP 24
ADDING NAME PREFIX4 + ADDRESS PREFIX6 + COUNTRY

✓ DuckDB settings configured
Existing candidates: 23,413,014

2. GENERATING S2 CANDIDATES


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Route A S2 pairs: 7,125,964

3. GENERATING S3 CANDIDATES


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Route A S3 pairs: 8,378,357

4. INSERTING NEW S2 PAIRS


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

TransactionException: TransactionContext Error: Failed to commit: Invalid node type for ARTOperator::Insert.

In [20]:
# ============================================================
# MATCHING — STEP 24A
# RECOVER AFTER FAILED INSERT
# ============================================================

print("=" * 80)
print("STEP 24A — RECOVER AFTER FAILED INSERT")
print("=" * 80)

# Roll back the failed transaction
try:
    conn.rollback()
    print("✓ Transaction rolled back")
except Exception as e:
    print("Rollback message:", e)

# Check current candidate count
current_count = conn.execute("""
    SELECT COUNT(*)
    FROM unique_candidates
""").fetchone()[0]

print(f"\nCurrent unique_candidates count: {current_count:,}")

# Check that the temporary route tables still exist
print("\nTemporary route tables:")

for table in ["route_a_s2", "route_a_s3"]:
    try:
        count = conn.execute(
            f"SELECT COUNT(*) FROM {table}"
        ).fetchone()[0]

        print(f"  {table}: {count:,}")

    except Exception as e:
        print(f"  {table}: NOT AVAILABLE")

print("\n" + "=" * 80)
print("STEP 24A COMPLETE")
print("=" * 80)

STEP 24A — RECOVER AFTER FAILED INSERT
Rollback message: TransactionContext Error: cannot rollback - no transaction is active

Current unique_candidates count: 23,413,014

Temporary route tables:
  route_a_s2: 7,125,964
  route_a_s3: 8,378,357

STEP 24A COMPLETE


In [21]:
# ============================================================
# MATCHING — STEP 24B
# COUNT NEW ROUTE A CANDIDATES
# ============================================================

print("=" * 80)
print("STEP 24B — COUNTING NEW ROUTE A CANDIDATES")
print("=" * 80)

conn.execute("SET threads=2")
conn.execute("SET preserve_insertion_order=false")
conn.execute("PRAGMA max_temp_directory_size='10GiB'")


# ------------------------------------------------------------
# S2 NEW PAIRS
# ------------------------------------------------------------

print("\nChecking S2...")

new_s2 = conn.execute("""
    SELECT COUNT(*)
    FROM route_a_s2 r
    LEFT JOIN unique_candidates u
        ON u.source1_entity_id = r.source1_entity_id
       AND u.candidate_entity_id = r.candidate_entity_id
    WHERE u.source1_entity_id IS NULL
""").fetchone()[0]

print(f"New S2 pairs: {new_s2:,}")


# ------------------------------------------------------------
# S3 NEW PAIRS
# ------------------------------------------------------------

print("\nChecking S3...")

new_s3 = conn.execute("""
    SELECT COUNT(*)
    FROM route_a_s3 r
    LEFT JOIN unique_candidates u
        ON u.source1_entity_id = r.source1_entity_id
       AND u.candidate_entity_id = r.candidate_entity_id
    WHERE u.source1_entity_id IS NULL
""").fetchone()[0]

print(f"New S3 pairs: {new_s3:,}")


# ------------------------------------------------------------
# TOTAL
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("RESULT")
print("=" * 80)

print(f"New S2:       {new_s2:,}")
print(f"New S3:       {new_s3:,}")
print(f"Total new:    {new_s2 + new_s3:,}")

print("\n" + "=" * 80)
print("STEP 24B COMPLETE")
print("=" * 80)

STEP 24B — COUNTING NEW ROUTE A CANDIDATES

Checking S2...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

New S2 pairs: 6,473,697

Checking S3...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

New S3 pairs: 7,936,744

RESULT
New S2:       6,473,697
New S3:       7,936,744
Total new:    14,410,441

STEP 24B COMPLETE


In [22]:
# ============================================================
# MATCHING — STEP 24C
# MATERIALIZE NEW ROUTE A PAIRS
# ============================================================

print("=" * 80)
print("STEP 24C — MATERIALIZING NEW ROUTE A PAIRS")
print("=" * 80)

conn.execute("SET threads=2")
conn.execute("SET preserve_insertion_order=false")
conn.execute("PRAGMA max_temp_directory_size='10GiB'")


# ------------------------------------------------------------
# DROP OLD TEMP TABLES IF THEY EXIST
# ------------------------------------------------------------

conn.execute("DROP TABLE IF EXISTS new_route_a_s2")
conn.execute("DROP TABLE IF EXISTS new_route_a_s3")


# ------------------------------------------------------------
# S2
# ------------------------------------------------------------

print("\nMaterializing new S2 pairs...")

conn.execute("""
    CREATE TEMP TABLE new_route_a_s2 AS
    SELECT
        r.source1_entity_id,
        r.candidate_entity_id
    FROM route_a_s2 r
    LEFT JOIN unique_candidates u
        ON u.source1_entity_id = r.source1_entity_id
       AND u.candidate_entity_id = r.candidate_entity_id
    WHERE u.source1_entity_id IS NULL
""")

count_s2 = conn.execute("""
    SELECT COUNT(*)
    FROM new_route_a_s2
""").fetchone()[0]

print(f"New S2 materialized: {count_s2:,}")


# ------------------------------------------------------------
# S3
# ------------------------------------------------------------

print("\nMaterializing new S3 pairs...")

conn.execute("""
    CREATE TEMP TABLE new_route_a_s3 AS
    SELECT
        r.source1_entity_id,
        r.candidate_entity_id
    FROM route_a_s3 r
    LEFT JOIN unique_candidates u
        ON u.source1_entity_id = r.source1_entity_id
       AND u.candidate_entity_id = r.candidate_entity_id
    WHERE u.source1_entity_id IS NULL
""")

count_s3 = conn.execute("""
    SELECT COUNT(*)
    FROM new_route_a_s3
""").fetchone()[0]

print(f"New S3 materialized: {count_s3:,}")


# ------------------------------------------------------------
# FINAL CHECK
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("MATERIALIZATION RESULT")
print("=" * 80)

print(f"S2 new pairs:    {count_s2:,}")
print(f"S3 new pairs:    {count_s3:,}")
print(f"Total new pairs: {count_s2 + count_s3:,}")

print("\nExpected:")
print("S2: 6,473,697")
print("S3: 7,936,744")
print("Total: 14,410,441")

print("\n" + "=" * 80)
print("STEP 24C COMPLETE")
print("=" * 80)

STEP 24C — MATERIALIZING NEW ROUTE A PAIRS

Materializing new S2 pairs...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

New S2 materialized: 6,473,697

Materializing new S3 pairs...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

New S3 materialized: 7,936,744

MATERIALIZATION RESULT
S2 new pairs:    6,473,697
S3 new pairs:    7,936,744
Total new pairs: 14,410,441

Expected:
S2: 6,473,697
S3: 7,936,744
Total: 14,410,441

STEP 24C COMPLETE


In [23]:
# ============================================================
# MATCHING — STEP 24D
# SAFE BATCH INSERT OF ROUTE A CANDIDATES
# ============================================================

print("=" * 80)
print("STEP 24D — SAFE BATCH INSERT")
print("=" * 80)

conn.execute("SET threads=2")
conn.execute("SET preserve_insertion_order=false")
conn.execute("PRAGMA max_temp_directory_size='10GiB'")

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

NUM_BUCKETS = 16

# ------------------------------------------------------------
# CHECK STARTING COUNT
# ------------------------------------------------------------

before_count = conn.execute("""
    SELECT COUNT(*)
    FROM unique_candidates
""").fetchone()[0]

print(f"\nStarting unique_candidates: {before_count:,}")
print(f"Expected starting count:    23,413,014")

if before_count != 23_413_014:
    raise RuntimeError(
        f"Unexpected starting count: {before_count:,}. "
        "Stopping before any insert."
    )

# ------------------------------------------------------------
# INSERT S2 IN HASH BUCKETS
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("INSERTING S2")
print("-" * 80)

s2_inserted = 0

for bucket in range(NUM_BUCKETS):

    conn.execute("BEGIN TRANSACTION")

    conn.execute(f"""
        INSERT INTO unique_candidates
        SELECT
            source1_entity_id,
            candidate_entity_id
        FROM new_route_a_s2
        WHERE
            MOD(
                HASH(source1_entity_id || '|' || candidate_entity_id),
                {NUM_BUCKETS}
            ) = {bucket}
    """)

    conn.execute("COMMIT")

    current = conn.execute("""
        SELECT COUNT(*)
        FROM unique_candidates
    """).fetchone()[0]

    added = current - before_count
    s2_inserted = added

    print(
        f"S2 bucket {bucket + 1:2}/{NUM_BUCKETS} | "
        f"current total: {current:,} | "
        f"Route A added so far: {added:,}"
    )


# ------------------------------------------------------------
# INSERT S3 IN HASH BUCKETS
# ------------------------------------------------------------

s2_total = conn.execute("""
    SELECT COUNT(*)
    FROM unique_candidates
""").fetchone()[0]

print("\n" + "-" * 80)
print("INSERTING S3")
print("-" * 80)

for bucket in range(NUM_BUCKETS):

    conn.execute("BEGIN TRANSACTION")

    conn.execute(f"""
        INSERT INTO unique_candidates
        SELECT
            source1_entity_id,
            candidate_entity_id
        FROM new_route_a_s3
        WHERE
            MOD(
                HASH(source1_entity_id || '|' || candidate_entity_id),
                {NUM_BUCKETS}
            ) = {bucket}
    """)

    conn.execute("COMMIT")

    current = conn.execute("""
        SELECT COUNT(*)
        FROM unique_candidates
    """).fetchone()[0]

    added = current - before_count

    print(
        f"S3 bucket {bucket + 1:2}/{NUM_BUCKETS} | "
        f"current total: {current:,} | "
        f"Route A added so far: {added:,}"
    )


# ------------------------------------------------------------
# FINAL VERIFICATION
# ------------------------------------------------------------

final_count = conn.execute("""
    SELECT COUNT(*)
    FROM unique_candidates
""").fetchone()[0]

actual_added = final_count - before_count

print("\n" + "=" * 80)
print("STEP 24D RESULT")
print("=" * 80)

print(f"Before:       {before_count:,}")
print(f"After:        {final_count:,}")
print(f"Actually added: {actual_added:,}")
print(f"Expected added: 14,410,441")
print(f"Expected final: 37,823,455")

if actual_added == 14_410_441 and final_count == 37_823_455:
    print("\n✓ INSERT SUCCESSFUL")
    print("✓ ALL 14,410,441 NEW ROUTE A PAIRS ADDED")
    print("✓ FINAL COUNT IS CORRECT")
else:
    print("\n⚠ COUNT DOES NOT MATCH EXPECTATION")
    print("STOP HERE — DO NOT CONTINUE TO THE NEXT STEP.")

print("=" * 80)
print("STEP 24D COMPLETE")
print("=" * 80)

STEP 24D — SAFE BATCH INSERT

Starting unique_candidates: 23,413,014
Expected starting count:    23,413,014

--------------------------------------------------------------------------------
INSERTING S2
--------------------------------------------------------------------------------


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

TransactionException: TransactionContext Error: Failed to commit: Invalid node type for ARTOperator::Insert.

In [24]:
# ============================================================
# STEP 24E — VERIFY AFTER FAILED BATCH INSERT
# ============================================================

print("=" * 80)
print("STEP 24E — VERIFYING DATABASE AFTER FAILED INSERT")
print("=" * 80)

# Check current candidate count
current_count = conn.execute("""
    SELECT COUNT(*)
    FROM unique_candidates
""").fetchone()[0]

print(f"\nCurrent unique_candidates: {current_count:,}")
print(f"Original count:            23,413,014")

# Check materialized tables
s2_count = conn.execute("""
    SELECT COUNT(*)
    FROM new_route_a_s2
""").fetchone()[0]

s3_count = conn.execute("""
    SELECT COUNT(*)
    FROM new_route_a_s3
""").fetchone()[0]

print(f"\nnew_route_a_s2: {s2_count:,}")
print(f"new_route_a_s3: {s3_count:,}")

# Check original route tables
route_s2_count = conn.execute("""
    SELECT COUNT(*)
    FROM route_a_s2
""").fetchone()[0]

route_s3_count = conn.execute("""
    SELECT COUNT(*)
    FROM route_a_s3
""").fetchone()[0]

print(f"\nroute_a_s2: {route_s2_count:,}")
print(f"route_a_s3: {route_s3_count:,}")

print("\n" + "=" * 80)
print("STEP 24E COMPLETE")
print("=" * 80)

STEP 24E — VERIFYING DATABASE AFTER FAILED INSERT

Current unique_candidates: 23,413,014
Original count:            23,413,014

new_route_a_s2: 6,473,697
new_route_a_s3: 7,936,744

route_a_s2: 7,125,964
route_a_s3: 8,378,357

STEP 24E COMPLETE


In [25]:
# ============================================================
# STEP 24F — BUILD NEW CANDIDATE TABLE
# ============================================================

print("=" * 80)
print("STEP 24F — BUILDING unique_candidates_v2")
print("=" * 80)

conn.execute("SET threads=2")
conn.execute("SET preserve_insertion_order=false")
conn.execute("PRAGMA max_temp_directory_size='10GiB'")

# ------------------------------------------------------------
# REMOVE ANY PREVIOUS ATTEMPT
# ------------------------------------------------------------

conn.execute("""
    DROP TABLE IF EXISTS unique_candidates_v2
""")

print("\nCreating unique_candidates_v2...")

# ------------------------------------------------------------
# BUILD NEW TABLE
#
# Existing candidates
#        +
# new Route A S2
#        +
# new Route A S3
#        ↓
# DISTINCT pairs
# ------------------------------------------------------------

conn.execute("""
    CREATE TABLE unique_candidates_v2 AS

    SELECT
        source1_entity_id,
        candidate_entity_id
    FROM unique_candidates

    UNION

    SELECT
        source1_entity_id,
        candidate_entity_id
    FROM new_route_a_s2

    UNION

    SELECT
        source1_entity_id,
        candidate_entity_id
    FROM new_route_a_s3
""")

print("\n✓ unique_candidates_v2 created")


# ------------------------------------------------------------
# COUNT
# ------------------------------------------------------------

v2_count = conn.execute("""
    SELECT COUNT(*)
    FROM unique_candidates_v2
""").fetchone()[0]

print("\n" + "=" * 80)
print("RESULT")
print("=" * 80)

print(f"Old unique_candidates: {23_413_014:,}")
print(f"New unique_candidates_v2: {v2_count:,}")
print(f"Expected: {37_823_455:,}")

if v2_count == 37_823_455:
    print("\n✓ COUNT IS EXACTLY CORRECT")
else:
    print("\n⚠ COUNT DOES NOT MATCH EXPECTATION")
    print("STOP — DO NOT DROP OR RENAME ANY TABLE.")

print("=" * 80)
print("STEP 24F COMPLETE")
print("=" * 80)

STEP 24F — BUILDING unique_candidates_v2

Creating unique_candidates_v2...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


✓ unique_candidates_v2 created

RESULT
Old unique_candidates: 23,413,014
New unique_candidates_v2: 37,823,455
Expected: 37,823,455

✓ COUNT IS EXACTLY CORRECT
STEP 24F COMPLETE


In [26]:
# ============================================================
# STEP 24G — VERIFY unique_candidates_v2 INTEGRITY
# ============================================================

print("=" * 80)
print("STEP 24G — VERIFYING unique_candidates_v2")
print("=" * 80)

# ------------------------------------------------------------
# 1. TOTAL ROWS
# ------------------------------------------------------------

total_rows = conn.execute("""
    SELECT COUNT(*)
    FROM unique_candidates_v2
""").fetchone()[0]

print(f"\nTotal rows: {total_rows:,}")


# ------------------------------------------------------------
# 2. DISTINCT PAIRS
# ------------------------------------------------------------

distinct_pairs = conn.execute("""
    SELECT COUNT(*)
    FROM (
        SELECT DISTINCT
            source1_entity_id,
            candidate_entity_id
        FROM unique_candidates_v2
    )
""").fetchone()[0]

print(f"Distinct pairs: {distinct_pairs:,}")


# ------------------------------------------------------------
# 3. DUPLICATE CHECK
# ------------------------------------------------------------

duplicates = total_rows - distinct_pairs

print(f"Duplicate rows: {duplicates:,}")


# ------------------------------------------------------------
# 4. S2 / S3 DISTRIBUTION
# ------------------------------------------------------------

s2_count = conn.execute("""
    SELECT COUNT(*)
    FROM unique_candidates_v2
    WHERE candidate_entity_id LIKE 'S2-%'
""").fetchone()[0]

s3_count = conn.execute("""
    SELECT COUNT(*)
    FROM unique_candidates_v2
    WHERE candidate_entity_id LIKE 'S3-%'
""").fetchone()[0]

print("\nCandidate source:")
print(f"S2 candidates: {s2_count:,}")
print(f"S3 candidates: {s3_count:,}")
print(f"S2 + S3:       {s2_count + s3_count:,}")


# ------------------------------------------------------------
# 5. NULL CHECK
# ------------------------------------------------------------

null_pairs = conn.execute("""
    SELECT COUNT(*)
    FROM unique_candidates_v2
    WHERE source1_entity_id IS NULL
       OR candidate_entity_id IS NULL
""").fetchone()[0]

print(f"\nNULL candidate pairs: {null_pairs:,}")


# ------------------------------------------------------------
# FINAL
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("INTEGRITY RESULT")
print("=" * 80)

if (
    total_rows == 37_823_455
    and distinct_pairs == 37_823_455
    and duplicates == 0
    and null_pairs == 0
    and s2_count + s3_count == 37_823_455
):
    print("✓ TABLE INTEGRITY PASSED")
    print("✓ 37,823,455 UNIQUE PAIRS")
    print("✓ NO DUPLICATES")
    print("✓ NO NULL PAIRS")
else:
    print("⚠ INTEGRITY CHECK FAILED")
    print("STOP — DO NOT SWITCH TABLES.")

print("=" * 80)
print("STEP 24G COMPLETE")
print("=" * 80)

STEP 24G — VERIFYING unique_candidates_v2

Total rows: 37,823,455


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Distinct pairs: 37,823,455
Duplicate rows: 0

Candidate source:
S2 candidates: 17,787,343
S3 candidates: 20,036,112
S2 + S3:       37,823,455

NULL candidate pairs: 0

INTEGRITY RESULT
✓ TABLE INTEGRITY PASSED
✓ 37,823,455 UNIQUE PAIRS
✓ NO DUPLICATES
✓ NO NULL PAIRS
STEP 24G COMPLETE


In [27]:
# ============================================================
# STEP 24H — RECALCULATE GROUND-TRUTH CANDIDATE RECALL
# ============================================================

print("=" * 80)
print("STEP 24H — CANDIDATE RECALL AFTER ROUTE A")
print("=" * 80)

conn.execute("SET threads=2")
conn.execute("SET preserve_insertion_order=false")
conn.execute("PRAGMA max_temp_directory_size='10GiB'")

# ------------------------------------------------------------
# DROP PREVIOUS TEMP RESULT
# ------------------------------------------------------------

conn.execute("""
    DROP TABLE IF EXISTS ground_truth_candidate_recall_v2
""")


# ------------------------------------------------------------
# COMPARE ALL GROUND-TRUTH PAIRS
# AGAINST NEW CANDIDATE TABLE
# ------------------------------------------------------------

print("\nChecking ground-truth pairs against unique_candidates_v2...")

conn.execute("""
    CREATE TEMP TABLE ground_truth_candidate_recall_v2 AS

    SELECT
        gt.source1_entity_id,
        gt.candidate_entity_id,

        CASE
            WHEN c.source1_entity_id IS NOT NULL
            THEN 1
            ELSE 0
        END AS found

    FROM ground_truth_pairs gt

    LEFT JOIN unique_candidates_v2 c
        ON c.source1_entity_id = gt.source1_entity_id
       AND c.candidate_entity_id = gt.candidate_entity_id
""")


# ------------------------------------------------------------
# PAIR-LEVEL RECALL
# ------------------------------------------------------------

stats = conn.execute("""
    SELECT
        COUNT(*) AS total_gt_pairs,
        SUM(found) AS found_pairs,
        COUNT(*) - SUM(found) AS missing_pairs,
        ROUND(
            100.0 * SUM(found) / COUNT(*),
            2
        ) AS pair_recall_pct
    FROM ground_truth_candidate_recall_v2
""").fetchone()

total_gt = stats[0]
found_gt = stats[1]
missing_gt = stats[2]
pair_recall = stats[3]

print("\n" + "=" * 80)
print("PAIR-LEVEL RECALL")
print("=" * 80)

print(f"Total ground-truth pairs: {total_gt:,}")
print(f"Found in candidates:      {found_gt:,}")
print(f"Missing:                  {missing_gt:,}")
print(f"Pair recall:              {pair_recall:.2f}%")


# ------------------------------------------------------------
# S1-LEVEL RECALL
# ------------------------------------------------------------

s1_stats = conn.execute("""
    WITH gt AS (
        SELECT
            source1_entity_id,
            COUNT(*) AS gt_count
        FROM ground_truth_pairs
        GROUP BY source1_entity_id
    ),

    found AS (
        SELECT
            source1_entity_id,
            SUM(found) AS found_count
        FROM ground_truth_candidate_recall_v2
        GROUP BY source1_entity_id
    )

    SELECT
        COUNT(*) AS s1_with_ground_truth,

        SUM(
            CASE
                WHEN found_count > 0 THEN 1
                ELSE 0
            END
        ) AS s1_any_candidate,

        SUM(
            CASE
                WHEN found_count = gt_count THEN 1
                ELSE 0
            END
        ) AS s1_all_candidates

    FROM gt
    LEFT JOIN found
        USING (source1_entity_id)
""").fetchone()

s1_total = s1_stats[0]
s1_any = s1_stats[1]
s1_all = s1_stats[2]

print("\n" + "=" * 80)
print("S1-LEVEL RECALL")
print("=" * 80)

print(f"S1 with ground truth: {s1_total:,}")
print(f"S1 with ≥1 true candidate: {s1_any:,}")
print(f"S1 with ALL true candidates: {s1_all:,}")

print(
    f"Any-candidate recall: "
    f"{100.0 * s1_any / s1_total:.2f}%"
)

print(
    f"All-candidate recall: "
    f"{100.0 * s1_all / s1_total:.2f}%"
)


# ------------------------------------------------------------
# COMPARE WITH OLD RECALL
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("COMPARISON WITH OLD CANDIDATE SET")
print("=" * 80)

print("OLD:")
print("  Pair recall:       29.87%")
print("  Any-candidate S1:  66.06%")
print("  All-candidate S1:   4.79%")

print("\nNEW:")
print(f"  Pair recall:       {pair_recall:.2f}%")
print(f"  Any-candidate S1:  {100.0 * s1_any / s1_total:.2f}%")
print(f"  All-candidate S1:  {100.0 * s1_all / s1_total:.2f}%")

print("\n" + "=" * 80)
print("STEP 24H COMPLETE")
print("=" * 80)

STEP 24H — CANDIDATE RECALL AFTER ROUTE A

Checking ground-truth pairs against unique_candidates_v2...


BinderException: Binder Error: Table "gt" does not have a column named "candidate_entity_id"

Candidate bindings: : "matched_entity_id"

LINE 18:        AND c.candidate_entity_id = gt.candidate_entity_id
                                            ^

In [28]:
# ============================================================
# STEP 24H — CANDIDATE RECALL AFTER ROUTE A
# CORRECTED FOR matched_entity_id
# ============================================================

print("=" * 80)
print("STEP 24H — CANDIDATE RECALL AFTER ROUTE A")
print("=" * 80)

conn.execute("SET threads=2")
conn.execute("SET preserve_insertion_order=false")
conn.execute("PRAGMA max_temp_directory_size='10GiB'")


# ------------------------------------------------------------
# DROP FAILED/OLD TEMP RESULT
# ------------------------------------------------------------

conn.execute("""
    DROP TABLE IF EXISTS ground_truth_candidate_recall_v2
""")


# ------------------------------------------------------------
# COMPARE GROUND TRUTH AGAINST NEW CANDIDATE TABLE
# ------------------------------------------------------------

print("\nChecking ground-truth pairs against unique_candidates_v2...")

conn.execute("""
    CREATE TEMP TABLE ground_truth_candidate_recall_v2 AS

    SELECT
        gt.source1_entity_id,
        gt.matched_entity_id,

        CASE
            WHEN c.source1_entity_id IS NOT NULL
            THEN 1
            ELSE 0
        END AS found

    FROM ground_truth_pairs gt

    LEFT JOIN unique_candidates_v2 c
        ON c.source1_entity_id = gt.source1_entity_id
       AND c.candidate_entity_id = gt.matched_entity_id
""")


print("✓ Ground-truth comparison table created")


# ------------------------------------------------------------
# PAIR-LEVEL RECALL
# ------------------------------------------------------------

stats = conn.execute("""
    SELECT
        COUNT(*) AS total_gt_pairs,
        SUM(found) AS found_pairs,
        COUNT(*) - SUM(found) AS missing_pairs,
        ROUND(
            100.0 * SUM(found) / COUNT(*),
            2
        ) AS pair_recall_pct
    FROM ground_truth_candidate_recall_v2
""").fetchone()

total_gt = stats[0]
found_gt = stats[1]
missing_gt = stats[2]
pair_recall = stats[3]


print("\n" + "=" * 80)
print("PAIR-LEVEL RECALL")
print("=" * 80)

print(f"Total ground-truth pairs: {total_gt:,}")
print(f"Found in candidates:      {found_gt:,}")
print(f"Missing:                  {missing_gt:,}")
print(f"Pair recall:              {pair_recall:.2f}%")


# ------------------------------------------------------------
# S1-LEVEL RECALL
# ------------------------------------------------------------

s1_stats = conn.execute("""
    WITH gt AS (
        SELECT
            source1_entity_id,
            COUNT(*) AS gt_count
        FROM ground_truth_pairs
        GROUP BY source1_entity_id
    ),

    found AS (
        SELECT
            source1_entity_id,
            SUM(found) AS found_count
        FROM ground_truth_candidate_recall_v2
        GROUP BY source1_entity_id
    )

    SELECT
        COUNT(*) AS s1_with_ground_truth,

        SUM(
            CASE
                WHEN COALESCE(found_count, 0) > 0
                THEN 1
                ELSE 0
            END
        ) AS s1_any_candidate,

        SUM(
            CASE
                WHEN COALESCE(found_count, 0) = gt_count
                THEN 1
                ELSE 0
            END
        ) AS s1_all_candidates

    FROM gt
    LEFT JOIN found
        USING (source1_entity_id)
""").fetchone()

s1_total = s1_stats[0]
s1_any = s1_stats[1]
s1_all = s1_stats[2]


print("\n" + "=" * 80)
print("S1-LEVEL RECALL")
print("=" * 80)

print(f"S1 with ground truth:       {s1_total:,}")
print(f"S1 with ≥1 true candidate:  {s1_any:,}")
print(f"S1 with ALL true candidates: {s1_all:,}")

print(
    f"Any-candidate recall: "
    f"{100.0 * s1_any / s1_total:.2f}%"
)

print(
    f"All-candidate recall: "
    f"{100.0 * s1_all / s1_total:.2f}%"
)


# ------------------------------------------------------------
# COMPARISON
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("COMPARISON WITH OLD CANDIDATE SET")
print("=" * 80)

print("OLD:")
print("  Pair recall:       29.87%")
print("  Any-candidate S1:  66.06%")
print("  All-candidate S1:   4.79%")

print("\nNEW:")
print(f"  Pair recall:       {pair_recall:.2f}%")
print(f"  Any-candidate S1:  {100.0 * s1_any / s1_total:.2f}%")
print(f"  All-candidate S1:  {100.0 * s1_all / s1_total:.2f}%")

print("\n" + "=" * 80)
print("STEP 24H COMPLETE")
print("=" * 80)

STEP 24H — CANDIDATE RECALL AFTER ROUTE A

Checking ground-truth pairs against unique_candidates_v2...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ Ground-truth comparison table created

PAIR-LEVEL RECALL
Total ground-truth pairs: 7,638,365
Found in candidates:      3,862,720
Missing:                  3,775,645
Pair recall:              50.57%


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


S1-LEVEL RECALL
S1 with ground truth:       2,083,574
S1 with ≥1 true candidate:  1,718,528
S1 with ALL true candidates: 365,871
Any-candidate recall: 82.48%
All-candidate recall: 17.56%

COMPARISON WITH OLD CANDIDATE SET
OLD:
  Pair recall:       29.87%
  Any-candidate S1:  66.06%
  All-candidate S1:   4.79%

NEW:
  Pair recall:       50.57%
  Any-candidate S1:  82.48%
  All-candidate S1:  17.56%

STEP 24H COMPLETE


In [29]:
# ============================================================
# STEP 24I — ANALYZE REMAINING MISSING GROUND-TRUTH PAIRS
# ============================================================

print("=" * 80)
print("STEP 24I — REMAINING MISSING GT PAIR ANALYSIS")
print("=" * 80)

# ------------------------------------------------------------
# BUILD REMAINING MISSING PAIRS
# ------------------------------------------------------------

conn.execute("""
    DROP TABLE IF EXISTS missing_gt_v2
""")

print("\nFinding remaining missing ground-truth pairs...")

conn.execute("""
    CREATE TEMP TABLE missing_gt_v2 AS

    SELECT
        r.source1_entity_id,
        r.matched_entity_id
    FROM ground_truth_candidate_recall_v2 r
    WHERE r.found = 0
""")

missing_count = conn.execute("""
    SELECT COUNT(*)
    FROM missing_gt_v2
""").fetchone()[0]

print(f"Remaining missing pairs: {missing_count:,}")


# ------------------------------------------------------------
# ATTACH BLOCKING DB
# ------------------------------------------------------------

try:
    conn.execute("DETACH DATABASE blocking_db")
except Exception:
    pass

conn.execute(
    f"ATTACH '{BLOCK_DB}' AS blocking_db (READ_ONLY)"
)

# ------------------------------------------------------------
# BUILD DETAIL TABLE
# ------------------------------------------------------------

conn.execute("""
    DROP TABLE IF EXISTS missing_gt_detail_v2
""")

print("\nBuilding blocking-key agreement analysis...")

conn.execute("""
    CREATE TEMP TABLE missing_gt_detail_v2 AS

    SELECT

        m.source1_entity_id,
        m.matched_entity_id,

        s1.country_norm AS s1_country,
        s2.country_norm AS s2_country,

        s1.name_norm AS s1_name,
        s2.name_norm AS s2_name,

        s1.address_norm AS s1_address,
        s2.address_norm AS s2_address,

        s1.name_core AS s1_name_core,
        s2.name_core AS s2_name_core,

        s1.name_prefix4 AS s1_name_prefix4,
        s2.name_prefix4 AS s2_name_prefix4,

        s1.name_prefix6 AS s1_name_prefix6,
        s2.name_prefix6 AS s2_name_prefix6,

        s1.address_prefix6 AS s1_address_prefix6,
        s2.address_prefix6 AS s2_address_prefix6,

        s1.address_prefix10 AS s1_address_prefix10,
        s2.address_prefix10 AS s2_address_prefix10

    FROM missing_gt_v2 m

    JOIN blocking_db.train_s1_blocking s1
        ON s1.entity_id = m.source1_entity_id

    JOIN (
        SELECT * FROM blocking_db.train_s2_blocking
        UNION ALL
        SELECT * FROM blocking_db.train_s3_blocking
    ) s2
        ON s2.entity_id = m.matched_entity_id
""")


# ------------------------------------------------------------
# AGREEMENT COUNTS
# ------------------------------------------------------------

result = conn.execute("""
    SELECT

        COUNT(*) AS total_pairs,

        SUM(
            s1_country = s2_country
        ) AS country_exact,

        SUM(
            s1_name = s2_name
        ) AS name_exact,

        SUM(
            s1_address = s2_address
        ) AS address_exact,

        SUM(
            s1_name_core = s2_name_core
        ) AS name_core_exact,

        SUM(
            s1_name_prefix4 = s2_name_prefix4
        ) AS name_prefix4_exact,

        SUM(
            s1_name_prefix6 = s2_name_prefix6
        ) AS name_prefix6_exact,

        SUM(
            s1_address_prefix6 = s2_address_prefix6
        ) AS address_prefix6_exact,

        SUM(
            s1_address_prefix10 = s2_address_prefix10
        ) AS address_prefix10_exact

    FROM missing_gt_detail_v2
""").fetchone()

labels = [
    "total_pairs",
    "country_exact",
    "name_exact",
    "address_exact",
    "name_core_exact",
    "name_prefix4_exact",
    "name_prefix6_exact",
    "address_prefix6_exact",
    "address_prefix10_exact"
]

print("\n" + "=" * 80)
print("BLOCKING KEY AGREEMENT")
print("=" * 80)

for label, value in zip(labels, result):
    print(f"{label:25s}: {value:,}")


# ------------------------------------------------------------
# COMBINATION ANALYSIS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("MAIN BLOCKING PATTERNS")
print("=" * 80)

patterns = conn.execute("""
    SELECT
        CASE

            WHEN s1_name_prefix6 = s2_name_prefix6
             AND s1_address_prefix6 = s2_address_prefix6
            THEN 'NAME PREFIX6 + ADDRESS PREFIX6'

            WHEN s1_name_prefix4 = s2_name_prefix4
             AND s1_address_prefix6 = s2_address_prefix6
            THEN 'NAME PREFIX4 + ADDRESS PREFIX6'

            WHEN s1_name_core = s2_name_core
            THEN 'NAME CORE ONLY'

            WHEN s1_name_prefix6 = s2_name_prefix6
            THEN 'NAME PREFIX6 ONLY'

            WHEN s1_address_prefix6 = s2_address_prefix6
            THEN 'ADDRESS PREFIX6 ONLY'

            WHEN s1_address_prefix10 = s2_address_prefix10
            THEN 'ADDRESS PREFIX10 ONLY'

            ELSE 'NO CURRENT KEY AGREEMENT'

        END AS pattern,

        COUNT(*) AS pair_count

    FROM missing_gt_detail_v2

    GROUP BY pattern

    ORDER BY pair_count DESC
""").fetchall()

for pattern, count in patterns:
    print(f"{pattern:45s} {count:,}")


# ------------------------------------------------------------
# DETACH
# ------------------------------------------------------------

try:
    conn.execute("DETACH DATABASE blocking_db")
except Exception:
    pass


print("\n" + "=" * 80)
print("STEP 24I COMPLETE")
print("=" * 80)

STEP 24I — REMAINING MISSING GT PAIR ANALYSIS

Finding remaining missing ground-truth pairs...
Remaining missing pairs: 3,775,645


NameError: name 'BLOCK_DB' is not defined

In [30]:
# ============================================================
# STEP 24I — ANALYZE REMAINING MISSING GROUND-TRUTH PAIRS
# CORRECTED BLOCK_DB PATH
# ============================================================

print("=" * 80)
print("STEP 24I — REMAINING MISSING GT PAIR ANALYSIS")
print("=" * 80)

# ------------------------------------------------------------
# DATABASE PATH
# ------------------------------------------------------------

BLOCK_DB = "/content/drive/MyDrive/er_checkpoint/er_blocking_train_fresh.duckdb"

# ------------------------------------------------------------
# BUILD REMAINING MISSING PAIRS
# ------------------------------------------------------------

conn.execute("""
    DROP TABLE IF EXISTS missing_gt_v2
""")

print("\nFinding remaining missing ground-truth pairs...")

conn.execute("""
    CREATE TEMP TABLE missing_gt_v2 AS

    SELECT
        source1_entity_id,
        matched_entity_id
    FROM ground_truth_candidate_recall_v2
    WHERE found = 0
""")

missing_count = conn.execute("""
    SELECT COUNT(*)
    FROM missing_gt_v2
""").fetchone()[0]

print(f"Remaining missing pairs: {missing_count:,}")


# ------------------------------------------------------------
# ATTACH BLOCKING DB
# ------------------------------------------------------------

try:
    conn.execute("DETACH DATABASE blocking_db")
except Exception:
    pass

conn.execute(
    f"ATTACH '{BLOCK_DB}' AS blocking_db (READ_ONLY)"
)

print("✓ Blocking database attached")


# ------------------------------------------------------------
# BUILD DETAIL TABLE
# ------------------------------------------------------------

conn.execute("""
    DROP TABLE IF EXISTS missing_gt_detail_v2
""")

print("\nBuilding blocking-key agreement analysis...")

conn.execute("""
    CREATE TEMP TABLE missing_gt_detail_v2 AS

    SELECT

        m.source1_entity_id,
        m.matched_entity_id,

        s1.country_norm AS s1_country,
        s2.country_norm AS s2_country,

        s1.name_norm AS s1_name,
        s2.name_norm AS s2_name,

        s1.address_norm AS s1_address,
        s2.address_norm AS s2_address,

        s1.name_core AS s1_name_core,
        s2.name_core AS s2_name_core,

        s1.name_prefix4 AS s1_name_prefix4,
        s2.name_prefix4 AS s2_name_prefix4,

        s1.name_prefix6 AS s1_name_prefix6,
        s2.name_prefix6 AS s2_name_prefix6,

        s1.address_prefix6 AS s1_address_prefix6,
        s2.address_prefix6 AS s2_address_prefix6,

        s1.address_prefix10 AS s1_address_prefix10,
        s2.address_prefix10 AS s2_address_prefix10

    FROM missing_gt_v2 m

    JOIN blocking_db.train_s1_blocking s1
        ON s1.entity_id = m.source1_entity_id

    JOIN (
        SELECT * FROM blocking_db.train_s2_blocking
        UNION ALL
        SELECT * FROM blocking_db.train_s3_blocking
    ) s2
        ON s2.entity_id = m.matched_entity_id
""")

print("✓ Detail table created")


# ------------------------------------------------------------
# AGREEMENT COUNTS
# ------------------------------------------------------------

result = conn.execute("""
    SELECT

        COUNT(*) AS total_pairs,

        SUM(
            s1_country = s2_country
        ) AS country_exact,

        SUM(
            s1_name = s2_name
        ) AS name_exact,

        SUM(
            s1_address = s2_address
        ) AS address_exact,

        SUM(
            s1_name_core = s2_name_core
        ) AS name_core_exact,

        SUM(
            s1_name_prefix4 = s2_name_prefix4
        ) AS name_prefix4_exact,

        SUM(
            s1_name_prefix6 = s2_name_prefix6
        ) AS name_prefix6_exact,

        SUM(
            s1_address_prefix6 = s2_address_prefix6
        ) AS address_prefix6_exact,

        SUM(
            s1_address_prefix10 = s2.address_prefix10
        ) AS address_prefix10_exact

    FROM missing_gt_detail_v2
""").fetchone()

STEP 24I — REMAINING MISSING GT PAIR ANALYSIS

Finding remaining missing ground-truth pairs...
Remaining missing pairs: 3,775,645


IOException: IO Error: Could not read enough bytes from file "/content/drive/MyDrive/er_checkpoint/er_blocking_train_fresh.duckdb": attempted to read 4 bytes from location 8

In [31]:
# ============================================================
# STEP 24I-A — CHECK BLOCKING DATABASE
# ============================================================

import os

BLOCK_DB = "/content/drive/MyDrive/er_checkpoint/er_blocking_train_fresh.duckdb"
LOCAL_BLOCK_DB = "/content/er_blocking_analysis.duckdb"

print("=" * 80)
print("STEP 24I-A — CHECKING BLOCKING DATABASE")
print("=" * 80)

if os.path.exists(BLOCK_DB):
    size_gb = os.path.getsize(BLOCK_DB) / (1024**3)

    print(f"\nGoogle Drive file exists: YES")
    print(f"Size: {size_gb:.2f} GB")
else:
    print("\nGoogle Drive file exists: NO")
    raise FileNotFoundError(BLOCK_DB)

print(f"\nPath:")
print(BLOCK_DB)

print("\n" + "=" * 80)
print("STEP 24I-A COMPLETE")
print("=" * 80)

STEP 24I-A — CHECKING BLOCKING DATABASE

Google Drive file exists: YES
Size: 8.74 GB

Path:
/content/drive/MyDrive/er_checkpoint/er_blocking_train_fresh.duckdb

STEP 24I-A COMPLETE


In [32]:
# ============================================================
# STEP 24I-B — COPY BLOCKING DB TO LOCAL DISK
# ============================================================

import os
import shutil
import time

BLOCK_DB = "/content/drive/MyDrive/er_checkpoint/er_blocking_train_fresh.duckdb"
LOCAL_BLOCK_DB = "/content/er_blocking_analysis.duckdb"

print("=" * 80)
print("STEP 24I-B — COPYING BLOCKING DATABASE LOCALLY")
print("=" * 80)

# ------------------------------------------------------------
# CHECK SOURCE
# ------------------------------------------------------------

if not os.path.exists(BLOCK_DB):
    raise FileNotFoundError(BLOCK_DB)

source_size = os.path.getsize(BLOCK_DB)

print(f"\nSource size: {source_size / (1024**3):.2f} GB")
print(f"Source: {BLOCK_DB}")
print(f"Destination: {LOCAL_BLOCK_DB}")

# ------------------------------------------------------------
# REMOVE OLD LOCAL COPY IF PRESENT
# ------------------------------------------------------------

if os.path.exists(LOCAL_BLOCK_DB):
    print("\nRemoving previous local copy...")
    os.remove(LOCAL_BLOCK_DB)

# ------------------------------------------------------------
# COPY
# ------------------------------------------------------------

print("\nCopying... This may take several minutes.")

start = time.time()

shutil.copy2(
    BLOCK_DB,
    LOCAL_BLOCK_DB
)

elapsed = time.time() - start

# ------------------------------------------------------------
# VERIFY
# ------------------------------------------------------------

local_size = os.path.getsize(LOCAL_BLOCK_DB)

print("\n" + "=" * 80)
print("COPY RESULT")
print("=" * 80)

print(f"Source size:      {source_size / (1024**3):.2f} GB")
print(f"Local copy size:  {local_size / (1024**3):.2f} GB")
print(f"Copy time:        {elapsed / 60:.2f} minutes")

if source_size == local_size:
    print("\n✓ LOCAL COPY SIZE MATCHES SOURCE")
else:
    raise RuntimeError(
        f"Size mismatch! Source={source_size}, Local={local_size}"
    )

print("\nLocal database:")
print(LOCAL_BLOCK_DB)

print("\n" + "=" * 80)
print("STEP 24I-B COMPLETE")
print("=" * 80)

STEP 24I-B — COPYING BLOCKING DATABASE LOCALLY

Source size: 8.74 GB
Source: /content/drive/MyDrive/er_checkpoint/er_blocking_train_fresh.duckdb
Destination: /content/er_blocking_analysis.duckdb

Copying... This may take several minutes.

COPY RESULT
Source size:      8.74 GB
Local copy size:  0.00 GB
Copy time:        0.00 minutes


RuntimeError: Size mismatch! Source=9380835328, Local=0

In [33]:
# ============================================================
# STEP 24I-C — CHECK COLAB STORAGE
# ============================================================

import os
import shutil

BLOCK_DB = "/content/drive/MyDrive/er_checkpoint/er_blocking_train_fresh.duckdb"
LOCAL_BLOCK_DB = "/content/er_blocking_analysis.duckdb"

print("=" * 80)
print("STEP 24I-C — STORAGE DIAGNOSTIC")
print("=" * 80)

# Local disk
total, used, free = shutil.disk_usage("/content")

print("\nLOCAL /content STORAGE")
print(f"Total: {total / (1024**3):.2f} GB")
print(f"Used:  {used / (1024**3):.2f} GB")
print(f"Free:  {free / (1024**3):.2f} GB")

# Drive file
print("\nGOOGLE DRIVE DATABASE")

if os.path.exists(BLOCK_DB):
    print("Exists: YES")
    print(f"Size: {os.path.getsize(BLOCK_DB) / (1024**3):.2f} GB")
else:
    print("Exists: NO")

# Local failed copy
print("\nLOCAL COPY")

if os.path.exists(LOCAL_BLOCK_DB):
    print("Exists: YES")
    print(f"Size: {os.path.getsize(LOCAL_BLOCK_DB) / (1024**3):.4f} GB")
else:
    print("Exists: NO")

print("\n" + "=" * 80)
print("STEP 24I-C COMPLETE")
print("=" * 80)

STEP 24I-C — STORAGE DIAGNOSTIC

LOCAL /content STORAGE
Total: 107.72 GB
Used:  57.73 GB
Free:  49.97 GB

GOOGLE DRIVE DATABASE
Exists: YES
Size: 8.74 GB

LOCAL COPY
Exists: YES
Size: 0.0000 GB

STEP 24I-C COMPLETE


In [34]:
# ============================================================
# STEP 24I-D — COPY BLOCKING DB USING SYSTEM COPY
# ============================================================

import os
import subprocess
import time

BLOCK_DB = "/content/drive/MyDrive/er_checkpoint/er_blocking_train_fresh.duckdb"
LOCAL_BLOCK_DB = "/content/er_blocking_analysis.duckdb"

print("=" * 80)
print("STEP 24I-D — COPYING BLOCKING DATABASE")
print("=" * 80)

# ------------------------------------------------------------
# REMOVE FAILED ZERO-BYTE COPY
# ------------------------------------------------------------

if os.path.exists(LOCAL_BLOCK_DB):
    print("\nRemoving failed local copy...")
    os.remove(LOCAL_BLOCK_DB)

# ------------------------------------------------------------
# SOURCE CHECK
# ------------------------------------------------------------

source_size = os.path.getsize(BLOCK_DB)

print(f"\nSource: {BLOCK_DB}")
print(f"Source size: {source_size / (1024**3):.2f} GB")
print(f"Destination: {LOCAL_BLOCK_DB}")

# ------------------------------------------------------------
# COPY USING SYSTEM cp
# ------------------------------------------------------------

print("\nStarting copy...")
print("Please wait. Do not interrupt the Colab runtime.")

start = time.time()

result = subprocess.run(
    [
        "cp",
        "--reflink=auto",
        BLOCK_DB,
        LOCAL_BLOCK_DB
    ],
    capture_output=True,
    text=True
)

elapsed = time.time() - start

# ------------------------------------------------------------
# CHECK COMMAND
# ------------------------------------------------------------

if result.returncode != 0:
    print("\nCOPY COMMAND FAILED")
    print("STDOUT:")
    print(result.stdout)
    print("STDERR:")
    print(result.stderr)

    raise RuntimeError(
        f"cp failed with return code {result.returncode}"
    )

# ------------------------------------------------------------
# VERIFY SIZE
# ------------------------------------------------------------

if not os.path.exists(LOCAL_BLOCK_DB):
    raise RuntimeError("Destination file was not created.")

local_size = os.path.getsize(LOCAL_BLOCK_DB)

print("\n" + "=" * 80)
print("COPY RESULT")
print("=" * 80)

print(f"Source size: {source_size / (1024**3):.2f} GB")
print(f"Local size:  {local_size / (1024**3):.2f} GB")
print(f"Time:        {elapsed / 60:.2f} minutes")

if source_size == local_size:
    print("\n✓ COPY SUCCESSFUL")
    print("✓ SOURCE AND LOCAL FILE SIZES MATCH")
else:
    print("\n⚠ SIZE MISMATCH")
    print(f"Difference: {(source_size - local_size) / (1024**2):.2f} MB")

print("\nLocal database:")
print(LOCAL_BLOCK_DB)

print("\n" + "=" * 80)
print("STEP 24I-D COMPLETE")
print("=" * 80)

STEP 24I-D — COPYING BLOCKING DATABASE

Removing failed local copy...

Source: /content/drive/MyDrive/er_checkpoint/er_blocking_train_fresh.duckdb
Source size: 8.74 GB
Destination: /content/er_blocking_analysis.duckdb

Starting copy...
Please wait. Do not interrupt the Colab runtime.

COPY RESULT
Source size: 8.74 GB
Local size:  0.00 GB
Time:        0.00 minutes

⚠ SIZE MISMATCH
Difference: 8946.26 MB

Local database:
/content/er_blocking_analysis.duckdb

STEP 24I-D COMPLETE


In [35]:
# ============================================================
# STEP 24I-E — TEST BLOCKING DB READABILITY
# ============================================================

print("=" * 80)
print("STEP 24I-E — TESTING BLOCKING DATABASE")
print("=" * 80)

# Remove failed local copy
import os

LOCAL_BLOCK_DB = "/content/er_blocking_analysis.duckdb"

if os.path.exists(LOCAL_BLOCK_DB):
    try:
        os.remove(LOCAL_BLOCK_DB)
        print("\n✓ Removed failed 0-byte local copy")
    except Exception as e:
        print(f"\nCould not remove local copy: {e}")


# ------------------------------------------------------------
# TRY ATTACH
# ------------------------------------------------------------

BLOCK_DB = "/content/drive/MyDrive/er_checkpoint/er_blocking_train_fresh.duckdb"

print("\nTesting DuckDB attachment...")

try:
    # Remove stale attachment if any
    try:
        conn.execute("DETACH DATABASE blocking_db")
    except Exception:
        pass

    conn.execute(
        f"ATTACH '{BLOCK_DB}' AS blocking_db (READ_ONLY)"
    )

    print("✓ ATTACH SUCCEEDED")

    # Very small read
    result = conn.execute("""
        SELECT COUNT(*)
        FROM blocking_db.train_s1_blocking
    """).fetchone()[0]

    print(f"S1 blocking rows: {result:,}")

    # Another small metadata query
    cols = conn.execute("""
        SELECT column_name
        FROM (
            DESCRIBE blocking_db.train_s1_blocking
        )
    """).fetchall()

    print(f"Columns found: {len(cols)}")

    print("\n✓ BLOCKING DATABASE IS READABLE")

except Exception as e:

    print("\n⚠ BLOCKING DATABASE READ FAILED")
    print(type(e).__name__)
    print(str(e))

print("\n" + "=" * 80)
print("STEP 24I-E COMPLETE")
print("=" * 80)

STEP 24I-E — TESTING BLOCKING DATABASE

✓ Removed failed 0-byte local copy

Testing DuckDB attachment...

⚠ BLOCKING DATABASE READ FAILED
IOException
IO Error: Could not read enough bytes from file "/content/drive/MyDrive/er_checkpoint/er_blocking_train_fresh.duckdb": attempted to read 4 bytes from location 8

STEP 24I-E COMPLETE


In [36]:
# ============================================================
# STEP 24I-F — VERIFY HEALTHY SOURCE DATABASE
# ============================================================

SOURCE_DB = "/content/drive/MyDrive/er_checkpoint/er_train_fresh.duckdb"

print("=" * 80)
print("STEP 24I-F — VERIFYING HEALTHY SOURCE DATABASE")
print("=" * 80)

# Remove any stale attachment
try:
    conn.execute("DETACH DATABASE source_check")
except Exception:
    pass

try:
    conn.execute(
        f"ATTACH '{SOURCE_DB}' AS source_check (READ_ONLY)"
    )

    print("\n✓ Source DB ATTACHED successfully")

    # Check tables
    tables = conn.execute("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_catalog = 'source_check'
        ORDER BY table_name
    """).fetchall()

    print("\nTables:")
    for t in tables:
        print("  -", t[0])

    # Check S1 schema
    print("\nS1 schema:")
    schema = conn.execute("""
        DESCRIBE source_check.train_source1
    """).fetchall()

    for row in schema:
        print(f"  {row[0]:25} {row[1]}")

    # Small count check
    s1_count = conn.execute("""
        SELECT COUNT(*)
        FROM source_check.train_source1
    """).fetchone()[0]

    s2_count = conn.execute("""
        SELECT COUNT(*)
        FROM source_check.train_source2
    """).fetchone()[0]

    s3_count = conn.execute("""
        SELECT COUNT(*)
        FROM source_check.train_source3
    """).fetchone()[0]

    print("\nRow counts:")
    print(f"  train_source1: {s1_count:,}")
    print(f"  train_source2: {s2_count:,}")
    print(f"  train_source3: {s3_count:,}")

    print("\n✓ SOURCE DATABASE IS HEALTHY")

except Exception as e:

    print("\n⚠ SOURCE DATABASE READ FAILED")
    print(type(e).__name__)
    print(str(e))

print("\n" + "=" * 80)
print("STEP 24I-F COMPLETE")
print("=" * 80)

STEP 24I-F — VERIFYING HEALTHY SOURCE DATABASE

⚠ SOURCE DATABASE READ FAILED
BinderException
Binder Error: Unique file handle conflict: Database "source_db" is already attached with path "/content/drive/MyDrive/er_checkpoint/er_train_fresh.duckdb", 

STEP 24I-F COMPLETE


In [37]:
# ============================================================
# STEP 24I-G — VERIFY EXISTING SOURCE_DB ATTACHMENT
# ============================================================

print("=" * 80)
print("STEP 24I-G — VERIFYING EXISTING source_db")
print("=" * 80)

try:
    # Check whether source_db is already attached
    attached = conn.execute("""
        SELECT database_name, path
        FROM duckdb_databases()
        WHERE database_name = 'source_db'
    """).fetchall()

    print("\nAttached database:")
    for row in attached:
        print(f"  Database: {row[0]}")
        print(f"  Path:     {row[1]}")

    # Check source tables
    print("\nChecking source tables...")

    tables = conn.execute("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_catalog = 'source_db'
        ORDER BY table_name
    """).fetchall()

    for row in tables:
        print("  ✓", row[0])

    # Read row counts
    print("\nRow counts:")

    for table in [
        "train_source1",
        "train_source2",
        "train_source3"
    ]:
        count = conn.execute(
            f"SELECT COUNT(*) FROM source_db.{table}"
        ).fetchone()[0]

        print(f"  {table}: {count:,}")

    print("\n✓ EXISTING source_db IS HEALTHY")

except Exception as e:
    print("\n⚠ SOURCE_DB CHECK FAILED")
    print(type(e).__name__)
    print(str(e))

print("\n" + "=" * 80)
print("STEP 24I-G COMPLETE")
print("=" * 80)

STEP 24I-G — VERIFYING EXISTING source_db

Attached database:
  Database: source_db
  Path:     /content/drive/MyDrive/er_checkpoint/er_train_fresh.duckdb

Checking source tables...
  ✓ test_s1
  ✓ test_s2
  ✓ test_s3
  ✓ train_ground_truth
  ✓ train_s1
  ✓ train_s2
  ✓ train_s3

Row counts:

⚠ SOURCE_DB CHECK FAILED
CatalogException
Catalog Error: Table with name train_source1 does not exist!
Did you mean "train_s1"?

LINE 1: SELECT COUNT(*) FROM source_db.train_source1
                             ^

STEP 24I-G COMPLETE


In [38]:
# ============================================================
# STEP 24I-H — CORRECT SOURCE DATABASE VERIFICATION
# ============================================================

print("=" * 80)
print("STEP 24I-H — CORRECT SOURCE DATABASE VERIFICATION")
print("=" * 80)

# ------------------------------------------------------------
# ROW COUNTS
# ------------------------------------------------------------

print("\nRow counts:")

for table in [
    "train_s1",
    "train_s2",
    "train_s3"
]:
    count = conn.execute(
        f"SELECT COUNT(*) FROM source_db.{table}"
    ).fetchone()[0]

    print(f"  {table}: {count:,}")

# ------------------------------------------------------------
# SCHEMAS
# ------------------------------------------------------------

for table in ["train_s1", "train_s2", "train_s3"]:

    print("\n" + "-" * 60)
    print(f"Schema: source_db.{table}")
    print("-" * 60)

    schema = conn.execute(
        f"DESCRIBE source_db.{table}"
    ).fetchall()

    for row in schema:
        print(f"  {row[0]:25} {row[1]}")

print("\n" + "=" * 80)
print("✓ SOURCE DATABASE VERIFIED")
print("=" * 80)
print("We can rebuild the required blocking information")
print("from the healthy source DB.")
print("=" * 80)

STEP 24I-H — CORRECT SOURCE DATABASE VERIFICATION

Row counts:
  train_s1: 2,206,821
  train_s2: 5,034,616
  train_s3: 5,285,603

------------------------------------------------------------
Schema: source_db.train_s1
------------------------------------------------------------
  entity_id                 VARCHAR
  business_name             VARCHAR
  business_address          VARCHAR
  country                   VARCHAR

------------------------------------------------------------
Schema: source_db.train_s2
------------------------------------------------------------
  entity_id                 VARCHAR
  business_name             VARCHAR
  business_address          VARCHAR
  country                   VARCHAR

------------------------------------------------------------
Schema: source_db.train_s3
------------------------------------------------------------
  entity_id                 VARCHAR
  business_name             VARCHAR
  business_address          VARCHAR
  country                

In [39]:
# ============================================================
# STEP 24I-I — FIND EXISTING NORMALIZATION LOGIC
# ============================================================

print("=" * 80)
print("STEP 24I-I — INSPECTING EXISTING NORMALIZATION LOGIC")
print("=" * 80)

# ------------------------------------------------------------
# 1. List macros/functions created in this DuckDB connection
# ------------------------------------------------------------

print("\n[1] User-defined macros/functions:")

try:
    macros = conn.execute("""
        SELECT *
        FROM duckdb_functions()
        WHERE schema_name NOT IN ('pg_catalog', 'information_schema')
          AND (
              function_name ILIKE '%norm%'
              OR function_name ILIKE '%clean%'
              OR function_name ILIKE '%block%'
              OR function_name ILIKE '%core%'
          )
        ORDER BY function_name
    """).fetchall()

    if macros:
        for row in macros:
            print(row)
    else:
        print("  No matching functions found.")

except Exception as e:
    print("  Could not inspect functions:")
    print(type(e).__name__, str(e))


# ------------------------------------------------------------
# 2. Inspect tables in the current candidate DB that contain
#    normalized columns.
# ------------------------------------------------------------

print("\n[2] Tables containing normalized/blocking columns:")

try:
    tables = conn.execute("""
        SELECT table_schema, table_name
        FROM information_schema.tables
        WHERE table_schema NOT IN ('information_schema', 'pg_catalog')
        ORDER BY table_schema, table_name
    """).fetchall()

    for schema, table in tables:
        try:
            cols = conn.execute(
                f'DESCRIBE "{schema}"."{table}"'
            ).fetchall()

            col_names = [r[0] for r in cols]

            interesting = [
                c for c in col_names
                if any(x in c.lower() for x in [
                    "norm",
                    "core",
                    "prefix",
                    "country_key",
                    "address_key"
                ])
            ]

            if interesting:
                print(
                    f"  {schema}.{table}: "
                    + ", ".join(interesting)
                )

        except Exception:
            pass

except Exception as e:
    print("  Table inspection failed:")
    print(type(e).__name__, str(e))


# ------------------------------------------------------------
# 3. Check the existing matching_candidates schema
# ------------------------------------------------------------

print("\n[3] Existing matching_candidates schema:")

try:
    schema = conn.execute("""
        DESCRIBE matching_candidates
    """).fetchall()

    for row in schema:
        print(f"  {row[0]:30} {row[1]}")

except Exception as e:
    print("  matching_candidates not available:")
    print(type(e).__name__, str(e))


print("\n" + "=" * 80)
print("STEP 24I-I COMPLETE")
print("=" * 80)

STEP 24I-I — INSPECTING EXISTING NORMALIZATION LOGIC

[1] User-defined macros/functions:
('system', '0', 'main', 'get_block_size', None, 'macro', None, None, {}, None, ['db_name'], [None], None, '(SELECT block_size FROM pragma_database_size() WHERE (database_name = db_name))', None, True, 4596, [], None, [])
('system', '0', 'main', 'nfc_normalize', None, 'scalar', 'Converts `string` to Unicode NFC normalized string. Useful for comparisons and ordering if text data is mixed between NFC normalized and not.', None, {}, 'VARCHAR', ['string'], ['VARCHAR'], None, None, False, True, 478, ["nfc_normalize('ardèch')"], 'CONSISTENT', [])
('system', '0', 'main', 'normalized_interval', None, 'scalar', 'Normalizes an INTERVAL to an equivalent interval', None, {}, 'INTERVAL', ['interval'], ['INTERVAL'], None, None, False, True, 1942, ["normalized_interval(INTERVAL '30 days')"], 'CONSISTENT', [])

[2] Tables containing normalized/blocking columns:
  main.calibrated_eval: best_score, score_margin
  mai

In [40]:
# ============================================================
# STEP 24I-J — CHECK EXISTING MISSING GT DETAIL
# ============================================================

print("=" * 80)
print("STEP 24I-J — CHECKING EXISTING missing_gt_detail")
print("=" * 80)

# ------------------------------------------------------------
# 1. Row count
# ------------------------------------------------------------

detail_count = conn.execute("""
    SELECT COUNT(*)
    FROM missing_gt_detail
""").fetchone()[0]

print(f"\nmissing_gt_detail rows: {detail_count:,}")


# ------------------------------------------------------------
# 2. Current remaining missing pairs
# ------------------------------------------------------------

remaining_count = conn.execute("""
    SELECT COUNT(*)
    FROM missing_gt_v2
""").fetchone()[0]

print(f"missing_gt_v2 rows:     {remaining_count:,}")


# ------------------------------------------------------------
# 3. Show schema
# ------------------------------------------------------------

print("\nmissing_gt_detail columns:")

schema = conn.execute("""
    DESCRIBE missing_gt_detail
""").fetchall()

for row in schema:
    print(f"  {row[0]:30} {row[1]}")


# ------------------------------------------------------------
# 4. Show one sample
# ------------------------------------------------------------

print("\nSample row:")

sample = conn.execute("""
    SELECT *
    FROM missing_gt_detail
    LIMIT 1
""").fetchone()

print(sample)


# ------------------------------------------------------------
# 5. Check whether the detail table contains the current
#    remaining missing pairs.
#
#    We use the two IDs that identify a GT pair.
# ------------------------------------------------------------

print("\nChecking overlap with current missing_gt_v2...")

try:

    # Discover ID column names
    detail_cols = [r[0] for r in schema]

    print("\nID-like columns in missing_gt_detail:")
    for c in detail_cols:
        if "entity_id" in c.lower():
            print(" ", c)

    overlap = conn.execute("""
        SELECT COUNT(*)
        FROM missing_gt_v2 m
        INNER JOIN missing_gt_detail d
          ON m.source1_entity_id = d.source1_entity_id
         AND m.matched_entity_id = d.matched_entity_id
    """).fetchone()[0]

    print(f"\nOverlap: {overlap:,}")

except Exception as e:

    print("\nOverlap check failed:")
    print(type(e).__name__, str(e))


print("\n" + "=" * 80)
print("STEP 24I-J COMPLETE")
print("=" * 80)

STEP 24I-J — CHECKING EXISTING missing_gt_detail

missing_gt_detail rows: 5,356,458
missing_gt_v2 rows:     3,775,645

missing_gt_detail columns:
  source1_entity_id              VARCHAR
  matched_entity_id              VARCHAR
  s1_name                        VARCHAR
  s1_address                     VARCHAR
  s1_country                     VARCHAR
  candidate_name                 VARCHAR
  candidate_address              VARCHAR
  candidate_country              VARCHAR
  s1_name_norm                   VARCHAR
  candidate_name_norm            VARCHAR
  s1_address_norm                VARCHAR
  candidate_address_norm         VARCHAR
  s1_country_norm                VARCHAR
  candidate_country_norm         VARCHAR
  s1_name_core                   VARCHAR
  candidate_name_core            VARCHAR
  s1_name_prefix4                VARCHAR
  candidate_name_prefix4         VARCHAR
  s1_name_prefix6                VARCHAR
  candidate_name_prefix6         VARCHAR
  s1_address_prefix6             V

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Overlap: 3,775,645

STEP 24I-J COMPLETE


In [41]:
# ============================================================
# STEP 24I-K — ANALYZE REMAINING MISSING GT PAIRS
# ============================================================

print("=" * 80)
print("STEP 24I-K — REMAINING MISSING-PAIR BLOCKING ANALYSIS")
print("=" * 80)

# ------------------------------------------------------------
# 1. BASIC AGREEMENT COUNTS
# ------------------------------------------------------------

result = conn.execute("""
SELECT
    COUNT(*) AS total_pairs,

    SUM(
        s1_name_norm = candidate_name_norm
    ) AS name_exact,

    SUM(
        s1_address_norm = candidate_address_norm
    ) AS address_exact,

    SUM(
        s1_country_norm = candidate_country_norm
    ) AS country_exact,

    SUM(
        s1_name_core = candidate_name_core
    ) AS name_core_exact,

    SUM(
        s1_name_prefix4 = candidate_name_prefix4
    ) AS name_prefix4_exact,

    SUM(
        s1_name_prefix6 = candidate_name_prefix6
    ) AS name_prefix6_exact,

    SUM(
        s1_address_prefix6 = candidate_address_prefix6
    ) AS address_prefix6_exact,

    SUM(
        s1_address_prefix10 = candidate_address_prefix10
    ) AS address_prefix10_exact

FROM missing_gt_v2 m
JOIN missing_gt_detail d
  ON m.source1_entity_id = d.source1_entity_id
 AND m.matched_entity_id = d.matched_entity_id
""").fetchone()

labels = [
    "total_pairs",
    "name_exact",
    "address_exact",
    "country_exact",
    "name_core_exact",
    "name_prefix4_exact",
    "name_prefix6_exact",
    "address_prefix6_exact",
    "address_prefix10_exact"
]

print("\nAgreement counts:")

for label, value in zip(labels, result):
    print(f"  {label:25} {value:,}")


# ------------------------------------------------------------
# 2. KEY COMBINATION ANALYSIS
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("KEY COMBINATION ANALYSIS")
print("-" * 80)

combo = conn.execute("""
WITH x AS (
    SELECT
        s1_name_core = candidate_name_core
            AS name_core,

        s1_name_prefix4 = candidate_name_prefix4
            AS prefix4,

        s1_name_prefix6 = candidate_name_prefix6
            AS prefix6,

        s1_address_prefix6 = candidate_address_prefix6
            AS addr6,

        s1_address_prefix10 = candidate_address_prefix10
            AS addr10

    FROM missing_gt_v2 m
    JOIN missing_gt_detail d
      ON m.source1_entity_id = d.source1_entity_id
     AND m.matched_entity_id = d.matched_entity_id
)

SELECT
    CASE
        WHEN name_core AND prefix6 AND addr6
            THEN 'NAME CORE + PREFIX6 + ADDRESS PREFIX6'

        WHEN name_core AND addr6
            THEN 'NAME CORE + ADDRESS PREFIX6'

        WHEN prefix6 AND addr6
            THEN 'NAME PREFIX6 + ADDRESS PREFIX6'

        WHEN prefix4 AND addr6
            THEN 'NAME PREFIX4 + ADDRESS PREFIX6'

        WHEN prefix6
            THEN 'NAME PREFIX6 ONLY'

        WHEN addr6
            THEN 'ADDRESS PREFIX6 ONLY'

        WHEN name_core
            THEN 'NAME CORE ONLY'

        WHEN addr10
            THEN 'ADDRESS PREFIX10 ONLY'

        ELSE 'NO KEY AGREEMENT'
    END AS agreement_type,

    COUNT(*) AS pairs

FROM x

GROUP BY agreement_type

ORDER BY pairs DESC
""").fetchall()

for agreement_type, pairs in combo:
    print(f"  {agreement_type:45} {pairs:,}")


# ------------------------------------------------------------
# 3. MORE PRECISE OVERLAP OF PROMISING ROUTES
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("PROMISING ROUTE OVERLAP")
print("-" * 80)

route_stats = conn.execute("""
WITH x AS (
    SELECT
        m.source1_entity_id,
        m.matched_entity_id,

        s1_name_core = candidate_name_core
            AS core,

        s1_name_prefix4 = candidate_name_prefix4
            AS p4,

        s1_name_prefix6 = candidate_name_prefix6
            AS p6,

        s1_address_prefix6 = candidate_address_prefix6
            AS a6,

        s1_address_prefix10 = candidate_address_prefix10
            AS a10

    FROM missing_gt_v2 m
    JOIN missing_gt_detail d
      ON m.source1_entity_id = d.source1_entity_id
     AND m.matched_entity_id = d.matched_entity_id
)

SELECT
    SUM(p4 AND a6) AS route_p4_a6,
    SUM(p6 AND a6) AS route_p6_a6,
    SUM(core AND a6) AS route_core_a6,
    SUM(p4 AND a10) AS route_p4_a10,
    SUM(p6 AND a10) AS route_p6_a10,
    SUM(core AND a10) AS route_core_a10,

    SUM(
        (p4 AND a6)
        AND NOT (core AND a6)
    ) AS p4_a6_only,

    SUM(
        (p6 AND a6)
        AND NOT (p4 AND a6)
    ) AS p6_a6_only,

    SUM(
        (core AND a6)
        AND NOT (p4 AND a6)
    ) AS core_a6_only

FROM x
""").fetchone()

names = [
    "route_p4_a6",
    "route_p6_a6",
    "route_core_a6",
    "route_p4_a10",
    "route_p6_a10",
    "route_core_a10",
    "p4_a6_only",
    "p6_a6_only",
    "core_a6_only"
]

for name, value in zip(names, route_stats):
    print(f"  {name:25} {value:,}")


print("\n" + "=" * 80)
print("STEP 24I-K COMPLETE")
print("=" * 80)

STEP 24I-K — REMAINING MISSING-PAIR BLOCKING ANALYSIS


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Agreement counts:
  total_pairs               3,775,645
  name_exact                0
  address_exact             0
  country_exact             3,775,645
  name_core_exact           618,668
  name_prefix4_exact        2,200,019
  name_prefix6_exact        2,028,540
  address_prefix6_exact     606,161
  address_prefix10_exact    514,192

--------------------------------------------------------------------------------
KEY COMBINATION ANALYSIS
--------------------------------------------------------------------------------


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  NAME PREFIX6 ONLY                             2,028,540
  NO KEY AGREEMENT                              1,136,165
  ADDRESS PREFIX6 ONLY                          605,900
  NAME CORE ONLY                                4,779
  NAME CORE + ADDRESS PREFIX6                   261

--------------------------------------------------------------------------------
PROMISING ROUTE OVERLAP
--------------------------------------------------------------------------------


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  route_p4_a6               0
  route_p6_a6               0
  route_core_a6             261
  route_p4_a10              0
  route_p6_a10              0
  route_core_a10            238
  p4_a6_only                0
  p6_a6_only                0
  core_a6_only              261

STEP 24I-K COMPLETE


In [42]:
# ============================================================
# STEP 24I-L — DISCOVER ADDITIONAL BLOCKING SIGNALS
# ============================================================

print("=" * 80)
print("STEP 24I-L — DISCOVERING ADDITIONAL BLOCKING SIGNALS")
print("=" * 80)

# ------------------------------------------------------------
# Build feature comparison for remaining missing GT pairs
# ------------------------------------------------------------

conn.execute("DROP TABLE IF EXISTS missing_signal_analysis")

conn.execute("""
CREATE TEMP TABLE missing_signal_analysis AS

WITH x AS (
    SELECT
        m.source1_entity_id,
        m.matched_entity_id,

        d.s1_name_norm,
        d.candidate_name_norm,

        d.s1_address_norm,
        d.candidate_address_norm,

        -- Existing categories
        d.s1_name_prefix6 = d.candidate_name_prefix6 AS name_p6,
        d.s1_address_prefix6 = d.candidate_address_prefix6 AS addr_p6,

        -- Name suffixes
        RIGHT(d.s1_name_norm, 4)
            = RIGHT(d.candidate_name_norm, 4)
            AS name_suffix4,

        RIGHT(d.s1_name_norm, 6)
            = RIGHT(d.candidate_name_norm, 6)
            AS name_suffix6,

        -- Address suffixes
        RIGHT(d.s1_address_norm, 6)
            = RIGHT(d.candidate_address_norm, 6)
            AS addr_suffix6,

        RIGHT(d.s1_address_norm, 10)
            = RIGHT(d.candidate_address_norm, 10)
            AS addr_suffix10,

        -- Complete digit sequence from address
        regexp_replace(
            d.s1_address_norm,
            '[^0-9]',
            '',
            'g'
        )
        =
        regexp_replace(
            d.candidate_address_norm,
            '[^0-9]',
            '',
            'g'
        )
        AND regexp_replace(
            d.s1_address_norm,
            '[^0-9]',
            '',
            'g'
        ) <> ''
        AS address_number_exact,

        -- Name lengths
        ABS(
            LENGTH(d.s1_name_norm)
            - LENGTH(d.candidate_name_norm)
        ) <= 2
        AS similar_name_length,

        -- Address lengths
        ABS(
            LENGTH(d.s1_address_norm)
            - LENGTH(d.candidate_address_norm)
        ) <= 5
        AS similar_address_length

    FROM missing_gt_v2 m

    JOIN missing_gt_detail d
      ON m.source1_entity_id = d.source1_entity_id
     AND m.matched_entity_id = d.matched_entity_id
)

SELECT *
FROM x
""")


# ------------------------------------------------------------
# Overall signal counts
# ------------------------------------------------------------

print("\nOVERALL SIGNAL AGREEMENT")
print("-" * 80)

signals = conn.execute("""
SELECT
    COUNT(*) AS total,

    SUM(name_suffix4) AS name_suffix4,
    SUM(name_suffix6) AS name_suffix6,

    SUM(addr_suffix6) AS addr_suffix6,
    SUM(addr_suffix10) AS addr_suffix10,

    SUM(address_number_exact) AS address_number_exact,

    SUM(similar_name_length) AS similar_name_length,
    SUM(similar_address_length) AS similar_address_length

FROM missing_signal_analysis
""").fetchone()

signal_names = [
    "total",
    "name_suffix4",
    "name_suffix6",
    "addr_suffix6",
    "addr_suffix10",
    "address_number_exact",
    "similar_name_length",
    "similar_address_length"
]

for name, value in zip(signal_names, signals):
    print(f"  {name:25} {value:,}")


# ------------------------------------------------------------
# Signal combinations
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("SIGNAL COMBINATIONS")
print("-" * 80)

combos = conn.execute("""
SELECT
    CASE
        WHEN name_p6 AND address_number_exact
            THEN 'NAME PREFIX6 + ADDRESS NUMBER'

        WHEN addr_p6 AND address_number_exact
            THEN 'ADDRESS PREFIX6 + ADDRESS NUMBER'

        WHEN name_p6 AND addr_suffix6
            THEN 'NAME PREFIX6 + ADDRESS SUFFIX6'

        WHEN addr_p6 AND name_suffix4
            THEN 'ADDRESS PREFIX6 + NAME SUFFIX4'

        WHEN name_p6 AND name_suffix4
            THEN 'NAME PREFIX6 + NAME SUFFIX4'

        WHEN name_p6 AND name_suffix6
            THEN 'NAME PREFIX6 + NAME SUFFIX6'

        WHEN addr_p6 AND addr_suffix6
            THEN 'ADDRESS PREFIX6 + ADDRESS SUFFIX6'

        WHEN name_suffix4 AND address_number_exact
            THEN 'NAME SUFFIX4 + ADDRESS NUMBER'

        WHEN name_suffix6 AND address_number_exact
            THEN 'NAME SUFFIX6 + ADDRESS NUMBER'

        WHEN address_number_exact
            THEN 'ADDRESS NUMBER ONLY'

        WHEN name_suffix4
            THEN 'NAME SUFFIX4 ONLY'

        WHEN addr_suffix6
            THEN 'ADDRESS SUFFIX6 ONLY'

        ELSE 'OTHER'
    END AS signal_group,

    COUNT(*) AS pairs

FROM missing_signal_analysis

GROUP BY signal_group

ORDER BY pairs DESC
""").fetchall()

for group, pairs in combos:
    print(f"  {group:45} {pairs:,}")


# ------------------------------------------------------------
# Analyze the three major remaining categories separately
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SIGNALS BY REMAINING PROBLEM CATEGORY")
print("=" * 80)

category_stats = conn.execute("""
WITH categorized AS (

    SELECT
        CASE
            WHEN name_p6 AND NOT addr_p6
                THEN 'NAME PREFIX6 ONLY'

            WHEN addr_p6 AND NOT name_p6
                THEN 'ADDRESS PREFIX6 ONLY'

            WHEN NOT name_p6 AND NOT addr_p6
                THEN 'NO PREFIX6 AGREEMENT'

            ELSE 'OTHER'
        END AS category,

        *

    FROM missing_signal_analysis
)

SELECT
    category,
    COUNT(*) AS total,

    SUM(name_suffix4) AS name_suffix4,
    SUM(name_suffix6) AS name_suffix6,

    SUM(addr_suffix6) AS addr_suffix6,
    SUM(addr_suffix10) AS addr_suffix10,

    SUM(address_number_exact) AS address_number_exact,

    SUM(
        name_suffix4 AND address_number_exact
    ) AS name_suffix4_number,

    SUM(
        name_p6 AND address_number_exact
    ) AS namep6_number,

    SUM(
        addr_p6 AND address_number_exact
    ) AS addrp6_number

FROM categorized

GROUP BY category

ORDER BY total DESC
""").fetchall()

headers = [
    "category",
    "total",
    "name_suffix4",
    "name_suffix6",
    "addr_suffix6",
    "addr_suffix10",
    "address_number_exact",
    "name_suffix4_number",
    "namep6_number",
    "addrp6_number"
]

for row in category_stats:

    print("\n" + row[0])

    for name, value in zip(headers[1:], row[1:]):
        print(f"  {name:25} {value:,}")


print("\n" + "=" * 80)
print("STEP 24I-L COMPLETE")
print("=" * 80)

STEP 24I-L — DISCOVERING ADDITIONAL BLOCKING SIGNALS


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


OVERALL SIGNAL AGREEMENT
--------------------------------------------------------------------------------
  total                     3,775,645
  name_suffix4              898,855
  name_suffix6              766,852
  addr_suffix6              775,653
  addr_suffix10             603,395
  address_number_exact      1,671,992
  similar_name_length       1,526,933
  similar_address_length    1,783,742

--------------------------------------------------------------------------------
SIGNAL COMBINATIONS
--------------------------------------------------------------------------------
  OTHER                                         1,234,632
  NAME PREFIX6 + ADDRESS NUMBER                 729,534
  ADDRESS PREFIX6 + ADDRESS NUMBER              466,825
  ADDRESS NUMBER ONLY                           332,443
  NAME PREFIX6 + ADDRESS SUFFIX6                281,357
  NAME PREFIX6 + NAME SUFFIX4                   214,372
  NAME SUFFIX4 ONLY                             179,902
  NAME SUFFIX4 + ADD

In [43]:
# ============================================================
# STEP 24I-M — ESTIMATE NEW ROUTE CANDIDATE VOLUME
# ============================================================

import time

print("=" * 80)
print("STEP 24I-M — ESTIMATING NEW BLOCKING ROUTE VOLUMES")
print("=" * 80)

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

conn.execute("SET threads=2")
conn.execute("SET preserve_insertion_order=false")

try:
    conn.execute("PRAGMA max_temp_directory_size='10GiB'")
except Exception:
    pass


# ------------------------------------------------------------
# NORMALIZATION
#
# The existing missing_gt_detail confirms that name_norm and
# address_norm use lowercase + non-alphanumeric removal.
# ------------------------------------------------------------

def build_key_table(source_table, output_table):
    print(f"\nBuilding {output_table} from {source_table}...")
    start = time.time()

    conn.execute(f"DROP TABLE IF EXISTS {output_table}")

    conn.execute(f"""
        CREATE TEMP TABLE {output_table} AS

        SELECT
            entity_id,

            lower(
                regexp_replace(
                    coalesce(business_name, ''),
                    '[^a-zA-Z0-9]',
                    '',
                    'g'
                )
            ) AS name_norm,

            lower(
                regexp_replace(
                    coalesce(business_address, ''),
                    '[^a-zA-Z0-9]',
                    '',
                    'g'
                )
            ) AS address_norm,

            lower(
                regexp_replace(
                    coalesce(country, ''),
                    '[^a-zA-Z0-9]',
                    '',
                    'g'
                )
            ) AS country_norm

        FROM source_db.{source_table}
    """)

    count = conn.execute(
        f"SELECT COUNT(*) FROM {output_table}"
    ).fetchone()[0]

    print(
        f"✓ {output_table}: {count:,} rows "
        f"({time.time() - start:.1f}s)"
    )


# ------------------------------------------------------------
# BUILD COMPACT KEY TABLES
# ------------------------------------------------------------

build_key_table("train_s1", "route_keys_s1")
build_key_table("train_s2", "route_keys_s2")
build_key_table("train_s3", "route_keys_s3")


# ------------------------------------------------------------
# ADD ROUTE KEYS
# ------------------------------------------------------------

print("\nCreating route keys...")

for table in [
    "route_keys_s1",
    "route_keys_s2",
    "route_keys_s3"
]:

    conn.execute(f"""
        ALTER TABLE {table}
        ADD COLUMN name_prefix6 VARCHAR
    """)

    conn.execute(f"""
        UPDATE {table}
        SET name_prefix6 = LEFT(name_norm, 6)
    """)

    conn.execute(f"""
        ALTER TABLE {table}
        ADD COLUMN address_prefix6 VARCHAR
    """)

    conn.execute(f"""
        UPDATE {table}
        SET address_prefix6 = LEFT(address_norm, 6)
    """)

    conn.execute(f"""
        ALTER TABLE {table}
        ADD COLUMN address_number VARCHAR
    """)

    conn.execute(f"""
        UPDATE {table}
        SET address_number =
            regexp_replace(
                address_norm,
                '[^0-9]',
                '',
                'g'
            )
    """)

print("✓ Route keys created")


# ------------------------------------------------------------
# REMOVE EMPTY ADDRESS NUMBERS
# ------------------------------------------------------------

print("\nChecking address-number quality...")

for table in [
    "route_keys_s1",
    "route_keys_s2",
    "route_keys_s3"
]:

    stats = conn.execute(f"""
        SELECT
            COUNT(*) AS total,
            SUM(address_number <> '') AS with_number,
            SUM(address_number = '') AS without_number
        FROM {table}
    """).fetchone()

    print(
        f"{table}: "
        f"total={stats[0]:,}, "
        f"with_number={stats[1]:,}, "
        f"without_number={stats[2]:,}"
    )


# ============================================================
# ROUTE 1
# NAME PREFIX6 + ADDRESS NUMBER + COUNTRY
# ============================================================

print("\n" + "-" * 80)
print("ROUTE 1: NAME PREFIX6 + ADDRESS NUMBER + COUNTRY")
print("-" * 80)

start = time.time()

s2_volume = conn.execute("""
    SELECT COUNT(*)
    FROM route_keys_s1 s1
    INNER JOIN route_keys_s2 s2
      ON s1.country_norm = s2.country_norm
     AND s1.name_prefix6 = s2.name_prefix6
     AND s1.address_number <> ''
     AND s1.address_number = s2.address_number
""").fetchone()[0]

print(
    f"S1 → S2 candidate rows: {s2_volume:,}"
)

print(
    f"Elapsed: {(time.time() - start):.1f}s"
)


start = time.time()

s3_volume = conn.execute("""
    SELECT COUNT(*)
    FROM route_keys_s1 s1
    INNER JOIN route_keys_s3 s3
      ON s1.country_norm = s3.country_norm
     AND s1.name_prefix6 = s3.name_prefix6
     AND s1.address_number <> ''
     AND s1.address_number = s3.address_number
""").fetchone()[0]

print(
    f"S1 → S3 candidate rows: {s3_volume:,}"
)

print(
    f"Elapsed: {(time.time() - start):.1f}s"
)

print(
    f"TOTAL ROUTE 1: {s2_volume + s3_volume:,}"
)


# ============================================================
# ROUTE 2
# ADDRESS PREFIX6 + ADDRESS NUMBER + COUNTRY
# ============================================================

print("\n" + "-" * 80)
print("ROUTE 2: ADDRESS PREFIX6 + ADDRESS NUMBER + COUNTRY")
print("-" * 80)

start = time.time()

s2_volume_addr = conn.execute("""
    SELECT COUNT(*)
    FROM route_keys_s1 s1
    INNER JOIN route_keys_s2 s2
      ON s1.country_norm = s2.country_norm
     AND s1.address_prefix6 = s2.address_prefix6
     AND s1.address_number <> ''
     AND s1.address_number = s2.address_number
""").fetchone()[0]

print(
    f"S1 → S2 candidate rows: {s2_volume_addr:,}"
)

print(
    f"Elapsed: {(time.time() - start):.1f}s"
)


start = time.time()

s3_volume_addr = conn.execute("""
    SELECT COUNT(*)
    FROM route_keys_s1 s1
    INNER JOIN route_keys_s3 s3
      ON s1.country_norm = s3.country_norm
     AND s1.address_prefix6 = s3.address_prefix6
     AND s1.address_number <> ''
     AND s1.address_number = s3.address_number
""").fetchone()[0]

print(
    f"S1 → S3 candidate rows: {s3_volume_addr:,}"
)

print(
    f"Elapsed: {(time.time() - start):.1f}s"
)

print(
    f"TOTAL ROUTE 2: "
    f"{s2_volume_addr + s3_volume_addr:,}"
)


print("\n" + "=" * 80)
print("STEP 24I-M COMPLETE")
print("=" * 80)

STEP 24I-M — ESTIMATING NEW BLOCKING ROUTE VOLUMES

Building route_keys_s1 from train_s1...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ route_keys_s1: 2,206,821 rows (17.0s)

Building route_keys_s2 from train_s2...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ route_keys_s2: 5,034,616 rows (41.1s)

Building route_keys_s3 from train_s3...


...

S1 → S3 candidate rows: 3,516,279
Elapsed: 3.1s
TOTAL ROUTE 1: 6,610,305

--------------------------------------------------------------------------------
ROUTE 2: ADDRESS PREFIX6 + ADDRESS NUMBER + COUNTRY
--------------------------------------------------------------------------------


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

S1 → S2 candidate rows: 12,081,551
Elapsed: 4.4s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

S1 → S3 candidate rows: 13,844,528
Elapsed: 3.5s
TOTAL ROUTE 2: 25,926,079

STEP 24I-M COMPLETE


In [44]:
# ============================================================
# STEP 24I-N — GENERATE ROUTE 1 CANDIDATES
# NAME PREFIX6 + ADDRESS NUMBER + COUNTRY
# ============================================================

import time

print("=" * 80)
print("STEP 24I-N — GENERATING ROUTE 1 CANDIDATES")
print("=" * 80)

conn.execute("SET threads=2")
conn.execute("SET preserve_insertion_order=false")

try:
    conn.execute("PRAGMA max_temp_directory_size='10GiB'")
except Exception:
    pass


# ------------------------------------------------------------
# CLEAN OLD TEMP TABLES
# ------------------------------------------------------------

for table in [
    "route_num_s2",
    "route_num_s3",
    "new_route_num_s2",
    "new_route_num_s3"
]:
    conn.execute(f"DROP TABLE IF EXISTS {table}")


# ============================================================
# S1 → S2
# ============================================================

print("\n[1/2] Generating S1 → S2 candidates...")

start = time.time()

conn.execute("""
CREATE TEMP TABLE route_num_s2 AS

SELECT DISTINCT
    s1.entity_id AS source1_entity_id,
    s2.entity_id AS candidate_entity_id

FROM route_keys_s1 s1

INNER JOIN route_keys_s2 s2
    ON s1.country_norm = s2.country_norm
   AND s1.name_prefix6 = s2.name_prefix6
   AND s1.address_number <> ''
   AND s1.address_number = s2.address_number
""")

count_s2 = conn.execute("""
    SELECT COUNT(*)
    FROM route_num_s2
""").fetchone()[0]

print(f"✓ S1 → S2 candidates: {count_s2:,}")
print(f"Time: {(time.time() - start) / 60:.2f} minutes")


# ============================================================
# S1 → S3
# ============================================================

print("\n[2/2] Generating S1 → S3 candidates...")

start = time.time()

conn.execute("""
CREATE TEMP TABLE route_num_s3 AS

SELECT DISTINCT
    s1.entity_id AS source1_entity_id,
    s3.entity_id AS candidate_entity_id

FROM route_keys_s1 s1

INNER JOIN route_keys_s3 s3
    ON s1.country_norm = s3.country_norm
   AND s1.name_prefix6 = s3.name_prefix6
   AND s1.address_number <> ''
   AND s1.address_number = s3.address_number
""")

count_s3 = conn.execute("""
    SELECT COUNT(*)
    FROM route_num_s3
""").fetchone()[0]

print(f"✓ S1 → S3 candidates: {count_s3:,}")
print(f"Time: {(time.time() - start) / 60:.2f} minutes")


# ============================================================
# CHECK TOTAL
# ============================================================

print("\n" + "-" * 80)
print("ROUTE 1 TOTAL")
print("-" * 80)

print(f"S2:   {count_s2:,}")
print(f"S3:   {count_s3:,}")
print(f"TOTAL: {count_s2 + count_s3:,}")


# ============================================================
# CHECK DUPLICATES
# ============================================================

print("\nChecking duplicates...")

dup_s2 = conn.execute("""
SELECT COUNT(*)
FROM (
    SELECT source1_entity_id, candidate_entity_id, COUNT(*) AS n
    FROM route_num_s2
    GROUP BY source1_entity_id, candidate_entity_id
    HAVING COUNT(*) > 1
)
""").fetchone()[0]

dup_s3 = conn.execute("""
SELECT COUNT(*)
FROM (
    SELECT source1_entity_id, candidate_entity_id, COUNT(*) AS n
    FROM route_num_s3
    GROUP BY source1_entity_id, candidate_entity_id
    HAVING COUNT(*) > 1
)
""").fetchone()[0]

print(f"S2 duplicate pairs: {dup_s2:,}")
print(f"S3 duplicate pairs: {dup_s3:,}")


print("\n" + "=" * 80)
print("STEP 24I-N COMPLETE")
print("=" * 80)

STEP 24I-N — GENERATING ROUTE 1 CANDIDATES

[1/2] Generating S1 → S2 candidates...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ S1 → S2 candidates: 3,094,026
Time: 0.12 minutes

[2/2] Generating S1 → S3 candidates...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ S1 → S3 candidates: 3,516,279
Time: 0.12 minutes

--------------------------------------------------------------------------------
ROUTE 1 TOTAL
--------------------------------------------------------------------------------
S2:   3,094,026
S3:   3,516,279
TOTAL: 6,610,305

Checking duplicates...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

S2 duplicate pairs: 0
S3 duplicate pairs: 0

STEP 24I-N COMPLETE


In [45]:
# ============================================================
# STEP 24I-O — FIND NEW ROUTE 1 CANDIDATES
# ============================================================

import time

print("=" * 80)
print("STEP 24I-O — FINDING NEW ROUTE 1 PAIRS")
print("=" * 80)

# ------------------------------------------------------------
# CLEAN OLD TEMP TABLES
# ------------------------------------------------------------

conn.execute("DROP TABLE IF EXISTS new_route_num_s2")
conn.execute("DROP TABLE IF EXISTS new_route_num_s3")


# ============================================================
# S2 NEW PAIRS
# ============================================================

print("\n[1/2] Checking S2 against unique_candidates_v2...")

start = time.time()

conn.execute("""
CREATE TEMP TABLE new_route_num_s2 AS

SELECT
    r.source1_entity_id,
    r.candidate_entity_id

FROM route_num_s2 r

ANTI JOIN unique_candidates_v2 u
    ON r.source1_entity_id = u.source1_entity_id
   AND r.candidate_entity_id = u.candidate_entity_id
""")

new_s2 = conn.execute("""
    SELECT COUNT(*)
    FROM new_route_num_s2
""").fetchone()[0]

print(f"✓ New S2 pairs: {new_s2:,}")
print(f"Time: {(time.time() - start) / 60:.2f} minutes")


# ============================================================
# S3 NEW PAIRS
# ============================================================

print("\n[2/2] Checking S3 against unique_candidates_v2...")

start = time.time()

conn.execute("""
CREATE TEMP TABLE new_route_num_s3 AS

SELECT
    r.source1_entity_id,
    r.candidate_entity_id

FROM route_num_s3 r

ANTI JOIN unique_candidates_v2 u
    ON r.source1_entity_id = u.source1_entity_id
   AND r.candidate_entity_id = u.candidate_entity_id
""")

new_s3 = conn.execute("""
    SELECT COUNT(*)
    FROM new_route_num_s3
""").fetchone()[0]

print(f"✓ New S3 pairs: {new_s3:,}")
print(f"Time: {(time.time() - start) / 60:.2f} minutes")


# ============================================================
# SUMMARY
# ============================================================

route_total = 3_094_026 + 3_516_279
new_total = new_s2 + new_s3

print("\n" + "-" * 80)
print("NEW ROUTE 1 SUMMARY")
print("-" * 80)

print(f"Route 1 total:       {route_total:,}")
print(f"Already existing:    {route_total - new_total:,}")
print(f"Genuinely new:       {new_total:,}")

if route_total > 0:
    print(
        f"New percentage:      "
        f"{new_total / route_total * 100:.2f}%"
    )


# ============================================================
# CHECK UNIQUE NEW PAIRS
# ============================================================

print("\nChecking new-pair uniqueness...")

new_duplicates = conn.execute("""
SELECT COUNT(*)
FROM (
    SELECT source1_entity_id, candidate_entity_id
    FROM new_route_num_s2

    UNION ALL

    SELECT source1_entity_id, candidate_entity_id
    FROM new_route_num_s3
)
GROUP BY source1_entity_id, candidate_entity_id
HAVING COUNT(*) > 1
""").fetchone()

# The query above returns a row only if duplicates exist.
# Use a safer explicit count too.

duplicate_count = conn.execute("""
SELECT COUNT(*)
FROM (
    SELECT
        source1_entity_id,
        candidate_entity_id,
        COUNT(*) AS n

    FROM (
        SELECT source1_entity_id, candidate_entity_id
        FROM new_route_num_s2

        UNION ALL

        SELECT source1_entity_id, candidate_entity_id
        FROM new_route_num_s3
    )

    GROUP BY source1_entity_id, candidate_entity_id

    HAVING COUNT(*) > 1
)
""").fetchone()[0]

print(f"Duplicate new pairs across S2/S3: {duplicate_count:,}")


print("\n" + "=" * 80)
print("STEP 24I-O COMPLETE")
print("=" * 80)

STEP 24I-O — FINDING NEW ROUTE 1 PAIRS

[1/2] Checking S2 against unique_candidates_v2...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ New S2 pairs: 1,868,044
Time: 0.15 minutes

[2/2] Checking S3 against unique_candidates_v2...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ New S3 pairs: 2,229,016
Time: 0.15 minutes

--------------------------------------------------------------------------------
NEW ROUTE 1 SUMMARY
--------------------------------------------------------------------------------
Route 1 total:       6,610,305
Already existing:    2,513,245
Genuinely new:       4,097,060
New percentage:      61.98%

Checking new-pair uniqueness...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate new pairs across S2/S3: 0

STEP 24I-O COMPLETE


In [46]:
import time

print("=" * 80)
print("STEP 24I-P — CREATING UNIQUE_CANDIDATES_V3")
print("=" * 80)

start = time.time()

# Drop only the new replacement table if it exists
conn.execute("DROP TABLE IF EXISTS unique_candidates_v3")

print("\n[1/3] Creating v3 using safe UNION...")

conn.execute("""
CREATE TABLE unique_candidates_v3 AS

SELECT
    source1_entity_id,
    candidate_entity_id
FROM unique_candidates_v2

UNION

SELECT
    source1_entity_id,
    candidate_entity_id
FROM new_route_num_s2

UNION

SELECT
    source1_entity_id,
    candidate_entity_id
FROM new_route_num_s3
""")

elapsed = (time.time() - start) / 60

print(f"✓ unique_candidates_v3 created")
print(f"Time: {elapsed:.2f} minutes")


# ------------------------------------------------------------------
# VERIFY COUNTS
# ------------------------------------------------------------------

print("\n[2/3] Verifying row counts...")

v2_count = conn.execute("""
SELECT COUNT(*)
FROM unique_candidates_v2
""").fetchone()[0]

v3_count = conn.execute("""
SELECT COUNT(*)
FROM unique_candidates_v3
""").fetchone()[0]

print(f"v2 candidates: {v2_count:,}")
print(f"v3 candidates: {v3_count:,}")
print(f"Increase:       {v3_count - v2_count:,}")


# ------------------------------------------------------------------
# VERIFY DISTINCTNESS
# ------------------------------------------------------------------

print("\n[3/3] Verifying uniqueness and NULLs...")

duplicate_count = conn.execute("""
SELECT COUNT(*)
FROM (
    SELECT
        source1_entity_id,
        candidate_entity_id,
        COUNT(*) AS n
    FROM unique_candidates_v3
    GROUP BY
        source1_entity_id,
        candidate_entity_id
    HAVING COUNT(*) > 1
)
""").fetchone()[0]

null_count = conn.execute("""
SELECT COUNT(*)
FROM unique_candidates_v3
WHERE source1_entity_id IS NULL
   OR candidate_entity_id IS NULL
""").fetchone()[0]

print(f"Duplicate pairs: {duplicate_count:,}")
print(f"NULL pairs:      {null_count:,}")

print("\n" + "=" * 80)
print("STEP 24I-P COMPLETE")
print("=" * 80)

if duplicate_count == 0 and null_count == 0:
    print("✓ v3 is clean")
else:
    print("⚠️ Verification problem detected — DO NOT continue")

STEP 24I-P — CREATING UNIQUE_CANDIDATES_V3

[1/3] Creating v3 using safe UNION...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FatalException: FATAL Error: Failed to create checkpoint because of error: Could not truncate file "/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb": No such file or directory

In [48]:
from google.colab import drive
import os

print("Mounting Google Drive...")

drive.mount("/content/drive")

print("\nChecking checkpoint folder...")

checkpoint_dir = "/content/drive/MyDrive/er_checkpoint"

if os.path.exists(checkpoint_dir):
    print("✓ er_checkpoint found")

    print("\nFiles:")
    for f in os.listdir(checkpoint_dir):
        print(" -", f)
else:
    print("❌ er_checkpoint folder not found")

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Checking checkpoint folder...
✓ er_checkpoint found

Files:
 - er_after_blocking_train.duckdb
 - CHECKPOINT.txt
 - er_train_fresh.duckdb
 - er_train_fresh_state.json
 - er_blocking_train_fresh.duckdb
 - er_blocking_state.json
 - er_candidates_state.json
 - er_candidates_train_fresh.duckdb.tmp


In [50]:
import os
import duckdb

tmp_db = "/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb.tmp"

print("=" * 80)
print("STEP 24I-Q — INSPECTING CANDIDATE DATABASE TEMP FILE")
print("=" * 80)

if not os.path.exists(tmp_db):
    raise FileNotFoundError(f"TMP database not found: {tmp_db}")

size_gb = os.path.getsize(tmp_db) / (1024**3)

print(f"\nTMP database:")
print(tmp_db)
print(f"Size: {size_gb:.2f} GB")

print("\nOpening TMP database read-only...")

try:
    test_conn = duckdb.connect(
        tmp_db,
        read_only=True
    )

    print("✓ TMP database opened successfully")

    print("\nChecking tables...")

    tables = test_conn.execute("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'main'
        ORDER BY table_name
    """).fetchall()

    for table in tables:
        print(" -", table[0])

    print("\nChecking unique_candidates_v2...")

    exists = test_conn.execute("""
        SELECT COUNT(*)
        FROM information_schema.tables
        WHERE table_schema = 'main'
          AND table_name = 'unique_candidates_v2'
    """).fetchone()[0]

    if exists:
        count = test_conn.execute("""
            SELECT COUNT(*)
            FROM unique_candidates_v2
        """).fetchone()[0]

        print(f"✓ unique_candidates_v2 exists")
        print(f"✓ Rows: {count:,}")
    else:
        print("❌ unique_candidates_v2 does NOT exist")

    test_conn.close()

except Exception as e:
    print("\n❌ Could not open TMP database")
    print(type(e).__name__)
    print(str(e))

print("\n" + "=" * 80)
print("STEP 24I-Q COMPLETE")
print("=" * 80)

STEP 24I-Q — INSPECTING CANDIDATE DATABASE TEMP FILE

TMP database:
/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb.tmp
Size: 0.00 GB

Opening TMP database read-only...

❌ Could not open TMP database
IOException
IO Error: Could not read from file "/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb.tmp": Is a directory

STEP 24I-Q COMPLETE


In [51]:
from pathlib import Path

tmp_path = Path("/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb.tmp")

print("Exists:", tmp_path.exists())
print("Is directory:", tmp_path.is_dir())
print("Is file:", tmp_path.is_file())

if tmp_path.is_dir():
    print("\nContents:")
    for p in tmp_path.iterdir():
        print(" -", p.name, " | ", "DIR" if p.is_dir() else "FILE")

Exists: True
Is directory: True
Is file: False

Contents:
 - duckdb_temp_storage_DEFAULT-0.tmp  |  FILE
 - duckdb_temp_storage_S160K-0.tmp  |  FILE
 - duckdb_temp_storage_S128K-0.tmp  |  FILE
 - duckdb_temp_storage_S96K-0.tmp  |  FILE
 - duckdb_temp_storage_S192K-0.tmp  |  FILE
 - duckdb_temp_storage_S224K-0.tmp  |  FILE
 - duckdb_temp_storage_S32K-0.tmp  |  FILE
 - duckdb_temp_storage_S64K-0.tmp  |  FILE


In [52]:
from pathlib import Path

checkpoint = Path("/content/drive/MyDrive/er_checkpoint")

for p in checkpoint.iterdir():
    print(
        p.name,
        "|",
        "DIR" if p.is_dir() else "FILE",
        "|",
        f"{p.stat().st_size / (1024**3):.3f} GB"
    )

er_after_blocking_train.duckdb | FILE | 3.121 GB
CHECKPOINT.txt | FILE | 0.000 GB
er_train_fresh.duckdb | FILE | 1.391 GB
er_train_fresh_state.json | FILE | 0.000 GB
er_blocking_train_fresh.duckdb | FILE | 8.737 GB
er_blocking_state.json | FILE | 0.000 GB
er_candidates_state.json | FILE | 0.000 GB
er_candidates_train_fresh.duckdb.tmp | DIR | 0.000 GB


In [53]:

import json
from pathlib import Path

state_path = Path(
    "/content/drive/MyDrive/er_checkpoint/er_candidates_state.json"
)

with open(state_path, "r") as f:
    state = json.load(f)

print(json.dumps(state, indent=2))


{
  "step": 15,
  "status": "complete",
  "database": "/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb",
  "tables": [
    "ranked_fuzzy_matches",
    "fuzzy_match_summary"
  ],
  "s1_records": 946356,
  "created_at": "2026-09-26 05:08:35"
}


In [54]:
from pathlib import Path

tmp_path = Path(
    "/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb.tmp"
)

print("TMP directory contents:\n")

for p in tmp_path.iterdir():
    size_mb = p.stat().st_size / (1024**2)
    print(f"{p.name:45s} {size_mb:10.2f} MB")

TMP directory contents:

duckdb_temp_storage_DEFAULT-0.tmp                 971.50 MB
duckdb_temp_storage_S160K-0.tmp                   432.97 MB
duckdb_temp_storage_S128K-0.tmp                   347.88 MB
duckdb_temp_storage_S96K-0.tmp                    145.88 MB
duckdb_temp_storage_S192K-0.tmp                   120.38 MB
duckdb_temp_storage_S224K-0.tmp                   185.28 MB
duckdb_temp_storage_S32K-0.tmp                     42.12 MB
duckdb_temp_storage_S64K-0.tmp                     27.25 MB


In [55]:
print(json.dumps(state, indent=2))

{
  "step": 15,
  "status": "complete",
  "database": "/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb",
  "tables": [
    "ranked_fuzzy_matches",
    "fuzzy_match_summary"
  ],
  "s1_records": 946356,
  "created_at": "2026-09-26 05:08:35"
}


In [56]:
import duckdb
from pathlib import Path

db_path = Path(
    "/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb"
)

print("Exists:", db_path.exists())
print("Is file:", db_path.is_file())
print("Size:", f"{db_path.stat().st_size / (1024**3):.3f} GB")

conn = duckdb.connect(str(db_path), read_only=True)

tables = conn.execute("SHOW TABLES").fetchall()

print("\nTables:")
for table in tables:
    print(" -", table[0])

conn.close()

Exists: False
Is file: False


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb'

In [57]:
from pathlib import Path

checkpoint = Path("/content/drive/MyDrive/er_checkpoint")

print("All candidate-related paths:\n")

for p in checkpoint.rglob("*"):
    if "candidate" in p.name.lower():
        if p.is_file():
            size_gb = p.stat().st_size / (1024**3)
            print(f"FILE : {p}")
            print(f"       Size = {size_gb:.3f} GB")
        else:
            print(f"DIR  : {p}")

All candidate-related paths:

FILE : /content/drive/MyDrive/er_checkpoint/er_candidates_state.json
       Size = 0.000 GB
DIR  : /content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb.tmp


In [58]:
tmp_dir = checkpoint / "er_candidates_train_fresh.duckdb.tmp"

print("\nTMP files:\n")

total = 0

for p in tmp_dir.iterdir():
    size_gb = p.stat().st_size / (1024**3)
    total += p.stat().st_size
    print(f"{p.name:50s} {size_gb:.3f} GB")

print(f"\nTotal TMP size: {total / (1024**3):.3f} GB")


TMP files:

duckdb_temp_storage_DEFAULT-0.tmp                  0.949 GB
duckdb_temp_storage_S160K-0.tmp                    0.423 GB
duckdb_temp_storage_S128K-0.tmp                    0.340 GB
duckdb_temp_storage_S96K-0.tmp                     0.142 GB
duckdb_temp_storage_S192K-0.tmp                    0.118 GB
duckdb_temp_storage_S224K-0.tmp                    0.181 GB
duckdb_temp_storage_S32K-0.tmp                     0.041 GB
duckdb_temp_storage_S64K-0.tmp                     0.027 GB

Total TMP size: 2.220 GB


In [59]:
import json

state_path = checkpoint / "er_candidates_state.json"

with open(state_path) as f:
    state = json.load(f)

print(json.dumps(state, indent=2))

print("\nState file modified:",
      state_path.stat().st_mtime)

{
  "step": 15,
  "status": "complete",
  "database": "/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb",
  "tables": [
    "ranked_fuzzy_matches",
    "fuzzy_match_summary"
  ],
  "s1_records": 946356,
  "created_at": "2026-09-26 05:08:35"
}

State file modified: 1790399315.0


In [60]:
import os
import subprocess

tmp_dir = "/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb.tmp"

print("Checking processes using candidate TMP directory...\n")

result = subprocess.run(
    ["bash", "-c", f"lsof +D '{tmp_dir}' 2>/dev/null | head -30"],
    capture_output=True,
    text=True
)

print(result.stdout if result.stdout else "No process currently has these files open.")

Checking processes using candidate TMP directory...

COMMAND   PID USER   FD   TYPE DEVICE   SIZE/OFF NODE NAME
python3 60914 root   51u   REG   0,45 1018691584 1289 /content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb.tmp/duckdb_temp_storage_DEFAULT-0.tmp
python3 60914 root   52u   REG   0,45  454000640 1292 /content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb.tmp/duckdb_temp_storage_S160K-0.tmp
python3 60914 root   54u   REG   0,45  364773376 1295 /content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb.tmp/duckdb_temp_storage_S128K-0.tmp
python3 60914 root   55u   REG   0,45  152961024 1298 /content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb.tmp/duckdb_temp_storage_S96K-0.tmp
python3 60914 root   56u   REG   0,45  126222336 1301 /content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb.tmp/duckdb_temp_storage_S192K-0.tmp
python3 60914 root   57u   REG   0,45  194281472 1304 /content/drive/MyDrive/er_checkpo

In [61]:
import subprocess

result = subprocess.run(
    ["bash", "-c", "ps -p 60914 -o pid,etime,%cpu,%mem,rss,cmd"],
    capture_output=True,
    text=True
)

print(result.stdout)

    PID     ELAPSED %CPU %MEM   RSS CMD
  60914    02:26:43 23.4 62.3 8289852 /usr/bin/python3 -m colab_kernel_launcher -f /root/.local/share/jupyter/runtime/kernel-70beaede-d42f-4171-ad48-4695bbf3e1b7.json



In [62]:
from pathlib import Path

tmp_dir = Path(
    "/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb.tmp"
)

total = sum(p.stat().st_size for p in tmp_dir.iterdir())

print(f"TMP size: {total / (1024**3):.3f} GB")

TMP size: 2.220 GB


In [63]:
from pathlib import Path

db = Path(
    "/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb"
)

print("Exists:", db.exists())
print("Is file:", db.is_file())

if db.exists():
    print("Size:", f"{db.stat().st_size / (1024**3):.3f} GB")

Exists: False
Is file: False


In [64]:
import subprocess

result = subprocess.run(
    ["bash", "-c", "ps -p 60914 -o pid,etime,%cpu,%mem,rss,stat,cmd"],
    capture_output=True,
    text=True
)

print(result.stdout)

    PID     ELAPSED %CPU %MEM   RSS STAT CMD
  60914    02:27:20 23.3 62.3 8289852 Ssl /usr/bin/python3 -m colab_kernel_launcher -f /root/.local/share/jupyter/runtime/kernel-70beaede-d42f-4171-ad48-4695bbf3e1b7.json



In [65]:
from pathlib import Path

tmp_dir = Path(
    "/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb.tmp"
)

total = sum(p.stat().st_size for p in tmp_dir.iterdir())

print(f"Current TMP size: {total / (1024**3):.3f} GB")

Current TMP size: 2.220 GB


In [66]:
import subprocess

print(subprocess.run(
    ["bash", "-c", "ps -p 60914 -o pid,etime,%cpu,%mem,rss,stat"],
    capture_output=True,
    text=True
).stdout)

    PID     ELAPSED %CPU %MEM   RSS STAT
  60914    02:28:17 23.1 62.3 8289972 Ssl



In [67]:
from pathlib import Path

tmp = Path(
    "/content/drive/MyDrive/er_checkpoint/"
    "er_candidates_train_fresh.duckdb.tmp"
)

total = sum(p.stat().st_size for p in tmp.iterdir())

print(f"TMP size: {total / (1024**3):.3f} GB")

TMP size: 2.220 GB


In [68]:
import subprocess

print(subprocess.run(
    ["bash", "-c", "ps -p 60914 -o pid,etime,%cpu,%mem,rss,stat"],
    capture_output=True,
    text=True
).stdout)

    PID     ELAPSED %CPU %MEM   RSS STAT
  60914    02:29:11 23.0 62.3 8289972 Ssl



In [69]:
from pathlib import Path

db = Path(
    "/content/drive/MyDrive/er_checkpoint/"
    "er_candidates_train_fresh.duckdb"
)

print("Exists:", db.exists())
print("Is file:", db.is_file())

if db.exists():
    print("Size:", f"{db.stat().st_size / (1024**3):.3f} GB")

Exists: False
Is file: False


In [70]:
import subprocess
from pathlib import Path

# Check Python process
print(subprocess.run(
    ["bash", "-c", "ps -p 60914 -o pid,etime,%cpu,%mem,rss,stat"],
    capture_output=True,
    text=True
).stdout)

# Check final database
db = Path("/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb")

print("\nFinal database:")
print("Exists:", db.exists())
print("Is file:", db.is_file())

if db.exists():
    print("Size:", f"{db.stat().st_size / (1024**3):.3f} GB")

    PID     ELAPSED %CPU %MEM   RSS STAT
  60914    02:30:22 22.8 62.3 8289972 Ssl


Final database:
Exists: False
Is file: False


In [71]:
from pathlib import Path
from datetime import datetime

tmp = Path(
    "/content/drive/MyDrive/er_checkpoint/"
    "er_candidates_train_fresh.duckdb.tmp"
)

print("Current temporary files:")

for p in tmp.iterdir():
    stat = p.stat()
    print(
        f"{p.name:45s} "
        f"{stat.st_size / (1024**3):.3f} GB   "
        f"modified: {datetime.fromtimestamp(stat.st_mtime)}"
    )

Current temporary files:
duckdb_temp_storage_DEFAULT-0.tmp             0.949 GB   modified: 2026-09-26 07:14:03
duckdb_temp_storage_S160K-0.tmp               0.423 GB   modified: 2026-09-26 07:14:03
duckdb_temp_storage_S128K-0.tmp               0.340 GB   modified: 2026-09-26 07:14:03
duckdb_temp_storage_S96K-0.tmp                0.142 GB   modified: 2026-09-26 07:14:03
duckdb_temp_storage_S192K-0.tmp               0.118 GB   modified: 2026-09-26 07:14:02
duckdb_temp_storage_S224K-0.tmp               0.181 GB   modified: 2026-09-26 07:14:02
duckdb_temp_storage_S32K-0.tmp                0.041 GB   modified: 2026-09-26 07:14:03
duckdb_temp_storage_S64K-0.tmp                0.027 GB   modified: 2026-09-26 07:14:03


In [77]:
import subprocess
from pathlib import Path

print("PROCESS:")
print(subprocess.run(
    ["bash", "-c", "ps -p 60914 -o pid,etime,%cpu,%mem,rss,stat"],
    capture_output=True,
    text=True
).stdout)

db = Path(
    "/content/drive/MyDrive/er_checkpoint/"
    "er_candidates_train_fresh.duckdb"
)

print("\nFINAL DATABASE:")
print("Exists:", db.exists())
print("Is file:", db.is_file())

if db.exists():
    print("Size:", f"{db.stat().st_size / (1024**3):.3f} GB")

PROCESS:
    PID     ELAPSED %CPU %MEM   RSS STAT
  60914    02:44:13 20.9 62.3 8290472 Ssl


FINAL DATABASE:
Exists: False
Is file: False


In [78]:
import subprocess

print(subprocess.run(
    ["bash", "-c", "ps -p 60914 -o pid,etime,%cpu,%mem,rss,stat,wchan:30,cmd"],
    capture_output=True,
    text=True
).stdout)

    PID     ELAPSED %CPU %MEM   RSS STAT WCHAN                          CMD
  60914    02:44:16 20.9 62.3 8290480 Ssl do_sys_poll                   /usr/bin/python3 -m colab_kernel_launcher -f /root/.local/share/jupyter/runtime/kernel-70beaede-d42f-4171-ad48-4695bbf3e1b7.json



In [79]:
import subprocess

print(subprocess.run(
    ["bash", "-c",
     "lsof -p 60914 | grep -E 'er_candidates|duckdb_temp|duckdb' | head -50"],
    capture_output=True,
    text=True
).stdout)

python3 60914 root  mem       REG                7,0             1200508 /usr/local/lib/python3.13/dist-packages/duckdb/duckdb.cpython-313-x86_64-linux-gnu.so (path dev=0,66)
python3 60914 root   41uW     REG               0,45 13414969344     414 /content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb (deleted)
python3 60914 root   44rR     REG               0,45  1493184512     337 /content/drive/MyDrive/er_checkpoint/er_train_fresh.duckdb
python3 60914 root   51u      REG               0,45  1018691584    1289 /content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb.tmp/duckdb_temp_storage_DEFAULT-0.tmp
python3 60914 root   52u      REG               0,45   454000640    1292 /content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb.tmp/duckdb_temp_storage_S160K-0.tmp
python3 60914 root   54u      REG               0,45   364773376    1295 /content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb.tmp/duckdb_temp_storage_S128K-0.tmp
p

In [80]:
from pathlib import Path

tmp = Path(
    "/content/drive/MyDrive/er_checkpoint/"
    "er_candidates_train_fresh.duckdb.tmp"
)

for p in tmp.iterdir():
    print(p.name, p.stat().st_size / (1024**3), p.stat().st_mtime)

duckdb_temp_storage_DEFAULT-0.tmp 0.94873046875 1790406843.0
duckdb_temp_storage_S160K-0.tmp 0.422821044921875 1790406843.0
duckdb_temp_storage_S128K-0.tmp 0.3397216796875 1790406843.0
duckdb_temp_storage_S96K-0.tmp 0.1424560546875 1790406843.0
duckdb_temp_storage_S192K-0.tmp 0.1175537109375 1790406842.0
duckdb_temp_storage_S224K-0.tmp 0.180938720703125 1790406842.0
duckdb_temp_storage_S32K-0.tmp 0.0411376953125 1790406843.0
duckdb_temp_storage_S64K-0.tmp 0.026611328125 1790406843.0


In [81]:
import subprocess

print(subprocess.run(
    ["bash", "-c", "lsof -p 60914 | grep 'er_candidates_train_fresh.duckdb'"],
    capture_output=True,
    text=True
).stdout)

python3 60914 root   41uW     REG               0,45 13414969344     414 /content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb (deleted)
python3 60914 root   51u      REG               0,45  1018691584    1289 /content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb.tmp/duckdb_temp_storage_DEFAULT-0.tmp
python3 60914 root   52u      REG               0,45   454000640    1292 /content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb.tmp/duckdb_temp_storage_S160K-0.tmp
python3 60914 root   54u      REG               0,45   364773376    1295 /content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb.tmp/duckdb_temp_storage_S128K-0.tmp
python3 60914 root   55u      REG               0,45   152961024    1298 /content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb.tmp/duckdb_temp_storage_S96K-0.tmp
python3 60914 root   56u      REG               0,45   126222336    1301 /content/drive/MyDrive/er_checkpoint/er_candidates_train_

In [82]:
import os

src = "/proc/60914/fd/41"
dst = "/content/drive/MyDrive/er_checkpoint/er_candidates_train_recovered.duckdb"

print("Source exists:", os.path.exists(src))

if os.path.exists(src):
    print("Recovering open DuckDB file...")
    with open(src, "rb") as r, open(dst, "wb") as w:
        while True:
            chunk = r.read(16 * 1024 * 1024)
            if not chunk:
                break
            w.write(chunk)

    print("Recovery copy completed.")
else:
    print("❌ File descriptor no longer exists.")

Source exists: True
Recovering open DuckDB file...
Recovery copy completed.


In [83]:
from pathlib import Path

recovered = Path(
    "/content/drive/MyDrive/er_checkpoint/"
    "er_candidates_train_recovered.duckdb"
)

print("Exists:", recovered.exists())
print("Is file:", recovered.is_file())

if recovered.exists():
    print("Size:",
          f"{recovered.stat().st_size / (1024**3):.3f} GB")

Exists: True
Is file: True
Size: 12.494 GB


In [84]:
import duckdb

recovered_path = (
    "/content/drive/MyDrive/er_checkpoint/"
    "er_candidates_train_recovered.duckdb"
)

try:
    con = duckdb.connect(recovered_path, read_only=True)

    print("✓ DuckDB opened successfully\n")

    tables = con.execute("SHOW TABLES").fetchall()

    print("Tables:")
    for table in tables:
        print(" -", table[0])

    con.close()

except Exception as e:
    print("❌ Could not open recovered database:")
    print(type(e).__name__, e)

✓ DuckDB opened successfully

Tables:
 - all_candidates
 - calibrated_fuzzy_matches
 - candidates_s1_s2
 - candidates_s1_s2_address
 - candidates_s1_s3
 - candidates_s1_s3_address
 - exact_match_signals
 - fuzzy_candidates
 - fuzzy_match_summary
 - fuzzy_scores
 - high_confidence_matches
 - matching_candidates
 - ranked_fuzzy_matches
 - s1_candidate_counts
 - s1_exact_address_counts
 - s1_exact_name_counts
 - s1_exact_summary
 - unique_candidates
 - unique_candidates_v2
 - unique_candidates_v3
 - unique_high_confidence_matches
 - unresolved_candidates
 - unresolved_s1_counts


In [85]:
import duckdb

db = "/content/drive/MyDrive/er_checkpoint/er_candidates_train_recovered.duckdb"

con = duckdb.connect(db, read_only=True)

for table in ["ranked_fuzzy_matches", "fuzzy_match_summary"]:
    count = con.execute(
        f"SELECT COUNT(*) FROM {table}"
    ).fetchone()[0]

    print(f"{table}: {count:,} rows")

con.close()

ranked_fuzzy_matches: 17,952,660 rows
fuzzy_match_summary: 946,356 rows


In [90]:
from pathlib import Path

checkpoint = Path("/content/drive/MyDrive/er_checkpoint")

for name in [
    "er_candidates_train_fresh.duckdb",
    "er_candidates_train_recovered.duckdb",
]:
    p = checkpoint / name
    print(name, "→", p.exists(),
          f"{p.stat().st_size / (1024**3):.3f} GB" if p.exists() else "")

er_candidates_train_fresh.duckdb → True 12.494 GB
er_candidates_train_recovered.duckdb → True 12.494 GB


In [87]:
from pathlib import Path
import shutil

checkpoint = Path("/content/drive/MyDrive/er_checkpoint")

src = checkpoint / "er_candidates_train_recovered.duckdb"
dst = checkpoint / "er_candidates_train_fresh.duckdb"

print("Source:", src)
print("Source exists:", src.exists())

if dst.exists():
    print("❌ Destination already exists. STOP.")
else:
    print("Creating official candidate database...")
    shutil.copy2(src, dst)
    print("✓ Official database created")

print("\nFinal check:")
print("Destination exists:", dst.exists())

if dst.exists():
    print(f"Destination size: {dst.stat().st_size / (1024**3):.3f} GB")

Source: /content/drive/MyDrive/er_checkpoint/er_candidates_train_recovered.duckdb
Source exists: True
Creating official candidate database...
✓ Official database created

Final check:
Destination exists: True
Destination size: 12.494 GB


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pathlib import Path

db = Path("/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb")

print("Exists:", db.exists())

if db.exists():
    print(f"Size: {db.stat().st_size / (1024**3):.3f} GB")

Exists: False


In [4]:
from pathlib import Path

checkpoint = Path("/content/drive/MyDrive/er_checkpoint")

print("Checkpoint folder exists:", checkpoint.exists())

if checkpoint.exists():
    for p in sorted(checkpoint.iterdir()):
        if p.is_file():
            size_gb = p.stat().st_size / (1024**3)
            print(f"{p.name:50} {size_gb:.3f} GB")
        elif p.is_dir():
            print(f"{p.name:50} [DIRECTORY]")

Checkpoint folder exists: True
CHECKPOINT.txt                                     0.000 GB
er_after_blocking_train.duckdb                     3.121 GB
er_blocking_state.json                             0.000 GB
er_blocking_train_fresh.duckdb                     0.000 GB
er_candidates_state.json                           0.000 GB
er_candidates_train_fresh.duckdb.tmp               [DIRECTORY]
er_train_fresh.duckdb                              1.391 GB
er_train_fresh_state.json                          0.000 GB


In [5]:
from pathlib import Path

tmp = Path("/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb.tmp")

print("TMP exists:", tmp.exists())
print("TMP is directory:", tmp.is_dir())

if tmp.exists():
    total = 0

    for p in sorted(tmp.iterdir()):
        if p.is_file():
            size = p.stat().st_size
            total += size
            print(f"{p.name:60} {size/(1024**3):.3f} GB")

    print(f"\nTotal TMP size: {total/(1024**3):.3f} GB")

TMP exists: True
TMP is directory: True
duckdb_temp_storage_DEFAULT-0.tmp                            0.949 GB
duckdb_temp_storage_S128K-0.tmp                              0.340 GB
duckdb_temp_storage_S160K-0.tmp                              0.423 GB
duckdb_temp_storage_S192K-0.tmp                              0.118 GB
duckdb_temp_storage_S224K-0.tmp                              0.181 GB
duckdb_temp_storage_S32K-0.tmp                               0.041 GB
duckdb_temp_storage_S64K-0.tmp                               0.027 GB
duckdb_temp_storage_S96K-0.tmp                               0.142 GB

Total TMP size: 2.220 GB


In [6]:
from pathlib import Path

checkpoint = Path("/content/drive/MyDrive/er_checkpoint")

print("DuckDB files in checkpoint:\n")

for p in checkpoint.rglob("*.duckdb"):
    print(f"{p}  →  {p.stat().st_size/(1024**3):.3f} GB")

DuckDB files in checkpoint:

/content/drive/MyDrive/er_checkpoint/er_after_blocking_train.duckdb  →  3.121 GB
/content/drive/MyDrive/er_checkpoint/er_train_fresh.duckdb  →  1.391 GB
/content/drive/MyDrive/er_checkpoint/er_blocking_train_fresh.duckdb  →  0.000 GB


In [7]:
from pathlib import Path

checkpoint = Path("/content/drive/MyDrive/er_checkpoint")

for filename in [
    "er_candidates_state.json",
    "CHECKPOINT.txt",
    "er_blocking_state.json",
    "er_train_fresh_state.json",
]:
    path = checkpoint / filename

    print("\n" + "=" * 80)
    print(filename)
    print("=" * 80)

    if path.exists():
        print(path.read_text())
    else:
        print("MISSING")


er_candidates_state.json
{
  "step": 15,
  "status": "complete",
  "database": "/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb",
  "tables": [
    "ranked_fuzzy_matches",
    "fuzzy_match_summary"
  ],
  "s1_records": 946356,
  "created_at": "2026-09-26 05:08:35"
}

CHECKPOINT.txt
Amazon ML Entity Resolution
Checkpoint: exact name + address candidates completed
Exact candidates: 117,932
Next step: optimized candidate generation for remaining records


er_blocking_state.json
{
  "completed_tables": [
    "train_s1_blocking",
    "train_s2_blocking",
    "train_s3_blocking"
  ],
  "last_completed": "train_s3_blocking",
  "updated_at": "2026-09-26 03:19:55"
}

er_train_fresh_state.json
{
  "completed": [
    "train_s1",
    "train_s2",
    "train_s3",
    "train_ground_truth",
    "test_s1",
    "test_s2",
    "test_s3"
  ],
  "database": "/content/drive/MyDrive/er_checkpoint/er_train_fresh.duckdb",
  "updated": "2026-09-26 03:00:33"
}


In [8]:
import duckdb

db = "/content/drive/MyDrive/er_checkpoint/er_after_blocking_train.duckdb"

con = duckdb.connect(db, read_only=True)

print("TABLES")
print("=" * 80)

tables = con.execute("SHOW TABLES").fetchall()

for (table,) in tables:
    print(table)

con.close()

TABLES
nrare_train
recs_test
recs_train


In [9]:
import duckdb

db = "/content/drive/MyDrive/er_checkpoint/er_after_blocking_train.duckdb"

con = duckdb.connect(db, read_only=True)

for table in [
    "train_s1",
    "train_s2",
    "train_s3",
    "train_s1_blocking",
    "train_s2_blocking",
    "train_s3_blocking",
]:
    try:
        n = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        print(f"{table:30} {n:,}")
    except Exception as e:
        print(f"{table:30} NOT FOUND")

con.close()

train_s1                       NOT FOUND
train_s2                       NOT FOUND
train_s3                       NOT FOUND
train_s1_blocking              NOT FOUND
train_s2_blocking              NOT FOUND
train_s3_blocking              NOT FOUND


In [10]:
import duckdb

db = "/content/drive/MyDrive/er_checkpoint/er_after_blocking_train.duckdb"

con = duckdb.connect(db, read_only=True)

tables = con.execute("SHOW TABLES").fetchall()

print("Tables in er_after_blocking_train.duckdb:")
print("=" * 60)

for row in tables:
    print(row[0])

con.close()

Tables in er_after_blocking_train.duckdb:
nrare_train
recs_test
recs_train


In [11]:
import duckdb

db = "/content/drive/MyDrive/er_checkpoint/er_after_blocking_train.duckdb"

con = duckdb.connect(db, read_only=True)

for table in ["nrare_train", "recs_test", "recs_train"]:
    print("\n" + "=" * 80)
    print(f"TABLE: {table}")
    print("=" * 80)

    count = con.execute(
        f"SELECT COUNT(*) FROM {table}"
    ).fetchone()[0]

    print(f"Rows: {count:,}")

    columns = con.execute(
        f"DESCRIBE {table}"
    ).fetchall()

    print("\nColumns:")
    for col in columns:
        print(f"  {col[0]:30} {col[1]}")

con.close()


TABLE: nrare_train
Rows: 12,524,513

Columns:
  entity_id                      VARCHAR
  r1                             VARCHAR
  r2                             VARCHAR

TABLE: recs_test
Rows: 11,702,133

Columns:
  src                            TINYINT
  entity_id                      VARCHAR
  country                        VARCHAR
  name_raw                       VARCHAR
  addr_raw                       VARCHAR
  non_ascii                      BOOLEAN
  name_norm                      VARCHAR
  addr_norm                      VARCHAR
  name_core                      VARCHAR
  name_skel                      VARCHAR
  addr_nums                      VARCHAR

TABLE: recs_train
Rows: 12,527,040

Columns:
  src                            TINYINT
  entity_id                      VARCHAR
  country                        VARCHAR
  name_raw                       VARCHAR
  addr_raw                       VARCHAR
  non_ascii                      BOOLEAN
  name_norm                      VARCHAR
 

In [12]:
import duckdb

for db in [
    "/content/drive/MyDrive/er_checkpoint/er_after_blocking_train.duckdb",
    "/content/drive/MyDrive/er_checkpoint/er_train_fresh.duckdb",
]:
    print("\n" + "=" * 80)
    print(db)
    print("=" * 80)

    con = duckdb.connect(db, read_only=True)

    tables = [x[0] for x in con.execute("SHOW TABLES").fetchall()]

    print("fuzzy_candidates exists:", "fuzzy_candidates" in tables)

    if "fuzzy_candidates" in tables:
        count = con.execute(
            "SELECT COUNT(*) FROM fuzzy_candidates"
        ).fetchone()[0]
        print("fuzzy_candidates rows:", f"{count:,}")

    con.close()


/content/drive/MyDrive/er_checkpoint/er_after_blocking_train.duckdb
fuzzy_candidates exists: False

/content/drive/MyDrive/er_checkpoint/er_train_fresh.duckdb
fuzzy_candidates exists: False


In [13]:
import duckdb
import os

CHECKPOINT_DIR = "/content/drive/MyDrive/er_checkpoint"

dbs = [
    "er_after_blocking_train.duckdb",
    "er_train_fresh.duckdb",
]

for db_name in dbs:
    path = os.path.join(CHECKPOINT_DIR, db_name)

    print("\n" + "=" * 80)
    print(db_name)
    print("=" * 80)

    conn_check = duckdb.connect(path, read_only=True)

    tables = conn_check.execute("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'main'
        ORDER BY table_name
    """).fetchall()

    for (table,) in tables:
        print(" -", table)

    conn_check.close()


er_after_blocking_train.duckdb
 - nrare_train
 - recs_test
 - recs_train

er_train_fresh.duckdb
 - test_s1
 - test_s2
 - test_s3
 - train_ground_truth
 - train_s1
 - train_s2
 - train_s3


In [14]:
import os
import glob

TMP_DIR = "/content/drive/MyDrive/er_checkpoint/er_candidates_train_fresh.duckdb.tmp"

print("=" * 80)
print("CANDIDATE TMP DIRECTORY")
print("=" * 80)

if os.path.isdir(TMP_DIR):
    files = glob.glob(os.path.join(TMP_DIR, "*"))

    total = 0

    for f in files:
        size = os.path.getsize(f)
        total += size

        print(f"{os.path.basename(f):45s} "
              f"{size / (1024**3):8.3f} GB")

    print("-" * 80)
    print(f"Total: {total / (1024**3):.3f} GB")
else:
    print("TMP directory does not exist")

CANDIDATE TMP DIRECTORY
duckdb_temp_storage_S32K-0.tmp                   0.041 GB
duckdb_temp_storage_S64K-0.tmp                   0.027 GB
duckdb_temp_storage_DEFAULT-0.tmp                0.949 GB
duckdb_temp_storage_S160K-0.tmp                  0.423 GB
duckdb_temp_storage_S128K-0.tmp                  0.340 GB
duckdb_temp_storage_S96K-0.tmp                   0.142 GB
duckdb_temp_storage_S192K-0.tmp                  0.118 GB
duckdb_temp_storage_S224K-0.tmp                  0.181 GB
--------------------------------------------------------------------------------
Total: 2.220 GB


In [15]:
import os

CHECKPOINT_DIR = "/content/drive/MyDrive/er_checkpoint"

print("\nCheckpoint files:")
for name in sorted(os.listdir(CHECKPOINT_DIR)):
    path = os.path.join(CHECKPOINT_DIR, name)

    if os.path.isfile(path):
        size = os.path.getsize(path) / (1024**3)
        print(f"{name:45s} {size:8.3f} GB")
    elif os.path.isdir(path):
        print(f"{name:45s} [DIRECTORY]")


Checkpoint files:
CHECKPOINT.txt                                   0.000 GB
er_after_blocking_train.duckdb                   3.121 GB
er_blocking_state.json                           0.000 GB
er_blocking_train_fresh.duckdb                   0.000 GB
er_candidates_state.json                         0.000 GB
er_candidates_train_fresh.duckdb.tmp          [DIRECTORY]
er_train_fresh.duckdb                            1.391 GB
er_train_fresh_state.json                        0.000 GB


In [16]:
import duckdb

DB = "/content/drive/MyDrive/er_checkpoint/er_after_blocking_train.duckdb"

con = duckdb.connect(DB, read_only=True)

print("=" * 80)
print("CHECKING SURVIVING BLOCKING CHECKPOINT")
print("=" * 80)

for table in ["recs_train", "recs_test", "nrare_train"]:
    print(f"\n--- {table} ---")

    print(
        con.execute(
            f"SELECT COUNT(*) FROM {table}"
        ).fetchone()[0]
    )

    print(
        con.execute(
            f"DESCRIBE {table}"
        ).fetchdf().to_string(index=False)
    )

con.close()

CHECKING SURVIVING BLOCKING CHECKPOINT

--- recs_train ---
12527040
column_name column_type null  key default extra
        src     TINYINT  YES None    None  None
  entity_id     VARCHAR  YES None    None  None
    country     VARCHAR  YES None    None  None
   name_raw     VARCHAR  YES None    None  None
   addr_raw     VARCHAR  YES None    None  None
  non_ascii     BOOLEAN  YES None    None  None
  name_norm     VARCHAR  YES None    None  None
  addr_norm     VARCHAR  YES None    None  None
  name_core     VARCHAR  YES None    None  None
  name_skel     VARCHAR  YES None    None  None
  addr_nums     VARCHAR  YES None    None  None

--- recs_test ---
11702133
column_name column_type null  key default extra
        src     TINYINT  YES None    None  None
  entity_id     VARCHAR  YES None    None  None
    country     VARCHAR  YES None    None  None
   name_raw     VARCHAR  YES None    None  None
   addr_raw     VARCHAR  YES None    None  None
  non_ascii     BOOLEAN  YES None    Non

In [17]:
import duckdb

DB = "/content/drive/MyDrive/er_checkpoint/er_after_blocking_train.duckdb"

con = duckdb.connect(DB, read_only=True)

print("=" * 80)
print("CHECKING ORIGINAL BLOCKING INTERMEDIATE TABLES")
print("=" * 80)

tables = con.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
    ORDER BY table_name
""").fetchdf()

print(tables.to_string(index=False))

print("\nChecking important tables:")

for table in [
    "nrare_train",
    "arare_train",
    "keys_train",
    "blocks_train",
    "train_s1_blocking",
    "train_s2_blocking",
    "train_s3_blocking"
]:
    exists = con.execute("""
        SELECT COUNT(*)
        FROM information_schema.tables
        WHERE table_schema = 'main'
          AND table_name = ?
    """, [table]).fetchone()[0]

    if exists:
        count = con.execute(
            f'SELECT COUNT(*) FROM "{table}"'
        ).fetchone()[0]

        print(f"✓ {table}: {count:,}")
    else:
        print(f"✗ {table}: MISSING")

con.close()

CHECKING ORIGINAL BLOCKING INTERMEDIATE TABLES
 table_name
nrare_train
  recs_test
 recs_train

Checking important tables:
✓ nrare_train: 12,524,513
✗ arare_train: MISSING
✗ keys_train: MISSING
✗ blocks_train: MISSING
✗ train_s1_blocking: MISSING
✗ train_s2_blocking: MISSING
✗ train_s3_blocking: MISSING


In [22]:
import duckdb
import os
import shutil

DB = "/content/drive/MyDrive/er_checkpoint/er_after_blocking_train.duckdb"
TEMP_DIR = "/content/arare_duckdb_tmp"

os.makedirs(TEMP_DIR, exist_ok=True)

print("=" * 80)
print("LOCAL DISK")
print("=" * 80)

total, used, free = shutil.disk_usage("/content")
print(f"Free space: {free / 1024**3:.2f} GB")

# =========================================================
# IMPORTANT:
# Start an IN-MEMORY DuckDB connection.
# Set temp directory BEFORE attaching the Drive database.
# =========================================================

con = duckdb.connect()

con.execute("SET threads=2")
con.execute("SET preserve_insertion_order=false")
con.execute("SET temp_directory='/content/arare_duckdb_tmp'")
con.execute("SET max_temp_directory_size='20GB'")

print("✓ Temporary storage configured")

# Attach the surviving checkpoint database
con.execute(f"""
ATTACH '{DB}' AS checkpoint (READ_WRITE)
""")

print("✓ Checkpoint database attached")

# =========================================================
# CHECK SOURCE
# =========================================================

count = con.execute("""
SELECT COUNT(*)
FROM checkpoint.recs_train
""").fetchone()[0]

print(f"✓ recs_train: {count:,}")

# =========================================================
# REBUILD arare_train
# =========================================================

print("\n" + "=" * 80)
print("REBUILDING arare_train")
print("=" * 80)

con.execute("""
CREATE OR REPLACE TABLE checkpoint.arare_train AS

WITH t AS (
    SELECT DISTINCT
        entity_id,
        country,
        tok
    FROM (
        SELECT
            entity_id,
            country,
            unnest(string_split(addr_norm, ' ')) AS tok
        FROM checkpoint.recs_train
    )
    WHERE length(tok) >= 3
      AND NOT regexp_matches(tok, '^[0-9]+$')
),

df AS (
    SELECT
        country,
        tok,
        count(*) AS df
    FROM t
    GROUP BY country, tok
),

r AS (
    SELECT
        t.entity_id,
        t.tok,
        row_number() OVER (
            PARTITION BY t.entity_id
            ORDER BY df.df, t.tok
        ) AS rn
    FROM t
    JOIN df
      ON t.country = df.country
     AND t.tok = df.tok
)

SELECT
    entity_id,
    max(tok) FILTER (WHERE rn = 1) AS r1,
    max(tok) FILTER (WHERE rn = 2) AS r2
FROM r
WHERE rn <= 2
GROUP BY entity_id
""")

print("✓ arare_train creation completed")

# =========================================================
# VERIFY
# =========================================================

count = con.execute("""
SELECT COUNT(*)
FROM checkpoint.arare_train
""").fetchone()[0]

print(f"✓ arare_train rows: {count:,}")

# Force database checkpoint
con.execute("CHECKPOINT checkpoint")

print("✓ CHECKPOINT completed")

con.close()

print("✓ Connection closed")

LOCAL DISK
Free space: 85.12 GB
✓ Temporary storage configured
✓ Checkpoint database attached
✓ recs_train: 12,527,040

REBUILDING arare_train


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ arare_train creation completed
✓ arare_train rows: 12,181,244
✓ CHECKPOINT completed
✓ Connection closed


In [24]:
import shutil
import os

SRC = "/content/drive/MyDrive/er_checkpoint/er_after_blocking_train.duckdb"
LOCAL = "/content/er_after_blocking_train.duckdb"

print("Copying checkpoint database to local disk...")

shutil.copy2(SRC, LOCAL)

print("✓ Copy completed")

size = os.path.getsize(LOCAL) / (1024**3)
print(f"Local DB size: {size:.3f} GB")

Copying checkpoint database to local disk...
✓ Copy completed
Local DB size: 3.121 GB


In [27]:
import duckdb
import os

DB = "/content/drive/MyDrive/er_checkpoint/er_train_fresh.duckdb"

print("=" * 80)
print("CHECKING HEALTHY TRAIN DATABASE")
print("=" * 80)

print(f"File size: {os.path.getsize(DB) / (1024**3):.3f} GB")

con = duckdb.connect(DB, read_only=True)

tables = con.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
    ORDER BY table_name
""").fetchdf()

print("\nTables:")
print(tables.to_string(index=False))

print("\nRow counts:")

for table in [
    "train_s1",
    "train_s2",
    "train_s3",
    "train_ground_truth"
]:
    count = con.execute(
        f'SELECT COUNT(*) FROM "{table}"'
    ).fetchone()[0]

    print(f"✓ {table}: {count:,}")

con.close()

print("\n✓ Database closed")

CHECKING HEALTHY TRAIN DATABASE
File size: 1.391 GB

Tables:
        table_name
           test_s1
           test_s2
           test_s3
train_ground_truth
          train_s1
          train_s2
          train_s3

Row counts:
✓ train_s1: 2,206,821
✓ train_s2: 5,034,616
✓ train_s3: 5,285,603
✓ train_ground_truth: 2,206,821

✓ Database closed


In [28]:
import duckdb

DB = "/content/drive/MyDrive/er_checkpoint/er_train_fresh.duckdb"

con = duckdb.connect(DB, read_only=True)

print("=" * 80)
print("CHECKING TRAIN TABLE SCHEMAS")
print("=" * 80)

for table in ["train_s1", "train_s2", "train_s3"]:

    print(f"\n{'=' * 40}")
    print(table)
    print("=" * 40)

    df = con.execute(f"""
        DESCRIBE "{table}"
    """).fetchdf()

    print(df.to_string(index=False))

con.close()

print("\n✓ Connection closed")

CHECKING TRAIN TABLE SCHEMAS

train_s1
     column_name column_type null  key default extra
       entity_id     VARCHAR  YES None    None  None
   business_name     VARCHAR  YES None    None  None
business_address     VARCHAR  YES None    None  None
         country     VARCHAR  YES None    None  None

train_s2
     column_name column_type null  key default extra
       entity_id     VARCHAR  YES None    None  None
   business_name     VARCHAR  YES None    None  None
business_address     VARCHAR  YES None    None  None
         country     VARCHAR  YES None    None  None

train_s3
     column_name column_type null  key default extra
       entity_id     VARCHAR  YES None    None  None
   business_name     VARCHAR  YES None    None  None
business_address     VARCHAR  YES None    None  None
         country     VARCHAR  YES None    None  None

✓ Connection closed


In [26]:
import duckdb

LOCAL = "/content/er_after_blocking_train.duckdb"

con = duckdb.connect(LOCAL, read_only=True)

print("=" * 80)
print("VERIFYING LOCAL CHECKPOINT")
print("=" * 80)

for table in ["recs_train", "nrare_train", "arare_train"]:
    count = con.execute(
        f'SELECT COUNT(*) FROM "{table}"'
    ).fetchone()[0]

    print(f"✓ {table}: {count:,}")

con.close()

print("✓ Local database verified")

IOException: IO Error: Could not read enough bytes from file "/content/er_after_blocking_train.duckdb": attempted to read 262144 bytes from location 3536334848

In [25]:
import duckdb
import os

DB = "/content/drive/MyDrive/er_checkpoint/er_after_blocking_train.duckdb"
TEMP_DIR = "/content/keys_duckdb_tmp"

os.makedirs(TEMP_DIR, exist_ok=True)

# =========================================================
# NEW IN-MEMORY CONNECTION
# =========================================================

con = duckdb.connect()

con.execute("SET threads=2")
con.execute("SET preserve_insertion_order=false")
con.execute("SET temp_directory='/content/keys_duckdb_tmp'")
con.execute("SET max_temp_directory_size='30GB'")

con.execute(f"""
ATTACH '{DB}' AS checkpoint (READ_WRITE)
""")

print("=" * 80)
print("REBUILDING keys_train")
print("=" * 80)

# =========================================================
# BUILD KEYS
# =========================================================

con.execute("""
CREATE OR REPLACE TABLE checkpoint.keys_train AS

WITH b AS (
    SELECT
        r.entity_id,
        r.src,
        r.country,

        replace(r.name_core, ' ', '') AS core_ns,
        replace(r.name_skel, ' ', '') AS skel_ns,

        n.r1 AS n1,
        n.r2 AS n2,

        a.r1 AS a1,
        a.r2 AS a2

    FROM checkpoint.recs_train r

    LEFT JOIN checkpoint.nrare_train n
        USING (entity_id)

    LEFT JOIN checkpoint.arare_train a
        USING (entity_id)
)

SELECT DISTINCT
    entity_id,
    src,
    kt,
    k
FROM (

    -- =====================================================
    -- n1 : each rare name token
    -- =====================================================

    SELECT
        entity_id,
        src,
        'n1' AS kt,
        'n1|' || country || '|' || n AS k
    FROM (
        SELECT
            *,
            unnest([n1, n2]) AS n
        FROM b
    )
    WHERE n IS NOT NULL


    UNION ALL


    -- =====================================================
    -- n2 : pair of rare name tokens
    -- =====================================================

    SELECT
        entity_id,
        src,
        'n2' AS kt,
        'n2|' ||
        country || '|' ||
        least(n1, n2) || ' ' ||
        greatest(n1, n2) AS k
    FROM b
    WHERE n1 IS NOT NULL
      AND n2 IS NOT NULL


    UNION ALL


    -- =====================================================
    -- px : first 6 characters of name_core
    -- =====================================================

    SELECT
        entity_id,
        src,
        'px' AS kt,
        'px|' || country || '|' ||
        left(core_ns, 6) AS k
    FROM b
    WHERE length(core_ns) >= 4


    UNION ALL


    -- =====================================================
    -- sk : first 8 characters of name_skel
    -- =====================================================

    SELECT
        entity_id,
        src,
        'sk' AS kt,
        'sk|' || country || '|' ||
        left(skel_ns, 8) AS k
    FROM b
    WHERE length(skel_ns) >= 3


    UNION ALL


    -- =====================================================
    -- a2 : pair of rare address tokens
    -- =====================================================

    SELECT
        entity_id,
        src,
        'a2' AS kt,
        'a2|' ||
        country || '|' ||
        least(a1, a2) || ' ' ||
        greatest(a1, a2) AS k
    FROM b
    WHERE a1 IS NOT NULL
      AND a2 IS NOT NULL


    UNION ALL


    -- =====================================================
    -- na : rare name token + rare address token
    -- =====================================================

    SELECT
        entity_id,
        src,
        'na' AS kt,
        'na|' ||
        country || '|' ||
        n || '|' ||
        a AS k

    FROM (
        SELECT
            entity_id,
            src,
            country,
            n,
            unnest([a1, a2]) AS a
        FROM (
            SELECT
                *,
                unnest([n1, n2]) AS n
            FROM b
        )
    )

    WHERE n IS NOT NULL
      AND a IS NOT NULL
)
""")

print("✓ keys_train created")

# =========================================================
# VERIFY
# =========================================================

total = con.execute("""
SELECT COUNT(*)
FROM checkpoint.keys_train
""").fetchone()[0]

print(f"✓ Total keys: {total:,}")

stats = con.execute("""
SELECT
    kt,
    COUNT(*) AS rows
FROM checkpoint.keys_train
GROUP BY kt
ORDER BY kt
""").fetchdf()

print("\nKey counts:")
print(stats.to_string(index=False))

# =========================================================
# CHECKPOINT
# =========================================================

con.execute("CHECKPOINT checkpoint")

print("\n✓ CHECKPOINT completed")

con.close()

print("✓ Connection closed")

IOException: IO Error: Could not read enough bytes from file "/content/drive/MyDrive/er_checkpoint/er_after_blocking_train.duckdb": attempted to read 262144 bytes from location 3536334848